# Multi-tool NER pipeline (standalone) v4 — body-only — PROVISIONAL

Self-contained notebook that runs NER tools over `segmented_sections_markdown/`
**body sections only** and writes slice-validated candidates to
`notebooks/demo_output/v4/` (one folder per run tag; never mixed with earlier runs).

Backmatter is excluded at tool input (extended list): References, acknowledgments,
funding, author contributions, data availability, conflicts of interest, associated
data, footnotes, supplementary materials, ORCID / corresponding-author lines.
`(preamble)` sections are KEPT as body: HMPAPER_000003 and HMPAPER_000009 carry their
entire body text in a single `(preamble)` section.

**STATUS: PROVISIONAL_UNVALIDATED** — every mention is machine slice-validated
(`text[start:end] == term` against the source section text), but no human review yet.

Tools: DNorm (disease), TaggerOne (disease + chemical), tmChem (chemical),
GNormPlus (gene/protein), HunFlair2 (multi-type),
BioBERT (disease / chemical / genetic), SciBERT (JNLPBA: DNA, RNA, protein,
cell type, cell line).
BERT models run on word-window chunks (<=2600 chars / <=450 subword tokens,
no overlap) via transformers in `.venv_markdown`; entity spans are absolute-
offset slice-validated against the raw section text.

SciSpacy (bc5cdr / bionlp13cg / craft / jnlpba) runs in the same subprocess env
via scispacy 0.6.2; every entity is linked to UMLS by the scispacy UMLS linker
(top CUI in `norm`, linker score in `score`). Unique scispacy terms are then
cross-linked to UTS (CUI name/TUI), BioPortal, EBI OLS and the local OMOP
vocabulary dump -> `newtools_scispacy_linking.csv`.

Verified conventions embedded here:
1. Section text = segment block with marker lines (`SEGMENT`, `SECTION_ID`, `SECTION_PATH`,
   `END_SEGMENT`) removed line-wise; `SEGMENT_ORDINAL_WITHIN_SECTION` line KEPT;
   sections joined with `\n\n`. Byte-equal to `staging/paper1_5tool_ner/aioner_input/*.xml`.
2. PubTator-format Java tools see `title + " " + abstract`; empty titles break them.
   We submit sentinel title = section_id and subtract `len(title)+1` from offsets.
3. five_paper_sample baseline text = harmonized markdown from
   `markdown_harmonization/artifacts/markdown/` with HTML comments KEPT,
   `\n\n+` -> two spaces, `\n` -> space, no strip, no NFKC (validated 2022/2022 hunflair2,
   6751/6828 scispacy — remainder are term-string `\n` artifacts, offsets correct).

## v2 changes (2026-10-17)

v1 outputs in `staging/newtools_ner_5paper/` are the baseline and are never written by this notebook.

1. **Placeholder IDs**: `-1` and empty values become `norm=""` (`norm_status` is PLACEHOLDER or MISSING). The raw value is kept in `norm_raw`. Dedup keys are `<vocab>:<id>` or `TERM:<surface>`, so unrelated chemicals can no longer merge through tmChem's `-1`.
2. **Ligatures**: `ﬁ`, `ﬂ` and the other U+FB00–FB06 characters are expanded in the *tool input only*. Every tool span is mapped back through `ToolText.to_orig()`, which rejects any span that cuts a ligature and re-verifies the original slice. `char_start`/`char_end` always index the original text; `tool_char_*` keep the tool's own offsets.
3. **BERT**: labels are decoded per word (first sub-token's label) instead of per sub-token. The old decoding is still emitted from the same logits as `legacy`, for tracing only.
4. **Mapping**: `cdm_type_proposed` is unchanged. A `mapping_status` column (MAPPED / REVIEW_REQUIRED / UNMAPPED) is checked against Microbiome-CDM `CDM_TYPES`. No types were added or renamed.

Cell 1c runs the regression tests and stops the notebook if any fail. Cell 10 compares v2 with v1 and checks that every row traces back to the original text. Cell 11 reports accuracy as unknown until human gold labels exist.

**v3 (Phase 1, staging only):** API keys from environment variables only; lookup caches separate real no-hits from failures (failures logged and retried); every tool runs through `run_tool_checked()` (stale outputs deleted first, exit code/timeout/output checked, FAILED tools write no CSV); LLM cache key covers text, prompt, schema, GGUF sha256, server build and settings; `llm_occurrence_expanded` column; five-paper checks skip cleanly. Output: `staging/newtools_ner_5paper_v3/`.

## Contents

1. Configuration (Cell 1)
2. v2 fixes and v3 run-safety machinery (Cell 1b)
3. Regression tests (Cell 1c)
4. v4 helpers and tests (Cell 1d)
5. Section parsing and tool-input texts (Cells 2, 2b)
6. PubTator batch and Java tools (Cells 3, 4, 5)
7. GPU and embedding tools (Cells 6, 6b, 7)
8. LLM NER - Qwen3.5-4B (Cell 7c)
9. Dictionary tools (Cell 7d, v4 N2)
10. Cross-vocabulary linking (Cell 7b)
11. Type-restricted linking, definitions, taxonomy (Cell 7e, v4 N5)
12. Outputs, audit and deduplication (Cells 8, 8b)
13. Junk filter, consensus, template tables, validation (Cell 8c, v4 N3/N4/N6/N7)
14. Baseline overlap (Cell 9)
15. v3 vs v2 comparison and traceability (Cell 10)
16. Evaluation status (Cell 11)
17. Gold sample, scoring hook and run manifest (Cell 11b, v4 N8)
18. Data catalogue (Cells 12a-12c, stage CAT)

**v4 (2026-10-18):** S7 section decisions drive tool input (N1); MeSH taxonomy + methods dictionary tools (N2); junk flags (N3); span-group consensus (N4); OLS4 type-restricted linking with verbatim definitions and taxonomy (N5); template tables with deduplicated frequency (N6); 13 build checks and review carry-over (N7); gold-sample hook (N8); data catalogue build (Cells 12a-12c). Outputs: `notebooks/demo_output/v4/`.

## 1. Configuration (Cell 1)

Paths, paper selection (`FULL_CORPUS`), the body-only backmatter filter, Java tool locations and the
per-tool timeout budget. All outputs go to `notebooks/demo_output/` (`OUT_DIR`); the v1/v2 staging
directories are read-only baselines and are never written to.


In [ ]:
# ===== Cell 1: config =====
import os, re, json, csv, subprocess, sys, time, unicodedata, tempfile, hashlib
from pathlib import Path

# POSIX host (login01). NB_ROOT overrides for a copy of the tree; the default is the live repo,
# which is where every artifact audited in Cells 14-19 lives.
# Repo root: $NB_ROOT, else walk up from the working directory to the .cdm_root marker.
ROOT = Path(os.environ.get("NB_ROOT") or next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / ".cdm_root").is_file())).resolve()
SEG_DIR = ROOT / "data/segmented_sections_markdown"
# DEAD PATH: markdown_harmonization/ does not exist on this host (only segmented_sections_markdown/
# survives). Kept as a name so downstream cells stay importable; nothing reads it any more.
ART_MD  = ROOT / "markdown_harmonization" / "artifacts" / "markdown"
assert SEG_DIR.exists(), f"segmented section dir not found under ROOT={ROOT}"
# Repo imports: `src.cdm_pipeline` / `common` are importable from the repo root with this insert.
for _p in (ROOT, ROOT / "src"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
# Stabilization: RUN_TAG/OUT_DIR are env-overridable (NB_RUN_TAG/NB_OUT_DIR) so a stabilization
# run writes into a fresh dedicated staging directory and never mixes with Phase A outputs.
RUN_TAG = os.environ.get("NB_RUN_TAG", "v4")   # v4 = N1-N8 + catalogue C1-C8 (first full run; v3 = Phase 1 run-safety machinery)
OUT_DIR = Path(os.environ.get("NB_OUT_DIR") or str(ROOT / "src/notebooks" / "demo_output" / RUN_TAG))
# DEAD PATHS (2026-10-28): staging/newtools_ner_5paper* were never carried over to this host. Both
# names are kept because Cell 7b/7c/8b still probe them for cache seeding; every reader is
# .exists()-guarded, so they degrade to "no seed" instead of failing.
V1_DIR  = ROOT / "staging" / "newtools_ner_5paper"        # read-only
V2_DIR  = ROOT / "staging" / "newtools_ner_5paper_v2"     # read-only (comparison + cache seeding)
print("staging seeds present:", V1_DIR.exists(), V2_DIR.exists())
assert OUT_DIR.resolve() not in {V1_DIR.resolve(), V2_DIR.resolve()}, "OUT_DIR must not be a v1/v2 directory"
# Prior-run protection: without an explicit NB_OUT_DIR redirect, refuse to write into a directory
# that already holds a completed run (run_manifest.json is written at the end of a successful run).
if "NB_OUT_DIR" not in os.environ and (OUT_DIR / "run_manifest.json").exists():
    raise RuntimeError(f"OUT_DIR {OUT_DIR} already contains a completed run; prior-run artifacts are "
                       f"read-only. Set NB_RUN_TAG to a fresh tag and/or NB_OUT_DIR to a fresh directory.")
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "_work").mkdir(exist_ok=True)
WORK = OUT_DIR / "_work"
PRODUCTION_WRITES = 0
RUN_STARTED_AT = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

# Paper set: NB_FULL_CORPUS=1 -> all papers in SEG_DIR; NB_PAPERS="A,B,C" -> explicit subset.
_fc_env = os.environ.get("NB_FULL_CORPUS")
FULL_CORPUS = (str(_fc_env).strip().lower() in ("1", "true", "yes")) if _fc_env is not None else False
DEFAULT_PAPERS = ["HMPAPER_000002", "HMPAPER_000003", "HMPAPER_000009",
                  "HMPAPER_000010", "HMPAPER_000014"]
_papers_env = os.environ.get("NB_PAPERS", "").strip()
if _papers_env:
    DEFAULT_PAPERS = [p.strip() for p in _papers_env.split(",") if p.strip()]
PAPERS = (sorted({p.name[:13] for p in SEG_DIR.glob("HMPAPER_*.md")}) if FULL_CORPUS
          else DEFAULT_PAPERS)
PAPERS = [p for p in PAPERS if (SEG_DIR / f"{p}.md").exists()]
print("papers:", PAPERS)

# Body-only tool input: extended backmatter exclusion (user-approved list).
# NOTE: '(preamble)' is KEPT as body: HMPAPER_000003/000009 carry their whole
# body text in a single '(preamble)' section (they have no other sections).
BODY_ONLY = True
BACKMATTER_TOKENS = ("references", "acknowledg", "funding", "author contribution",
                     "data availability", "conflict", "associated data", "footnote",
                     "supplementary", "orcid", "corresponding author",
                     "institutional", "informed consent", "competing interest")

def is_backmatter(path):
    """True if a SECTION_PATH marks this section as backmatter/meta (non-body)."""
    last = (path or "").split(">")[-1].strip().lower()
    if not last or last == "(preamble)":
        return False
    return any(tok in last for tok in BACKMATTER_TOKENS)

print("body-only filter:", BODY_ONLY, "| backmatter tokens:", len(BACKMATTER_TOKENS))

NER_TOOLS = Path(os.environ.get("CDM_NER_TOOLS", Path.home() / "tools" / "ner_tools"))
JDK     = NER_TOOLS / "java" / "jdk8u504-b01"
JAVA    = str(JDK / "bin" / "java")            # POSIX JDK (was bin/java.exe on Windows)
TOOLS   = NER_TOOLS / "tools"
DNORM   = TOOLS / "DNorm-0.0.7"
TGONE   = TOOLS / "TaggerOne-0.2.1"
TMCHEM  = TOOLS / "tmChemM1-0.0.2"             # available: self-contained M1-0.0.2 from the CBBresearch mirror
AB3P    = TOOLS / "Ab3P-v1.5"                   # built from ftp.ncbi.nlm.nih.gov/pub/wilbur; make test = NCBI gold
GNORM   = TOOLS / "GNormPlusJava"
# Tool interpreter: the only venvs on this host are ~/venvs/ner_variants (spacy 3.7.5 + flair 0.15.1 +
# transformers 4.57.6) and gliner_overlay. NB_ENV_PY overrides; falls back to the running kernel.
ENV_PY  = os.environ.get("NB_ENV_PY")
if not ENV_PY:
    for cand in (Path.home() / "venvs" / "ner_variants" / "bin" / "python",
                 Path.home() / "venvs" / "gliner_overlay" / "bin" / "python",
                 Path(sys.executable)):
        if Path(cand).exists():
            ENV_PY = str(cand); break
print("java:", Path(JAVA).exists(), "| env_py:", ENV_PY)

# Per-paper timeout per tool (seconds). Phase 1 still runs each tool ONCE over all papers,
# so the effective limit is TOOL_STARTUP_S + per-paper value x number of papers, and a
# timeout fails that tool for EVERY paper in the run (no per-paper isolation until Phase 2).
TOOL_TIMEOUT_PER_PAPER = {"tmChem": 300, "DNorm": 300, "TaggerOne-Disease": 300,
                          "TaggerOne-Chemical": 300, "GNormPlus": 300, "HunFlair2": 300,
                          "BERT": 300, "SciSpacy": 300, "LLM": 300}
TOOL_STARTUP_S = 600   # JVM/model load, once per tool run

def tool_timeout(tool_key):
    """Whole-run timeout for one tool: startup allowance plus per-paper seconds times len(PAPERS)."""
    return TOOL_STARTUP_S + TOOL_TIMEOUT_PER_PAPER[tool_key] * len(PAPERS)

# Per-tool run status: tool_key -> {status OK|FAILED, reason, log_tail, outputs{path: sha256}, ...}
TOOL_STATUS = {}

# ---------------------------------------------------------------------------
# CDM type mapping registry (stabilization Phase C). Versioned, auditable, test-pinned.
# Non-rule entries trace to verdict rows in MAPPING_AUDIT_CSV. Rule entries (source="rule")
# are pipeline-owned label spaces (N2 dictionary tools, Cell 1d).
#   status MAPPED          - audited OK: raw label -> real CDM type
#   status REVIEW_REQUIRED - target kept, mapping needs human sign-off (audit TOO_BROAD etc.)
#   status UNMAPPED        - no schema type exists; target is a UNMAPPED_* sentinel, never a
#                            made-up real type (BODY_SITE/SPECIMEN are NOT in the schema).
# NOTE (LLM label precedence): the LLM (Cell 7c) emits CDM-type strings directly; a label that
# collides with an audited registry key (e.g. ORGANISM, PROTEIN) inherits that key's status.
# The reasons behind those statuses (HOST-vs-MICROBE split, abbreviation noise) are
# model-agnostic, so registry-first is the conservative, audit-faithful choice.
MAPPING_REGISTRY_VERSION = "registry-0.2"
MAPPING_AUDIT_CSV = ROOT / "staging" / "ner_audit" / "cdm_mapping_audit.csv"
CDM_MAPPING_REGISTRY = [
    # label, target, status, source  -- audited labels first
    dict(label="disease",               target="DISEASE",      status="MAPPED",          source="audit:OK"),
    dict(label="chemical",              target="CHEMICAL",     status="MAPPED",          source="audit:OK"),
    dict(label="simple_chemical",       target="CHEMICAL",     status="MAPPED",          source="audit:OK"),
    dict(label="amino_acid",            target="CHEMICAL",     status="MAPPED",          source="audit:OK"),
    dict(label="chebi",                 target="CHEMICAL",     status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="cancer",                target="DISEASE",      status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="pathological_formation", target="DISEASE",     status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="gene",                  target="GENOMICS",     status="MAPPED",          source="audit:OK"),
    dict(label="gene/protein",          target="GENOMICS",     status="MAPPED",          source="audit:OK"),
    dict(label="gene/family",           target="GENOMICS",     status="MAPPED",          source="audit:OK"),
    dict(label="gene/family/protein",   target="GENOMICS",     status="MAPPED",          source="audit:OK"),
    dict(label="domain",                target="GENOMICS",     status="MAPPED",          source="audit:OK"),
    dict(label="genetic",               target="GENOMICS",     status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="dna",                    target="GENOMICS",    status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="protein",               target="PROTEIN",      status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="gene_or_gene_product",  target="PROTEIN",      status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="ggp",                   target="PROTEIN",      status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="go",                    target="PATHWAY",      status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="so",                    target="DNA_RNA",      status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="cell_type",             target="CELL_TYPE",    status="MAPPED",          source="audit:OK"),
    dict(label="cell type",             target="CELL_TYPE",    status="MAPPED",          source="audit:OK"),
    dict(label="cell",                  target="CELL_TYPE",    status="REVIEW_REQUIRED", source="audit:WRONG_hybrid"),
    dict(label="cellular_component",    target="CELL_TYPE",    status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="cl",                    target="CELL_TYPE",    status="MAPPED",          source="audit:OK"),
    dict(label="species",               target="ORGANISM",     status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="taxon",                target="ORGANISM",     status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    dict(label="organism",             target="ORGANISM",     status="REVIEW_REQUIRED", source="audit:TOO_BROAD"),
    # SILENT_PASSTHROUGH fixed: genus/strain previously passed through unmapped; both are ORGANISM.
    dict(label="genus",                target="ORGANISM",     status="MAPPED",          source="audit:SILENT_PASSTHROUGH_FIX"),
    dict(label="strain",               target="ORGANISM",     status="MAPPED",          source="audit:SILENT_PASSTHROUGH_FIX"),
    # wrong or no schema slot -> UNMAPPED_* sentinel (NEEDS_SCHEMA proposes BODY_SITE/SPECIMEN in rationale)
    dict(label="rna",                   target="UNMAPPED_RNA", status="UNMAPPED",       source="audit:WRONG"),
    dict(label="cell_line",             target="UNMAPPED_CELL_LINE", status="UNMAPPED",  source="audit:WRONG"),
    dict(label="cell line",             target="UNMAPPED_CELL_LINE", status="UNMAPPED",  source="audit:WRONG"),
    dict(label="organ",                 target="UNMAPPED_ORGAN", status="UNMAPPED",      source="audit:WRONG"),
    dict(label="tissue",                target="UNMAPPED_TISSUE", status="UNMAPPED",     source="audit:WRONG"),
    dict(label="multi_tissue_structure", target="UNMAPPED_MULTI_TISSUE_STRUCTURE", status="UNMAPPED", source="audit:WRONG"),
    dict(label="organism_substance",    target="UNMAPPED_SPECIMEN", status="UNMAPPED",   source="audit:NEEDS_SCHEMA:proposed_BODY_SITE"),
    dict(label="organism_subdivision",  target="UNMAPPED_ORGANISM_SUBDIVISION", status="UNMAPPED", source="audit:NEEDS_SCHEMA"),
    dict(label="anatomical_system",     target="UNMAPPED_ANATOMICAL_SYSTEM", status="UNMAPPED", source="audit:NEEDS_SCHEMA"),
    dict(label="immaterial_anatomical_entity", target="UNMAPPED_IMMATERIAL_ANATOMICAL_ENTITY", status="UNMAPPED", source="audit:NEEDS_SCHEMA"),
    # rule labels (pipeline-owned label spaces)
    dict(label="kit",                  target="LAB_METHOD",    status="MAPPED",          source="rule"),
    dict(label="platform",             target="LAB_METHOD",    status="MAPPED",          source="rule"),
    dict(label="primer",               target="LAB_METHOD",    status="MAPPED",          source="rule"),
    dict(label="assay",                target="LAB_METHOD",    status="MAPPED",          source="rule"),
    dict(label="amplicon_region",       target="LAB_METHOD",    status="MAPPED",          source="rule"),
    dict(label="software",             target="DATA_ANALYSIS", status="MAPPED",          source="rule"),
    dict(label="reference_db",         target="DATA_ANALYSIS", status="MAPPED",          source="rule"),
    dict(label="taxon_mesh",            target="ORGANISM",     status="MAPPED",          source="rule"),
    dict(label="taxon_abbrev",         target="ORGANISM",      status="MAPPED",          source="rule"),
]
_CDM_MAP = {e["label"].lower(): e for e in CDM_MAPPING_REGISTRY}

def cdm_of(label, model):
    """Map a raw tool label to a proposed Microbiome-CDM type via the versioned registry.
    Unknown labels pass through unchanged (flagged UNMAPPED by mapping_status);
    an empty label becomes UNMAPPED_<model>."""
    ent = _CDM_MAP.get((label or "").strip().lower())
    if ent is not None:
        return ent["target"]
    L = (label or "").strip()
    return L if L else "UNMAPPED_" + model

def load_csv(p):
    """Read a UTF-8 CSV into a list of dicts."""
    with open(p, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))

def _assert_output_path(path):
    """Stabilization (Phase A): every CSV the pipeline writes must land inside OUT_DIR."""
    p = Path(path).resolve()
    od = OUT_DIR.resolve()
    if p != od and od not in p.parents:
        raise RuntimeError(f"refusing to write outside OUT_DIR: {p}")

def dump_csv(path, rows, fields):
    """Write rows to a UTF-8 CSV under OUT_DIR; missing keys become empty strings."""
    _assert_output_path(path)
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        for r in rows: w.writerow({k: r.get(k, "") for k in fields})

FIELDS = ["candidate_id", "cid", "section_id", "model", "term", "label_raw", "norm", "norm_raw",
          "norm_status", "norm_vocab", "cdm_type_proposed", "mapping_status", "char_start", "char_end",
          "tool_char_start", "tool_char_end", "partial_word", "score", "review_status",
          "llm_occurrence_expanded"]   # v3: appended only (1 = LLM row from a further occurrence)
print("OUT_DIR:", OUT_DIR)
print("read-only:", V1_DIR, "|", V2_DIR)


# ---- v4: pipeline-map changes (N1-N8). Every new input is read-only; outputs stay in OUT_DIR. ----
# N1: tools read the S7 decision first; the backmatter list above is the second guard.
# 2026-10-28: template_mapped_corpus/ does not exist on this host. The surviving decision table is
# corpus_pipeline/manifests/section_selection.csv (3,105 rows / 435 papers). It uses
# `selection_status`, NOT `final_status` - select_section() normalises the two names (see Cell 1d).
SEL_CSV  = Path(os.environ.get("NB_SEL_CSV")
                or str(ROOT / "pipelines/corpus_pipeline" / "manifests" / "section_selection.csv"))
SEL_KEEP = {"RELEVANT", "RELEVANT_CONTEXT", "UNCERTAIN"}   # IRRELEVANT / CONTAINER_ONLY never reach tools
# N2: dictionary tools. DEAD PATH: terms/ (and terms/cache) is absent on this host, so the MeSH B03
# lexicon and the abbreviation stoplist are NOT_ATTEMPTED; the MeSH gazetteer that does exist is
# results/mesh_ner/mesh_gazetteer.jsonl (Cell 14), a different, flat schema.
MESH_TAXON_LEXICON = ROOT / "terms" / "cache" / "mesh_lexicon_7f5eb3d02b7087b1.json"
TERMS_CONFIG = ROOT / "terms" / "config.json"              # abbreviation_stoplist
NCBI_TAXDUMP = NER_TOOLS / "downloads" / "taxdump"
MESH_GAZETTEER = ROOT / "outputs/results" / "mesh_ner" / "mesh_gazetteer.jsonl"   # what actually exists
# N3: general-English NER for the junk filter, in its own interpreter (keeps spaCy out of the kernel).
# CAVEAT: no en_core_* spaCy model is installed in any venv on this host, so this stays NOT_ATTEMPTED
# until one is downloaded; GENERAL_NER_PY now points at a real interpreter.
GENERAL_NER_MODEL = "en_core_web_trf"
GENERAL_NER_PY = os.environ.get("GENERAL_NER_PY") or ENV_PY
# N6/N7: template file (if present its field lists are enforced) and read-only gold folder (N8).
NER_TEMPLATE_YAML = ROOT / "template_mapped_corpus" / "config" / "ner_entity_template.yaml"
GOLD_DIR = ROOT / "staging" / "ner_audit"
print("v4 inputs:", {k: Path(v).exists() for k, v in dict(
    SEL_CSV=SEL_CSV, MESH_TAXON_LEXICON=MESH_TAXON_LEXICON, NCBI_TAXDUMP=NCBI_TAXDUMP,
    NER_TEMPLATE_YAML=NER_TEMPLATE_YAML, GOLD_DIR=GOLD_DIR).items()})

## 2. v2 fixes and v3 run-safety machinery (Cell 1b)

- **v2 fixes**: placeholder normalization IDs (tmChem `-1` and friends), ligature-safe offset mapping
  (`ToolText` expands U+FB00..FB06 in tool input only and rejects spans that cut a ligature),
  word-level BERT label aggregation (`BERT_AGG_SRC`, shared verbatim with the Cell 6b runner), and
  `mapping_status` checked against Microbiome-CDM `CDM_TYPES` (`cdm_type_proposed` unchanged).
- **v3 run safety**: secrets from the environment only (`get_secret`, `redact`);
  `run_tool_checked` / `consume_output` / `register` (stale outputs deleted first, status plus sha256
  provenance, FAILED tools write no CSV); `LookupCache` (FOUND and genuine NOT_FOUND cached, failures
  logged separately and retried); the settings-complete LLM cache key; and the
  `five_paper_check_ready` gate that skips five-paper-only checks with a logged reason.


In [ ]:
# ===== Cell 1b: v2 fixes - ID placeholders, ligature offset map, BERT word aggregation, mapping status =====
# FIX 1 (IDs): tool placeholders are MISSING, never entity IDs. tmChem writes "-1" for
# "not normalized"; v1 Cell 8b used it as a dedup key, merging unrelated chemicals.
# Raw value kept in `norm_raw`, cleaned value in `norm`, provenance in `norm_status`/`norm_vocab`.
PLACEHOLDER_IDS = {"", "-1", "-", "none", "null", "n/a", "nan", "cui-less"}
CUI_RE = re.compile(r"^C\d{7}$")

def clean_norm(raw):
    """-> (norm, status): status VALID | PLACEHOLDER (documented sentinel) | MISSING (empty)."""
    s = (raw or "").strip()
    if not s:
        return "", "MISSING"
    if s.lower() in PLACEHOLDER_IDS:
        return "", "PLACEHOLDER"
    return s, "VALID"

def norm_vocab(model, label, norm):
    """Vocabulary of a normalized ID: from an ID prefix, CUI shape means UMLS, bare GNormPlus numbers get NCBIGene/NCBITaxon by label, else UNKNOWN."""
    if not norm:
        return ""
    if ":" in norm:
        return norm.split(":", 1)[0].upper()          # MESH:, CHEBI:, OMIM:
    if CUI_RE.match(norm):
        return "UMLS"
    if model == "GNormPlus":                          # bare NCBI numbers: namespace by label
        return "NCBIGene" if label.lower() == "gene" else "NCBITaxon"
    return "UNKNOWN"

def entity_key(term_clean, norm, vocab):
    """Dedup key: vocabulary-qualified ID when valid, else the cleaned surface form."""
    return f"{vocab}:{norm}" if norm else "TERM:" + term_clean.lower()

# FIX 2 (ligatures): tools see text with U+FB00-FB06 expanded (ﬁ -> fi). Expansion changes
# length, so every tool offset is mapped back through ToolText.to_orig(), which REJECTS spans
# that cut a ligature instead of widening them, and re-verifies the original slice.
LIGATURES = {"ﬀ": "ff", "ﬁ": "fi", "ﬂ": "fl", "ﬃ": "ffi",
             "ﬄ": "ffl", "ﬅ": "st", "ﬆ": "st"}

def lig_normalize(text):
    """Expand U+FB00..FB06 ligature characters to ASCII (applied to TOOL input text only)."""
    return "".join(LIGATURES.get(ch, ch) for ch in text)

class ToolText:
    """Original section text + ligature-normalized tool text + normalized->original offset map."""
    def __init__(self, orig):
        """Build the ligature-normalized tool text and its per-char map back to original offsets (identity when no ligatures)."""
        self.orig = orig
        self.identity = not any(ch in LIGATURES for ch in orig)
        if self.identity:
            self.norm = orig
            return
        n2o, first, last, parts = [], [], [], []
        for i, ch in enumerate(orig):
            rep = LIGATURES.get(ch, ch)
            parts.append(rep)
            for k in range(len(rep)):
                n2o.append(i); first.append(k == 0); last.append(k == len(rep) - 1)
        self.norm = "".join(parts)
        self.n2o, self.first, self.last = n2o, first, last

    def to_orig(self, s, e):
        """Map tool span [s, e) to original offsets. Returns (os, oe) or a reject reason string."""
        if not (0 <= s < e <= len(self.norm)):
            return "OUT_OF_TEXT"
        if self.identity:
            os_, oe = s, e
        else:
            if not (self.first[s] and self.last[e - 1]):
                return "LIGATURE_SPLIT"
            os_, oe = self.n2o[s], self.n2o[e - 1] + 1
        if lig_normalize(self.orig[os_:oe]) != self.norm[s:e]:   # validate the mapping, never assume it
            return "MAP_VERIFY_FAIL"
        return os_, oe

def partial_word(text, s, e):
    """True if [s, e) starts or ends inside an alphanumeric run."""
    return bool((s > 0 and text[s - 1].isalnum() and text[s].isalnum()) or
                (e < len(text) and text[e - 1].isalnum() and text[e].isalnum()))

# FIX 3 (BERT): v1 decoded BIO labels per SUB-TOKEN, so a B-/O label on a non-initial
# word piece cut words ("Bifidobacte|rium"). v2 takes each word's label from its FIRST
# sub-token (the HF "first" aggregation these token-classification heads are trained for)
# and emits spans on word boundaries from the tokenizer's own word_ids/offsets.
# Shared verbatim with the subprocess runner (Cell 6b) so the tested code is the run code.
BERT_AGG_SRC = r'''
def word_level(word_ids, labels, offsets):
    """Collapse sub-tokens to words: label = first sub-token's label; span = union of pieces."""
    words, prev = [], None
    for wid, lab, (s, e) in zip(word_ids, labels, offsets):
        if wid is None or s == e:
            continue
        if wid != prev:
            words.append([s, e, lab]); prev = wid
        else:
            words[-1][1] = max(words[-1][1], e)
    return words

def words_to_spans(words):
    """BIO decode over words -> [(type, start, end)]. I-X continues only an open X entity."""
    ents, cur = [], None
    for s, e, lab in words:
        if lab == "O":
            if cur: ents.append(cur); cur = None
            continue
        b, t = lab.split("-", 1)
        if b == "I" and cur and cur[0] == t:
            cur[2] = e
        else:
            if cur: ents.append(cur)
            cur = [t, s, e]
    if cur: ents.append(cur)
    return [tuple(x) for x in ents]
'''
exec(BERT_AGG_SRC)

# Mapping STATUS (stabilization Phase C): registry-driven. cdm_type_proposed unchanged
# (no CDM types added or renamed); statuses come from CDM_MAPPING_REGISTRY (Cell 1, registry-0.2)
# whose non-rule entries trace to staging/ner_audit/cdm_mapping_audit.csv.
# Schema authority: src/cdm_pipeline/config.py CDM_TYPES (14 types). The old path was a Windows
# absolute path (C:\Users\jkalra\...\cdm_pipeline\config.py); the file lives under src/ on this host.
# The literal below is kept on purpose: the assert after _read_schema_types() is a drift check.
_CDM_SCHEMA_PATH = ROOT / "src" / "cdm_pipeline" / "config.py"
CDM_TYPES = {"ORGANISM", "CHEMICAL", "DISEASE", "PROTEIN", "DNA_RNA", "CELL_TYPE",
             "GENOMICS", "PATHWAY", "LAB_METHOD", "DATA_ANALYSIS", "SAMPLE_METHOD",
             "CLINICAL", "EXPERIMENTAL_METHOD", "STUDY_DESIGN"}

def _read_schema_types():
    """Parse CDM_TYPES from the authoritative config.py; hard-fail (never degrade silently)."""
    txt = _CDM_SCHEMA_PATH.read_text(encoding="utf-8", errors="replace")
    m = re.search(r"CDM_TYPES\s*=\s*[\[(](.+?)[\])]", txt, re.S)
    if not m:
        raise RuntimeError(f"CDM_TYPES not found in {_CDM_SCHEMA_PATH}")
    types = set(re.findall(r"['\"]([A-Z][A-Z_]+)['\"]", m.group(1)))
    if not types:
        raise RuntimeError(f"CDM_TYPES in {_CDM_SCHEMA_PATH} parsed empty")
    return types

CDM_TYPES_SCHEMA = _read_schema_types()
assert CDM_TYPES == CDM_TYPES_SCHEMA, (
    f"CDM_TYPES drifted from the schema file {_CDM_SCHEMA_PATH}: "
    f"missing={sorted(CDM_TYPES_SCHEMA - CDM_TYPES)} extra={sorted(CDM_TYPES - CDM_TYPES_SCHEMA)}")
_SCHEMA_CHECK = dict(ok=True, source=str(_CDM_SCHEMA_PATH), n_types=len(CDM_TYPES_SCHEMA))

# derived, not hand-maintained; TestMappingRegistry cross-checks against the registry
REVIEW_LABELS = {e["label"] for e in CDM_MAPPING_REGISTRY if e["status"] == "REVIEW_REQUIRED"}
UNMAPPED_LABELS = {e["label"] for e in CDM_MAPPING_REGISTRY if e["status"] == "UNMAPPED"}

def mapping_status(label, cdm_type):
    """Registry-driven (Phase C): an audited label carries its own status (MAPPED |
    REVIEW_REQUIRED | UNMAPPED); an unknown label is MAPPED only when its pass-through
    target is a real schema type (e.g. LLM rows that emit CDM-type strings directly)."""
    ent = _CDM_MAP.get((label or "").strip().lower())
    if ent is not None:
        return ent["status"]
    return "MAPPED" if cdm_type in CDM_TYPES else "UNMAPPED"

REJECTS = {}   # model -> {reason: count}

def make_row(model, cid, sid, s, e, label, norm_raw="", score=""):
    """Tool span in TOOL-TEXT coordinates -> candidate row in ORIGINAL coordinates, or None."""
    tt = TT[cid][sid]
    m = tt.to_orig(s, e)
    if isinstance(m, str):
        REJECTS.setdefault(model, {}).setdefault(m, 0)
        REJECTS[model][m] += 1
        return None
    os_, oe = m
    norm, status = clean_norm(norm_raw)
    cdm = cdm_of(label, model)
    return dict(candidate_id=f"{model}|{cid}|{sid}|{os_}|{oe}", cid=cid, section_id=sid,
                model=model, term=tt.orig[os_:oe], label_raw=label,
                norm=norm, norm_raw=norm_raw or "", norm_status=status,
                norm_vocab=norm_vocab(model, label, norm),
                cdm_type_proposed=cdm, mapping_status=mapping_status(label, cdm),
                char_start=os_, char_end=oe, tool_char_start=s, tool_char_end=e,
                partial_word=int(partial_word(tt.orig, os_, oe)),
                score=score, review_status="PROVISIONAL_UNVALIDATED")

print("v2 fixes loaded: placeholders", sorted(PLACEHOLDER_IDS), "| ligatures", len(LIGATURES))


# ---------------------------------------------------------------------------------------
# v3 (Phase 1) run-safety helpers. Tested in Cell 1c before any tool runs.
# ---------------------------------------------------------------------------------------
import shutil, datetime

def _now():
    """Current UTC time as an ISO string with second precision (audit timestamps)."""
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")

# --- secrets: environment only; values are never printed, logged or written ---
SECRET_ENV_VARS = ("UMLS_API_KEY", "BIOPORTAL_API_KEY")

def get_secret(name):
    """Return a required API key from the environment; raise a clear error if unset (the value is never printed or logged)."""
    v = os.environ.get(name, "").strip()
    if not v:
        raise RuntimeError(f"{name} is not set. Set it as an environment variable "
                           f"(the value is never printed or logged), then re-run this cell.")
    return v

def redact(text):
    """Remove any secret value from text before it reaches a log, summary or report."""
    s = str(text)
    for n in SECRET_ENV_VARS:
        v = os.environ.get(n, "").strip()
        if v:
            s = s.replace(v, f"<{n}>")
    return s

def file_sha256(path):
    """Streamed sha256 of a file's bytes."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

# --- tool execution with status, stale-output removal and output provenance ---
def _remove(p):
    """Delete a file or a whole directory tree; missing paths are ignored."""
    p = Path(p)
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()

def _log_tail(log_path, n=1500):
    """Last n characters of a log file with secret values redacted."""
    try:
        return redact(Path(log_path).read_text(encoding="utf-8", errors="replace")[-n:])
    except Exception as ex:
        return f"<log unreadable: {ex!r}>"

def run_tool_checked(tool_key, args, cwd, log_path, outputs, timeout, create_dirs=()):
    """Run one tool. `outputs` = files (or dirs that must contain >=1 file) the tool must produce.
    They are deleted BEFORE the run so a failed run can never leave old results behind.
    `create_dirs` are recreated EMPTY after deletion, for tools that write into an existing
    output folder (GNormPlus). Sets TOOL_STATUS[tool_key]; returns True only when rc == 0 and
    every output exists."""
    for o in outputs:
        _remove(o)
    for d in create_dirs:
        Path(d).mkdir(parents=True)
    t0 = time.time()
    st = dict(status="FAILED", reason="", started_at=_now(), timeout_s=timeout,
              log=str(log_path), outputs={})
    try:
        with open(log_path, "w", encoding="utf-8") as log:
            cp = subprocess.run(args, cwd=str(cwd), stdout=log, stderr=subprocess.STDOUT,
                                timeout=timeout, shell=False)
        rc = cp.returncode
    except subprocess.TimeoutExpired:
        rc, st["reason"] = None, f"TIMEOUT after {timeout}s (whole-run limit; all papers in this run fail for this tool)"
    except Exception as ex:
        rc, st["reason"] = None, redact(f"LAUNCH_ERROR {ex!r}")
    st["elapsed_s"] = round(time.time() - t0, 1)
    st["returncode"] = rc
    if rc is not None and rc != 0:
        st["reason"] = f"EXIT_CODE {rc}"
    if rc == 0:
        missing = []
        for o in outputs:
            o = Path(o)
            f = next((c for c in sorted(o.iterdir()) if c.is_file()), None) if o.is_dir() else (o if o.exists() else None)
            if f is None:
                missing.append(str(o))
            else:
                st["outputs"][str(o)] = dict(file=str(f), sha256=file_sha256(f), mtime=f.stat().st_mtime)
        if missing:
            st["reason"] = f"NO_OUTPUT {missing}"
        else:
            st["status"] = "OK"
    if st["status"] != "OK":
        st["log_tail"] = _log_tail(log_path)
    st["ended_at"] = _now()
    TOOL_STATUS[tool_key] = st
    print(f"{tool_key}: {st['status']} rc={rc} elapsed={st['elapsed_s']}s {st['reason']}")
    return st["status"] == "OK"

def mark_tool(tool_key, status, reason="", **extra):
    """Status for tools that are not a single subprocess (e.g. the LLM server loop)."""
    TOOL_STATUS[tool_key] = dict(status=status, reason=redact(reason), ended_at=_now(), outputs={}, **extra)
    print(f"{tool_key}: {status} {reason}")

def consume_output(tool_key, output):
    """Return the produced file only if the tool succeeded IN THIS RUN and the file is
    byte-identical to what was recorded right after it ran (no stale or replaced output)."""
    st = TOOL_STATUS.get(tool_key)
    if not st or st["status"] != "OK":
        raise RuntimeError(f"{tool_key}: refusing to read output; tool status is "
                           f"{(st or {}).get('status', 'NOT_RUN')}")
    rec = st["outputs"].get(str(output))
    if rec is None:
        raise RuntimeError(f"{tool_key}: {output} was not recorded as an output of this run")
    if not Path(rec["file"]).exists() or file_sha256(rec["file"]) != rec["sha256"]:
        raise RuntimeError(f"{tool_key}: {rec['file']} changed or vanished after the tool ran")
    return Path(rec["file"])

all_rows = []   # rows from every tool that finished OK
def register(rows, model, tool_key):
    """Write <model>_candidates.csv only for a tool whose status is OK in this run."""
    st = TOOL_STATUS.get(tool_key)
    if not st or st["status"] != "OK":
        print(f"{model}: NOT registered (tool {tool_key} status {(st or {}).get('status', 'NOT_RUN')}); no CSV written")
        return []
    all_rows.extend(rows)
    p = OUT_DIR / f"{model}_candidates.csv"
    dump_csv(p, rows, FIELDS)
    st.setdefault("models", {})[model] = len(rows)
    print(f"{model}: {len(rows)} rows -> {p.name} | rejected: {REJECTS.get(model, {})}")
    return rows

# make_row v3: extra columns pass through (llm_occurrence_expanded defaults to 0)
_make_row_v2 = make_row
def make_row(model, cid, sid, s, e, label, norm_raw="", score="", **extra):
    """v3 wrapper around _make_row_v2: defaults llm_occurrence_expanded to 0 and passes extra columns through."""
    row = _make_row_v2(model, cid, sid, s, e, label, norm_raw, score)
    if row is not None:
        row["llm_occurrence_expanded"] = 0
        row.update(extra)
    return row

# --- lookup caches: FOUND / NOT_FOUND are cached; failures are logged and retried ---
class LookupFailure(Exception):
    """Raised by fetchers on network or server errors; a failure is never cached as NOT_FOUND."""
    pass

class LookupCache:
    """entries[key] = {status FOUND|NOT_FOUND, value, provenance reused_v2|refreshed|new, fetched_at, query_sig}.
    Failures never enter `entries`; they go to the failures file and are retried next run."""
    def __init__(self, source, path, failures_path, query_sig):
        """Load entries (rejecting a mismatched query signature) and this source's failure log."""
        self.source, self.path, self.failures_path, self.query_sig = source, Path(path), Path(failures_path), query_sig
        d = json.loads(self.path.read_text(encoding="utf-8")) if self.path.exists() else {}
        if d and d.get("meta", {}).get("query_sig") != query_sig:
            raise RuntimeError(f"{source}: cache {self.path.name} was built with a different query "
                               f"signature; use a new run tag or delete it deliberately")
        self.entries = d.get("entries", {})
        self.v2_ambiguous = set(d.get("meta", {}).get("v2_ambiguous", []))
        allf = json.loads(self.failures_path.read_text(encoding="utf-8")) if self.failures_path.exists() else {}
        self.failures = allf.get(source, {})
        self.counts = dict(cached=0, reused_v2=0, refreshed=0, new=0, failed=0, seeded_reused_v2=0,
                           seeded_skipped_ambiguous=0, seeded_skipped_incompatible=0)

    def seed_from_v2(self, v2_path, v2_query_sig, convert):
        """Reuse a v2 entry only if it is a real hit made with the same query signature.
        v2 cached failures and no-hits identically (None) -> those are re-queried ('refreshed')."""
        if self.entries or not Path(v2_path).exists():
            return
        v2 = json.loads(Path(v2_path).read_text(encoding="utf-8"))   # read-only
        for k, v in v2.items():
            if v is None:
                self.v2_ambiguous.add(k); self.counts["seeded_skipped_ambiguous"] += 1
            elif v2_query_sig != self.query_sig:
                self.counts["seeded_skipped_incompatible"] += 1
            else:
                self.entries[k] = dict(status="FOUND", value=convert(v), provenance="reused_v2",
                                       fetched_at="v2", query_sig=v2_query_sig)
                self.counts["seeded_reused_v2"] += 1

    def get(self, key, fetch):
        """fetch(key) -> ("FOUND", value) | ("NOT_FOUND", None); raises on any failure."""
        if key in self.entries:
            e = self.entries[key]
            self.counts["reused_v2" if e.get("provenance") == "reused_v2" else "cached"] += 1
            return e
        try:
            status, value = fetch(key)
            assert status in ("FOUND", "NOT_FOUND"), status
        except Exception as ex:
            f = self.failures.setdefault(key, dict(attempts=0))
            f.update(attempts=f["attempts"] + 1, error=redact(repr(ex))[:300], last_at=_now())
            self.counts["failed"] += 1
            return None
        prov = "refreshed" if key in self.v2_ambiguous else "new"
        self.counts[prov] += 1
        self.failures.pop(key, None)
        e = self.entries[key] = dict(status=status, value=value, provenance=prov,
                                     fetched_at=_now(), query_sig=self.query_sig)
        return e

    def save(self):
        """Write entries plus meta, and merge failures back into the shared failure file (retried next run)."""
        self.path.write_text(json.dumps(dict(meta=dict(source=self.source, query_sig=self.query_sig,
                                                       v2_ambiguous=sorted(self.v2_ambiguous)),
                                             entries=self.entries)), encoding="utf-8")
        allf = json.loads(self.failures_path.read_text(encoding="utf-8")) if self.failures_path.exists() else {}
        allf[self.source] = self.failures
        self.failures_path.write_text(json.dumps(allf, indent=1), encoding="utf-8")

    def provenance_counts(self):
        """Entry counts grouped as provenance:status pairs, sorted."""
        c = {}
        for e in self.entries.values():
            k = f'{e["provenance"]}:{e["status"]}'
            c[k] = c.get(k, 0) + 1
        return dict(sorted(c.items()))

# --- LLM cache key and occurrence flag ---
def llm_cache_key(chunk_text, system_prompt, schema, model_name, model_sha256, server_build,
                  temperature, max_tokens, prompt_version, retry_policy):
    """sha256 over chunk text, system prompt, schema, GGUF name and sha256, server build and generation settings; v2 keys recorded none of these, so no v2 entry is reused."""
    payload = json.dumps(dict(text_sha256=hashlib.sha256(chunk_text.encode("utf-8")).hexdigest(),
                              system_prompt_sha256=hashlib.sha256(system_prompt.encode("utf-8")).hexdigest(),
                              schema_sha256=hashlib.sha256(json.dumps(schema, sort_keys=True).encode("utf-8")).hexdigest(),
                              model_name=model_name, model_sha256=model_sha256, server_build=server_build,
                              temperature=temperature, max_tokens=max_tokens,
                              prompt_version=prompt_version, retry_policy=retry_policy), sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def llm_locate(chunk, term):
    """Whole-word occurrences of term in chunk -> [(pos, expanded)]; expanded=0 for the first match only."""
    if not term:
        return []
    hits = [m.start() for m in re.finditer(r"(?<!\w)" + re.escape(term) + r"(?!\w)", chunk)]
    return [(h, 0 if i == 0 else 1) for i, h in enumerate(hits)]

# --- five-paper-only checks ---
CHECK_LOG = []
def five_paper_check_ready(name, required_paths=(), required_papers=()):
    """True if a five-paper-only check may run; otherwise logs why it is skipped."""
    why = None
    if FULL_CORPUS:
        why = "FULL_CORPUS=True"
    else:
        miss = [str(p) for p in required_paths if not Path(p).exists()]
        missp = [p for p in required_papers if p not in PAPERS]
        if miss or missp:
            why = f"missing inputs {miss + missp}"
    CHECK_LOG.append(dict(check=name, ran=why is None, reason=why or ""))
    if why:
        print(f"[skip] {name}: {why}")
    return why is None

print("v3 run-safety helpers loaded")

# --- Phase C: registry export (the artifact the audit and Phase E reports point at) ---
from collections import Counter as _Ctr
from collections import Counter as _C  # alias used by LLM/catalogue cells
dump_csv(OUT_DIR / "cdm_mapping_registry.csv", CDM_MAPPING_REGISTRY,
         ["label", "target", "status", "source"])
_registry_meta = dict(
    version=MAPPING_REGISTRY_VERSION, generated_at=_now(),
    audit_csv=str(MAPPING_AUDIT_CSV), audit_csv_exists=MAPPING_AUDIT_CSV.exists(),
    audit_csv_sha256=(file_sha256(MAPPING_AUDIT_CSV) if MAPPING_AUDIT_CSV.exists() else ""),
    n_entries=len(CDM_MAPPING_REGISTRY),
    status_counts=dict(sorted(_Ctr(e["status"] for e in CDM_MAPPING_REGISTRY).items())),
    schema_check=dict(_SCHEMA_CHECK, hard_drift_check="CDM_TYPES == parsed config.py types (asserted)"),
    llm_label_precedence=(
        "Cell 7c LLM rows emit CDM-type strings directly; a label colliding with an audited "
        "registry key inherits that key's status (ORGANISM/PROTEIN -> REVIEW_REQUIRED). The "
        "HOST-vs-MICROBE split and abbreviation-noise reviews are model-agnostic; registry-first "
        "is the conservative default. Untested label spaces remain PROVISIONAL_UNVALIDATED."),
)
(OUT_DIR / "cdm_mapping_registry_meta.json").write_text(json.dumps(_registry_meta, indent=1), encoding="utf-8")
print("mapping registry:", _registry_meta["version"], "| entries:", _registry_meta["n_entries"],
      "| statuses:", _registry_meta["status_counts"])


## 3. Regression tests (Cell 1c)

Unittest coverage for the machinery above: placeholder IDs, ligature round-trips, BERT word
aggregation, mapping status, secrets, lookup cache, tool status, LLM cache key and occurrences,
and the five-paper check gate. The cell **fails the notebook** if any test fails, so it runs before
any tool work.


In [ ]:
# ===== Cell 1c: regression tests for the v2 + v3 fixes (cell fails if any test fails) =====
import unittest

class TestPlaceholderIDs(unittest.TestCase):
    def test_placeholders_are_missing(self):
        for raw in ["-1", " -1 ", "", "-", "None", "CUI-less"]:
            self.assertEqual(clean_norm(raw)[0], "", raw)
        self.assertEqual(clean_norm("-1")[1], "PLACEHOLDER")
        self.assertEqual(clean_norm("")[1], "MISSING")

    def test_valid_ids_kept(self):
        for raw in ["MESH:D002241", "CHEBI:17968", "C0009402", "9606", "OMIM:114500"]:
            self.assertEqual(clean_norm(raw), (raw, "VALID"))

    def test_unrelated_chemicals_with_minus1_do_not_merge(self):
        norm = clean_norm("-1")[0]
        vocab = norm_vocab("tmChem", "Chemical", norm)
        keys = {entity_key(t.lower(), norm, vocab) for t in ["butyrate", "UDCA", "formate"]}
        self.assertEqual(len(keys), 3)

    def test_same_valid_id_merges_across_surface_forms(self):
        k1 = entity_key("butyrate", "MESH:D020148", norm_vocab("tmChem", "Chemical", "MESH:D020148"))
        k2 = entity_key("butyric acid", "MESH:D020148", norm_vocab("TaggerOne-Chemical", "Chemical", "MESH:D020148"))
        self.assertEqual(k1, k2)

    def test_gnormplus_namespaces(self):
        self.assertEqual(norm_vocab("GNormPlus", "Gene", "1234"), "NCBIGene")
        self.assertEqual(norm_vocab("GNormPlus", "Species", "1234"), "NCBITaxon")
        self.assertNotEqual(entity_key("x", "1234", "NCBIGene"), entity_key("x", "1234", "NCBITaxon"))

class TestLigatureOffsets(unittest.TestCase):
    def roundtrip(self, orig, tool_term, want_orig_term):
        tt = ToolText(orig)
        s = tt.norm.index(tool_term)
        m = tt.to_orig(s, s + len(tool_term))
        self.assertIsInstance(m, tuple, m)
        self.assertEqual(orig[m[0]:m[1]], want_orig_term)

    def test_ligature_inside_word(self):
        self.roundtrip("Notably, Biﬁdobacterium breve grew.", "Bifidobacterium breve", "Biﬁdobacterium breve")

    def test_text_after_ligature_shifts_correctly(self):
        self.roundtrip("dietary ﬁbers and ﬂavonoids; butyrate.", "butyrate", "butyrate")
        self.roundtrip("dietary ﬁbers and ﬂavonoids; butyrate.", "flavonoids", "ﬂavonoids")

    def test_multi_char_ligatures(self):
        self.roundtrip("eﬃcacy of the eﬄuent", "efficacy", "eﬃcacy")
        self.roundtrip("eﬃcacy of the eﬄuent", "effluent", "eﬄuent")

    def test_span_cutting_a_ligature_is_rejected_not_widened(self):
        tt = ToolText("Biﬁdobacterium")
        self.assertEqual(tt.to_orig(0, 3), "LIGATURE_SPLIT")   # 'Bif' ends inside 'ﬁ'

    def test_punctuation_hyphen_unicode_untouched(self):
        orig = "5′-TCG-3′ (C. difficile), anti-TNF α–β µg/mL ³H"
        tt = ToolText(orig)
        self.assertTrue(tt.identity)
        self.assertEqual(tt.norm, orig)                         # no NFKC: µ, ′, ³ kept
        for term in ["C. difficile", "anti-TNF", "α–β", "µg/mL"]:
            s = orig.index(term)
            self.assertEqual(tt.to_orig(s, s + len(term)), (s, s + len(term)))

    def test_out_of_text_rejected(self):
        tt = ToolText("abc")
        self.assertEqual(tt.to_orig(1, 5), "OUT_OF_TEXT")

    def test_every_mapped_span_verifies(self):
        orig = "ﬁ ﬂ ﬃ x ﬁber-rich (ﬂ) end"
        tt = ToolText(orig)
        for s in range(len(tt.norm)):
            for e in range(s + 1, len(tt.norm) + 1):
                m = tt.to_orig(s, e)
                if isinstance(m, tuple):
                    self.assertEqual(lig_normalize(orig[m[0]:m[1]]), tt.norm[s:e])

class TestBertAggregation(unittest.TestCase):
    def test_bplus_subword_labels_do_not_cut_words(self):
        # "Bifidobacterium breve" -> pieces Bi ##fi ##do ##bacterium breve; 2nd piece mislabeled O
        wids = [None, 0, 0, 0, 0, 1, None]
        offs = [(0, 0), (0, 2), (2, 4), (4, 6), (6, 15), (16, 21), (0, 0)]
        labs = ["O", "B-SPECIES", "O", "B-SPECIES", "I-SPECIES", "I-SPECIES", "O"]
        self.assertEqual(words_to_spans(word_level(wids, labs, offs)), [("SPECIES", 0, 21)])

    def test_entity_spanning_token_boundaries(self):
        # "16S rRNA gene" = 3 words; punctuation word ends the entity
        wids = [0, 1, 1, 2, 3]
        offs = [(0, 3), (4, 5), (5, 8), (9, 13), (13, 14)]
        labs = ["B-GENETIC", "I-GENETIC", "I-GENETIC", "I-GENETIC", "O"]
        self.assertEqual(words_to_spans(word_level(wids, labs, offs)), [("GENETIC", 0, 13)])

    def test_type_change_starts_new_entity(self):
        words = [[0, 5, "B-DISEASE"], [6, 10, "I-CHEMICAL"], [11, 14, "I-CHEMICAL"]]
        self.assertEqual(words_to_spans(words), [("DISEASE", 0, 5), ("CHEMICAL", 6, 14)])

    def test_word_spans_are_never_partial(self):
        text = "Biﬁdobacterium"
        self.assertTrue(partial_word(text, 0, 7))
        self.assertFalse(partial_word(text, 0, len(text)))

# Pins registry-0.2 (audited; see CDM_MAPPING_REGISTRY + cdm_mapping_audit.csv):
#   Genus -> ORGANISM/MAPPED (audit:SILENT_PASSTHROUGH_FIX)
#   ORGAN -> UNMAPPED_ORGAN/UNMAPPED (audit:WRONG); sentinel, never a made-up real type
class TestMappingStatus(unittest.TestCase):
    def test_status_only(self):
        self.assertEqual(mapping_status("Genus", cdm_of("Genus", "GNormPlus")), "MAPPED")
        self.assertEqual(cdm_of("Genus", "GNormPlus"), "ORGANISM")
        self.assertEqual(mapping_status("CELL_LINE", cdm_of("CELL_LINE", "SciBERT-JNLPBA")), "UNMAPPED")
        self.assertEqual(mapping_status("ORGAN", cdm_of("ORGAN", "x")), "UNMAPPED")
        self.assertEqual(mapping_status("Disease", cdm_of("Disease", "DNorm")), "MAPPED")
        self.assertEqual(cdm_of("ORGAN", "x"), "UNMAPPED_ORGAN")   # UNMAPPED_* sentinel, not a real type

# ---------------------------- v3 (Phase 1) regression tests ----------------------------
_T = Path(tempfile.mkdtemp(prefix="v3tests_"))
_PY = sys.executable

class TestSecrets(unittest.TestCase):
    def test_missing_key_raises_clear_error(self):
        old = os.environ.pop("ZZ_TEST_KEY", None)
        with self.assertRaises(RuntimeError) as cm:
            get_secret("ZZ_TEST_KEY")
        self.assertIn("ZZ_TEST_KEY is not set", str(cm.exception))

    def test_redact_removes_secret_values(self):
        for n in SECRET_ENV_VARS:
            v = os.environ.get(n, "")
            if v:
                self.assertNotIn(v, redact(f"url?apikey={v}&x=1"))
        os.environ["UMLS_API_KEY_TMP"] = "s3cr3t-value"
        g = globals(); saved = g["SECRET_ENV_VARS"]
        try:
            g["SECRET_ENV_VARS"] = saved + ("UMLS_API_KEY_TMP",)
            self.assertEqual(redact("k=s3cr3t-value"), "k=<UMLS_API_KEY_TMP>")
        finally:
            g["SECRET_ENV_VARS"] = saved
            os.environ.pop("UMLS_API_KEY_TMP")

class TestLookupCache(unittest.TestCase):
    def mk(self, name):
        return LookupCache("T", _T / f"{name}.json", _T / f"{name}_fail.json", "sig1")

    def test_no_hit_is_cached(self):
        c = self.mk("nohit"); calls = []
        fetch = lambda k: (calls.append(k), ("NOT_FOUND", None))[1]
        self.assertEqual(c.get("x", fetch)["status"], "NOT_FOUND")
        c.get("x", fetch)
        self.assertEqual(len(calls), 1)

    def test_failure_logged_not_cached_and_retried(self):
        c = self.mk("fail"); calls = []
        def boom(k):
            calls.append(k); raise LookupFailure("HTTP 503")
        self.assertIsNone(c.get("y", boom))
        self.assertNotIn("y", c.entries)
        self.assertEqual(c.failures["y"]["attempts"], 1)
        c.save()
        c2 = self.mk("fail")                                 # next run
        self.assertEqual(c2.get("y", lambda k: ("FOUND", {"id": 1}))["status"], "FOUND")
        self.assertNotIn("y", c2.failures)
        self.assertEqual(c2.counts["new"], 1)

    def test_v2_seeding_rules(self):
        v2 = _T / "v2cache.json"
        v2.write_text(json.dumps({"hit": {"id": "A"}, "amb": None}), encoding="utf-8")
        c = self.mk("seed"); c.seed_from_v2(v2, "sig1", lambda v: v)
        self.assertEqual(c.entries["hit"]["provenance"], "reused_v2")
        self.assertNotIn("amb", c.entries)                   # ambiguous v2 None never trusted
        self.assertEqual(c.get("amb", lambda k: ("NOT_FOUND", None))["provenance"], "refreshed")
        c3 = self.mk("seed_incompat"); c3.seed_from_v2(v2, "OTHER_SIG", lambda v: v)
        self.assertEqual(c3.entries, {})
        self.assertEqual(c3.counts["seeded_skipped_incompatible"], 1)

    def test_query_signature_change_refused(self):
        c = self.mk("sigchk"); c.get("a", lambda k: ("FOUND", 1)); c.save()
        with self.assertRaises(RuntimeError):
            LookupCache("T", _T / "sigchk.json", _T / "sigchk_fail.json", "sig2")

class TestToolStatus(unittest.TestCase):
    def tearDown(self):
        for k in [k for k in TOOL_STATUS if k.startswith("_t")]:
            TOOL_STATUS.pop(k)

    def test_nonzero_exit_is_failed_and_not_registered(self):
        out = _T / "o1.json"
        ok = run_tool_checked("_t1", [_PY, "-c", "import sys; sys.exit(3)"], _T, _T / "t1.log", [out], 30)
        self.assertFalse(ok)
        self.assertEqual(TOOL_STATUS["_t1"]["reason"], "EXIT_CODE 3")
        n = len(all_rows)
        self.assertEqual(register([{"x": 1}], "_t1model", "_t1"), [])
        self.assertEqual(len(all_rows), n)
        self.assertFalse((OUT_DIR / "_t1model_candidates.csv").exists())

    def test_timeout_is_failed(self):
        ok = run_tool_checked("_t2", [_PY, "-c", "import time; time.sleep(30)"], _T, _T / "t2.log", [_T / "o2"], 1)
        self.assertFalse(ok)
        self.assertTrue(TOOL_STATUS["_t2"]["reason"].startswith("TIMEOUT"))

    def test_stale_output_is_removed_and_not_reused(self):
        stale = _T / "o3.json"; stale.write_text("old", encoding="utf-8")
        ok = run_tool_checked("_t3", [_PY, "-c", "pass"], _T, _T / "t3.log", [stale], 30)
        self.assertFalse(ok)                                  # rc 0 but no fresh output
        self.assertFalse(stale.exists())
        self.assertIn("NO_OUTPUT", TOOL_STATUS["_t3"]["reason"])
        with self.assertRaises(RuntimeError):
            consume_output("_t3", stale)

    def test_output_changed_after_run_is_rejected(self):
        out = _T / "o4.json"
        code = f"open(r'{out}', 'w').write('fresh')"
        self.assertTrue(run_tool_checked("_t4", [_PY, "-c", code], _T, _T / "t4.log", [out], 30))
        self.assertEqual(consume_output("_t4", out), out)
        out.write_text("tampered", encoding="utf-8")
        with self.assertRaises(RuntimeError):
            consume_output("_t4", out)

    def test_tool_writing_into_precreated_dir(self):
        d = _T / "o6dir"; d.mkdir(); (d / "stale.txt").write_text("old", encoding="utf-8")
        code = f"open(r'{d / 'fresh.txt'}', 'w').write('new')"   # tool needs the folder to exist
        self.assertTrue(run_tool_checked("_t6", [_PY, "-c", code], _T, _T / "t6.log", [d], 30, create_dirs=[d]))
        self.assertFalse((d / "stale.txt").exists())                # old content removed
        self.assertEqual(consume_output("_t6", d).name, "fresh.txt")

    def test_output_dir_needs_a_file(self):
        d = _T / "o5dir"
        code = f"import os; os.makedirs(r'{d}', exist_ok=True)"
        self.assertFalse(run_tool_checked("_t5", [_PY, "-c", code], _T, _T / "t5.log", [d], 30))

class TestLLMKeyAndOccurrences(unittest.TestCase):
    base = dict(chunk_text="Butyrate and butyrate.", system_prompt="P", schema={"a": 1},
                model_name="m.gguf", model_sha256="abc", server_build="b1", temperature=0,
                max_tokens=2048, prompt_version="v1", retry_policy="halves-1")

    def test_key_changes_with_every_component(self):
        k0 = llm_cache_key(**self.base)
        for f, v in [("chunk_text", "x"), ("system_prompt", "Q"), ("schema", {"a": 2}),
                     ("model_name", "n.gguf"), ("model_sha256", "def"), ("server_build", "b2"),
                     ("temperature", 0.1), ("max_tokens", 1024), ("prompt_version", "v2"),
                     ("retry_policy", "none")]:
            self.assertNotEqual(llm_cache_key(**dict(self.base, **{f: v})), k0, f)
        self.assertEqual(llm_cache_key(**self.base), k0)

    def test_first_match_zero_further_matches_one(self):
        self.assertEqual(llm_locate("SCFA and SCFAs; SCFA.", "SCFA"), [(0, 0), (16, 1)])
        self.assertEqual(llm_locate("abc", "zz"), [])

    def test_make_row_default_flag(self):
        self.assertIn("llm_occurrence_expanded", FIELDS)

class TestFivePaperChecks(unittest.TestCase):
    def test_skipped_when_inputs_missing(self):
        self.assertFalse(five_paper_check_ready("_t", [_T / "nope.csv"]))
        self.assertFalse(five_paper_check_ready("_t", [], ["HMPAPER_999999"]))
        self.assertTrue(five_paper_check_ready("_t", [_T]))
        CHECK_LOG[:] = [c for c in CHECK_LOG if c["check"] != "_t"]

_suite = unittest.TestSuite()
for _tc in (TestPlaceholderIDs, TestLigatureOffsets, TestBertAggregation, TestMappingStatus,
            TestSecrets, TestLookupCache, TestToolStatus, TestLLMKeyAndOccurrences, TestFivePaperChecks):
    _suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(_tc))
_res = unittest.TextTestRunner(verbosity=2, stream=sys.stdout).run(_suite)
TEST_RESULTS = dict(ran=_res.testsRun, failures=len(_res.failures), errors=len(_res.errors),
                    ok=_res.wasSuccessful(),
                    failed_tests=[str(t) for t, _ in _res.failures + _res.errors])
shutil.rmtree(_T, ignore_errors=True)
(OUT_DIR / "test_results.json").write_text(json.dumps(TEST_RESULTS, indent=1), encoding="utf-8")
assert _res.wasSuccessful(), "regression tests failed - run stopped"

## 3b. v4 helpers and tests (Cell 1d)

S7 section selection, PROPOSED section-class crosswalk, dictionary matcher, junk flags and span groups, with 13 regression tests (the cell stops the run if any fails).

In [ ]:
# ===== Cell 1d: v4 helpers - S7 section decisions, section classes, dictionary matchers, junk flags, span groups =====
# Pure functions used by Cells 2b, 7d, 7e and 8c, with their own regression tests at the end of this cell.
#   N1  select_section(): the S7 decision (section_selection_final.csv) decides first; the backmatter
#       list stays as a second guard. Every exclusion is logged with its reason.
#   N6  section_class_for(): PROPOSED crosswalk S5 rules R01-R17 -> section class
#       (staging/ner_audit/section_class_crosswalk_PROPOSED.yaml, option B = inherit from nearest ancestor).
#       Every class carries status PROPOSED_CROSSWALK until the crosswalk is approved.
#   N2  DictMatcher: whole-word, case-insensitive, longest-first dictionary matcher (taxa, methods).
#   N3  junk_reason(): flags only; rows are never deleted.
#   N4  span_groups(): identical or overlapping spans in one section share a span_group_id.
import hashlib as _hl

# ---------- N1: S7 decisions ----------
def load_section_decisions(path):
    out = {}
    if Path(path).exists():
        with open(path, encoding="utf-8", newline="") as f:
            for r in csv.DictReader(f):
                # 2026-10-28: the surviving decision table is section_selection.csv, which names the
                # verdict `selection_status`; older tables used `final_status`. Normalise once, here,
                # so select_section()/section_class_for() keep reading a single key.
                r["final_status"] = (r.get("final_status") or r.get("selection_status") or "").strip().upper()
                out[(r["corpus_id"], r["section_id"])] = r
    return out

def select_section(cid, sid, path, decisions):
    """-> (keep, reason). S7 final_status first, backmatter list second."""
    d = decisions.get((cid, sid))
    back = BODY_ONLY and is_backmatter(path)
    if d is None:
        return (not back), ("NO_S7_DECISION_BACKMATTER" if back else "NO_S7_DECISION_KEPT")
    st = d.get("final_status", "")
    if st not in SEL_KEEP:
        return False, f"S7_{st or 'BLANK'}"
    if back:
        return False, f"S7_{st}_BACKMATTER_GUARD"
    return True, f"S7_{st}"

# ---------- N6: section classes (PROPOSED crosswalk, mirrors the yaml) ----------
XWALK_VERSION = "xwalk-0.1"
XWALK = [  # (rule_id, alias regex on the section's own title or None, class) - first match wins
    ("R16_methods_container", None, "METHODS"),
    ("R17_results_container", None, "RESULTS"),
    ("R01_research_question", r"^abstract$", "TITLE_ABSTRACT"),
    ("R01_research_question", r"^(introduction|background|aims?|objectives?|rationale)$", "INTRODUCTION"),
    ("R02_study_design", None, "METHODS"), ("R03_population_recruitment", None, "METHODS"),
    ("R04_eligibility_criteria", None, "METHODS"), ("R05_sampling_specimen", None, "METHODS"),
    ("R06_lab_methods_extraction", None, "METHODS"), ("R07_bioinformatics_profiling", None, "METHODS"),
    ("R08_statistics", None, "METHODS"), ("R09_findings_results", None, "RESULTS"),
    ("R10_limitations_interpretation", r"conclusions?", "CONCLUSION"),
    ("R10_limitations_interpretation", r"limitations?|discussion|interpretation|future", "DISCUSSION"),
    ("R11_data_availability", None, "NOT_APPLICABLE_EXCLUDED"),
    ("R12_document_statements", None, "NOT_APPLICABLE_EXCLUDED"),
    ("R13_supplementary_material", None, "NOT_APPLICABLE_EXCLUDED"),
    ("R14_references", None, "NOT_APPLICABLE_EXCLUDED"),
    ("R15_ethics_author_statements", None, "NOT_APPLICABLE_EXCLUDED"),
]

def _own_title(h):
    return re.sub(r"^[\d.\s]+", "", h or "").strip().lower()

def own_section_class(rule_ids, heading):
    ids = {x for x in (rule_ids or "").split(";") if x}
    for rid, pat, cls in XWALK:
        if rid in ids and (pat is None or re.search(pat, _own_title(heading))):
            return cls, f"{rid}@{XWALK_VERSION}"
    return None, None

def section_class_for(cid, sid, path, decisions):
    """-> (section_class, section_class_rule). Option B: no own match -> nearest matched ancestor."""
    if path.split(" > ")[-1].strip() == "(preamble)" or path.strip() == "(preamble)":
        return "PREAMBLE", f"PREAMBLE@{XWALK_VERSION}"
    d = decisions.get((cid, sid))
    if d is None:
        return "UNKNOWN_NO_S7_DECISION", ""
    cls, rule = own_section_class(d.get("rule_ids", ""), d.get("heading", ""))
    if cls:
        return cls, rule
    by_path = {r["heading_path"]: r for (c, _), r in decisions.items() if c == cid}
    parts = d.get("heading_path", "").split(" > ")
    for k in range(len(parts) - 1, 0, -1):
        anc = by_path.get(" > ".join(parts[:k]))
        if anc:
            cls, rule = own_section_class(anc.get("rule_ids", ""), anc.get("heading", ""))
            if cls:
                return cls, "ANCESTOR:" + rule
    return "OTHER_BODY", f"NO_MATCH@{XWALK_VERSION}"

# ---------- N2: dictionary matcher ----------
class DictMatcher:
    """Whole-word, case-insensitive, longest-first. find() -> [(start, end, text, payloads)]."""
    def __init__(self, entries, case_sensitive=False):
        self.cs = case_sensitive
        self.by_key = {}
        for surface, payload in entries:
            s = surface.strip()
            if s:
                self.by_key.setdefault(s if case_sensitive else s.lower(), []).append(payload)
        alts = sorted(self.by_key, key=len, reverse=True)
        self.rx = (re.compile(r"(?<![\w-])(?:" + "|".join(re.escape(a) for a in alts) + r")(?![\w-])",
                              0 if case_sensitive else re.I) if alts else None)

    def find(self, text):
        if not self.rx:
            return []
        out = []
        for m in self.rx.finditer(text):
            k = m.group(0) if self.cs else m.group(0).lower()
            out.append((m.start(), m.end(), m.group(0), self.by_key[k]))
        return out

TAXON_COLLECTIVE = {"bacteria", "bacterium", "bacterial", "microbiota", "microbiome", "flora",
                    "gut flora", "prokaryotes", "eubacteria"}

def load_mesh_taxa(path):
    """MeSH B03 lexicon (terms/cache) -> [(surface, payload)]; collective nouns left out (guide rule)."""
    d = json.loads(Path(path).read_text(encoding="utf-8"))
    out = []
    for e in d["entries"]:
        for low, s in e["surfaces"].items():
            if low in TAXON_COLLECTIVE:
                continue
            out.append((s["surface"], dict(id="MESH:" + e["concept_id"], name=e["preferred_label"],
                                           kind=s["kind"], tree=e["tree_numbers"][0] if e["tree_numbers"] else "")))
    return out, d.get("xml_sha256", "")

ABBREV_RX = re.compile(r"(?<![\w.])([A-Z])\.\s?([a-z]{3,})\b")

def expand_abbreviations(text, full_names, stoplist=()):
    """'F. prausnitzii' -> the full 'Faecalibacterium prausnitzii' seen in the same paper, if unique.
    full_names: set of 'Genus species' strings found in the paper. -> [(s, e, text, expansion|None)]."""
    by_key = {}
    for n in full_names:
        p = n.split()
        if len(p) >= 2:
            by_key.setdefault((p[0][0], p[1].lower()), set()).add(n)
    out = []
    for m in ABBREV_RX.finditer(text):
        if m.group(2).lower() in stoplist:
            continue
        hits = by_key.get((m.group(1), m.group(2).lower()), set())
        out.append((m.start(), m.end(), m.group(0), next(iter(hits)) if len(hits) == 1 else None))
    return out

# methods dictionary (N2): (surface, label). Labels map to CDM types in V4_LABELS below.
METHOD_TERMS = [(s, "kit") for s in (
    "QIAamp DNA Stool Mini Kit", "QIAamp Fast DNA Stool Mini Kit", "QIAamp DNA Mini Kit",
    "QIAamp PowerFecal DNA Kit", "QIAamp PowerFecal Pro DNA Kit", "DNeasy PowerSoil Kit",
    "DNeasy PowerSoil Pro Kit", "PowerSoil DNA Isolation Kit", "MoBio PowerSoil", "PowerFecal",
    "PowerSoil", "MagAttract PowerSoil DNA Kit", "ZymoBIOMICS DNA Miniprep Kit", "FastDNA SPIN Kit",
    "FastDNA SPIN Kit for Soil", "NucleoSpin Soil", "E.Z.N.A. Stool DNA Kit", "Nextera XT",
    "TruSeq", "KAPA HiFi HotStart ReadyMix", "Qubit dsDNA HS Assay Kit", "AMPure XP")] + \
  [(s, "platform") for s in (
    "Illumina MiSeq", "Illumina HiSeq", "Illumina NovaSeq", "Illumina NextSeq", "MiSeq", "HiSeq",
    "NovaSeq", "NextSeq", "Illumina", "Ion Torrent", "Ion PGM", "Ion S5", "454 pyrosequencing",
    "Roche 454", "GS FLX", "PacBio", "Oxford Nanopore", "MinION", "BGISEQ-500", "BGISEQ", "DNBSEQ")] + \
  [(s, "primer") for s in (
    "515F", "806R", "341F", "785R", "805R", "27F", "338F", "1492R", "926R", "907R", "799F", "1193R",
    "8F", "533R", "319F", "U515F", "515FB", "806RB", "ITS1F", "ITS2")] + \
  [(s, "assay") for s in (
    "16S rRNA gene sequencing", "16S rRNA sequencing", "16S rDNA sequencing", "amplicon sequencing",
    "shotgun metagenomic sequencing", "shotgun metagenomics", "metagenomic sequencing",
    "whole-genome shotgun sequencing", "qPCR", "quantitative PCR", "real-time PCR")] + \
  [(s, "software") for s in (
    "QIIME 2", "QIIME2", "QIIME", "DADA2", "mothur", "LEfSe", "MetaPhlAn", "MetaPhlAn2", "MetaPhlAn3",
    "MetaPhlAn4", "HUMAnN", "HUMAnN2", "HUMAnN3", "PICRUSt", "PICRUSt2", "Kraken", "Kraken2",
    "Bracken", "USEARCH", "VSEARCH", "UPARSE", "Deblur", "phyloseq", "vegan", "MaAsLin2", "ANCOM",
    "DESeq2")] + \
  [(s, "reference_db") for s in ("SILVA", "Greengenes", "Greengenes2", "RDP", "GTDB", "UNITE")]

V_REGION_RX = re.compile(r"(?<![\w-])V[1-9](?:\s?[-–/]\s?V[1-9])?(?![\w-])")
V_REGION_CONTEXT = re.compile(r"16S|rRNA|rDNA|region|amplicon|hypervariable", re.I)

def find_v_regions(text, window=80):
    """V-region mentions with 16S/region context within `window` chars (no context -> not emitted)."""
    out = []
    for m in V_REGION_RX.finditer(text):
        ctx = text[max(0, m.start() - window): m.end() + window]
        if V_REGION_CONTEXT.search(ctx):
            out.append((m.start(), m.end(), m.group(0)))
    return out

V4_LABELS = {"kit": "LAB_METHOD", "platform": "LAB_METHOD", "primer": "LAB_METHOD",
             "amplicon_region": "LAB_METHOD", "assay": "LAB_METHOD", "software": "DATA_ANALYSIS",
             "reference_db": "DATA_ANALYSIS", "taxon_mesh": "ORGANISM", "taxon_abbrev": "ORGANISM"}
if "_cdm_of_v3" not in globals():
    _cdm_of_v3 = cdm_of
def cdm_of(label, model):
    return V4_LABELS.get((label or "").strip().lower()) or _cdm_of_v3(label, model)

# ---------- N4: span groups ----------
def span_groups(rows):
    """Assign span_group_id: identical or overlapping spans in one (cid, section) chain into one group.
    Adds group_tools, group_n_tools, group_types, type_conflict. Returns {gid: summary}."""
    by_sec = {}
    for r in rows:
        by_sec.setdefault((r["cid"], r["section_id"]), []).append(r)
    groups = {}
    for (cid, sid), rs in by_sec.items():
        rs.sort(key=lambda r: (int(r["char_start"]), -int(r["char_end"])))
        cur, cur_end = [], -1
        def close(members):
            if not members:
                return
            s0 = min(int(m["char_start"]) for m in members); e0 = max(int(m["char_end"]) for m in members)
            gid = f"SG|{cid}|{sid}|{s0}|{e0}"
            tools = sorted({m["model"] for m in members})
            types = sorted({m["cdm_type_proposed"] for m in members if m.get("mapping_status") != "UNMAPPED"})
            for m in members:
                m.update(span_group_id=gid, group_tools="|".join(tools), group_n_tools=len(tools),
                         group_types="|".join(types), type_conflict=int(len(types) > 1))
            groups[gid] = dict(cid=cid, section_id=sid, start=s0, end=e0, tools=tools, types=types,
                               n_rows=len(members))
        for r in rs:
            s, e = int(r["char_start"]), int(r["char_end"])
            if cur and s < cur_end:
                cur.append(r); cur_end = max(cur_end, e)
            else:
                close(cur); cur, cur_end = [r], e
        close(cur)
    return groups

# ---------- N3: junk flags (flag, never delete) ----------
GENERAL_NER_LABELS = {"PERSON", "DATE", "CARDINAL", "ORDINAL", "PERCENT", "QUANTITY"}
DROP_REASONS = {"", "JUNK_NUMERIC", "JUNK_SHORT", "GENERAL_NER_OVERLAP", "LOW_CONFIDENCE"}

def junk_reason(term, general_label, has_valid_id, n_tools, has_link):
    """-> (drop_reason, rescued_from). Rescue: valid ID, >=2 tools in the span group, or an ontology link."""
    t = re.sub(r"\s+", " ", term or "").strip()
    flag = ""
    if not re.search(r"[^\W\d_]", t):
        flag = "JUNK_NUMERIC"
    elif len(t) < 2:
        flag = "JUNK_SHORT"
    elif general_label in GENERAL_NER_LABELS:
        flag = "GENERAL_NER_OVERLAP"
    rescue = has_valid_id or n_tools >= 2 or has_link
    if flag:
        return ("", flag) if rescue else (flag, "")
    if not rescue:
        return "LOW_CONFIDENCE", ""
    return "", ""

def overlap_label(spans, s, e):
    """First general-NER label in GENERAL_NER_LABELS overlapping [s, e), else ''."""
    for a, b, lab in spans:
        if a < e and s < b and lab in GENERAL_NER_LABELS:
            return lab
    return ""

def stable_id(prefix, *parts):
    return prefix + _hl.sha1("|".join(map(str, parts)).encode("utf-8")).hexdigest()[:12]

# ---------- tests (cell stops the run if any fails) ----------
class TestV4Selection(unittest.TestCase):
    D = {("P", "S1"): dict(final_status="RELEVANT", rule_ids="R16_methods_container", heading="2. Methods",
                           heading_path="T > 2. Methods"),
         ("P", "S2"): dict(final_status="UNCERTAIN", rule_ids="", heading="Stool collection",
                           heading_path="T > 2. Methods > Stool collection"),
         ("P", "S3"): dict(final_status="IRRELEVANT", rule_ids="", heading="Footnotes", heading_path="T > Footnotes"),
         ("P", "S4"): dict(final_status="RELEVANT", rule_ids="R01_research_question", heading="Abstract",
                           heading_path="T > Abstract"),
         ("P", "S5"): dict(final_status="RELEVANT_CONTEXT", rule_ids="", heading="References",
                           heading_path="T > References")}
    def test_s7_decides_first(self):
        self.assertEqual(select_section("P", "S3", "T > Footnotes", self.D), (False, "S7_IRRELEVANT"))
        self.assertEqual(select_section("P", "S2", "T > 2. Methods > Stool collection", self.D), (True, "S7_UNCERTAIN"))
    def test_backmatter_guard_still_applies(self):
        self.assertEqual(select_section("P", "S5", "T > References", self.D)[0], False)
    def test_missing_decision_falls_back(self):
        self.assertEqual(select_section("P", "SX", "T > Results", self.D), (True, "NO_S7_DECISION_KEPT"))
    def test_section_class_own_ancestor_preamble(self):
        self.assertEqual(section_class_for("P", "S1", "", self.D)[0], "METHODS")
        c, rule = section_class_for("P", "S2", "", self.D)
        self.assertEqual((c, rule.startswith("ANCESTOR:R16")), ("METHODS", True))
        self.assertEqual(section_class_for("P", "S4", "", self.D)[0], "TITLE_ABSTRACT")
        self.assertEqual(section_class_for("P", "S9", "(preamble)", self.D)[0], "PREAMBLE")

class TestV4Dictionaries(unittest.TestCase):
    def test_longest_first_whole_word(self):
        m = DictMatcher([("Illumina", "a"), ("Illumina MiSeq", "b"), ("MiSeq", "c")])
        self.assertEqual([x[2] for x in m.find("on an Illumina MiSeq and Illuminas")], ["Illumina MiSeq"])
    def test_case_insensitive_offsets_exact(self):
        t = "using qiime 2 and DADA2."
        for s, e, txt, _ in DictMatcher([(x, y) for x, y in METHOD_TERMS]).find(t):
            self.assertEqual(t[s:e], txt)
    def test_v_region_needs_context(self):
        self.assertEqual([x[2] for x in find_v_regions("the V3-V4 region of the 16S rRNA gene")], ["V3-V4"])
        self.assertEqual(find_v_regions("patients with V4 status"), [])
    def test_abbreviation_expanded_only_if_unique(self):
        full = {"Faecalibacterium prausnitzii", "Bacteroides fragilis"}
        r = expand_abbreviations("F. prausnitzii and B. fragilis and E. coli", full)
        self.assertEqual([(x[2], x[3]) for x in r],
                         [("F. prausnitzii", "Faecalibacterium prausnitzii"),
                          ("B. fragilis", "Bacteroides fragilis"), ("E. coli", None)])
    def test_new_labels_map_to_cdm(self):
        self.assertEqual(cdm_of("kit", "Methods-Dict"), "LAB_METHOD")
        self.assertEqual(cdm_of("Disease", "DNorm"), "DISEASE")
        self.assertEqual(mapping_status("software", cdm_of("software", "x")), "MAPPED")

class TestV4JunkAndGroups(unittest.TestCase):
    def test_flags_and_rescue(self):
        self.assertEqual(junk_reason("14", "", False, 1, False), ("JUNK_NUMERIC", ""))
        self.assertEqual(junk_reason("0.05", "", False, 3, False), ("", "JUNK_NUMERIC"))
        self.assertEqual(junk_reason("Smith", "PERSON", False, 1, False), ("GENERAL_NER_OVERLAP", ""))
        self.assertEqual(junk_reason("Escherichia", "PERSON", True, 1, False), ("", "GENERAL_NER_OVERLAP"))
        self.assertEqual(junk_reason("widget", "", False, 1, False), ("LOW_CONFIDENCE", ""))
        self.assertEqual(junk_reason("butyrate", "", False, 1, True), ("", ""))
    def test_overlap_label(self):
        self.assertEqual(overlap_label([(0, 5, "PERSON")], 3, 8), "PERSON")
        self.assertEqual(overlap_label([(0, 5, "ORG")], 3, 8), "")
        self.assertEqual(overlap_label([(0, 5, "PERSON")], 5, 8), "")
    def test_span_groups(self):
        rows = [dict(cid="P", section_id="S", model=m, char_start=s, char_end=e, cdm_type_proposed=t,
                     mapping_status="MAPPED") for m, s, e, t in
                [("A", 0, 17, "DISEASE"), ("B", 11, 17, "DISEASE"), ("C", 0, 17, "CHEMICAL"), ("A", 30, 35, "DISEASE")]]
        g = span_groups(rows)
        self.assertEqual(len(g), 2)
        self.assertEqual(rows[0]["span_group_id"], rows[1]["span_group_id"])
        self.assertEqual(rows[0]["type_conflict"], 1)
        self.assertEqual(rows[3]["group_n_tools"], 1)
    def test_stable_id(self):
        self.assertEqual(stable_id("E_", "DISEASE", "MESH:D1"), stable_id("E_", "DISEASE", "MESH:D1"))
        self.assertNotEqual(stable_id("E_", "DISEASE", "MESH:D1"), stable_id("E_", "CHEMICAL", "MESH:D1"))

_suite4 = unittest.TestSuite()
for _tc in (TestV4Selection, TestV4Dictionaries, TestV4JunkAndGroups):
    _suite4.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(_tc))
_res4 = unittest.TextTestRunner(verbosity=2, stream=sys.stdout).run(_suite4)
TEST_RESULTS["v4"] = dict(ran=_res4.testsRun, failures=len(_res4.failures), errors=len(_res4.errors),
                          ok=_res4.wasSuccessful(),
                          failed_tests=[str(t) for t, _ in _res4.failures + _res4.errors])
(OUT_DIR / "test_results.json").write_text(json.dumps(TEST_RESULTS, indent=1), encoding="utf-8")
assert _res4.wasSuccessful(), "v4 regression tests failed - run stopped"

## 4. Section parsing and tool-input texts (Cells 2, 2b)

`parse_sections` turns the `<!-- SEGMENT / SECTION_ID / ... -->` markdown into the verified section
texts (marker lines removed line-wise, segments joined with blank lines; self-check anchored on
paper 1 and the AIONER XML). Cell 2b builds `TT`: body sections only (backmatter dropped,
`(preamble)` kept), ligatures expanded for the tools, with `ToolText` keeping the offset map back to
the original text.


In [ ]:
# ===== Cell 2: section parser + self-check =====
MARKER_LINE = re.compile(r"^\s*<!-- (SECTION_PATH: .*?|SEGMENT: \S+|SECTION_ID: \S+|END_SEGMENT: \S+) -->\s*$")
# 2026-10-28: every one of the 4,006 segments also carries a
# `<!-- SEGMENT_ORDINAL_WITHIN_SECTION: n -->` line that MARKER_LINE above does NOT match, so it
# used to survive into the section body as a 43-char HTML comment at the top of every section.
# cdm_pipeline.ner.runner_sections (which produced every NER/LLM artifact audited in Cells 14-19)
# strips it. With this second filter the notebook parser is byte-identical to the runner on all
# 4,006 sections; without it every char_start/char_end would be off by 43.
ORDINAL_LINE = re.compile(r"^\s*<!-- SEGMENT_ORDINAL_WITHIN_SECTION: \d+ -->\s*$")

def parse_sections(md_text):
    """Parse one segmented-sections markdown file into section_id -> joined body text, marker lines removed; the self-check below verifies lengths against the verified paper-1 baseline."""
    txt = md_text
    if txt.startswith("---"):
        txt = re.sub(r"\A---\n.*?\n---\n", "", txt, flags=re.S)
    out = {}
    for m in re.finditer(
            r"<!-- SEGMENT: (\S+) -->\n<!-- SECTION_ID: (\S+) -->\n(.*?)\n<!-- END_SEGMENT: \1 -->",
            txt, flags=re.S):
        seg_id, sec_id, body = m.group(1), m.group(2), m.group(3)
        lines = [ln for ln in body.split("\n")
                 if not MARKER_LINE.match(ln) and not ORDINAL_LINE.match(ln)]
        out.setdefault(sec_id, []).append("\n".join(lines).strip())
    return {s: "\n\n".join(v) for s, v in out.items()}

secs = {cid: parse_sections((SEG_DIR / f"{cid}.md").read_text(encoding="utf-8")) for cid in PAPERS}
sec_paths = {}  # cid -> {sec_id: SECTION_PATH} (used only for body filtering)
for cid in PAPERS:
    _md = (SEG_DIR / f"{cid}.md").read_text(encoding="utf-8")
    for _m in re.finditer(r"<!-- SEGMENT: \S+ -->\n<!-- SECTION_ID: (\S+) -->\n(.*?)\n<!-- END_SEGMENT: \S+ -->", _md, flags=re.S):
        _pm = re.search(r"<!-- SECTION_PATH: (.*?) -->", _m.group(2))
        sec_paths.setdefault(cid, {}).setdefault(_m.group(1), _pm.group(1) if _pm else "")
print("section paths extracted:", sum(len(v) for v in sec_paths.values()))
for cid in PAPERS:
    print(cid, "sections:", len(secs[cid]), "total_chars:", sum(len(t) for t in secs[cid].values()))

# --- self-check vs the corpus's own section inventory ---
# 2026-10-28: the old anchor (PRJDB11845_SEC_0001, a 193-char slice of the colorectal-cancer paper)
# no longer exists. corpus 2.2.0 renumbered papers: that paper is now HMPAPER_000005 and every
# section id is sec_NNN. A hard-coded byte anchor is therefore not maintainable here.
# Instead: (a) the parser must be byte-identical to cdm_pipeline.ner.runner_sections (the component
# that produced every artifact Cells 14-19 audit), and (b) each section must start on real text,
# not on a leftover HTML comment.
if five_paper_check_ready("paper1_anchor_and_aioner_check", required_papers=["HMPAPER_000002"]):
    import sys as _sys
    for _p in (ROOT / "src", ROOT):
        if str(_p) not in _sys.path:
            _sys.path.insert(0, str(_p))
    from cdm_pipeline.ner.runner_sections import parse_markdown as _runner_parse
    _same = _diff = 0
    for _cid in PAPERS:
        _raw = (SEG_DIR / f"{_cid}.md").read_text(encoding="utf-8", errors="replace")
        _rn = {s["section_id"]: s["body"] for s in _runner_parse(_raw)[0]}
        for _sid, _t in secs[_cid].items():
            if _rn.get(_sid) == _t:
                _same += 1
            else:
                _diff += 1
                print("  PARSER MISMATCH", _cid, _sid, len(_t), len(_rn.get(_sid, "")))
    print(f"parser vs runner_sections: {_same} identical / {_diff} differ")
    assert _diff == 0, "parse_sections disagrees with runner_sections - offsets would not match the NER artifacts"
    _bad_head = [(c, s, t[:40]) for c in PAPERS for s, t in secs[c].items()
                 if t.lstrip().startswith("<!--")]
    print("sections starting on a leftover HTML marker:", len(_bad_head))
    assert not _bad_head, f"un-stripped marker at section head: {_bad_head[:3]}"
    # DEAD PATH: staging/paper1_5tool_ner was never carried to this host. Both blocks below are
    # .exists()-guarded so a missing baseline degrades to a printed note, never a crash.
    sp1 = ROOT / "staging" / "paper1_5tool_ner" / "sections_paper1.json"
    if sp1.exists():
        j = json.loads(sp1.read_text(encoding="utf-8"))
        entries = j.get("sections", j) if isinstance(j, dict) else j
        n_ok = n_bad = 0
        for s in entries:
            sid = s.get("section_id") or s.get("id")
            expect = s.get("chars") or s.get("len")
            if sid is None or expect is None: continue
            if len(secs.get("HMPAPER_000002", {}).get(sid, "")) == expect: n_ok += 1
            else: n_bad += 1; print("  LEN MISMATCH", sid, expect)
        print(f"paper-1 chars self-check: {n_ok} OK / {n_bad} mismatch")
    else:
        print("staging/paper1_5tool_ner absent on this host; parser/runner equality check above is the anchor")
    # aioner XML byte-check for one section
    import xml.etree.ElementTree as ET
    xmls = sorted((ROOT / "staging" / "paper1_5tool_ner" / "aioner_input").glob("*.xml"))
    if xmls:
        xf = xmls[0]
        passages = ET.parse(xf).getroot().findall(".//passage")
        xml_texts = ["".join(p.itertext()).strip() for p in passages] if passages else []
        # fall back: compare raw text content of whole doc when structure differs
        print("aioner_input xml present:", xf.name)
print("parser ready")

In [ ]:
# ===== Cell 2b: tool-input texts (v4 N1: S7 decisions select sections; ligatures expanded, offset map kept) =====
# Every tool below reads TT[cid][sid].norm; every emitted span goes back through make_row()
# -> ToolText.to_orig(), so stored char_start/char_end always index the ORIGINAL section text.
# v4: a section reaches the tools only if its S7 final_status is in SEL_KEEP AND it is not backmatter.
# NER and retrieval therefore read the same section decision. Every decision is written to
# tool_input_selection.csv (one row per section, with heading path and reason).
SECTION_DECISIONS = load_section_decisions(SEL_CSV)
print("S7 decisions loaded:", len(SECTION_DECISIONS), "from", SEL_CSV.name)
SEL_LOG, TT, SEC_CLASS = [], {}, {}
for cid in PAPERS:
    TT[cid] = {}
    for sid, t in secs[cid].items():
        path = sec_paths[cid].get(sid, "")
        keep, reason = select_section(cid, sid, path, SECTION_DECISIONS)
        cls, rule = section_class_for(cid, sid, path, SECTION_DECISIONS)
        SEC_CLASS[(cid, sid)] = dict(section_class=cls, section_class_rule=rule,
                                     section_selection_status=reason)
        d = SECTION_DECISIONS.get((cid, sid), {})
        SEL_LOG.append(dict(cid=cid, section_id=sid, heading_path=path, s7_final_status=d.get("final_status", ""),
                            kept=int(keep), reason=reason, section_class=cls, section_class_rule=rule,
                            chars=len(t)))
        if keep:
            TT[cid][sid] = ToolText(t)
dump_csv(OUT_DIR / "tool_input_selection.csv", SEL_LOG,
         ["cid", "section_id", "heading_path", "s7_final_status", "kept", "reason", "section_class",
          "section_class_rule", "chars"])
body_secs = {cid: {sid: tt.orig for sid, tt in m.items()} for cid, m in TT.items()}
tool_secs = {cid: {sid: tt.norm for sid, tt in m.items()} for cid, m in TT.items()}
for cid in PAPERS:
    n_lig = sum(sum(t.orig.count(ch) for ch in LIGATURES) for t in TT[cid].values())
    excl = [r for r in SEL_LOG if r["cid"] == cid and not r["kept"]]
    print(cid, "body sections:", len(TT[cid]), "| excluded:", len(excl), "| ligatures expanded:", n_lig)
    for r in excl:
        print("   excluded", r["section_id"], r["reason"], "|", r["heading_path"][-70:])
assert all(TT[cid] for cid in PAPERS), "a paper has no body section left - check tool_input_selection.csv"

## 5. PubTator batch and Java tools (Cells 3, 4, 5)

One global PubTator batch: sentinel title = section id (empty titles break the Java parsers),
literal pipes sanitized to `!` (same length, offsets unchanged), a terminal `.` appended to avoid
empty sentences. `java_tool` runs tmChem, DNorm, TaggerOne (disease + chemical) and GNormPlus
(folder API, output folder pre-created) through `run_tool_checked`; every mention is
offset-shifted, slice-validated on the sanitized tool text, and mapped back to the original text
via `make_row` / `ToolText.to_orig`.


In [ ]:
# ===== Cell 3: PubTator batch writers (verified convention; v2: ligature-normalized input) =====
# Java PubTator-format tools see text = title + " " + abstract. Empty titles break
# DNorm/TaggerOne parsing. We submit title = section_id (non-empty sentinel) and
# subtract len(title)+1 from returned offsets. Tool text (TT .norm) is flattened to one
# line (\n -> space, same length); offsets are then mapped back to the original text.

def flat1(s):
    """Replace newlines with spaces (same total length) so a section fits PubTator's one-line abstract field."""
    return re.sub(r"\n", " ", s)

# Build one global batch: one article per section across all papers.
# PubTator uses '|' as field delimiter: literal pipes inside text (markdown tables)
# break Java parsers. Replace '|' with '!' (same length => offsets identical).
BATCH = []  # (idx, cid, sec_id, title, abstract_sanitized, n_tool_chars)
for cid in PAPERS:
    for sid, tt in sorted(TT[cid].items()):
        idx = len(BATCH) + 1
        title = sid  # sentinel title = section id (no spaces; non-empty)
        san = flat1(tt.norm).replace("|", "!")
        # banner sentence breaker can emit an empty Sentence for certain trailing
        # punctuation runs (e.g. '...patients. .;,;;' -> java crash). Appending a
        # single terminal '.' after the last real character fixes it; spans reaching
        # into that '.' are rejected as OUT_OF_TEXT by to_orig().
        if san and san[-1] not in '.!?)]\"\'':
            san = san + "."
        BATCH.append((idx, cid, sid, title, san, len(tt.norm)))
BATCH_MAP = {t[0]: t for t in BATCH}
_n_all = sum(len(v) for v in secs.values())
print(f"batch articles: {len(BATCH)} (body-only; {_n_all - len(BATCH)} backmatter sections skipped)")

def write_pubtator_batch(path):
    """Write BATCH as PubTator records: id|t|title then id|a|abstract, LF line endings."""
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        for idx, cid, sid, title, abstract, _n in BATCH:
            f.write(f"{idx}|t|{title}\n{idx}|a|{abstract}\n\n")

def parse_pubtator_output(path):
    """Return list of dicts: idx, start, end, term, label, norm in title+abstract coords."""
    recs = []
    with open(path, encoding="utf-8", errors="replace") as f:
        for ln in f:
            ln = ln.rstrip("\n")
            if not ln or "|" in ln and ln.count("\t") == 0: continue
            parts = ln.split("\t")
            if len(parts) >= 5 and parts[1].isdigit() and parts[2].isdigit():
                recs.append(dict(idx=int(parts[0]), start=int(parts[1]), end=int(parts[2]),
                                 term=parts[3], label=parts[4], norm=parts[5] if len(parts) > 5 else ""))
    return recs

def pubtator_to_candidates(recs, model, skip=None):
    """PubTator records to candidate rows: shift by the sentinel title, slice-verify against the sanitized tool text, then map back to original offsets via make_row(); returns (rows, slice_fails, out_of_text)."""
    rows, bad, oot = [], 0, 0
    for r in recs:
        if skip and skip(r):
            continue
        meta = BATCH_MAP.get(r["idx"])
        if meta is None: continue
        _, cid, sid, title, abstract_s, _n = meta
        st0, en0 = r["start"] - (len(title) + 1), r["end"] - (len(title) + 1)
        if st0 < 0:                                  # mention inside sentinel title
            oot += 1; continue
        if abstract_s[st0:en0] != r["term"]:         # tool saw sanitized text
            bad += 1; continue
        row = make_row(model, cid, sid, st0, en0, r["label"], r["norm"])
        if row: rows.append(row)
    return rows, bad, oot

print("pubtator helpers ready; batch size", len(BATCH))

In [ ]:
# ===== Cell 4: run Java tools (subprocess, batch per tool) =====
# v3: every tool goes through run_tool_checked(): old outputs deleted first, exit code and
# output existence checked, timeout from TOOL_TIMEOUT_PER_PAPER; outputs are read only via
# consume_output() (status OK + sha256 recorded right after the run). A failed tool is
# marked FAILED in TOOL_STATUS and nothing is registered for it; the run continues.
BATCH_IN  = WORK / "batch_pubtator.txt"
write_pubtator_batch(BATCH_IN)
IN_NAME, TMP_NAME = f"input_{RUN_TAG}.txt", f"temp_{RUN_TAG}"

def java_tool(tool_key, model, tool_dir, args, out_name, log_name):
    """Copy the batch into a tool folder, run it via run_tool_checked(), and register parsed rows only when the run succeeded."""
    shutil.copy(BATCH_IN, tool_dir / IN_NAME)
    out = tool_dir / out_name
    if run_tool_checked(tool_key, args, tool_dir, WORK / log_name, [out], tool_timeout(tool_key)):
        rows, bad, oot = pubtator_to_candidates(parse_pubtator_output(consume_output(tool_key, out)), model)
        TOOL_STATUS[tool_key].update(slice_fails=bad, out_of_text=oot)
        register(rows, model, tool_key)
        print(model, "slice-fails:", bad, "out-of-text:", oot)
    else:
        register([], model, tool_key)

# ---- tmChem (chemical, MeSH/CHEBI normalization) ----
(TMCHEM / TMP_NAME).mkdir(exist_ok=True); (DNORM / TMP_NAME).mkdir(exist_ok=True)
out_name = f"output_{RUN_TAG}.txt"
    java_tool("tmChem", "tmChem", TMCHEM,
          [JAVA, "-cp", "libs/*", "ncbi.Run", "config/banner_JOINT.xml",
           "data/dict.txt", str(AB3P), TMP_NAME, IN_NAME, out_name], out_name, "tmchem.log")

# ---- DNorm (disease, CTD normalization; Ab3P dir supplied -> abbreviations expand) ----
java_tool("DNorm", "DNorm", DNORM,
          [JAVA, "-Xmx6G", "-cp", "libs/*", "dnorm.ApplyDNorm",
           "config/banner_BC5CDR_UMLS2013AA.xml", "data/CTD_diseases.tsv",
           "output/simmatrix_BC5CDR_e4.bin",
           str(AB3P), TMP_NAME, IN_NAME, out_name], out_name, "dnorm.log")

# ---- TaggerOne disease (model_BC5CDRD.bin) and chemical (model_BC5CDRC.bin) ----
for model, mfile, oname, log in [("TaggerOne-Disease", "model_BC5CDRD.bin", f"output_disease_{RUN_TAG}.txt", "taggerone_disease.log"),
                                 ("TaggerOne-Chemical", "model_BC5CDRC.bin", f"output_chem_{RUN_TAG}.txt", "taggerone_chem.log")]:
    java_tool(model, model, TGONE,
              [JAVA, "-cp", "libs/*", "ncbi.taggerOne.ProcessText",
               "--fileFormat", "Pubtator", "--inputFilename", IN_NAME,
               "--outputFilename", oname,
               "--modelInputFilename", f"output/{mfile}",
               "--useSentenceBreaker", "true"], oname, log)
print("java tools done:", {k: TOOL_STATUS[k]["status"] for k in ("tmChem", "DNorm", "TaggerOne-Disease", "TaggerOne-Chemical")})

In [ ]:
# ===== Cell 5: GNormPlus (gene/protein; folder API, PubTator flat format) =====
# GNormPlus takes an input FOLDER of PubTator-format files and an output FOLDER.
# Output line format: <idx>\t<start>\t<end>\t<term>\t<Type>\t<ID> in title+abstract
# coords, same as our batch, so we reuse the batch + offset-shift machinery.
# v3: g_in/g_out are deleted before the run; the output folder must contain a fresh file.
g_in  = GNORM / f"gnorm_batch_in_{RUN_TAG}"
g_out = WORK / "gnorm_out"
for _d in (g_in, g_out):
    if _d.exists(): shutil.rmtree(_d)
g_in.mkdir()
shutil.copy(BATCH_IN, g_in / "batch.txt")
g_args = [JAVA, "-Xmx30G", "-Xms8G", "-jar", "GNormPlus.jar", g_in.name, str(g_out), "setup.txt"]
# run_tool_checked deletes g_out, then recreates it EMPTY: GNormPlus does not create its output
# folder (v3 run 1 failed with FileNotFoundException ...\gnorm_outatch.txt without this)
if run_tool_checked("GNormPlus", g_args, GNORM, WORK / "gnormplus.log", [g_out], tool_timeout("GNormPlus"),
                    create_dirs=[g_out]):
    # Species 9606 (human) rows are dropped as in v1/v2 (unchanged; see audit issue 7)
    rows_g, bad_g, oot_g = pubtator_to_candidates(
        parse_pubtator_output(consume_output("GNormPlus", g_out)), "GNormPlus",
        skip=lambda r: r["label"] == "Species" and r["norm"] == "9606")
    TOOL_STATUS["GNormPlus"].update(slice_fails=bad_g, out_of_text=oot_g)
    register(rows_g, "GNormPlus", "GNormPlus")
    print("GNormPlus slice-fails:", bad_g, "out-of-text:", oot_g)
else:
    register([], "GNormPlus", "GNormPlus")

## 6. GPU and embedding tools (Cells 6, 6b, 7)

HunFlair2 (3000-char paragraph-boundary chunks), BioBERT and SciBERT (word windows of at most 2600
chars / 450 subword tokens, word-level BIO decode via the shared `BERT_AGG_SRC`; v1 sub-token
decoding kept as `legacy` for tracing), and four scispacy NER models with the UMLS linker
(scispacy 0.6.2) - each in a `.venv_markdown` subprocess so the tool environments never touch this
kernel. All spans come back in tool-text coordinates and go through `make_row`.


In [ ]:
# ===== Cell 6: HunFlair2 (chunked, via .venv_markdown subprocess to avoid kernel GPU/env conflicts) =====
# v2: the runner reads the ligature-normalized body texts (tool_input.json) instead of
# re-parsing SEG_DIR itself; offsets come back in tool-text coords and go through make_row().
TOOL_IN = WORK / "tool_input.json"
TOOL_IN.write_text(json.dumps(tool_secs), encoding="utf-8")

HF_SCRIPT = WORK / "hunflair_runner.py"
HF_SCRIPT.write_text(r"""
import sys, json
from pathlib import Path
from flair.models import SequenceTagger
from flair.data import Sentence

model = SequenceTagger.load("hunflair/hunflair2-ner")
INP, OUTJSON = Path(sys.argv[1]), Path(sys.argv[2])
data = json.loads(INP.read_text(encoding="utf-8"))

def para_chunks(text, maxlen=3000):
    # split on paragraph boundaries, pack to <= maxlen; return (offset, chunk) pairs
    paras = text.split("\n\n")
    chunks, cur, cur_off, pos = [], "", 0, 0
    for p in paras:
        if cur and len(cur) + len(p) + 2 > maxlen:
            chunks.append((cur_off, cur))
            cur, cur_off = p, pos
        else:
            if not cur: cur_off = pos
            cur = cur + "\n\n" + p if cur else p
        pos += len(p) + 2
    if cur: chunks.append((cur_off, cur))
    return chunks

out = {}
for cid, secmap in data.items():
    result = {}
    for sid, text in secmap.items():
        ents = []
        for off, ch in para_chunks(text):
            s = Sentence(ch, use_tokenizer=False)
            model.predict(s)
            for e in s.get_spans("ner"):
                start = off + e.tokens[0].start_position
                end = off + e.tokens[-1].end_position
                ents.append([start, end, e.text, e.get_label("ner").value])
        result[sid] = ents
    out[cid] = result
    print(cid, sum(len(x) for x in result.values()), "entities", flush=True)
OUTJSON.write_text(json.dumps(out), encoding="utf-8")
""", encoding="utf-8")

hf_out = WORK / "hunflair_out.json"
rows_hf, bad_hf = [], 0
# v3: checked run (old hf_out deleted first; exit code, timeout, output provenance)
if run_tool_checked("HunFlair2", [ENV_PY or "MISSING_ENV_PY", "-X", "utf8", str(HF_SCRIPT), str(TOOL_IN), str(hf_out)],
                    WORK, WORK / "hunflair_run.log", [hf_out], tool_timeout("HunFlair2")):
    hf = json.loads(consume_output("HunFlair2", hf_out).read_text(encoding="utf-8"))
    for cid, secmap in hf.items():
        for sid, ents in secmap.items():
            for st, en, term, label in ents:
                if tool_secs[cid][sid][st:en] != term:     # slice-check in tool-text coords
                    bad_hf += 1; continue
                row = make_row("HunFlair2", cid, sid, st, en, label)
                if row: rows_hf.append(row)
    TOOL_STATUS["HunFlair2"]["slice_fails"] = bad_hf
register(rows_hf, "HunFlair2", "HunFlair2"); print("HunFlair2 slice-fails:", bad_hf)

In [ ]:
# ===== Cell 6b: BioBERT + SciBERT (HF token-classification via .venv_markdown/transformers) =====
# Input = ligature-normalized body sections. Word-window chunking (<=2600 chars / <=450
# sub-tokens, no overlap). v2 decoding: word-level, first-sub-token label (BERT_AGG_SRC,
# tested in Cell 1c). The runner ALSO emits the v1 sub-token decoding from the same
# logits ("legacy") so the partial-word source can be traced on identical predictions.
BERT_SCRIPT = WORK / "bert_runner.py"
BERT_SCRIPT.write_text(r"""
import sys, json, re, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification
""" + BERT_AGG_SRC + r"""
INP, OUTP = Path(sys.argv[1]), Path(sys.argv[2])
data = json.loads(INP.read_text(encoding="utf-8"))
MODELS = [("BioBERT-Disease", "alvaroalon2/biobert_diseases_ner"),
          ("BioBERT-Chemical", "alvaroalon2/biobert_chemical_ner"),
          ("BioBERT-Genetic", "alvaroalon2/biobert_genetic_ner"),
          ("SciBERT-JNLPBA", "fran-martinez/scibert_scivocab_cased_ner_jnlpba")]
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(8)
print("device:", device, flush=True)

def norm_label(raw):
    s = (raw or "").strip().upper()
    if s in ("O", "0", ""):
        return "O"
    if "-" in s:
        b, t = s.split("-", 1)
        if b in ("B", "I") and t:
            return s
    return "O"

def legacy_decode(labels, offsets, base):
    # v1 behaviour: BIO over sub-tokens (kept only for the before/after trace)
    ents, cur = [], None
    for lab, (s, e) in zip(labels, offsets):
        if s == e or lab == "O":
            if cur: ents.append(cur); cur = None
            continue
        b, t = lab.split("-", 1)
        if b == "B" or not (cur and cur[0] == t):
            if cur: ents.append(cur)
            cur = [t, base + s, base + e]
        else:
            cur[2] = base + e
    if cur: ents.append(cur)
    return ents

def predict(model, tok, text, max_tok=450, max_words=220, max_chars=2600):
    v2, legacy = [], []
    id2 = {int(k): norm_label(v) for k, v in model.config.id2label.items()}
    words = [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    n, i = len(words), 0
    while i < n:
        j = min(i + max_words, n)
        while j - i > 1 and words[j - 1][1] - words[i][0] > max_chars:
            j -= 1
        while j - i > 1 and len(tok(text[words[i][0]:words[j - 1][1]])["input_ids"]) > max_tok:
            j = max(i + 1, j - max(1, (j - i) // 2))
        base = words[i][0]
        chunk = text[base:words[j - 1][1]]
        enc = tok(chunk, return_tensors="pt", return_offsets_mapping=True,
                  truncation=True, max_length=512)
        wids = enc.word_ids(0)
        off = enc.pop("offset_mapping")[0].tolist()
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits[0]
        labels = [id2[li] for li in logits.argmax(-1).tolist()]
        for t, s, e in words_to_spans(word_level(wids, labels, off)):
            v2.append([base + s, base + e, t])
        legacy.extend(legacy_decode(labels, off, base))
        i = j
    return ([[st, en, text[st:en], ty] for st, en, ty in v2],
            [[st, en, text[st:en], ty] for ty, st, en in legacy])

out = {"v2": {}, "legacy": {}}
for name, mid in MODELS:
    tok = AutoTokenizer.from_pretrained(mid)
    assert tok.is_fast, f"{mid}: word_ids() needs a fast tokenizer"
    mdl = AutoModelForTokenClassification.from_pretrained(mid).to(device).eval()
    for cid, secmap in data.items():
        for sid, text in secmap.items():
            a, b = predict(mdl, tok, text)
            out["v2"].setdefault(name, {}).setdefault(cid, {})[sid] = a
            out["legacy"].setdefault(name, {}).setdefault(cid, {})[sid] = b
        print(name, cid, sum(len(v) for v in out["v2"][name][cid].values()), "ents", flush=True)
    del mdl
    if device == "cuda":
        torch.cuda.empty_cache()
OUTP.write_text(json.dumps(out), encoding="utf-8")
print("bert runner done", flush=True)
""", encoding="utf-8")

bert_out = WORK / "bert_out.json"
BERT_MODELS = ["BioBERT-Disease", "BioBERT-Chemical", "BioBERT-Genetic", "SciBERT-JNLPBA"]
BERT_TRACE = {}   # model -> partial-word counts for legacy vs v2 decoding on the same logits
# v3: checked run (old bert_out deleted first); one status for the four models (one subprocess)
bert_ok = run_tool_checked("BERT", [ENV_PY or "MISSING_ENV_PY", "-X", "utf8", str(BERT_SCRIPT), str(TOOL_IN), str(bert_out)],
                           WORK, WORK / "bert_run.log", [bert_out], tool_timeout("BERT"))
bert = json.loads(consume_output("BERT", bert_out).read_text(encoding="utf-8")) if bert_ok else {}
for model in BERT_MODELS:
    rows_b, bad_b = [], 0
    for cid, secmap in bert.get("v2", {}).get(model, {}).items():
        for sid, ents in secmap.items():
            for st, en, term, label in ents:
                if tool_secs[cid][sid][st:en] != term:
                    bad_b += 1; continue
                row = make_row(model, cid, sid, st, en, label)
                if row: rows_b.append(row)
    register(rows_b, model, "BERT")
    if not bert_ok:
        continue
    leg = [(cid, sid, st, en) for cid, sm in bert["legacy"].get(model, {}).items()
           for sid, ents in sm.items() for st, en, _t, _l in ents]
    leg_pw = sum(partial_word(tool_secs[c][s], a, b) for c, s, a, b in leg)
    BERT_TRACE[model] = dict(legacy_spans=len(leg), legacy_partial=leg_pw,
                             v2_spans=len(rows_b), v2_partial=sum(r["partial_word"] for r in rows_b),
                             slice_fails=bad_b)
    print(model, BERT_TRACE[model])

In [ ]:
# ===== Cell 7: SciSpacy (4 NER models + scispacy UMLS linker, via .venv_markdown/scispacy) =====
# Reads WORK/tool_input.json (ligature-normalized body sections; fresh v2 WORK => no stale cache).
# scispacy 0.6.2 registers the Span extension 'kb_ents' (docs say 'umls_ents' - wrong for
# this version); the top linked CUI goes to `norm`, its linker score to `score`.
SCISPACY_SCRIPT = WORK / "scispacy_runner.py"
SCISPACY_SCRIPT.write_text(r"""
import sys, json, time
from pathlib import Path
import spacy
import scispacy.linking

INP, OUTP = Path(sys.argv[1]), Path(sys.argv[2])
data = json.loads(INP.read_text(encoding="utf-8"))
MODELS = ["en_ner_bc5cdr_md", "en_ner_bionlp13cg_md", "en_ner_craft_md", "en_ner_jnlpba_md"]

out = {}
for name in MODELS:
    t0 = time.time()
    nlp = spacy.load(name)
    nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})
    res = {}
    for cid, secmap in data.items():
        total = 0
        for sid, text in secmap.items():
            doc = nlp(text)
            ents = []
            for e in doc.ents:
                cui, score = "", 0.0
                kb = list(getattr(e._, "kb_ents", []) or [])
                if kb:
                    cui, score = kb[0][0], round(float(kb[0][1]), 4)
                ents.append([e.start_char, e.end_char, e.text, e.label_, cui, score])
            res.setdefault(cid, {})[sid] = ents
            total += len(ents)
        print(name, cid, total, "ents", flush=True)
    out[name] = res
    print(name, "done in", round(time.time() - t0, 1), "s", flush=True)
OUTP.write_text(json.dumps(out), encoding="utf-8")
print("scispacy runner done", flush=True)
""", encoding="utf-8")

scispacy_out = WORK / "scispacy_out.json"
# v3: checked run (old scispacy_out deleted first); one status for the four models
sc_ok = run_tool_checked("SciSpacy", [ENV_PY or "MISSING_ENV_PY", "-X", "utf8", str(SCISPACY_SCRIPT), str(TOOL_IN), str(scispacy_out)],
                         WORK, WORK / "scispacy_run.log", [scispacy_out], tool_timeout("SciSpacy"))
scispacy = json.loads(consume_output("SciSpacy", scispacy_out).read_text(encoding="utf-8")) if sc_ok else {}
for model in ["en_ner_bc5cdr_md", "en_ner_bionlp13cg_md", "en_ner_craft_md", "en_ner_jnlpba_md"]:
    rows_s, bad_s = [], 0
    for cid, secmap in scispacy.get(model, {}).items():
        for sid, ents in secmap.items():
            for st, en, term, label, cui, score in ents:
                if tool_secs[cid][sid][st:en] != term:
                    bad_s += 1; continue
                row = make_row(model, cid, sid, st, en, label, cui, score)
                if row: rows_s.append(row)
    register(rows_s, model, "SciSpacy")
    print(model, "slice-fails:", bad_s)

## 7. LLM NER - Qwen3.5-4B (Cell 7c)

One llama-server (CUDA, 8 parallel slots, thinking off, JSON schema limited to `CDM_TYPES`) started
fresh on a free port and stopped afterwards. Chunks of at most 1500 chars cut on sentence or
whitespace boundaries; introduction, background and backmatter sections are skipped. Returned
texts are located by exact whole-word match in their chunk (unlocatable returns dropped) and
expanded to further occurrences (`llm_occurrence_expanded` = 1). Per-chunk disk cache keyed by
text, prompt, schema, GGUF sha256, server build and settings (`llm_cache_key`); a timeout or chunk
errors mark LLM FAILED while completed chunks stay cached.


In [ ]:
# ===== Cell 7c: LLM NER - Qwen3.5-4B (GGUF) via llama.cpp llama-server, thinking off =====
# Efficiency: one CUDA llama-server, N parallel slots (continuous batching), the shared system
# prompt stays in each slot's KV cache, JSON-schema-constrained output, per-chunk disk cache.
# v3 cache key (llm_cache_key, tested in 1c): sha256 of the chunk text, SYSTEM_PROMPT, LLM_SCHEMA,
# GGUF file name AND file sha256, llama-server build, temperature, max_tokens, prompt version and
# retry policy. v2 cache entries recorded none of the settings, so none are reused.
# The LLM returns only (text, type); offsets come from whole-word matches in the chunk
# (llm_locate) and go through make_row(). llm_occurrence_expanded = 1 marks rows from a further
# occurrence of a returned term (not the first match). Phase 1 emits the same rows as v2.
import socket, urllib.request
from concurrent.futures import ThreadPoolExecutor

LLM_MODEL_NAME = "Qwen3.5-4B"
# DEAD PATH on this host: models/ does not exist and there is no llama-server binary anywhere on
# disk, so Cell 7c cannot produce rows here. The GGUF/server template stays env-overridable
# (NB_LLM_GGUF / NB_LLM_SERVER); the block below already degrades to TOOL_STATUS=FAILED rather than
# crashing, and LLM_CACHE AUDIT reports to_run>0 with 0 new entries when the files are missing.
LLM_GGUF   = Path(os.environ.get("NB_LLM_GGUF")
                  or str(ROOT / "models" / "Qwen3.5-4B-Q4_K_M.gguf"))
LLM_SERVER = Path(os.environ.get("NB_LLM_SERVER")
                  or str(ROOT / "models" / "llama-bin" / "llama-server"))   # no .exe on POSIX
LLM_SLOTS, LLM_SLOT_CTX = 8, 4096

def _free_port():
    """Ask the OS for a free localhost port; a fixed port could collide with an already-running llama-server."""
    with socket.socket() as so:
        so.bind(("127.0.0.1", 0))
        return so.getsockname()[1]
LLM_PORT = _free_port()  # never reuse a fixed port: another llama-server may already be on it
LLM_CHUNK_CHARS = 1500
LLM_MAX_TOKENS = 2048
LLM_TEMPERATURE = 0
LLM_PROMPT_VERSION = "cdm-ner-v1"
LLM_RETRY_POLICY = "split-halves-once"
LLM_CACHE = WORK / "llm_ner_cache_v3.json"
# LLM-only section skip: backmatter (Cell 1) plus introduction/background.
# (preamble) stays in: 000003/000009 have their whole text there, intro included.
LLM_SKIP_TOKENS = BACKMATTER_TOKENS + ("introduction", "background")

def llm_skip(path):
    """LLM-only section filter: backmatter tokens plus introduction or background; (preamble) sections are always kept."""
    last = (path or "").split(">")[-1].strip().lower()
    if not last or last == "(preamble)":
        return False
    return any(tok in last for tok in LLM_SKIP_TOKENS)

SYSTEM_PROMPT = """You extract named entities from microbiome research papers for a common data model.
Return every entity mention in the passage, copying the text EXACTLY as written (same spelling, case, spacing).
Use only these types:
ORGANISM: bacteria, archaea, fungi, viruses, named eukaryotic species and model organisms (not generic words like patients, participants, subjects)
CHEMICAL: small molecules, drugs, metabolites
DISEASE: human diseases and conditions
PROTEIN: proteins, enzymes, genes and gene products
DNA_RNA: DNA/RNA elements (e.g. 16S rRNA gene, primers, plasmids)
CELL_TYPE: cell types
GENOMICS: genomics concepts and sequencing approaches (e.g. shotgun metagenomics, 16S rRNA sequencing)
PATHWAY: biological pathways and processes
LAB_METHOD: laboratory techniques
DATA_ANALYSIS: statistical or computational methods and software
SAMPLE_METHOD: sample collection procedures
CLINICAL: clinical metrics and measurements
EXPERIMENTAL_METHOD: experimental design elements
STUDY_DESIGN: study structure (e.g. cohort study, randomized controlled trial)
Do not tag figure/table references, citations, author names, or licence text. If there are none, return an empty list."""

LLM_SCHEMA = {"type": "object", "properties": {"entities": {"type": "array", "items": {
    "type": "object", "properties": {"text": {"type": "string"},
                                     "type": {"type": "string", "enum": sorted(CDM_TYPES)}},
    "required": ["text", "type"]}}}, "required": ["entities"]}

def llm_chunks(text, limit=LLM_CHUNK_CHARS):
    """(start, end) windows on TOOL text, cut after sentence ends / whitespace, never mid-word."""
    out, s, n = [], 0, len(text)
    while s < n:
        e = min(s + limit, n)
        if e < n:
            cut = max(text.rfind(". ", s, e), text.rfind("\n", s, e))
            if cut <= s:
                cut = text.rfind(" ", s, e)
            e = cut + 1 if cut > s else e
        if text[s:e].strip():
            out.append((s, e))
        s = e
    return out

def llm_call(chunk):
    """One llama-server chat completion (temperature 0, thinking off, JSON schema constrained to CDM types); returns the parsed entity list."""
    body = json.dumps({
        "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                     {"role": "user", "content": chunk}],
        "temperature": LLM_TEMPERATURE, "max_tokens": LLM_MAX_TOKENS, "cache_prompt": True,
        "chat_template_kwargs": {"enable_thinking": False},
        "response_format": {"type": "json_object", "schema": LLM_SCHEMA}}).encode("utf-8")
    req = urllib.request.Request(f"http://127.0.0.1:{LLM_PORT}/v1/chat/completions", data=body,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=600) as resp:
        msg = json.loads(resp.read())["choices"][0]["message"]["content"]
    return json.loads(msg).get("entities", [])

# model identity: file sha256 (memoised by path/size/mtime) + server build string
_fh_path = WORK / "file_hashes.json"
_fh = json.loads(_fh_path.read_text(encoding="utf-8")) if _fh_path.exists() else {}
_st = LLM_GGUF.stat() if LLM_GGUF.exists() else None
_fkey = f"{LLM_GGUF}|{_st.st_size}|{_st.st_mtime}" if _st else ""
if _st and _fkey not in _fh:
    _fh[_fkey] = file_sha256(LLM_GGUF); _fh_path.write_text(json.dumps(_fh, indent=1), encoding="utf-8")
LLM_GGUF_SHA256 = _fh.get(_fkey, "MISSING")
try:
    _v = subprocess.run([str(LLM_SERVER), "--version"], capture_output=True, text=True, timeout=60)
    LLM_SERVER_BUILD = next((l.strip() for l in (_v.stdout + _v.stderr).splitlines() if l.startswith("version:")), "UNKNOWN")
except Exception as ex:
    LLM_SERVER_BUILD = f"UNKNOWN ({ex!r})"
print("GGUF:", LLM_GGUF.name, LLM_GGUF_SHA256[:16], "| server:", LLM_SERVER_BUILD)

# 1) chunk the eligible sections
jobs = []  # (cid, sid, chunk_start, chunk_end, key)
skipped_secs = {}
for cid in PAPERS:
    for sid, t in tool_secs[cid].items():
        if llm_skip(sec_paths[cid].get(sid, "")):
            skipped_secs.setdefault(cid, []).append(sec_paths[cid].get(sid, ""))
            continue
        for s, e in llm_chunks(t):
            key = llm_cache_key(t[s:e], SYSTEM_PROMPT, LLM_SCHEMA, LLM_GGUF.name, LLM_GGUF_SHA256,
                                LLM_SERVER_BUILD, LLM_TEMPERATURE, LLM_MAX_TOKENS,
                                LLM_PROMPT_VERSION, LLM_RETRY_POLICY)
            jobs.append((cid, sid, s, e, key))
llm_cache = json.loads(LLM_CACHE.read_text(encoding="utf-8")) if LLM_CACHE.exists() else {}
todo = [j for j in jobs if j[4] not in llm_cache]
_v2_llm = V2_DIR / "_work" / "llm_ner_cache.json"
LLM_CACHE_AUDIT = dict(chunks=len(jobs), cached_v3=len(jobs) - len(todo), to_run=len(todo),
                       v2_entries_available=len(json.loads(_v2_llm.read_text(encoding="utf-8"))) if _v2_llm.exists() else 0,
                       reused_v2=0,
                       reuse_rule="v2 keys = sha1(prompt version|GGUF name|chunk); no model hash, schema, "
                                  "max_tokens or retry policy recorded -> not demonstrably compatible, none reused")
print(f"LLM chunks: {len(jobs)} | cached (v3): {len(jobs) - len(todo)} | to run: {len(todo)} | "
      f"sections skipped (intro/backmatter): {sum(len(v) for v in skipped_secs.values())}")

# 2) run uncached chunks on one parallel llama-server (started here, stopped afterwards)
server, server_log, llm_errors = None, None, []
llm_deadline = time.time() + tool_timeout("LLM")
try:
    if todo:
        if not (LLM_GGUF.exists() and LLM_SERVER.exists()):
            raise RuntimeError(f"GGUF or llama-server missing (looked for {LLM_GGUF} and {LLM_SERVER}); "
                               f"set NB_LLM_GGUF / NB_LLM_SERVER to run Cell 7c on this host")
        server_log = open(WORK / "llama_server.log", "w", encoding="utf-8")
        server = subprocess.Popen([str(LLM_SERVER), "-m", str(LLM_GGUF), "-ngl", "99",
                                   "-c", str(LLM_SLOTS * LLM_SLOT_CTX), "--parallel", str(LLM_SLOTS),
                                   "--jinja", "--reasoning-budget", "0",
                                   "--host", "127.0.0.1", "--port", str(LLM_PORT)],
                                  stdout=server_log, stderr=subprocess.STDOUT)
        healthy = False
        for _ in range(300):
            try:
                with urllib.request.urlopen(f"http://127.0.0.1:{LLM_PORT}/health", timeout=2) as r:
                    if r.status == 200: healthy = True; break
            except Exception:
                if server.poll() is not None:
                    break
                time.sleep(1)
        if not healthy:
            raise RuntimeError("llama-server did not become healthy - see _work/llama_server.log")
        t0 = time.time()
        def _run(job):
            """Worker for one uncached chunk; on truncated or invalid JSON retry once as two halves; returns (key, entities, error)."""
            cid, sid, s, e, key = job
            if time.time() > llm_deadline:
                return key, None, "TIMEOUT"
            text = tool_secs[cid][sid][s:e]
            try:
                return key, llm_call(text), None
            except Exception:
                # truncated/invalid JSON on a dense chunk: retry once as two halves
                cut = text.rfind(" ", 0, len(text) // 2)
                try:
                    return key, llm_call(text[:cut]) + llm_call(text[cut:]), None
                except Exception as ex2:
                    return key, None, repr(ex2)
        with ThreadPoolExecutor(LLM_SLOTS) as pool:
            for i, (key, ents, err) in enumerate(pool.map(_run, todo), 1):
                if err: llm_errors.append(err)
                else: llm_cache[key] = dict(entities=ents, created_at=_now(), provenance="new")
                if i % 50 == 0:
                    LLM_CACHE.write_text(json.dumps(llm_cache, ensure_ascii=False), encoding="utf-8")
                    print(f"  {i}/{len(todo)} chunks, {time.time() - t0:.0f}s", flush=True)
        LLM_CACHE.write_text(json.dumps(llm_cache, ensure_ascii=False), encoding="utf-8")
        print(f"LLM run: {len(todo)} chunks in {time.time() - t0:.0f}s | errors: {len(llm_errors)}", llm_errors[:3])
    if llm_errors:
        reason = ("TIMEOUT (whole-run limit)" if "TIMEOUT" in llm_errors else "CHUNK_ERRORS") + \
                 f": {len(llm_errors)} chunk(s) failed; completed chunks stay cached and are not re-run"
        mark_tool("LLM", "FAILED", reason, errors=[redact(e) for e in llm_errors[:5]])
    else:
        mark_tool("LLM", "OK", model_sha256=LLM_GGUF_SHA256, server_build=LLM_SERVER_BUILD)
except Exception as ex:
    mark_tool("LLM", "FAILED", f"{ex!r}",
              log_tail=_log_tail(WORK / "llama_server.log") if (WORK / "llama_server.log").exists() else "")
finally:
    if server is not None:
        server.terminate(); server.wait(timeout=30)
    if server_log is not None:
        server_log.close()
LLM_CACHE_AUDIT.update(new_entries=sum(1 for j in todo if j[4] in llm_cache), failed=len(llm_errors))

# 3) locate returned texts in their chunk (whole-word, every occurrence) -> rows
LLM_STATS = dict(returned=0, not_in_text=0, bad_type=0, located_spans=0,
                 first_match_rows=0, expanded_rows=0, expanded_by_type={})
rows_llm, seen = [], set()
if TOOL_STATUS["LLM"]["status"] == "OK":
    for cid, sid, s, e, key in jobs:
        chunk = tool_secs[cid][sid][s:e]
        for ent in llm_cache[key]["entities"]:
            LLM_STATS["returned"] += 1
            term, typ = (ent.get("text") or "").strip(), ent.get("type", "")
            if typ not in CDM_TYPES:
                LLM_STATS["bad_type"] += 1; continue
            hits = llm_locate(chunk, term)
            if not hits:
                LLM_STATS["not_in_text"] += 1; continue
            for h, expanded in hits:
                ts, te = s + h, s + h + len(term)
                if (cid, sid, ts, te) in seen:
                    continue
                seen.add((cid, sid, ts, te))
                row = make_row(LLM_MODEL_NAME, cid, sid, ts, te, typ, llm_occurrence_expanded=expanded)
                if row:
                    rows_llm.append(row); LLM_STATS["located_spans"] += 1
                    LLM_STATS["expanded_rows" if expanded else "first_match_rows"] += 1
                    if expanded:
                        LLM_STATS["expanded_by_type"][typ] = LLM_STATS["expanded_by_type"].get(typ, 0) + 1
register(rows_llm, LLM_MODEL_NAME, "LLM")
TOOL_STATUS["LLM"].update(stats=LLM_STATS, cache=LLM_CACHE_AUDIT)
print("LLM stats:", LLM_STATS)
print("LLM cache:", LLM_CACHE_AUDIT)

## 7b. Dictionary tools (Cell 7d, v4 N2)

MeSH B03 taxonomy matcher with in-document abbreviation expansion, and methods extractors (kits, platforms, primers, assays, software, V-regions).

In [ ]:
# ===== Cell 7d: dictionary tools (v4 N2) - MeSH taxonomy matcher + methods extractors =====
# Two in-process tools over the same tool text as every other tool, so offsets go through make_row():
#   MeSH-Taxon-Dict : MeSH 2026 B03 (Bacteria) preferred labels + entry terms, whole word, longest first;
#                     abbreviations ('F. prausnitzii') expanded only when the full name occurs in the same
#                     paper and is unique, otherwise kept with label taxon_abbrev and no ID.
#   Methods-Dict    : kits, sequencing platforms, primers, assays, software, reference databases
#                     (METHOD_TERMS, Cell 1d) + V-regions that have 16S/region context nearby.
# NCBI Taxonomy / GTDB: no dump in the repository, so organism IDs here are MeSH only (recorded in TOOL_STATUS).
DICT_STATUS = {}
for _k in ("MeSH-Taxon-Dict", "Methods-Dict"):
    for _p in OUT_DIR.glob(f"{_k}_candidates.csv"):
        _p.unlink()                                   # stale output never survives a failed run

try:
    _taxa, _mesh_sha = load_mesh_taxa(MESH_TAXON_LEXICON)
    _stop = set()
    if TERMS_CONFIG.exists():
        _stop = {s.lower() for s in json.loads(TERMS_CONFIG.read_text(encoding="utf-8")).get("abbreviation_stoplist", [])}
    TAXON_MATCHER = DictMatcher(_taxa)
    rows_tax, n_abbrev_exp, n_abbrev_unexp = [], 0, 0
    for cid in PAPERS:
        full_names, found = {}, []
        for sid, tt in TT[cid].items():
            for s, e, txt, pays in TAXON_MATCHER.find(tt.norm):
                p = pays[0]
                found.append((sid, s, e, p))
                if len(txt.split()) >= 2:
                    full_names[txt] = p
        for sid, s, e, p in found:
            r = make_row("MeSH-Taxon-Dict", cid, sid, s, e, "taxon_mesh", p["id"], p["kind"])
            if r: rows_tax.append(r)
        for sid, tt in TT[cid].items():
            for s, e, txt, exp in expand_abbreviations(tt.norm, set(full_names), _stop):
                p = full_names.get(exp) if exp else None
                n_abbrev_exp += bool(p); n_abbrev_unexp += not p
                r = make_row("MeSH-Taxon-Dict", cid, sid, s, e, "taxon_abbrev",
                             p["id"] if p else "", "abbreviation_expanded_within_document" if p else
                             "abbreviated_name_not_expanded")
                if r: rows_tax.append(r)
    TOOL_STATUS["DICT_TAXON"] = dict(status="OK", reason="", vocabulary="MeSH 2026 B03",
                                     mesh_xml_sha256=_mesh_sha, surfaces=len(_taxa),
                                     abbreviations_expanded=n_abbrev_exp, abbreviations_not_expanded=n_abbrev_unexp,
                                     ncbi_taxonomy="NOT_ATTEMPTED: no taxdump at " + str(NCBI_TAXDUMP)
                                     if not (NCBI_TAXDUMP / "names.dmp").exists() else "present (used in Cell 7e)")
    register(rows_tax, "MeSH-Taxon-Dict", "DICT_TAXON")
except Exception as ex:
    TOOL_STATUS["DICT_TAXON"] = dict(status="FAILED", reason=f"{type(ex).__name__}: {ex}"[:300])
    print("MeSH-Taxon-Dict FAILED:", TOOL_STATUS["DICT_TAXON"]["reason"])

try:
    METHOD_MATCHER = DictMatcher(METHOD_TERMS)
    rows_m = []
    for cid in PAPERS:
        for sid, tt in TT[cid].items():
            for s, e, txt, labs in METHOD_MATCHER.find(tt.norm):
                r = make_row("Methods-Dict", cid, sid, s, e, labs[0])
                if r: rows_m.append(r)
            for s, e, txt in find_v_regions(tt.norm):
                r = make_row("Methods-Dict", cid, sid, s, e, "amplicon_region")
                if r: rows_m.append(r)
    TOOL_STATUS["DICT_METHODS"] = dict(status="OK", reason="", terms=len(METHOD_TERMS),
                                       by_label={l: sum(1 for r in rows_m if r["label_raw"] == l)
                                                 for l in sorted({r["label_raw"] for r in rows_m})})
    register(rows_m, "Methods-Dict", "DICT_METHODS")
except Exception as ex:
    TOOL_STATUS["DICT_METHODS"] = dict(status="FAILED", reason=f"{type(ex).__name__}: {ex}"[:300])
    print("Methods-Dict FAILED:", TOOL_STATUS["DICT_METHODS"]["reason"])

## 8. Cross-vocabulary linking (Cell 7b)

Unique cleaned terms from **all** tools: UTS (CUI-shaped norms only -> name + semantic type),
BioPortal and EBI OLS (top hit only, provisional), and exact matching against the local OMOP dump
(CONCEPT + CONCEPT_SYNONYM, standard concepts preferred). API keys come only from the environment;
FOUND and genuine NOT_FOUND answers are cached, failures go to `_work/link_failures.json` and are
retried next run; v2 cache reuse requires the same query signature.


In [ ]:
# ===== Cell 7b: cross-vocabulary linking (UTS/UMLS + BioPortal + EBI OLS + OMOP) =====
# Scope: ALL tool rows. Unique terms are keyed on the cleaned surface form (same rule as
# Cell 8b clean_term) so the entity CSVs can join on term_lower.
# 1) UTS REST resolves unique UMLS CUIs -> preferred name + semantic type. Only `norm`
#    values shaped like a CUI (C + 7 digits) are treated as UMLS; tool-native IDs
#    (MESH:, CHEBI:, OMIM:, NCBI Gene/Taxonomy numbers) are kept in `native_ids`.
# 2) BioPortal + EBI OLS search each unique term (top hit only - provisional).
# 3) exact lowercase match against the local OMOP vocabulary dump
#    (CONCEPT.csv + CONCEPT_SYNONYM.csv, prefer standard 'S').
# v3: API keys come ONLY from the environment (UMLS_API_KEY, BIOPORTAL_API_KEY) and are never
# printed or logged. LookupCache (Cell 1b) caches FOUND and genuine NOT_FOUND answers; failures
# (network, 5xx, 401/403, unexpected payload) go to _work/link_failures.json and are retried
# next run. v2 caches are read-only seeds: only real v2 hits with the same query signature are
# reused; v2 None entries (no-hit OR failure - indistinguishable) are re-queried ("refreshed").
import urllib.request, urllib.parse, urllib.error
from concurrent.futures import ThreadPoolExecutor

UMLS_API_KEY = get_secret("UMLS_API_KEY")
BIOPORTAL_API_KEY = get_secret("BIOPORTAL_API_KEY")
print("API keys loaded from environment (values not shown)")
OMOP_DIR = Path(os.environ.get("NB_OMOP_DIR") or str(ROOT / "OMOP"))
# No OMOP dump on this host (OMOP/CONCEPT.csv absent). Rather than abort the notebook, mark the
# OMOP source UNAVAILABLE and let the UTS/BioPortal/OLS sources carry the cell; the OMOP loop below
# is skipped when OMOP_AVAILABLE is False.
OMOP_AVAILABLE = (OMOP_DIR / "CONCEPT.csv").exists() and (OMOP_DIR / "CONCEPT_SYNONYM.csv").exists()
if not OMOP_AVAILABLE:
    print(f"OMOP: UNAVAILABLE (no vocabulary dump at {OMOP_DIR}); local-vocab linking skipped")
LINK_FAILURES = WORK / "link_failures.json"

CUI_RE = re.compile(r"^C\d{7}$")
def link_clean(t):
    """Canonical surface form for linking: single-spaced, surrounding punctuation stripped (same rule as clean_term in Cell 8b)."""
    return re.sub(r"\s+", " ", t or "").strip().strip('.,;:)}]"%').strip()

term_count, term_models, term_cuis, term_native = {}, {}, {}, {}
for r in all_rows:
    t = link_clean(r["term"])
    if not t or len(t) < 2 or re.fullmatch(r"[\d\W]+", t):
        continue
    term_count[t] = term_count.get(t, 0) + 1
    term_models.setdefault(t, set()).add(r["model"])
    n = (r.get("norm") or "").strip()
    if CUI_RE.match(n):
        d = term_cuis.setdefault(t, {})
        d[n] = d.get(n, 0) + 1
    elif n and n != "-1":
        term_native.setdefault(t, set()).add(f"{r['model']}:{n}")
terms = sorted(term_count)
print("rows:", len(all_rows), "| unique terms:", len(terms),
      "| terms with CUI:", len(term_cuis), "| terms with native IDs:", len(term_native))

def http_json(url, tries=3):
    """-> (http_status, payload). 404 is returned (a real 'not found'); every other error raises
    LookupFailure after retries (4xx other than 404 are not retried). Messages never carry the URL."""
    last = None
    for i in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "ner-pipeline/1.0"})
            with urllib.request.urlopen(req, timeout=45) as resp:
                return resp.status, json.loads(resp.read().decode("utf-8"))
        except urllib.error.HTTPError as ex:
            if ex.code == 404:
                return 404, None
            last = f"HTTP {ex.code}"
            if 400 <= ex.code < 500 and ex.code != 429:
                break
        except Exception as ex:
            last = type(ex).__name__ + ": " + redact(str(ex))[:120]
        time.sleep(1 + 2 * i)
    raise LookupFailure(last)

def _replay(o):
    """Re-raise an exception captured by _safe so cache.get records it as a failure."""
    if isinstance(o, Exception):
        raise o
    return o

def _safe(fetch):
    """Wrap a fetcher so pool workers return exceptions instead of raising inside the pool."""
    def f(k):
        """Call fetch(k) and capture any exception as the return value."""
        try:
            return fetch(k)
        except Exception as ex:
            return ex
    return f

LINK_CACHE_AUDIT = {}

# --- UTS: unique CUIs -> name + semantic type ---
UTS_SIG = "GET uts-ws.nlm.nih.gov/rest/content/current/CUI/{cui}"
uts_c = LookupCache("UTS", WORK / "uts_cache_v3.json", LINK_FAILURES, UTS_SIG)
uts_c.seed_from_v2(V2_DIR / "_work" / "uts_cache.json", UTS_SIG, lambda v: v)
def uts_fetch(c):
    """Resolve one UMLS CUI via UTS REST into name plus semantic types, or NOT_FOUND; unexpected payloads raise LookupFailure."""
    code, d = http_json(f"https://uts-ws.nlm.nih.gov/rest/content/current/CUI/{c}?apiKey={UMLS_API_KEY}")
    if code == 404:
        return "NOT_FOUND", None
    rr = (d or {}).get("result")
    if not isinstance(rr, dict):
        raise LookupFailure("UNEXPECTED_PAYLOAD")
    return "FOUND", {"name": rr.get("name", ""),
                     "tui": "|".join(sorted({t.get("name", "") for t in rr.get("semanticTypes", [])}))}
cuis = sorted({c for d in term_cuis.values() for c in d})
uts = {}
for i, c in enumerate(cuis):
    e = uts_c.get(c, uts_fetch)
    uts[c] = e["value"] if e and e["status"] == "FOUND" else None
    if i == 9 and uts_c.counts["failed"] == 10:
        uts_c.save()
        raise RuntimeError("UTS: first 10 lookups all failed - check UMLS_API_KEY / network "
                           "(see _work/link_failures.json; key not shown)")
    time.sleep(0.05)
uts_c.save()
LINK_CACHE_AUDIT["UTS"] = dict(queried=len(cuis), **uts_c.counts, stored=uts_c.provenance_counts())
print("UTS CUIs resolved:", sum(v is not None for v in uts.values()), "/", len(cuis), "|", uts_c.counts)

def parallel_lookup(cache, keys, fetch, label):
    """Cache-backed lookup over keys: cached hits counted, misses fetched with 6 threads, cache saved every 100 keys; returns key -> value or None."""
    todo = [k for k in keys if k not in cache.entries]
    for k in keys:
        if k in cache.entries:
            cache.get(k, fetch)                           # counts cached / reused_v2
    with ThreadPoolExecutor(max_workers=6) as ex:
        for i, (k, o) in enumerate(zip(todo, ex.map(_safe(fetch), todo))):
            cache.get(k, lambda _k, o=o: _replay(o))
            if i % 100 == 0:
                cache.save(); print(label, i, "/", len(todo), flush=True)
    cache.save()
    return {k: (cache.entries[k]["value"] if k in cache.entries and cache.entries[k]["status"] == "FOUND" else None)
            for k in keys}

# --- BioPortal search per unique term ---
BP_SIG = "GET data.bioontology.org/search?q={term}&pagesize=1"
bp_c = LookupCache("BioPortal", WORK / "bioportal_cache_v3.json", LINK_FAILURES, BP_SIG)
bp_c.seed_from_v2(V2_DIR / "_work" / "bioportal_cache.json", BP_SIG, lambda v: v)
def bp_fetch(term):
    """BioPortal search for one term, top hit only, giving id, label, ontology (provisional); an empty collection is NOT_FOUND."""
    code, d = http_json("https://data.bioontology.org/search?q=" + urllib.parse.quote(term) +
                        "&pagesize=1&apikey=" + BIOPORTAL_API_KEY)
    if code == 404 or not isinstance(d, dict) or "collection" not in d:
        raise LookupFailure(f"UNEXPECTED_PAYLOAD http={code}")
    col = d["collection"]
    if not col:
        return "NOT_FOUND", None
    c0 = col[0]
    m = re.match(r"https?://[^/]+/ontology/([^/]+)/", c0.get("@id", "") + "/")
    return "FOUND", {"id": c0.get("@id", ""), "label": c0.get("prefLabel", ""),
                     "ontology": m.group(1) if m else ""}
bp = parallel_lookup(bp_c, terms, bp_fetch, "BioPortal")
LINK_CACHE_AUDIT["BioPortal"] = dict(queried=len(terms), **bp_c.counts, stored=bp_c.provenance_counts())
print("BioPortal hits:", sum(v is not None for v in bp.values()), "/", len(terms), "|", bp_c.counts)

# --- EBI OLS search per unique term ---
OLS_SIG = "GET www.ebi.ac.uk/ols4/api/search?q={term}&rows=1"   # v4: OLS4 (the /ols/api endpoint is retired)
ols_c = LookupCache("OLS", WORK / "ols_cache_v4.json", LINK_FAILURES, OLS_SIG)
# v4: no v2 seeding for OLS - the v2 cache was built against the old /ols/api endpoint
def ols_fetch(term):
    """EBI OLS search for one term, top hit only, giving iri, curie, label, ontology (provisional); empty docs is NOT_FOUND."""
    code, d = http_json("https://www.ebi.ac.uk/ols4/api/search?q=" + urllib.parse.quote(term) + "&rows=1")
    resp = (d or {}).get("response") if isinstance(d, dict) else None
    if code == 404 or not isinstance(resp, dict) or "docs" not in resp:
        raise LookupFailure(f"UNEXPECTED_PAYLOAD http={code}")
    docs = resp["docs"]
    if not docs:
        return "NOT_FOUND", None
    d0 = docs[0]
    return "FOUND", {"iri": d0.get("iri", ""), "curie": d0.get("curie", "") or d0.get("obo_id", ""),
                     "label": d0.get("label", ""), "ontology": d0.get("ontology_prefix", "")}
ols = parallel_lookup(ols_c, terms, ols_fetch, "OLS")
LINK_CACHE_AUDIT["OLS"] = dict(queried=len(terms), **ols_c.counts, stored=ols_c.provenance_counts())
print("OLS hits:", sum(v is not None for v in ols.values()), "/", len(terms), "|", ols_c.counts)

# --- OMOP: one local pass over CONCEPT.csv + CONCEPT_SYNONYM.csv (skipped: no dump on this host) ---
keys = {}
for t in terms:
    keys.setdefault(t.lower(), []).append(t)
name_hits, syn_hits, want_ids, id_meta = {}, {}, set(), {}
if OMOP_AVAILABLE:
    with open(OMOP_DIR / "CONCEPT_SYNONYM.csv", encoding="utf-8", newline="") as f:
        for line in f:
            parts = line.rstrip("\r\n").split("\t")
            if len(parts) < 2:
                continue
            k = parts[1].strip().lower()
            for t in keys.get(k, ()):
                syn_hits.setdefault(t, set()).add(parts[0])
                want_ids.add(parts[0])
    with open(OMOP_DIR / "CONCEPT.csv", encoding="utf-8", newline="") as f:
        next(f, None)
        for line in f:
            parts = line.rstrip("\r\n").split("\t")
            if len(parts) < 6:
                continue
            k = parts[1].strip().lower()
            for t in keys.get(k, ()):
                name_hits.setdefault(t, []).append(
                    (parts[0], parts[1], parts[3], parts[2], parts[5] == "S"))
            if parts[0] in want_ids:
                id_meta[parts[0]] = (parts[1], parts[3], parts[2], parts[5] == "S")
omop_best = {}
for t in terms:
    cands = list(name_hits.get(t, []))
    seen = {c[0] for c in cands}
    cands += [(cid, meta[0], meta[1], meta[2], meta[3])
              for cid in syn_hits.get(t, set())
              if cid in id_meta and cid not in seen for meta in [id_meta[cid]]]
    cands.sort(key=lambda x: (not x[4], x[0]))
    omop_best[t] = "; ".join(f"{c[0]}|{c[1]}|{c[2]}|{c[3]}" for c in cands[:3])
print("OMOP hits:", sum(bool(v) for v in omop_best.values()), "/", len(terms))

LINK_FIELDS = ["term", "n_mentions", "models", "native_ids", "umls_cui", "cui_mentions",
               "uts_name", "uts_semantic_type", "bioportal_id", "bioportal_label",
               "bioportal_ontology", "ols_curie", "ols_label", "ols_ontology",
               "omop_concepts", "n_link_sources"]
rows_l, status_rows = [], []
def _st(cache, k):
    """Lookup state for the status CSV: FOUND / NOT_FOUND / LOOKUP_FAILED / NOT_ATTEMPTED."""
    return cache.entries[k]["status"] if k in cache.entries else ("LOOKUP_FAILED" if k in cache.failures else "NOT_ATTEMPTED")
for t in terms:
    cuis_t = term_cuis.get(t, {})
    cui_top = max(cuis_t, key=cuis_t.get) if cuis_t else ""
    u = uts.get(cui_top) or {}
    b = bp.get(t) or {}
    o = ols.get(t) or {}
    row = dict(term=t, n_mentions=term_count[t],
               models=";".join(sorted(term_models[t])),
               native_ids=";".join(sorted(term_native.get(t, ()))),
               umls_cui=cui_top, cui_mentions=sum(cuis_t.values()) if cuis_t else 0,
               uts_name=u.get("name", ""), uts_semantic_type=u.get("tui", ""),
               bioportal_id=b.get("id", ""), bioportal_label=b.get("label", ""),
               bioportal_ontology=b.get("ontology", ""),
               ols_curie=o.get("curie", "") or o.get("iri", ""),
               ols_label=o.get("label", ""), ols_ontology=o.get("ontology", ""),
               omop_concepts=omop_best.get(t, ""))
    row["n_link_sources"] = sum(bool(row[k]) for k in
                                ("umls_cui", "bioportal_id", "ols_curie", "omop_concepts"))
    rows_l.append(row)
    status_rows.append(dict(term=t, uts_status=_st(uts_c, cui_top) if cui_top else "NOT_APPLICABLE",
                            bioportal_status=_st(bp_c, t), ols_status=_st(ols_c, t),
                            omop_status="FOUND" if omop_best.get(t) else "NOT_FOUND"))
dump_csv(OUT_DIR / "newtools_linking.csv", rows_l, LINK_FIELDS)
# lookup states in a separate file so newtools_linking.csv keeps its v2 columns
dump_csv(OUT_DIR / "link_lookup_status.csv", status_rows,
         ["term", "uts_status", "bioportal_status", "ols_status", "omop_status"])
print("linking CSV:", OUT_DIR / "newtools_linking.csv", "|", len(rows_l), "rows")
covered = sum(1 for r in rows_l if r["n_link_sources"])
print(f"terms covered by >=1 source: {covered}/{len(rows_l)} ({covered/max(len(rows_l),1):.1%})")
print("lookup failures (retried next run):", {s: len(c.failures) for s, c in
                                              (("UTS", uts_c), ("BioPortal", bp_c), ("OLS", ols_c))})

# lowercase-term index for Cell 8b joins; first case variant with most mentions wins
LINK_BY_TERM = {}
for r in sorted(rows_l, key=lambda r: -r["n_mentions"]):
    LINK_BY_TERM.setdefault(r["term"].lower(), r)

## 8b. Type-restricted linking, definitions, taxonomy (Cell 7e, v4 N5)

OLS4 search restricted by CDM type, verbatim definitions (OLS4, then UTS), and NCBITaxon rank + lineage for organisms. Failures are retried next run, never cached as misses.

In [ ]:
# ===== Cell 7e: type-restricted linking, definitions, taxonomy (v4 N5) =====
# Runs after Cell 7b. Per (cdm_type, term):
#   - EBI OLS4 search restricted to the ontologies for that type (TYPE_ONTOLOGIES). The definition is the
#     ontology's own text, copied verbatim, with its source ontology and term ID. Never model-written.
#   - UTS definitions for the term's top CUI (from Cell 7b), verbatim, with the UMLS root source.
#   - Organisms: NCBITaxon rank + lineage from OLS4 (has_rank annotation, hierarchicalAncestors).
# Caching uses LookupCache (Cell 1b): FOUND / NOT_FOUND are cached, failures go to link_failures.json and
# are retried on the next run. Every field gets a status: FOUND, NOT_FOUND, LOOKUP_FAILED, NOT_ATTEMPTED
# or NOT_APPLICABLE.
TYPE_ONTOLOGIES = {"CHEMICAL": "chebi", "DISEASE": "mondo,doid", "ORGANISM": "ncbitaxon",
                   "PROTEIN": "pr", "GENOMICS": "so,pr", "DNA_RNA": "so", "CELL_TYPE": "cl",
                   "PATHWAY": "go", "LAB_METHOD": "obi", "DATA_ANALYSIS": "edam,obi",
                   "SAMPLE_METHOD": "obi", "EXPERIMENTAL_METHOD": "obi", "CLINICAL": "ncit",
                   "STUDY_DESIGN": "obi"}
OLS4 = "https://www.ebi.ac.uk/ols4/api"

def iri_to_curie(iri):
    """http://purl.obolibrary.org/obo/PR_000001 -> PR:000001 (some OLS4 docs carry no obo_id)."""
    m = re.search(r"/obo/([A-Za-z]+)_(\w+)$", iri or "")
    return f"{m.group(1)}:{m.group(2)}" if m else (iri or "")

typed_terms = {}
for r in all_rows:
    t = link_clean(r["term"])
    if not t or len(t) < 2 or re.fullmatch(r"[\d\W]+", t) or r["mapping_status"] == "UNMAPPED":
        continue
    if r["cdm_type_proposed"] in TYPE_ONTOLOGIES:
        typed_terms.setdefault((r["cdm_type_proposed"], t.lower()), t)
print("typed (cdm_type, term) pairs:", len(typed_terms))

OLS4_SIG = OLS4 + "/search?q={term}&ontology={onts}&rows=1&queryFields=label,synonym"
ols4_c = LookupCache("OLS4_TYPED", WORK / "ols4_typed_cache_v4.json", LINK_FAILURES, OLS4_SIG)
def ols4_fetch(key):
    onts, term = key.split("|", 1)
    code, d = http_json(f"{OLS4}/search?q={urllib.parse.quote(term)}&ontology={onts}&rows=1"
                        "&queryFields=label,synonym")
    resp = (d or {}).get("response") if isinstance(d, dict) else None
    if code == 404 or not isinstance(resp, dict) or "docs" not in resp:
        raise LookupFailure(f"UNEXPECTED_PAYLOAD http={code}")
    if not resp["docs"]:
        return "NOT_FOUND", None
    d0 = resp["docs"][0]
    desc = [x for x in (d0.get("description") or []) if isinstance(x, str) and x.strip()]
    syns = sorted(set((d0.get("exact_synonyms") or []) + (d0.get("related_synonyms") or [])))[:20]
    return "FOUND", dict(curie=d0.get("obo_id") or iri_to_curie(d0.get("iri", "")), iri=d0.get("iri", ""), label=d0.get("label", ""),
                         ontology=d0.get("ontology_prefix", ""), synonyms=syns,
                         definition=desc[0] if desc else "")
ols4_keys = sorted({f"{TYPE_ONTOLOGIES[ct]}|{t}" for (ct, _), t in typed_terms.items()})
ols4 = parallel_lookup(ols4_c, ols4_keys, ols4_fetch, "OLS4 typed")
LINK_CACHE_AUDIT["OLS4_TYPED"] = dict(queried=len(ols4_keys), **ols4_c.counts, stored=ols4_c.provenance_counts())
print("OLS4 typed hits:", sum(v is not None for v in ols4.values()), "/", len(ols4_keys), "|", ols4_c.counts)

# --- UTS definitions for CUIs found in Cell 7b ---
UTSDEF_SIG = "GET uts-ws.nlm.nih.gov/rest/content/current/CUI/{cui}/definitions"
utsdef_c = LookupCache("UTS_DEF", WORK / "uts_def_cache_v4.json", LINK_FAILURES, UTSDEF_SIG)
UTS_DEF_SOURCES = ("MSH", "NCI", "CSP", "MEDLINEPLUS", "HPO", "GO", "CHV")   # preference order
def utsdef_fetch(c):
    code, d = http_json(f"https://uts-ws.nlm.nih.gov/rest/content/current/CUI/{c}/definitions?apiKey={UMLS_API_KEY}")
    if code == 404:
        return "NOT_FOUND", None
    res = (d or {}).get("result")
    if not isinstance(res, list):
        raise LookupFailure("UNEXPECTED_PAYLOAD")
    if not res:
        return "NOT_FOUND", None
    res.sort(key=lambda x: UTS_DEF_SOURCES.index(x.get("rootSource")) if x.get("rootSource") in UTS_DEF_SOURCES else 99)
    return "FOUND", dict(definition=res[0].get("value", ""), source="UMLS:" + res[0].get("rootSource", ""))
def_cuis = sorted({r["umls_cui"] for r in rows_l if r["umls_cui"]})
uts_defs = parallel_lookup(utsdef_c, def_cuis, utsdef_fetch, "UTS definitions")
LINK_CACHE_AUDIT["UTS_DEF"] = dict(queried=len(def_cuis), **utsdef_c.counts, stored=utsdef_c.provenance_counts())
print("UTS definitions:", sum(v is not None for v in uts_defs.values()), "/", len(def_cuis))

# --- NCBITaxon rank + lineage (OLS4) for organism links ---
RANK_ORDER = ["domain", "superkingdom", "kingdom", "phylum", "class", "order", "family", "genus", "species",
              "subspecies", "strain"]
TAXRANK = {"0000037": "domain", "0000017": "kingdom", "0000001": "phylum", "0000002": "class",
           "0000003": "order", "0000004": "family", "0000005": "genus", "0000006": "species",
           "0000023": "subspecies", "0000060": "strain"}
def _rank_of(term_json):
    ann = term_json.get("annotation") or {}
    for v in ann.get("obsolete has_rank", []):
        if "NCBITaxon_" in v:
            return v.rsplit("NCBITaxon_", 1)[1]
    for v in ann.get("has_rank", []):
        return TAXRANK.get(v.rsplit("_", 1)[-1], "")
    return ""
TAX_SIG = OLS4 + "/ontologies/ncbitaxon/terms/{iri}(+/hierarchicalAncestors)"
tax_c = LookupCache("OLS4_TAXON", WORK / "ols4_taxon_cache_v4.json", LINK_FAILURES, TAX_SIG)
def tax_fetch(iri):
    enc = urllib.parse.quote(urllib.parse.quote(iri, safe=""), safe="")
    code, t = http_json(f"{OLS4}/ontologies/ncbitaxon/terms/{enc}")
    if code == 404:
        return "NOT_FOUND", None
    code2, a = http_json(f"{OLS4}/ontologies/ncbitaxon/terms/{enc}/hierarchicalAncestors?size=100")
    anc = ((a or {}).get("_embedded") or {}).get("terms", []) if code2 != 404 else []
    lin = sorted(((_rank_of(x), x.get("label", "")) for x in anc if _rank_of(x) in RANK_ORDER),
                 key=lambda p: RANK_ORDER.index(p[0]))
    return "FOUND", dict(rank=_rank_of(t), lineage="; ".join(f"{r}:{n}" for r, n in lin))
tax_iris = sorted({v["iri"] for k, v in ols4.items() if v and k.startswith("ncbitaxon|")})
taxa_info = parallel_lookup(tax_c, tax_iris, tax_fetch, "OLS4 taxonomy")
LINK_CACHE_AUDIT["OLS4_TAXON"] = dict(queried=len(tax_iris), **tax_c.counts, stored=tax_c.provenance_counts())
print("taxa with rank/lineage:", sum(v is not None for v in taxa_info.values()), "/", len(tax_iris))

# --- one row per (cdm_type, term_lower) ---
TYPED_FIELDS = ["cdm_type", "term_lower", "term", "ontologies_searched", "ols4_status", "ols4_curie", "ols4_label",
                "ols4_ontology", "synonyms", "definition", "definition_source", "definition_id",
                "definition_status", "taxon_rank", "taxon_lineage", "taxonomy_status"]
TYPED_LINKS, typed_rows = {}, []
for (ct, tl), t in sorted(typed_terms.items()):
    k = f"{TYPE_ONTOLOGIES[ct]}|{t}"
    o = ols4.get(k)
    if o and not o.get("curie"):
        o = dict(o, curie=iri_to_curie(o.get("iri", "")))     # entries cached before the obo_id fallback
    lk = LINK_BY_TERM.get(tl) or {}
    ud = uts_defs.get(lk.get("umls_cui", "")) if lk.get("umls_cui") else None
    row = dict(cdm_type=ct, term_lower=tl, term=t, ontologies_searched=TYPE_ONTOLOGIES[ct],
               ols4_status=_st(ols4_c, k), ols4_curie=(o or {}).get("curie", ""),
               ols4_label=(o or {}).get("label", ""), ols4_ontology=(o or {}).get("ontology", ""),
               synonyms="|".join((o or {}).get("synonyms", [])))
    if o and o.get("definition"):
        row.update(definition=o["definition"], definition_source="OLS4:" + o["ontology"],
                   definition_id=o["curie"], definition_status="FOUND")
    elif ud and ud.get("definition"):
        row.update(definition=ud["definition"], definition_source=ud["source"],
                   definition_id=lk["umls_cui"], definition_status="FOUND")
    else:
        failed = _st(ols4_c, k) == "LOOKUP_FAILED" or (lk.get("umls_cui") and _st(utsdef_c, lk["umls_cui"]) == "LOOKUP_FAILED")
        row.update(definition="", definition_source="", definition_id="",
                   definition_status="LOOKUP_FAILED" if failed else "NOT_FOUND")
    if ct == "ORGANISM":
        ti = taxa_info.get((o or {}).get("iri", "")) if o and o.get("ontology") == "NCBITaxon" else None
        row.update(taxon_rank=(ti or {}).get("rank", ""), taxon_lineage=(ti or {}).get("lineage", ""),
                   taxonomy_status=("FOUND" if ti else _st(tax_c, o["iri"])) if o and o.get("ontology") == "NCBITaxon"
                   else ("LOOKUP_FAILED" if row["ols4_status"] == "LOOKUP_FAILED" else "NOT_FOUND"))
    else:
        row.update(taxon_rank="", taxon_lineage="", taxonomy_status="NOT_APPLICABLE")
    TYPED_LINKS[(ct, tl)] = row
    typed_rows.append(row)
dump_csv(OUT_DIR / "newtools_typed_links.csv", typed_rows, TYPED_FIELDS)
print("typed links:", len(typed_rows), "| definitions:", sum(r["definition_status"] == "FOUND" for r in typed_rows),
      "| lookup failures (retried next run):", {s: len(c.failures) for s, c in
      (("OLS4_TYPED", ols4_c), ("UTS_DEF", utsdef_c), ("OLS4_TAXON", tax_c))})

## 9. Outputs, audit and deduplication (Cells 8, 8b)

`newtools_candidates.csv` (rows from OK tools only) plus `run_summary.json` and `audit_log.json`
(secrets redacted). Cell 8b builds the entity view: mapping -> normalization (junk dropped,
`entity_key` as dedup key) -> per-paper and per-batch entities, joined with the Cell 7b
vocabulary links. The mention table stays the source of truth; entities are a view.


In [ ]:
# ===== Cell 8: combined CSV + audit log =====
# Built only from rows of tools whose status is OK in THIS run (register() refuses others).
dump_csv(OUT_DIR / "newtools_candidates.csv", all_rows, FIELDS)
by_model, by_type, by_paper, by_map = {}, {}, {}, {}
for r in all_rows:
    by_model[r["model"]] = by_model.get(r["model"], 0) + 1
    by_type[r["cdm_type_proposed"]] = by_type.get(r["cdm_type_proposed"], 0) + 1
    by_paper[r["cid"]] = by_paper.get(r["cid"], 0) + 1
    by_map[r["mapping_status"]] = by_map.get(r["mapping_status"], 0) + 1

def _write_redacted(path, obj):
    """Write pretty-printed JSON with secret values removed; used for every summary and audit output."""
    path.write_text(redact(json.dumps(obj, indent=1, ensure_ascii=False)), encoding="utf-8")

RUN_STARTED_AT = globals().get("RUN_STARTED_AT", "NOT RECORDED")
tool_counts = {}
for st in TOOL_STATUS.values():
    tool_counts[st["status"]] = tool_counts.get(st["status"], 0) + 1
summary = dict(run_tag=RUN_TAG, status="PROVISIONAL_UNVALIDATED", production_writes=PRODUCTION_WRITES,
               written_at=_now(), papers=PAPERS, full_corpus=FULL_CORPUS,
               candidate_rows=len(all_rows), by_model=by_model, by_cdm_type=by_type,
               by_paper=by_paper, by_mapping_status=by_map, rejected_spans=REJECTS,
               bert_partial_word_trace=BERT_TRACE,
               tool_status=TOOL_STATUS, tool_status_counts=tool_counts,
               timeouts=dict(per_paper_s=TOOL_TIMEOUT_PER_PAPER, startup_s=TOOL_STARTUP_S,
                             note="Phase 1: one run per tool over all papers; a timeout fails the tool "
                                  "for every paper in the run. No per-paper isolation yet."),
               llm=dict(stats=LLM_STATS, cache=LLM_CACHE_AUDIT, gguf=LLM_GGUF.name,
                        gguf_sha256=LLM_GGUF_SHA256, server_build=LLM_SERVER_BUILD),
               link_cache_audit=LINK_CACHE_AUDIT, tests=TEST_RESULTS, five_paper_checks=CHECK_LOG)
_write_redacted(OUT_DIR / "run_summary.json", summary)
print(json.dumps(dict(candidate_rows=len(all_rows), by_model=by_model, tool_status_counts=tool_counts,
                      by_mapping_status=by_map), indent=1))

audit = dict(run_type="NEWTOOLS_NER_MULTI_TOOL", run_tag=RUN_TAG, status="PROVISIONAL_UNVALIDATED",
             accuracy="UNKNOWN - no adjudicated human gold labels yet",
             read_only_dirs=[str(V1_DIR), str(V2_DIR)], body_only=BODY_ONLY,
             body_filter="extended backmatter list; (preamble) kept (single-section papers 000003/000009)",
             papers=PAPERS, full_corpus=FULL_CORPUS, inputs=["segmented_sections_markdown"],
             v2_fixes=dict(
                 placeholder_ids=f"norm in {sorted(PLACEHOLDER_IDS)} -> norm='' (norm_status PLACEHOLDER/MISSING); raw kept in norm_raw",
                 ligatures="U+FB00-FB06 expanded in tool input only; offsets mapped back via ToolText.to_orig (rejects ligature-cutting spans, verifies every slice); stored offsets index ORIGINAL text",
                 bert_decoding="word-level BIO, first-sub-token label per word (tokenizer word_ids); v1 sub-token decoding emitted alongside as 'legacy' for tracing only",
                 mapping="cdm_type_proposed unchanged; mapping_status MAPPED/REVIEW_REQUIRED/UNMAPPED vs Microbiome-CDM config.py CDM_TYPES"),
             v3_phase1_fixes=dict(
                 secrets="UMLS_API_KEY/BIOPORTAL_API_KEY from environment only; never printed; redact() applied to every summary/log tail",
                 lookup_caches="FOUND/NOT_FOUND cached; failures -> _work/link_failures.json, retried next run; v2 seeds: hits with same query signature only, v2 None re-queried",
                 tool_status="run_tool_checked: old outputs deleted first, exit code + timeout + output existence checked; consume_output requires status OK and unchanged sha256; FAILED tools write no CSV",
                 timeouts="TOOL_STARTUP_S + per-paper x n_papers, whole-run (no per-paper isolation yet)",
                 llm_cache_key="sha256(chunk text, prompt, schema, GGUF name + sha256, server build, temperature, max_tokens, prompt version, retry policy)",
                 llm_occurrence_expanded="1 = row from a further whole-word occurrence of a returned term; same rows as v2"),
             conventions=dict(
                 section_text="marker lines removed line-wise; ORDINAL kept; sections joined \\n\\n (verified vs aioner_input XML)",
                 pubtator="sentinel title=section_id; offsets shifted by len(title)+1; slice-validated on tool text, then mapped to original",
                 hunflair2="3000-char paragraph-boundary chunking over tool text",
                 scispacy="4 models via .venv_markdown/scispacy 0.6.2; UMLS linker top CUI -> norm, linker score -> score",
                 llm="Qwen3.5-4B Q4_K_M GGUF via llama.cpp llama-server (CUDA, 8 slots, thinking off, JSON schema limited to CDM_TYPES); intro/background + backmatter sections skipped; returned text located by exact whole-word match in its chunk; not_in_text dropped",
                 linking="unique cleaned terms from ALL tools: UTS (CUI-shaped norms only) + BioPortal + EBI OLS (top hit, provisional) + OMOP exact match",
                 baseline_flatten="harmonized md; HTML comments KEPT; \\n\\n+ -> 2 spaces; \\n -> space; no strip; no NFKC"),
             outputs=dict(candidates_csv=str(OUT_DIR / "newtools_candidates.csv"),
                          linking_csv=str(OUT_DIR / "newtools_linking.csv"),
                          link_lookup_status_csv=str(OUT_DIR / "link_lookup_status.csv"),
                          entities_paper_csv=str(OUT_DIR / "newtools_entities_paper.csv"),
                          entities_batch_csv=str(OUT_DIR / "newtools_entities_batch.csv"),
                          dedup_stats=str(OUT_DIR / "dedup_stats.json"),
                          comparison=str(OUT_DIR / "v3_vs_v2_comparison.json")),
             production_writes=PRODUCTION_WRITES)
_write_redacted(OUT_DIR / "audit_log.json", audit)
print("audit written")

In [ ]:
# ===== Cell 8b: mapping -> normalization -> deduplication =====
# Mention table (newtools_candidates.csv) stays the source of truth; entities are a view.
#   1) MAPPING      - cdm_of (Cell 1) + mapping_status (Cell 1b); unmapped/review counted
#   2) NORMALIZATION- surface cleanup; junk rows (empty, <2 chars, numeric/punct-only) dropped
#                     key = entity_key(): "<vocab>:<id>" for VALID ids, else "TERM:<clean lower>"
#                     (v2: placeholder ids such as tmChem '-1' can never be a key)
#   3) DEDUPLICATION- tier 1: per paper (cid, cdm_type, key); tier 2: per batch (cdm_type, key)
#   4) LINKING      - join Cell 7b vocabulary links (LINK_BY_TERM) on term_lower
def clean_term(t):
    """Canonical surface form for dedup: single-spaced, surrounding punctuation stripped; joins use term_lower."""
    return re.sub(r"\s+", " ", t or "").strip().strip('.,;:)}]"%').strip()

def is_junk(s):
    """True for empty, one-character, or purely numeric or punctuation surface forms; those rows are dropped."""
    return (not s) or len(s) < 2 or re.fullmatch(r"[\d\W]+", s)

norm_stats = dict(input_rows=len(all_rows), dropped_junk=0, unmapped_labels={},
                  review_required_labels={}, norm_status={}, cleaned_examples=[])
ent_paper, ent_batch = {}, {}
for r in all_rows:
    lab = r.get("label_raw", "") or r["model"]
    if r["mapping_status"] == "UNMAPPED":
        norm_stats["unmapped_labels"][lab] = norm_stats["unmapped_labels"].get(lab, 0) + 1
    elif r["mapping_status"] == "REVIEW_REQUIRED":
        norm_stats["review_required_labels"][lab] = norm_stats["review_required_labels"].get(lab, 0) + 1
    norm_stats["norm_status"][r["norm_status"]] = norm_stats["norm_status"].get(r["norm_status"], 0) + 1
    term = clean_term(r["term"])
    if is_junk(term):
        norm_stats["dropped_junk"] += 1
        if len(norm_stats["cleaned_examples"]) < 10:
            norm_stats["cleaned_examples"].append([r["model"], r["term"], term])
        continue
    key_norm = entity_key(term, r["norm"], r["norm_vocab"])
    pkey = (r["cid"], r["cdm_type_proposed"], key_norm)
    e = ent_paper.setdefault(pkey, dict(cid=r["cid"], cdm_type=r["cdm_type_proposed"],
                                         norm_key=key_norm, term=term, term_lower=term.lower(),
                                         sources=[], mention_count=0, section_ids=[],
                                         ids=set(), terms=set(), mapping=set()))
    e["sources"].append(r["model"]); e["mention_count"] += 1
    e["section_ids"].append(r["section_id"]); e["terms"].add(term.lower())
    e["mapping"].add(r["mapping_status"])
    if r["norm"]:
        e["ids"].add(f'{r["norm_vocab"]}:{r["norm"]}')
    bkey = (r["cdm_type_proposed"], key_norm)
    b = ent_batch.setdefault(bkey, dict(cdm_type=r["cdm_type_proposed"], norm_key=key_norm,
                                         term=term, term_lower=term.lower(), sources=[],
                                         mention_count=0, cids=set(), terms=set(),
                                         mapping=set(), first_model=r["model"]))
    b["sources"].append(r["model"]); b["mention_count"] += 1
    b["cids"].add(r["cid"]); b["terms"].add(term.lower()); b["mapping"].add(r["mapping_status"])

LINK_COLS = ["native_ids", "umls_cui", "uts_name", "uts_semantic_type", "bioportal_id",
             "bioportal_label", "bioportal_ontology", "ols_curie", "ols_label", "ols_ontology",
             "omop_concepts", "n_link_sources"]
def link_cols(term_lower):
    """Vocabulary-link columns for a lowercase term from the Cell 7b index; unknown terms get empty strings."""
    l = LINK_BY_TERM.get(term_lower) or {}
    return {k: l.get(k, "") for k in LINK_COLS}

PAPER_FIELDS = ["cid", "cdm_type", "norm_key", "term", "term_lower", "entity_id", "sources",
                "n_sources", "mention_count", "n_sections", "section_ids", "ids", "n_ids",
                "n_surface_forms", "mapping_status"] + LINK_COLS
paper_rows = []
for i, (k, e) in enumerate(sorted(ent_paper.items()), 1):
    srcs = sorted(set(e["sources"]))
    paper_rows.append(dict(cid=e["cid"], cdm_type=e["cdm_type"], norm_key=e["norm_key"],
                           term=e["term"], term_lower=e["term_lower"], entity_id=f"ENTP_{i:05d}",
                           sources="|".join(srcs), n_sources=len(srcs),
                           mention_count=e["mention_count"],
                           n_sections=len(set(e["section_ids"])),
                           section_ids="|".join(sorted(set(e["section_ids"]))),
                           ids="|".join(sorted(e["ids"])), n_ids=len(e["ids"]),
                           n_surface_forms=len(e["terms"]),
                           mapping_status="|".join(sorted(e["mapping"])),
                           **link_cols(e["term_lower"])))
dump_csv(OUT_DIR / "newtools_entities_paper.csv", paper_rows, PAPER_FIELDS)

BATCH_FIELDS = ["cdm_type", "norm_key", "term", "term_lower", "entity_id", "sources", "n_sources",
                "n_papers", "cids", "mention_count", "n_surface_forms", "mapping_status",
                "first_model"] + LINK_COLS
batch_rows = []
for i, (k, b) in enumerate(sorted(ent_batch.items()), 1):
    srcs = sorted(set(b["sources"]))
    batch_rows.append(dict(cdm_type=b["cdm_type"], norm_key=b["norm_key"],
                           term=b["term"], term_lower=b["term_lower"], entity_id=f"ENTB_{i:05d}",
                           sources="|".join(srcs), n_sources=len(srcs),
                           n_papers=len(b["cids"]), cids="|".join(sorted(b["cids"])),
                           mention_count=b["mention_count"], n_surface_forms=len(b["terms"]),
                           mapping_status="|".join(sorted(b["mapping"])),
                           first_model=b["first_model"], **link_cols(b["term_lower"])))
dump_csv(OUT_DIR / "newtools_entities_batch.csv", batch_rows, BATCH_FIELDS)

norm_stats["paper_level_entities"] = len(paper_rows)
norm_stats["batch_level_entities"] = len(batch_rows)
(OUT_DIR / "dedup_stats.json").write_text(json.dumps(norm_stats, indent=1, ensure_ascii=False), encoding="utf-8")
print("normalization stats:", json.dumps({k: v for k, v in norm_stats.items() if k != "cleaned_examples"}, indent=1))
print("paper-level entities:", len(paper_rows), "| batch-level entities:", len(batch_rows))
print("batch entities with >=1 vocabulary link:",
      sum(1 for b in batch_rows if b["n_link_sources"]), "/", len(batch_rows))

## 8c. Normalization and deduplication rules (Cell 8b2)

Ligature, case and abbreviation-aware entity keys; type conflicts listed for review; LLM and value deduplication. Used by the entity tables in Cell 9b and the catalogue (Cell 12b).

In [ ]:
# ===== Cell 8b2: normalization and deduplication rules for the v4 entity tables =====
# Used by Cell 8c (entity keys, type conflicts) and Cell 12b (LLM and value deduplication).
# Nothing here changes a mention's offsets or its stored term; only the keys used to merge rows change.
#   norm_surface     NFKC, ligature folding, whitespace and edge punctuation removed, lowercase
#   entity_key_v4    a valid vocabulary ID is used as before; otherwise TERM:<norm_surface>, so
#                    'Biﬁdobacterium' and 'biﬁdobacterium' merge
#   abbreviations    'F. prausnitzii' merges into 'Faecalibacterium prausnitzii' when that full name
#                    occurs exactly once in the same paper (per-paper map built from the MeSH rows)
#   type conflicts   one ID under several CDM types stays split, and Cell 8c writes it to type_conflicts.csv
#                    for review; it is never merged silently
import unicodedata
LIG_FOLD = {"ﬀ": "ff", "ﬁ": "fi", "ﬂ": "fl", "ﬃ": "ffi", "ﬄ": "ffl", "ﬅ": "st", "ﬆ": "st"}

def norm_surface(s):
    t = unicodedata.normalize("NFKC", s or "")
    for k, v in LIG_FOLD.items():
        t = t.replace(k, v)
    t = re.sub(r"\s+", " ", t).strip().strip(".,;:()[]{}\"'").strip()
    return t.lower()

def build_full_names(rows):
    """cid -> {(initial, second_word): {normalized full names}} from MeSH taxon rows with 2+ words."""
    out = {}
    for r in rows:
        if r.get("model") == "MeSH-Taxon-Dict" and r.get("label_raw") == "taxon_mesh":
            words = norm_surface(r["term"]).split()
            if len(words) >= 2:
                out.setdefault(r["cid"], {}).setdefault((words[0][:1], words[1]), set()).add(" ".join(words))
    return out

FULL_NAMES = {}

def entity_key_v4(cid, term, norm, vocab):
    if norm:
        return f"{vocab}:{norm}"
    s = norm_surface(term)
    m = re.fullmatch(r"([a-z])\. (\S+)", s)
    if m:
        hits = FULL_NAMES.get(cid, {}).get((m.group(1), m.group(2)), set())
        if len(hits) == 1:
            return "TERM:" + next(iter(hits))
    return "TERM:" + s

def dedupe_by(rows, key):
    """Keep the first row for each key; returns (kept, n_dropped)."""
    seen, kept = set(), []
    for r in rows:
        k = key(r)
        if k not in seen:
            seen.add(k); kept.append(r)
    return kept, len(rows) - len(kept)

# ---------- tests (the cell stops the run if any fails) ----------
class TestV4Normalization(unittest.TestCase):
    def test_surface_folds_ligatures_case_and_edges(self):
        self.assertEqual(norm_surface("Biﬁdobacterium"), "bifidobacterium")
        self.assertEqual(norm_surface("  B. biﬁdum. "), "b. bifidum")
        self.assertEqual(norm_surface("ＡＢＣ"), "abc")                      # NFKC full-width
    def test_ligature_and_case_variants_merge_without_id(self):
        self.assertEqual(entity_key_v4("P", "Biﬁdobacterium", "", ""), entity_key_v4("P", "bifidobacterium", "", ""))
    def test_id_key_unchanged(self):
        self.assertEqual(entity_key_v4("P", "Butyrate", "MESH:D020148", "MESH"), "MESH:MESH:D020148")   # unchanged from Cell 8b
    def test_abbreviation_merges_only_when_unique(self):
        rows = [dict(model="MeSH-Taxon-Dict", label_raw="taxon_mesh", cid="P", term="Faecalibacterium prausnitzii"),
                dict(model="MeSH-Taxon-Dict", label_raw="taxon_mesh", cid="Q", term="Bacteroides fragilis"),
                dict(model="MeSH-Taxon-Dict", label_raw="taxon_mesh", cid="Q", term="Bacteroides fragilis subsp. x")]
        global FULL_NAMES
        saved, FULL_NAMES = FULL_NAMES, build_full_names(rows)
        try:
            self.assertEqual(entity_key_v4("P", "F. prausnitzii", "", ""), "TERM:faecalibacterium prausnitzii")
            self.assertEqual(entity_key_v4("Q", "B. fragilis", "", ""), "TERM:b. fragilis")       # two full names: keep
            self.assertEqual(entity_key_v4("P", "E. coli", "", ""), "TERM:e. coli")
        finally:
            FULL_NAMES = saved
    def test_dedupe_keeps_first(self):
        kept, n = dedupe_by([dict(k=1, v="a"), dict(k=1, v="b"), dict(k=2, v="c")], lambda r: r["k"])
        self.assertEqual((len(kept), n, kept[0]["v"]), (2, 1, "a"))

_suite8 = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromTestCase(TestV4Normalization))
_res8 = unittest.TextTestRunner(verbosity=1, stream=sys.stdout).run(_suite8)
TEST_RESULTS["v4_normalization"] = dict(ran=_res8.testsRun, ok=_res8.wasSuccessful(),
                                        failed_tests=[str(t) for t, _ in _res8.failures + _res8.errors])
assert _res8.wasSuccessful(), "v4 normalization tests failed - run stopped"

FULL_NAMES = build_full_names(all_rows)

## 9b. Junk filter, consensus, template tables, validation (Cell 8c, v4 N3/N4/N6/N7)

Flags (never deletes) junk mentions, groups overlapping spans across tools, builds `ner_mentions` / `ner_entities_paper` / `ner_entities_corpus`, carries review fields over by `entity_id`, and stops the run if any of the 13 build checks fails.

In [ ]:
# ===== Cell 8c: junk filter, consensus, template tables, validation (v4 N3, N4, N6, N7) =====
#   N3  general-English NER (spaCy GENERAL_NER_MODEL, run under GENERAL_NER_PY, cached per section) +
#       junk_reason() flags. Flagged rows stay in ner_mentions with drop_reason; entity tables skip them.
#   N4  span_groups(): one span_group_id per set of identical/overlapping spans; frequency counts groups.
#   N6  ner_mentions (source of truth) -> ner_entities_paper -> ner_entities_corpus, rebuilt every run.
#       section_class comes from the PROPOSED crosswalk (Cell 1d) and is marked as such.
#   N7  13 build checks; any violation writes the report and stops the run. Review fields are carried
#       over by entity_id from the previous build; curated values never overwrite machine values.
# Tables follow the notebook's draft schema (NER_SCHEMA_VERSION) because config/ner_entity_template.yaml
# is not on disk. When that file exists, its field lists are compared with these and a mismatch fails.

# ---------- N3a: general NER (resumable per section) ----------
GEN_CACHE = WORK / f"general_ner_{GENERAL_NER_MODEL}.json"
gen_cache = json.loads(GEN_CACHE.read_text(encoding="utf-8")) if GEN_CACHE.exists() else {}
def _sec_sha(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()
todo = {}
for cid in PAPERS:
    for sid, text in body_secs[cid].items():
        h = _sec_sha(text)
        if h not in gen_cache:
            todo[h] = text
GENERAL_NER = dict(model=GENERAL_NER_MODEL, python=GENERAL_NER_PY,
                   sections_cached=sum(_sec_sha(t) in gen_cache for c in PAPERS for t in body_secs[c].values()),
                   sections_new=len(todo), status="OK", reason="")
if todo:
    runner = WORK / "general_ner_runner.py"
    runner.write_text(
        "import sys, json\n"
        "inp, out, model = sys.argv[1:4]\n"
        "try:\n"
        "    import spacy\n"
        "    nlp = spacy.load(model)\n"
        "except Exception as e:\n"
        "    json.dump({'status': 'NOT_ATTEMPTED', 'reason': (type(e).__name__ + ': ' + str(e))[:300]}, open(out, 'w'))\n"
        "    sys.exit(0)\n"
        "data = json.load(open(inp, encoding='utf-8'))\n"
        "res = {h: [[e.start_char, e.end_char, e.label_] for e in nlp(t).ents] for h, t in data.items()}\n"
        "json.dump({'status': 'OK', 'spacy': spacy.__version__, 'spans': res}, open(out, 'w', encoding='utf-8'))\n",
        encoding="utf-8")
    gin, gout = WORK / "general_ner_in.json", WORK / "general_ner_out.json"
    gin.write_text(json.dumps(todo, ensure_ascii=False), encoding="utf-8")
    if gout.exists():
        gout.unlink()
    if not GENERAL_NER_PY:
        GENERAL_NER.update(status="NOT_ATTEMPTED", reason="no interpreter (set GENERAL_NER_PY)")
    else:
        try:
            p = subprocess.run([GENERAL_NER_PY, str(runner), str(gin), str(gout), GENERAL_NER_MODEL],
                               capture_output=True, text=True, timeout=600 + 120 * len(PAPERS))
            o = json.loads(gout.read_text(encoding="utf-8")) if gout.exists() else \
                dict(status="FAILED", reason=f"exit {p.returncode}: {redact(p.stderr[-300:])}")
        except Exception as ex:
            o = dict(status="FAILED", reason=f"{type(ex).__name__}: {ex}"[:300])
        GENERAL_NER.update(status=o["status"], reason=o.get("reason", ""), spacy=o.get("spacy", ""))
        if o["status"] == "OK":
            gen_cache.update(o["spans"])
            GEN_CACHE.write_text(json.dumps(gen_cache), encoding="utf-8")
print("general NER:", {k: v for k, v in GENERAL_NER.items() if k != "python"})
if GENERAL_NER["status"] != "OK":
    print("  -> GENERAL_NER_OVERLAP flag NOT applied this run. To enable: install spaCy's",
          GENERAL_NER_MODEL, "in a separate venv and set GENERAL_NER_PY to its python")

# ---------- N4 + N3b: mentions, span groups, flags ----------
mentions = []
for r in all_rows:
    m = {k: r.get(k, "") for k in FIELDS}
    m.update(SEC_CLASS.get((r["cid"], r["section_id"]), dict(section_class="UNKNOWN", section_class_rule="",
                                                             section_selection_status="UNKNOWN")))
    m["mention_id"] = stable_id("MEN_", r["candidate_id"])
    mentions.append(m)
SPAN_GROUPS = span_groups(mentions)

def _has_link(m):
    tl = clean_term(m["term"]).lower()
    l = LINK_BY_TERM.get(tl) or {}
    t = TYPED_LINKS.get((m["cdm_type_proposed"], tl)) or {}
    return bool(l.get("n_link_sources")) or t.get("ols4_status") == "FOUND"

for m in mentions:
    spans = gen_cache.get(_sec_sha(body_secs[m["cid"]][m["section_id"]]), []) if GENERAL_NER["status"] == "OK" else []
    glab = overlap_label(spans, int(m["char_start"]), int(m["char_end"]))
    m["general_ner_label"] = glab
    m["drop_reason"], m["rescued_from"] = junk_reason(m["term"], glab, m["norm_status"] == "VALID",
                                                      int(m["group_n_tools"]), _has_link(m))

# ---------- N6: entity keys and tables ----------
N_PROCESSED = sum(1 for c in PAPERS if TT[c])
NER_SCHEMA_VERSION = "notebook-draft-0.4"
for m in mentions:
    if m["drop_reason"]:
        m["entity_key"] = m["entity_id_paper"] = m["entity_id"] = ""
        continue
    k = entity_key_v4(m["cid"], m["term"], m["norm"], m["norm_vocab"])   # v4: ligature/case/abbreviation-aware
    m["entity_key"] = k
    m["entity_id"] = stable_id("ENT_", m["cdm_type_proposed"], k)
    m["entity_id_paper"] = stable_id("ENTP_", m["cid"], m["cdm_type_proposed"], k)
    m["run_tag"] = RUN_TAG

MENTION_FIELDS = (["mention_id"] + FIELDS + ["section_class", "section_class_rule", "section_selection_status",
                  "span_group_id", "group_tools", "group_n_tools", "group_types", "type_conflict",
                  "general_ner_label", "drop_reason", "rescued_from", "entity_key", "entity_id",
                  "entity_id_paper", "run_tag"])
dump_csv(OUT_DIR / "ner_mentions.csv", mentions, MENTION_FIELDS)

kept = [m for m in mentions if not m["drop_reason"]]
# v4: one ID or surface name under several CDM types stays split; listed for review, never merged silently
_types_by_key = {}
for m in kept:
    _types_by_key.setdefault(m["entity_key"], set()).add(m["cdm_type_proposed"])
_conflicts = sorted((k, sorted(t)) for k, t in _types_by_key.items() if len(t) > 1)
dump_csv(OUT_DIR / "type_conflicts.csv", [dict(entity_key=k, cdm_types="|".join(t), n_types=len(t)) for k, t in _conflicts],
         ["entity_key", "cdm_types", "n_types"])
print("type conflicts (kept split, for review):", len(_conflicts))
def _agg(ms):
    groups = {m["span_group_id"] for m in ms}
    by_cls = {}
    for m in ms:
        by_cls.setdefault(m["section_class"], set()).add(m["span_group_id"])
    tools = sorted({m["model"] for m in ms})
    surf = {}
    for m in ms:
        surf[clean_term(m["term"])] = surf.get(clean_term(m["term"]), 0) + 1
    return dict(frequency_tool_rows=len(ms), frequency=len(groups),
                frequency_by_section_class=json.dumps({k: len(v) for k, v in sorted(by_cls.items())}),
                tools="|".join(tools), n_tools=len(tools),
                type_conflict=int(any(int(m["type_conflict"]) for m in ms)),
                surface_forms="|".join(sorted(surf, key=lambda s: (-surf[s], s))),
                ids="|".join(sorted({f'{m["norm_vocab"]}:{m["norm"]}' for m in ms if m["norm"]})))

by_ent_paper, by_ent = {}, {}
for m in kept:
    by_ent_paper.setdefault(m["entity_id_paper"], []).append(m)
    by_ent.setdefault(m["entity_id"], []).append(m)

PAPER_TABLE_FIELDS = ["entity_id_paper", "entity_id", "cid", "cdm_type", "normalized_term", "entity_key",
                      "frequency", "frequency_tool_rows", "frequency_by_section_class", "section_class_status",
                      "n_sections", "section_ids", "tools", "n_tools", "type_conflict", "surface_forms", "ids",
                      "run_tag"]
ner_paper = []
for eid, ms in sorted(by_ent_paper.items()):
    a = _agg(ms)
    ner_paper.append(dict(entity_id_paper=eid, entity_id=ms[0]["entity_id"], cid=ms[0]["cid"],
                          cdm_type=ms[0]["cdm_type_proposed"], normalized_term=a["surface_forms"].split("|")[0],
                          entity_key=ms[0]["entity_key"], section_class_status="PROPOSED_CROSSWALK",
                          n_sections=len({m["section_id"] for m in ms}),
                          section_ids="|".join(sorted({m["section_id"] for m in ms})), run_tag=RUN_TAG, **a))

REVIEW_FIELDS = ["review_status", "curated_preferred_name", "curated_cdm_type", "curated_ids", "reviewer",
                 "review_note", "reviewed_at"]
CORPUS_TABLE_FIELDS = (["entity_id", "cdm_type", "normalized_term", "entity_key", "preferred_name",
                        "preferred_name_source", "synonyms", "definition", "definition_source", "definition_id",
                        "definition_status", "umls_cui", "uts_semantic_type", "mesh_id", "chebi_id",
                        "ncbi_gene_id", "ncbitaxon_id", "other_ids", "ols4_curie", "ols4_ontology", "ols4_status",
                        "bioportal_id", "bioportal_ontology", "omop_concepts", "taxon_rank", "taxon_lineage",
                        "taxonomy_status", "link_quality", "n_link_sources", "frequency", "frequency_tool_rows",
                        "document_frequency", "n_papers_processed", "document_frequency_ratio",
                        "frequency_by_section_class", "section_class_status", "tools", "n_tools", "type_conflict",
                        "surface_forms", "ids"] + REVIEW_FIELDS + ["run_tag", "built_at", "schema_version"])

# N7 review carry-over: read the previous build's review columns before overwriting
prev_review, CORPUS_CSV = {}, OUT_DIR / "ner_entities_corpus.csv"
if CORPUS_CSV.exists():
    for r in load_csv(CORPUS_CSV):
        if any(r.get(f) for f in REVIEW_FIELDS if f != "review_status") or \
           r.get("review_status", "PROVISIONAL_UNVALIDATED") != "PROVISIONAL_UNVALIDATED":
            prev_review[r["entity_id"]] = {f: r.get(f, "") for f in REVIEW_FIELDS}

def _ids_of(ids, vocab):
    return "|".join(sorted(i for i in ids.split("|") if i.upper().startswith(vocab.upper() + ":")))

ner_corpus, built_at = [], _now()
for eid, ms in sorted(by_ent.items()):
    a = _agg(ms)
    ct, nterm = ms[0]["cdm_type_proposed"], a["surface_forms"].split("|")[0]
    tl = nterm.lower()
    l = LINK_BY_TERM.get(tl) or {}
    t = TYPED_LINKS.get((ct, tl)) or {}
    if l.get("uts_name"):
        pn, pns = l["uts_name"], "UMLS:" + l["umls_cui"]
    elif t.get("ols4_label"):
        pn, pns = t["ols4_label"], "OLS4:" + t["ols4_curie"]
    elif l.get("bioportal_label"):
        pn, pns = l["bioportal_label"], "BioPortal:" + l["bioportal_id"]
    else:
        pn, pns = nterm, "SURFACE_FORM"
    tool_ids = a["ids"]
    known = {"MESH", "CHEBI", "NCBIGENE", "NCBITAXON", "UMLS"}
    has_onto = bool(l.get("n_link_sources")) or t.get("ols4_status") == "FOUND"
    row = dict(entity_id=eid, cdm_type=ct, normalized_term=nterm, entity_key=ms[0]["entity_key"],
               preferred_name=pn, preferred_name_source=pns,
               synonyms="|".join(sorted(set(a["surface_forms"].split("|")) |
                                        set(filter(None, t.get("synonyms", "").split("|"))))),
               definition=t.get("definition", ""), definition_source=t.get("definition_source", ""),
               definition_id=t.get("definition_id", ""),
               definition_status=t.get("definition_status") or ("NOT_ATTEMPTED" if ct not in TYPE_ONTOLOGIES
                                                                else "NOT_FOUND"),
               umls_cui=l.get("umls_cui", ""), uts_semantic_type=l.get("uts_semantic_type", ""),
               mesh_id=_ids_of(tool_ids, "MESH"), chebi_id=_ids_of(tool_ids, "CHEBI"),
               ncbi_gene_id=_ids_of(tool_ids, "NCBIGene"),
               ncbitaxon_id=_ids_of(tool_ids, "NCBITaxon") or (t.get("ols4_curie", "") if t.get("ols4_ontology") == "NCBITaxon" else ""),
               other_ids="|".join(i for i in tool_ids.split("|") if i and i.split(":")[0].upper() not in known),
               ols4_curie=t.get("ols4_curie", ""), ols4_ontology=t.get("ols4_ontology", ""),
               ols4_status=t.get("ols4_status") or "NOT_ATTEMPTED",
               bioportal_id=l.get("bioportal_id", ""), bioportal_ontology=l.get("bioportal_ontology", ""),
               omop_concepts=l.get("omop_concepts", ""),
               taxon_rank=t.get("taxon_rank", ""), taxon_lineage=t.get("taxon_lineage", ""),
               taxonomy_status=t.get("taxonomy_status") or ("NOT_ATTEMPTED" if ct == "ORGANISM" else "NOT_APPLICABLE"),
               link_quality=("TOOL_ID_AND_ONTOLOGY" if tool_ids and has_onto else "TOOL_ID_ONLY" if tool_ids
                             else "ONTOLOGY_ONLY" if has_onto else "SURFACE_ONLY"),
               n_link_sources=(int(l.get("n_link_sources") or 0) + (t.get("ols4_status") == "FOUND")),
               document_frequency=len({m["cid"] for m in ms}), n_papers_processed=N_PROCESSED,
               section_class_status="PROPOSED_CROSSWALK", run_tag=RUN_TAG, built_at=built_at,
               schema_version=NER_SCHEMA_VERSION, **a)
    row["document_frequency_ratio"] = round(row["document_frequency"] / max(N_PROCESSED, 1), 4)
    row.update({f: "" for f in REVIEW_FIELDS}, review_status="PROVISIONAL_UNVALIDATED")
    row.update(prev_review.get(eid, {}))           # curated_* columns only; machine values untouched
    ner_corpus.append(row)
review_orphans = [dict(entity_id=e, **v) for e, v in prev_review.items() if e not in by_ent]
dump_csv(OUT_DIR / "ner_entities_paper.csv", ner_paper, PAPER_TABLE_FIELDS)
dump_csv(OUT_DIR / "ner_entities_corpus.csv", ner_corpus, CORPUS_TABLE_FIELDS)
if review_orphans:
    dump_csv(OUT_DIR / "review_orphans.csv", review_orphans, ["entity_id"] + REVIEW_FIELDS)

# ---------- reports: junk filter + consensus ----------
from collections import Counter as _C
flagged = [m for m in mentions if m["drop_reason"]]
def _examples(rows, n=30):
    return [dict(model=m["model"], cdm_type=m["cdm_type_proposed"], term=m["term"], reason=m["drop_reason"] or
                 ("rescued:" + m["rescued_from"] if m["rescued_from"] else "kept")) for m in rows[:n]]
junk_report = dict(status="PROVISIONAL_UNVALIDATED", general_ner=GENERAL_NER, mentions=len(mentions),
                   flagged=len(flagged), kept=len(kept),
                   by_reason=dict(_C(m["drop_reason"] for m in flagged)),
                   by_reason_tool={f"{k[0]}|{k[1]}": v for k, v in _C((m["drop_reason"], m["model"]) for m in flagged).items()},
                   by_reason_type={f"{k[0]}|{k[1]}": v for k, v in _C((m["drop_reason"], m["cdm_type_proposed"]) for m in flagged).items()},
                   rescued=dict(_C(m["rescued_from"] for m in mentions if m["rescued_from"])),
                   flagged_examples=_examples(flagged[::max(1, len(flagged) // 30)]),
                   kept_examples=_examples(kept[::max(1, len(kept) // 30)]),
                   watch="taxa named after people (Escherichia, Shigella) can overlap PERSON; check rescued examples")
_write_redacted(OUT_DIR / "junk_filter_report.json", junk_report)
consensus = dict(span_groups=len(SPAN_GROUPS),
                 groups_by_n_tools=dict(_C(len(g["tools"]) for g in SPAN_GROUPS.values())),
                 type_conflict_groups=sum(len(g["types"]) > 1 for g in SPAN_GROUPS.values()),
                 tool_weights="NOT_AVAILABLE - weights need per-tool, per-type scores from the gold sample (Cell 11b)")
_write_redacted(OUT_DIR / "consensus_report.json", consensus)
print("mentions:", len(mentions), "| flagged:", junk_report["by_reason"], "| kept:", len(kept))
print("span groups:", consensus["span_groups"], "| by n tools:", consensus["groups_by_n_tools"],
      "| type conflicts:", consensus["type_conflict_groups"])
print("entities: paper", len(ner_paper), "| corpus", len(ner_corpus), "| review rows carried:",
      sum(1 for r in ner_corpus if r["entity_id"] in prev_review), "| orphaned reviews:", len(review_orphans))

# ---------- N7: build checks ----------
STATUS_VALUES = {"FOUND", "NOT_FOUND", "LOOKUP_FAILED", "NOT_ATTEMPTED", "NOT_APPLICABLE"}
kept_secs = {(r["cid"], r["section_id"]) for r in SEL_LOG if r["kept"]}
corpus_ids = {r["entity_id"] for r in ner_corpus}
def _bad(rows, pred):
    return [r for r in rows if pred(r)]
CHECKS = [
    ("01_mention_slice_exact", _bad(mentions, lambda m: body_secs[m["cid"]][m["section_id"]]
                                   [int(m["char_start"]):int(m["char_end"])] != m["term"])),
    ("02_mention_id_unique", [k for k, v in _C(m["mention_id"] for m in mentions).items() if v > 1]),
    ("03_every_mention_has_span_group", _bad(mentions, lambda m: not m.get("span_group_id"))),
    ("04_drop_reason_allowed", _bad(mentions, lambda m: m["drop_reason"] not in DROP_REASONS)),
    ("05_kept_have_entity_dropped_have_none", _bad(mentions, lambda m: bool(m["drop_reason"]) == bool(m["entity_id"]))),
    ("06_cdm_type_valid_or_unmapped", _bad(mentions, lambda m: m["cdm_type_proposed"] not in CDM_TYPES
                                           and m["mapping_status"] != "UNMAPPED")),
    ("07_mentions_only_from_selected_sections", _bad(mentions, lambda m: (m["cid"], m["section_id"]) not in kept_secs)),
    ("08_mentions_only_from_run_papers", _bad(mentions, lambda m: m["cid"] not in PAPERS)),
    ("09_entity_ids_unique", [k for k, v in _C(r["entity_id"] for r in ner_corpus).items() if v > 1] +
                             [k for k, v in _C(r["entity_id_paper"] for r in ner_paper).items() if v > 1]),
    ("10_paper_entities_resolve", _bad(ner_paper, lambda r: r["entity_id"] not in corpus_ids)),
    ("11_frequency_bounds", _bad(ner_corpus, lambda r: not (1 <= r["frequency"] <= r["frequency_tool_rows"]
                                                           and 1 <= r["document_frequency"] <= N_PROCESSED))),
    ("12_status_values_allowed", _bad(ner_corpus, lambda r: not {r["definition_status"], r["taxonomy_status"],
                                                                 r["ols4_status"]} <= STATUS_VALUES)),
    ("13_definitions_have_source_and_id", _bad(ner_corpus, lambda r: bool(r["definition"]) and
                                               not (r["definition_source"] and r["definition_id"]))),
]
template_check = "SKIPPED: " + str(NER_TEMPLATE_YAML) + " not on disk (draft schema " + NER_SCHEMA_VERSION + ")"
if NER_TEMPLATE_YAML.exists():
    import yaml
    _tpl = yaml.safe_load(NER_TEMPLATE_YAML.read_text(encoding="utf-8")) or {}
    _want = {t: [f["name"] if isinstance(f, dict) else f for f in (spec or {}).get("fields", [])]
             for t, spec in (_tpl.get("tables") or {}).items()}
    _have = dict(ner_mentions=MENTION_FIELDS, ner_entities_paper=PAPER_TABLE_FIELDS,
                 ner_entities_corpus=CORPUS_TABLE_FIELDS)
    diff = {t: dict(missing=sorted(set(_want.get(t, [])) - set(f)), extra=sorted(set(f) - set(_want.get(t, []))))
            for t, f in _have.items() if t in _want}
    CHECKS.append(("14_template_fields_match", [t for t, d in diff.items() if d["missing"] or d["extra"]]))
    template_check = diff
validation = dict(status="PASS" if all(not v for _, v in CHECKS) else "FAIL", built_at=built_at,
                  checks={k: dict(violations=len(v), examples=[str(x)[:200] for x in v[:5]]) for k, v in CHECKS},
                  template=template_check, schema_version=NER_SCHEMA_VERSION)
_write_redacted(OUT_DIR / "ner_validation_report.json", validation)
for k, v in CHECKS:
    print(f"  {k:45s} {'ok' if not v else f'FAIL ({len(v)})'}")
assert validation["status"] == "PASS", "N7 build checks failed - see ner_validation_report.json"

## 10. Baseline overlap (Cell 9)

Five-paper-only check (skipped with a logged reason when inputs are missing or `FULL_CORPUS`):
surface-form overlap of this run against the verified `five_paper_sample` hunflair2 and scispacy
baselines, with novel contributions broken down per tool.


In [ ]:
# ===== Cell 9: overlap / contribution analysis vs existing baselines =====
# v3: five-paper-only check; skipped with a logged reason when inputs are missing or FULL_CORPUS
if five_paper_check_ready("baseline_overlap", [ROOT / "staging/five_paper_sample/hunflair2/hunflair2_candidates.csv",
                                               ROOT / "staging/five_paper_sample/scispacy/scispacy_candidates.csv"]) \
        and all(any(ART_MD.glob(f"{cid}_*.md")) for cid in PAPERS):
    def baseline_flatten(md_path):
        """Baseline paper text: harmonized markdown with blank-line runs collapsed to two spaces and newlines to single spaces (verified convention; HTML comments kept)."""
        t = md_path.read_text(encoding="utf-8")
        t = re.sub(r"\n\n+", "  ", t)
        return re.sub(r"\n", " ", t)

    # load baselines
    def load_candidates(path, cols):
        """Load a baseline candidates CSV into minimal dicts (model, term, label, cid, char_start, char_end)."""
        return [dict(model=c[cols["model"]], term=c[cols["term"]], label=c[cols["label"]],
                     cid=c[cols["cid"]], char_start=int(c[cols["cs"]]), char_end=int(c[cols["ce"]]))
                for c in load_csv(path)]

    hf_base = load_candidates(ROOT / "staging/five_paper_sample/hunflair2/hunflair2_candidates.csv",
                              dict(model="model", term="term", label="cdm_type_proposed", cid="cid", cs="char_start", ce="char_end"))
    sc_base = load_candidates(ROOT / "staging/five_paper_sample/scispacy/scispacy_candidates.csv",
                              dict(model="model", term="term", label="cdm_type_proposed", cid="cid", cs="char_start", ce="char_end"))

    # paper-level flattened baselines
    BL = {}
    for cid in PAPERS:
        _m = next(p.name[len(cid) + 1:] for p in ART_MD.glob(f"{cid}_" + "*.md"))
        src = ART_MD / f"{cid}_{_m}"
        BL[cid] = baseline_flatten(src)

    def spanset(rows, texts):
        """Baseline rows to a set of (cid, term slice, label) resolved against the flattened paper text."""
        s = set()
        for r in rows:
            t = texts.get(r["cid"])
            if t is None: continue
            s.add((r["cid"], t[r["char_start"]:r["char_end"]], r["label"]))
        return s

    base_spans = spanset(hf_base + sc_base, BL)

    # our spans — SECTION-level: normalize by (cid, normalized term-lower, cdm type) since
    # baselines are paper-level and section offsets differ (comments kept). Use surface match.
    def section_spans(rows):
        """Our section-level rows to a set of (cid, term, cdm_type); baselines are paper-level, so comparison is on surface form."""
        s = set()
        for r in rows:
            s.add((r["cid"], r["term"], r["cdm_type_proposed"]))
        return s

    ours = section_spans(all_rows)
    base = set()
    for cid, term, label in base_spans:
        base.add((cid, term, label))
    # term-level fuzzy pass: strip whitespace defects
    norm = lambda s: " ".join(s.split())
    ours_n = {(cid, norm(t), l) for cid, t, l in ours}
    base_n = {(cid, norm(t), l) for cid, t, l in base}
    overlap = ours_n & base_n
    novel = ours_n - base_n
    print(f"our unique (paper, term, type): {len(ours_n)}")
    print(f"baseline unique (paper, term, type): {len(base_n)}")
    print(f"overlap: {len(overlap)}  novel(non-overlapping): {len(novel)}")
    print(f"novel ratio: {len(novel) / max(len(ours_n),1):.1%}")
    by_tool_novel = {}
    for r in all_rows:
        k = (r["cid"], norm(r["term"]), r["cdm_type_proposed"])
        if k in novel:
            by_tool_novel[r["model"]] = by_tool_novel.get(r["model"], 0) + 1
    print("novel contributions by tool:", dict(sorted(by_tool_novel.items(), key=lambda x: -x[1])))
    novel_sample = sorted(novel)[:40]
    print("sample novel terms:", novel_sample[:15])

## 11. v3 vs v2 comparison and traceability (Cell 10)

Per-model row diffs against the read-only v2 CSVs (added / removed / label-or-ID changes),
duplicate-id counts, slice health, coordinate conflicts and linking coverage. The hard traceability
check asserts that every v3 row slice-verifies against the original section text, matches its
candidate_id formula and round-trips through `ToolText.to_orig`.


In [ ]:
# ===== Cell 10: v3 vs v2 comparison + traceability check =====
# v2 = read-only per-model CSVs in V2_DIR (same 5 papers, same original-text offsets).
# Differences are explained at row level: candidate_id sets per model (added / removed).
import glob as _glob

CMP = None
_v2_csvs = sorted(_glob.glob(str(V2_DIR / "*_candidates.csv")))
if five_paper_check_ready("v3_vs_v2_comparison", [V2_DIR / "newtools_candidates.csv"] + [Path(p) for p in _v2_csvs[:1]]):
    v2_rows = []
    for p in _v2_csvs:
        if not Path(p).name.startswith("newtools_"):
            v2_rows.extend(load_csv(p))
    v2_combined_models = sorted({r["model"] for r in load_csv(V2_DIR / "newtools_candidates.csv")})
    v3_rows = all_rows

    def span_health(rows):
        """Per model: row count, slice mismatches and partial-word spans, all checked against the parsed section text."""
        out = {}
        for r in rows:
            m = out.setdefault(r["model"], dict(n=0, slice_mismatch=0, partial_word=0))
            t = secs[r["cid"]][r["section_id"]]
            s, e = int(r["char_start"]), int(r["char_end"])
            m["n"] += 1
            m["slice_mismatch"] += t[s:e] != r["term"]
            m["partial_word"] += partial_word(t, s, e)
        return out

    def dup_ids(rows):
        """Number of duplicate candidate_id rows beyond the first occurrence."""
        c = {}
        for r in rows:
            c[r["candidate_id"]] = c.get(r["candidate_id"], 0) + 1
        return sum(v - 1 for v in c.values() if v > 1)

    def conflicts(rows):
        """Exact-coordinate spans carrying more than one CDM type or more than one distinct normalization."""
        by = {}
        for r in rows:
            by.setdefault((r["cid"], r["section_id"], r["char_start"], r["char_end"]), []).append(r)
        tc = sum(1 for rs in by.values() if len({x["cdm_type_proposed"] for x in rs}) > 1)
        idc = sum(1 for rs in by.values() if len({clean_norm(x.get("norm_raw", x["norm"]))[0] for x in rs} - {""}) > 1)
        return dict(exact_spans=len(by), type_conflict_spans=tc, id_conflict_spans=idc)

    def trace_check(rows):
        """Every row must slice-verify, match its candidate_id formula and round-trip through ToolText.to_orig; returns counts plus examples."""
        bad = []
        for r in rows:
            t = secs[r["cid"]][r["section_id"]]; s, e = int(r["char_start"]), int(r["char_end"])
            ok = (t[s:e] == r["term"]
                  and r["candidate_id"] == f'{r["model"]}|{r["cid"]}|{r["section_id"]}|{s}|{e}'
                  and TT[r["cid"]][r["section_id"]].to_orig(int(r["tool_char_start"]), int(r["tool_char_end"])) == (s, e))
            if not ok: bad.append(r["candidate_id"])
        return dict(rows=len(rows), untraceable=len(bad), examples=bad[:5])

    models = sorted({r["model"] for r in v2_rows} | {r["model"] for r in v3_rows})
    per_model = {}
    for m in models:
        a = {r["candidate_id"]: r for r in v2_rows if r["model"] == m}
        b = {r["candidate_id"]: r for r in v3_rows if r["model"] == m}
        added, removed = sorted(set(b) - set(a)), sorted(set(a) - set(b))
        changed = sorted(k for k in set(a) & set(b)
                         if (a[k]["label_raw"], a[k]["norm_raw"], a[k]["cdm_type_proposed"]) !=
                            (b[k]["label_raw"], b[k]["norm_raw"], b[k]["cdm_type_proposed"]))
        per_model[m] = dict(v2=len(a), v3=len(b), diff=len(b) - len(a), added=len(added),
                            removed=len(removed), label_or_id_changed=len(changed),
                            in_v2_combined_csv=m in v2_combined_models,
                            v3_tool_status=next((st["status"] for st in TOOL_STATUS.values()
                                                 if m in st.get("models", {})), "NOT_REGISTERED"),
                            examples_added=[b[k]["term"] for k in added[:5]],
                            examples_removed=[a[k]["term"] for k in removed[:5]])

    def link_cov(path):
        """Coverage of a linking CSV: total terms, covered terms and per-source hit counts; None when the file is missing."""
        if not Path(path).exists():
            return None
        rows = load_csv(path)
        return dict(terms=len(rows),
                    covered=sum(1 for r in rows if int(r["n_link_sources"] or 0) > 0),
                    umls=sum(1 for r in rows if r["umls_cui"]), bioportal=sum(1 for r in rows if r["bioportal_id"]),
                    ols=sum(1 for r in rows if r["ols_curie"]), omop=sum(1 for r in rows if r["omop_concepts"]))

    def ds(path):
        """Load a JSON file if it exists, else None (used for v2 vs v3 dedup stats)."""
        return json.loads(Path(path).read_text(encoding="utf-8")) if Path(path).exists() else None

    CMP = dict(v2_dir=str(V2_DIR), per_model=per_model,
               totals=dict(v2_per_model_csvs=len(v2_rows), v3=len(v3_rows),
                           v2_combined_csv=len(load_csv(V2_DIR / "newtools_candidates.csv"))),
               duplicate_candidate_ids=dict(v2=dup_ids(v2_rows), v3=dup_ids(v3_rows)),
               span_health=dict(v2=span_health(v2_rows), v3=span_health(v3_rows)),
               conflicts=dict(v2=conflicts(v2_rows), v3=conflicts(v3_rows)),
               traceability_v3=trace_check(v3_rows),
               linking_coverage=dict(v2=link_cov(V2_DIR / "newtools_linking.csv"),
                                     v3=link_cov(OUT_DIR / "newtools_linking.csv")),
               dedup_stats=dict(v2=ds(V2_DIR / "dedup_stats.json"), v3=ds(OUT_DIR / "dedup_stats.json")),
               llm_occurrence=dict(first_match=LLM_STATS["first_match_rows"], expanded=LLM_STATS["expanded_rows"],
                                   expanded_by_type=LLM_STATS["expanded_by_type"]))
    _write_redacted(OUT_DIR / "v3_vs_v2_comparison.json", CMP)
    print(f"{'model':24s} {'v2':>6s} {'v3':>6s} {'diff':>6s} {'+':>5s} {'-':>5s} {'chg':>4s}  status")
    for m, d in per_model.items():
        print(f"{m:24s} {d['v2']:6d} {d['v3']:6d} {d['diff']:6d} {d['added']:5d} {d['removed']:5d} "
              f"{d['label_or_id_changed']:4d}  {d['v3_tool_status']}{'' if d['in_v2_combined_csv'] else '  (not in v2 combined CSV)'}")
    for k in ("totals", "duplicate_candidate_ids", "conflicts", "traceability_v3", "linking_coverage", "llm_occurrence"):
        print(k, json.dumps(CMP[k]))
    assert CMP["traceability_v3"]["untraceable"] == 0, "v3 rows failed the original-text traceability check"
    assert all(v["slice_mismatch"] == 0 for v in CMP["span_health"]["v3"].values())

## 12. Evaluation status (Cell 11)

Accuracy stays **UNKNOWN** until the independent, adjudicated human gold labels in
`staging/ner_audit` exist. The dev/test split is used unchanged, no tool output is ever treated as
gold, and metrics are reported only for units a human marked DONE.


In [ ]:
# ===== Cell 11: evaluation status (independent gold only) =====
# The 25 review units and the paper-level dev/test split in staging/ner_audit are used
# unchanged (gold_units.csv is never regenerated here). No tool output is used as gold.
# Metrics are reported only for units a human marked DONE after adjudication; tune on dev only.
# Host note: staging/ner_audit/ is not on every machine this notebook runs on; when it is
# absent the cell reports UNKNOWN instead of crashing (the gold standard itself is unchanged).
AUDIT_DIR = ROOT / "staging" / "ner_audit"
UNITS_CSV = AUDIT_DIR / "gold_units.csv"
if not UNITS_CSV.exists():
    units, n_gold = [], 0
    print(f"gold standard not on this host: {UNITS_CSV} is absent - no evaluation is possible here")
else:
    units = load_csv(UNITS_CSV)
gold_path = AUDIT_DIR / "gold_spans.csv"
n_gold = len(load_csv(gold_path)) if gold_path.exists() else 0
n_done = sum(u["review_status"].upper() == "DONE" for u in units)
print(f"review units: {len(units)} (dev {sum(u['split']=='dev' for u in units)}, "
      f"test {sum(u['split']=='test' for u in units)}) | DONE: {n_done} | gold rows: {n_gold}")
if n_done == 0 or n_gold == 0:
    print("ACCURACY: UNKNOWN - no independent, adjudicated human annotation yet. No P/R/F1 reported.")
else:
    cp = subprocess.run([sys.executable, str(AUDIT_DIR / "evaluate.py"), "--gold", str(gold_path),
                         "--units", str(AUDIT_DIR / "gold_units.csv"),
                         "--pred", str(OUT_DIR / "newtools_candidates.csv"),
                         "--split", "dev", "--out", str(OUT_DIR / "metrics_dev")],
                        capture_output=True, text=True)
    print(cp.stdout or cp.stderr)

## 13. Gold sample, scoring hook and run manifest (Cell 11b, v4 N8)

Writes a CDM-type annotation guide draft, scores against `staging/ner_audit` gold once any unit is DONE, and records a per-paper run manifest.

In [ ]:
# ===== Cell 11b: gold sample + scoring hook (v4 N8) and run manifest =====
# N8: the gold set lives in staging/ner_audit (read-only here): gold_units.csv (25 locked dev/test units)
# and gold_spans.csv. This cell
#   1) writes ANNOTATION_GUIDE_CDM14.md to OUT_DIR: the draft guide re-based on the 14 CDM types, with
#      body site / specimen / host / cohort / exposure recorded as flags, not new types. It still needs
#      sign-off before anyone annotates; the original guide is not changed.
#   2) scores this run with staging/ner_audit/evaluate.py, but only when at least one unit is DONE.
#      Until then accuracy stays UNKNOWN and consensus has no tool weights.
# Run manifest: per-paper input sha256 and kept sections, tool status, model files - the record a
# per-paper resume (Phase 2) needs. Tools still run once over all papers in this notebook.
GUIDE_CDM14 = OUT_DIR / "ANNOTATION_GUIDE_CDM14.md"
GUIDE_CDM14.write_text("""# NER gold annotation guide (draft v0.3, CDM types) - NEEDS SIGN-OFF

Re-based from `staging/ner_audit/ANNOTATION_GUIDE.md` (v0.1). Workflow, span rules and file formats are
unchanged; only the type set changed. `gold_type` must be one of the 14 CDM types below, so gold labels
and predictions share one vocabulary and `evaluate.py` can score span + type directly.

## gold_type (the 14 CDM types)

| gold_type | Include | Exclude / note |
|---|---|---|
| `ORGANISM` | Named taxa at any rank incl. abbreviations (`F. nucleatum`), strains; host species (`human`, `mice`) | Collective nouns (`microbiota`, `bacteria`). Host species: add flag `host`. Rank in `note` (`rank=strain`). |
| `DISEASE` | Diseases, disorders, phenotypes (`colorectal cancer`, `PBC`, `T2D`) | Label the whole disease name only. |
| `CHEMICAL` | Drugs, nutrients, small molecules, metabolites (`butyrate`, `SCFA`, `bile acids`) | Metabolites: add flag `metabolite`. |
| `PROTEIN` | Proteins and enzymes (`fucosidase`, `BSH`) | |
| `GENOMICS` | Genes, gene families, marker genes used as genes (`bsh`, `16S rRNA gene` as a gene) | `16S rRNA sequencing` as an assay is `LAB_METHOD`. |
| `DNA_RNA` | Nucleic-acid entities that are not genes (`plasmid DNA`, `mRNA`) | |
| `CELL_TYPE` | Cell types (`T cells`, `colonocytes`) | |
| `PATHWAY` | Metabolic or signalling pathways, GO processes | GO cell components |
| `LAB_METHOD` | Wet-lab methods: extraction kits, primers, V-regions, sequencing platforms and assays (`MiSeq`, `515F`, `V4 region`, `qPCR`) | |
| `DATA_ANALYSIS` | Analysis software, pipelines, reference databases, statistical methods (`QIIME 2`, `LEfSe`, `SILVA`) | |
| `SAMPLE_METHOD` | Sample collection, storage and handling methods (`stool collected in RNAlater`, `stored at -80 C`) | The specimen itself is not a method: see flags. |
| `EXPERIMENTAL_METHOD` | Experimental procedures and models (`fecal microbiota transplantation`, `germ-free mice`) | |
| `CLINICAL` | Clinical measures, interventions and exposures (`BMI`, `probiotic supplementation`, `diet`) | Exposures: add flag `exposure`. |
| `STUDY_DESIGN` | Design terms (`case-control`, `cross-sectional`, `randomized controlled trial`) | |

## Flags (column `note`, `;`-separated) - gaps in the CDM, recorded, not new types

`body_site` (gut, oral cavity) and `specimen` (stool, saliva, biopsy) have no CDM type: label the span with
`gold_type` left blank and the flag in `note`; `evaluate.py` then ignores it for scoring but it is counted.
Other flags: `host`, `cohort` (study groups such as `CRC patients`), `metabolite`, `exposure`, `ambiguous`.

## Unchanged from v0.1
Workflow steps 1-6, span rules (maximal span, nested allowed, abbreviations as separate mentions, ligatures as
written, leave out citations/authors/figure refs/accessions), and gold_norm vocabularies (NCBI Taxonomy, MeSH or
MONDO, ChEBI, NCBI Gene or UniProt).
""", encoding="utf-8")
print("annotation guide (CDM types, needs sign-off):", GUIDE_CDM14)

gold_units = load_csv(GOLD_DIR / "gold_units.csv") if (GOLD_DIR / "gold_units.csv").exists() else []
done = [u for u in gold_units if u.get("review_status") == "DONE"]
GOLD_STATUS = dict(units=len(gold_units), units_done=len(done),
                   splits=dict(_C(u.get("split", "") for u in gold_units)),
                   scoring="NOT_RUN - no reviewed units (accuracy UNKNOWN)" if not done else "")
if done:
    metrics_dir = OUT_DIR / "metrics"
    p = subprocess.run([sys.executable, str(GOLD_DIR / "evaluate.py"), "--gold", str(GOLD_DIR / "gold_spans.csv"),
                        "--units", str(GOLD_DIR / "gold_units.csv"), "--pred", str(OUT_DIR / "newtools_candidates.csv"),
                        "--out", str(metrics_dir)], capture_output=True, text=True, timeout=600)
    GOLD_STATUS["scoring"] = f"exit {p.returncode}; metrics in {metrics_dir}" if p.returncode == 0 else \
        f"FAILED exit {p.returncode}: {p.stderr[-300:]}"
print("gold sample:", GOLD_STATUS)

# ---------- run manifest (Phase 2 groundwork) ----------
run_manifest = dict(
    run_tag=RUN_TAG, written_at=_now(), status="PROVISIONAL_UNVALIDATED", out_dir=str(OUT_DIR),
    papers={cid: dict(input=str(SEG_DIR / f"{cid}.md"), input_sha256=file_sha256(SEG_DIR / f"{cid}.md"),
                      sections_total=len(secs[cid]), sections_kept=len(TT[cid]),
                      mentions=sum(1 for m in mentions if m["cid"] == cid),
                      kept_mentions=sum(1 for m in kept if m["cid"] == cid)) for cid in PAPERS},
    section_decisions=dict(path=str(SEL_CSV), sha256=file_sha256(SEL_CSV) if SEL_CSV.exists() else ""),
    crosswalk=dict(version=XWALK_VERSION, status="PROPOSED_NOT_APPROVED"),
    mesh_taxon_lexicon=dict(path=str(MESH_TAXON_LEXICON), sha256=file_sha256(MESH_TAXON_LEXICON)),
    tool_status={k: {kk: vv for kk, vv in v.items() if kk not in ("log_tail",)} for k, v in TOOL_STATUS.items()},
    general_ner={k: v for k, v in GENERAL_NER.items() if k != "python"},
    llm=(dict(gguf=LLM_GGUF.name, gguf_sha256=LLM_GGUF_SHA256, server_build=LLM_SERVER_BUILD,
              note="local GGUF pin only; extraction backend/status refreshed by Cell 12L2")
         if "LLM_GGUF" in globals() else dict(status="NOT_RUN")),
    validation=validation["status"], gold=GOLD_STATUS,
    not_yet=["per-paper tool execution with resume (tools still run once over all papers)",
             "Hugging Face model revisions and Java tool versions pinned in this manifest"])
_write_redacted(OUT_DIR / "run_manifest.json", run_manifest)
print("run manifest:", OUT_DIR / "run_manifest.json")

## 14. Data catalogue (Cells 12a-12c, stage CAT)

Builds the catalogue defined in `notebooks/cdm_catalogue_schema.yaml` (0.1.0 DRAFT): studies, publications, files, sections, field values with evidence, controlled terms, flags, findings, catalogue entries and facets. Validates against the schema's 12 rules, then cuts an immutable DRAFT release (approved_by = NOT_APPROVED) under `demo_output/catalogue/releases/`. Curators record decisions in `demo_output/catalogue/review_log.csv`; the build only reads it.

In [ ]:
# ===== Cell 12a: data catalogue (stage CAT) - C1 studies, C2 publications + files, C3 sections =====
# Builds the Microbiome-CDM data catalogue defined in notebooks/cdm_catalogue_schema.yaml (0.1.0 DRAFT)
# from pipeline outputs. Reads only; writes only under OUT_DIR/catalogue/. Scope:
#   - study / publication / source_file / section: the whole corpus (482 projects.csv rows)
#   - values from paper text and NER content: the papers in THIS run (PAPERS); other studies show
#     those fields as NOT_ATTEMPTED until FULL_CORPUS = True
# DUPLICATE_RECORD rows are separate studies that share their canonical row's publication; the duplicate's
# own HMPAPER copy is not a publication (its text is processed once, under the canonical paper).
#
# 2026-10-28 REBUILD for corpus 2.2.0. The four legacy inputs this cell used to read are gone from the
# host (ROOT/projects.csv, audit/final/FINAL_DOWNLOAD_AUDIT.csv,
# segmented_sections_markdown/reports/eligibility_audit.csv, markdown_harmonization/.../provenance),
# so every shim below is derived from the surviving, authoritative exports - nothing is guessed:
#   PROJECTS       168000-samples-paper/projects.csv   482 rows  study spine (cell50 V03)
#   STUDIES        frozen_output/01_corpus/studies.csv 453 rows  HMPAPER_*_STUDY_001 ids, curated fields
#   PAPERS_META    data_catalogue/tables/papers.csv    453 rows  43 cols, keyed by paper_id
#   PAPERS_JSON    corpus_v2/data/papers.jsonl         453 rows  dedup / bioproject_link / integrity_checks
#   SEC_ROWS       corpus_v2/data/sections.jsonl     4,006 rows  section spine (order/type/tokens)
# Key decisions downstream cells depend on (each is forced by the data, not a preference):
#   - publication.corpus_id := paper_id (HMPAPER_*). corpus 2.2.0's own `corpus_id` field is the
#     RELEASE id (HMCV2) on every row, so it cannot key a publication; `paper_id` is what SEG_DIR
#     files, PAPERS and every NER artifact are keyed by.
#   - study.study_id := bioproject accession (cell50 V03: one study per projects.csv row = 482).
#     studies.csv's HMPAPER_*_STUDY_001 id is a different namespace and is never used as a key.
#   - section.section_id := "<paper_id>:<section_id_original>". section_id_original repeats across
#     papers (sec_preface x453), so the bare id cannot be a primary key. PUB_TEXT and
#     SECTION_DECISIONS accept both spellings, so Cells 12b-13c keep working unchanged.
#   - ELIG["final_category"] mirrors papers.jsonl inclusion_decision (pending 447 / exclude 6):
#     no curator decision has been recorded, so none is claimed here. catalogue_status=INCLUDED
#     means "a converted full text exists in corpus 2.2.0", not "eligibility approved".
import yaml
CAT_SCHEMA_PATH = ROOT / "src/notebooks" / "cdm_catalogue_schema.yaml"
CAT_SCHEMA = yaml.safe_load(CAT_SCHEMA_PATH.read_text(encoding="utf-8"))
CAT_VERSION = CAT_SCHEMA["schema_version"]
CAT_DIR = OUT_DIR / "catalogue"
CAT_DIR.mkdir(exist_ok=True)
CAT_CREATED_AT = _now()
CAT_RUN_ID = "CAT-" + CAT_CREATED_AT.replace(":", "").replace("-", "")[:15] + "Z"
NER_RUN_ID = f"NER-{RUN_TAG}-" + RUN_STARTED_AT.replace(":", "").replace("-", "")[:15] + "Z"
RELEASE_ID = "draft-" + CAT_CREATED_AT[:10] + "-" + CAT_RUN_ID[4:]
CAT = {}                                   # table name -> list of rows
CAT_FLAGS = []

def cat_stamp(rows, run_id=None):
    """Common fields on every row. pipeline_run rows keep their own run_id (it is their primary key)."""
    for r in rows:
        r.setdefault("run_id", run_id or CAT_RUN_ID)
        r.update(release_id=RELEASE_ID, schema_version=CAT_VERSION, created_at=CAT_CREATED_AT)
    return rows

def raise_flag(table, target, code, severity, detail, field="", raised_by="CAT"):
    CAT_FLAGS.append(dict(flag_id=stable_id("FLG_", table, target, code, field), target_table=table,
                          target_id=target, field=field, code=code, severity=severity, detail=detail,
                          raised_by=raised_by, flag_status="OPEN", resolution_review_id=""))

# ---------- inputs (read-only; the CSVs start with a byte-order mark) ----------
def load_csv_bom(p):
    with open(p, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))
STUDIES_CSV   = ROOT / "releases/frozen_output" / "01_corpus" / "studies.csv"
PROJECTS_SRC  = ROOT / "data/168000-samples-paper" / "projects.csv"
PAPERS_CSV    = ROOT / "releases/data_catalogue" / "tables" / "papers.csv"
PAPERS_JSONL  = ROOT / "pipelines/corpus_v2" / "data" / "papers.jsonl"
SECTIONS_JSONL = ROOT / "pipelines/corpus_v2" / "data" / "sections.jsonl"
STUDIES  = load_csv_bom(STUDIES_CSV) if STUDIES_CSV.exists() else []
PROJECTS = load_csv_bom(PROJECTS_SRC) if PROJECTS_SRC.exists() else []
PAPERS_META = {r["paper_id"]: r for r in (load_csv_bom(PAPERS_CSV) if PAPERS_CSV.exists() else [])}
PAPERS_JSON = {}
if PAPERS_JSONL.exists():
    for _line in open(PAPERS_JSONL, encoding="utf-8"):
        if _line.strip():
            _d = json.loads(_line)
            PAPERS_JSON[_d["paper_id"]] = _d
SEC_ROWS = []
if SECTIONS_JSONL.exists():
    for _line in open(SECTIONS_JSONL, encoding="utf-8"):
        if _line.strip():
            SEC_ROWS.append(json.loads(_line))
BP2PID = {p["bioproject"]: pid for pid, p in PAPERS_JSON.items() if p.get("bioproject")}
STUDY_BY_BP = {r["bioproject"]: r for r in STUDIES}
PROJ_BY_BP = {r["project"]: r for r in PROJECTS}
PROV_DIR = ROOT / "markdown_harmonization" / "artifacts" / "provenance"   # DEAD PATH (never read; reported below)
SEGV_DIR = ROOT / "segmentation" / "validation"                            # DEAD PATH (never read; reported below)
CAT_INPUT_STATUS = dict(studies=len(STUDIES), projects=len(PROJECTS), papers_meta=len(PAPERS_META),
                        papers_jsonl=len(PAPERS_JSON), sections_jsonl=len(SEC_ROWS),
                        provenance=PROV_DIR.exists(), segmentation=SEGV_DIR.exists())
print("catalogue inputs:", CAT_INPUT_STATUS)
if not (PROJECTS and STUDIES and PAPERS_META and PAPERS_JSON):
    raise RuntimeError("catalogue inputs missing on this host: need frozen_output/01_corpus/studies.csv, "
                       f"168000-samples-paper/projects.csv, data_catalogue/tables/papers.csv and "
                       f"corpus_v2/data/papers.jsonl under ROOT={ROOT} (got {CAT_INPUT_STATUS})")

# ---------- C0: canonical studies (bioproject -> canonical bioproject) ----------
# corpus 2.2.0 dedup: 5 duplicate groups / 11 papers, 6 non-canonical copies. The duplicate copy's
# bioproject maps to the canonical paper's bioproject; everything else maps to itself.
CANONICAL = {}
for s in PROJ_BY_BP:
    p = PAPERS_JSON.get(BP2PID.get(s, ""), {})
    if p and not p.get("is_canonical", True):
        _cp = PAPERS_JSON.get(p.get("canonical_paper_id") or "", {})
        CANONICAL[s] = _cp.get("bioproject") or s
    else:
        CANONICAL[s] = s

# ---------- C0b: ELIG / DL_AUDIT shims (the two tables Cells 12b and 13a read) ----------
# ELIG mirrors the paper-level eligibility state that corpus 2.2.0 actually records:
#   pending            447 canonical papers (inclusion_decision=pending, nothing decided)
#   excluded             6 duplicate copies (inclusion_decision=exclude, reason kept verbatim)
#   placeholder_no_full_text  30 projects.csv rows with no paper in the corpus
ELIG, DL_AUDIT = {}, {}
for s in PROJ_BY_BP:
    pid = BP2PID.get(s, "")
    p, pm, st = PAPERS_JSON.get(pid, {}), PAPERS_META.get(pid, {}), STUDY_BY_BP.get(s, {})
    dup = bool(pid) and not p.get("is_canonical", True)
    md = SEG_DIR / f"{pid}.md" if pid else None
    has_text = bool(md and md.exists())
    title = p.get("title") or pm.get("title", "")
    doi = p.get("DOI") or pm.get("doi", "")
    year = p.get("publication_year") or pm.get("publication_year", "")
    _au = p.get("authors") or []
    if not _au and pm.get("authors"):
        _au = [a.strip() for a in str(pm["authors"]).split(";") if a.strip()]
    first_author = _au[0] if _au else ""
    tmatch = (p.get("integrity_checks") or {}).get("title_matches_source", "")
    if dup:
        final_category, outcome = "excluded", "DUPLICATE_RECORD"
        reason = pm.get("exclusion_reason") or f"duplicate copy of {p.get('canonical_paper_id', '')}"
    elif not pid:
        final_category, outcome = "placeholder_no_full_text", "NO_PAPER_RECORD"
        reason = "projects.csv row with no paper record in corpus 2.2.0"
    else:
        final_category, outcome = "pending", ("FULL_TEXT_CONVERTED" if has_text else "NO_FULL_TEXT")
        reason = "" if has_text else "no converted full text in corpus 2.2.0"
    ELIG[s] = dict(expected_study_id=s, corpus_id=(f"DUPLICATE:{pid}" if dup else pid),
                   final_category=final_category, current_outcome=outcome,
                   exclusion_or_review_reason=reason, expected_title=title, expected_doi=doi,
                   expected_year=year, expected_first_author=first_author,
                   status="EXCLUDED" if dup else ("NO_PAPER" if not pid else "PENDING"))
    DL_AUDIT[s] = dict(expected_study_id=s, expected_title=title, expected_doi=doi, expected_year=year,
                       expected_first_author=first_author, link=p.get("full_text_url") or "",
                       matched_local_file=(f"segmented_sections_markdown/{pid}.md" if has_text else ""),
                       doc_title_matched={"pass": "True", "fail": "False"}.get(tmatch, ""),
                       doc_doi_in_header="", publication_identity="",
                       final_status="DUPLICATE_RECORD" if dup else ("NO_LOCAL_TEXT" if not pid else "FULL_TEXT_PRESENT"),
                       canonical_publication_row=CANONICAL[s],
                       evidence_and_reason=(p.get("bioproject_link") or {}).get("link_evidence", ""))

# ---------- C1: studies ----------
ARCHIVE = {"PRJEB": ("ENA", "https://www.ebi.ac.uk/ena/browser/view/{}"),
           "PRJNA": ("NCBI_SRA", "https://www.ncbi.nlm.nih.gov/bioproject/{}"),
           "PRJDB": ("DDBJ", "https://ddbj.nig.ac.jp/resource/bioproject/{}")}
def catalogue_status(study_id):
    """INCLUDED = a converted full text exists for the canonical publication of this study."""
    canon_pid = BP2PID.get(CANONICAL[study_id], "")
    if not canon_pid:
        return "NO_FULL_TEXT"
    return "INCLUDED" if (SEG_DIR / f"{canon_pid}.md").exists() else "HELD_FOR_REVIEW"

STUDY = {}
for p in PROJECTS:
    s = p["project"]
    arch, url = ARCHIVE[s[:5]]
    pid = BP2PID.get(s, "")
    e, a = ELIG[s], DL_AUDIT[s]
    STUDY[s] = dict(study_id=s, archive=arch, archive_url=url.format(s),
                    title=a["expected_title"], title_status="NOT_ATTEMPTED",
                    catalogue_status=catalogue_status(s), csv_row_status=a["final_status"],
                    notes=p.get("notes", ""), n_runs="", n_publications=0,
                    audit_summary="{}", open_flags="{}")
    for _f, _spec in ((k, v) for k, v in CAT_SCHEMA["tables"]["study"]["fields"].items() if v.get("curated")):
        STUDY[s][_f] = (json.dumps([]) if str(_spec.get("type", "")).startswith("list")
                        else json.dumps(dict(value=None, term_id=None, status="NOT_ATTEMPTED", source=None, value_id=None)))
    if a["final_status"] == "DUPLICATE_RECORD":
        raise_flag("study", s, "DUPLICATE_RECORD", "INFO",
                   f"shares the publication of {CANONICAL[s]} (canonical paper "
                   f"{PAPERS_JSON.get(BP2PID.get(CANONICAL[s], ''), {}).get('paper_id', '')}); "
                   "text processed once under the canonical paper", raised_by="S0")
    if e["final_category"] == "placeholder_no_full_text":
        raise_flag("study", s, "NO_FULL_TEXT", "BLOCKER", e["exclusion_or_review_reason"], raised_by="S0")
    if not a["expected_doi"]:
        raise_flag("study", s, "NO_DOI", "WARNING", "no DOI on the paper record; identity rests on title + first author",
                   raised_by="S0")
    if a["doc_title_matched"] == "False":
        raise_flag("study", s, "METADATA_MISMATCH", "WARNING",
                   "corpus integrity check title_matches_source=fail; identity NOT_CHECKED until a curator looks",
                   raised_by="S0")
print("C1 studies:", len(STUDY), "|", dict(_C(x["catalogue_status"] for x in STUDY.values())),
      "| duplicate groups:", len({s for s in STUDY if CANONICAL[s] != s}))

# ---------- C2: publications, files, study links ----------
PUB, FILES, PROV = {}, {}, {}
PTYPE = {"primary_research": "RESEARCH_ARTICLE", "review": "REVIEW", "other": "OTHER"}
for pid, p in sorted(PAPERS_JSON.items()):
    if not p.get("is_canonical", True):
        continue                                          # duplicates link to the canonical publication below
    pm = PAPERS_META.get(pid, {})
    md = SEG_DIR / f"{pid}.md"
    ic = p.get("integrity_checks") or {}
    _au = p.get("authors") or []
    files, chosen = [], ""
    for fmt, key in (("JATS_XML", "original_xml_path"), ("PDF", "original_pdf_path")):
        _path = p.get(key)
        if _path and Path(_path).exists():
            files.append(dict(file_id=stable_id("SF_", pid, fmt, _path), corpus_id=pid, format=fmt,
                              local_path=str(_path), size_bytes=Path(_path).stat().st_size, origin="UNKNOWN",
                              retrieved_at="", chosen=False,
                              choice_reason="original source kept for audit; text and offsets come from the converted markdown",
                              converter="corpus 2.2.0 papers_txt->SEGMENT/SECTION", conversion_status="CONVERTED"))
    if md.exists():
        chosen = stable_id("SF_", pid, "TXT", "segmented_sections_markdown")
        files.append(dict(file_id=chosen, corpus_id=pid, format="TXT", local_path=str(md),
                          size_bytes=md.stat().st_size, origin="PMC" if (pm.get("pmcid") or p.get("PMCID")) else "UNKNOWN",
                          retrieved_at="", chosen=True,
                          choice_reason="chosen text: every NER/LLM artifact in this repository indexes this file",
                          converter="corpus 2.2.0 papers_txt->SEGMENT/SECTION", conversion_status="CONVERTED"))
    doi = p.get("DOI") or pm.get("doi", "")
    ptype = PTYPE.get(p.get("publication_type") or pm.get("article_type") or "", "OTHER")
    if doi.startswith("10.1038/sdata"):
        ptype = "DATA_DESCRIPTOR"
    tmatch = ic.get("title_matches_source", "")
    PUB[pid] = dict(corpus_id=pid, doi=doi, pmid=p.get("PubMed") or pm.get("pmid", ""),
                    pmcid=p.get("PMCID") or pm.get("pmcid", ""), title=p.get("title") or pm.get("title", ""),
                    journal=p.get("journal") or pm.get("journal", ""), year=p.get("publication_year") or pm.get("publication_year", ""),
                    first_author=(_au[0] if _au else ""), publication_type=ptype,
                    oa_status={"yes": "OPEN", "no": "CLOSED"}.get(str((p.get("bioproject_link") or {}).get("bundle_is_oa")), "UNKNOWN"),
                    licence="", identity_status={"pass": "VERIFIED_TITLE_AUTHOR"}.get(tmatch, "NOT_CHECKED"),
                    full_text_status="CONVERTED" if md.exists() else "NO_SOURCE",
                    body_status={"complete": "PASS", "partial": "PASS_WITH_WARNINGS"}.get(ic.get("body_text_completeness", ""), "NOT_CHECKED"),
                    segmentation_status={"yes": "OK", "no": "NEEDS_REVIEW"}.get(ic.get("section_hierarchy_preserved", ""), "NOT_RUN"),
                    chosen_file_id=chosen, text_sha256=p.get("raw_file_hash", ""),
                    n_sections=0, n_sections_selected=0, n_findings=0, n_entities=0)
    for f in files:
        FILES[f["file_id"]] = f
    if tmatch == "fail":
        raise_flag("publication", pid, "METADATA_MISMATCH", "WARNING",
                   "integrity check title_matches_source=fail; identity left NOT_CHECKED", field="title", raised_by="S1")
    raise_flag("publication", pid, "LICENCE_UNKNOWN", "INFO", "licence not looked up yet (Unpaywall / Europe PMC)")
    if not md.exists():
        raise_flag("publication", pid, "CONVERSION_BLOCKED", "BLOCKER", "no converted markdown for this publication", raised_by="S1")
    PROV[pid] = dict(status=("validated" if p.get("validation_status") == "COMPLETE" else "needs_review") if md.exists() else "no_full_text_source",
                     chosen_source=str(md) if md.exists() else "", chosen_format="md" if md.exists() else "",
                     source_sha256=p.get("raw_file_hash", ""),
                     all_sources=[dict(format=f["format"].lower().replace("jats_xml", "xml"), path=f["local_path"],
                                       sha256=f["file_id"]) for f in files],
                     warnings=[w for w in [ic.get("known_extraction_problems")] if w],
                     pipeline_version=f"corpus-2.2.0/{p.get('schema_version', '')}")

def _prov(study_id):
    """Provenance shim for Cells 13a/13b: the markdown_harmonization provenance dir is gone, so the
    same keys are served from corpus 2.2.0's papers.jsonl (status/chosen_source/all_sources/...)."""
    return PROV.get(BP2PID.get(study_id, ""), {})

def sec_key(cid, sid):
    """Catalogue section id: `<paper_id>:<section_id_original>`. The bare section_id_original repeats
    across papers (sec_preface x453), so study_field_value rows must carry the composite too."""
    return sid if isinstance(sid, str) and ":" in sid else f"{cid}:{sid}"

class _SecText(dict):
    """Section text keyed by the bare section_id_original; the catalogue's composite
    `<paper_id>:<section_id_original>` resolves to the same text, so Cells 12b-13c can slice with
    either spelling (SECTION rows carry the composite as their primary key)."""
    def _bare(self, k):
        """-> the bare key a composite key resolves to, or None."""
        if isinstance(k, str) and ":" in k:
            return k.split(":", 1)[1]
        return None

    def __missing__(self, k):
        _b = self._bare(k)
        if _b is not None and dict.__contains__(self, _b):
            return dict.__getitem__(self, _b)
        raise KeyError(k)

    # dict.get() and `in` bypass __missing__ (both are C-level), so the composite spelling must
    # be handled here too - otherwise Cells 12b-13c silently read empty text and V07 fails.
    def get(self, k, default=None):
        if dict.__contains__(self, k):
            return dict.__getitem__(self, k)
        _b = self._bare(k)
        if _b is not None and dict.__contains__(self, _b):
            return dict.__getitem__(self, _b)
        return default

    def __contains__(self, k):
        if dict.__contains__(self, k):
            return True
        _b = self._bare(k)
        return _b is not None and dict.__contains__(self, _b)

PUB_TEXT = {}                                             # paper_id -> {section_id: text} (markers removed)
for cid in PUB:
    if cid in secs:
        PUB_TEXT[cid] = _SecText(secs[cid])
    else:
        _md = SEG_DIR / f"{cid}.md"
        PUB_TEXT[cid] = _SecText(parse_sections(_md.read_text(encoding="utf-8"))) if _md.exists() else _SecText()

LINKS = []
for s in STUDY:
    canon = CANONICAL[s]
    cid = BP2PID.get(canon, "")
    if not cid or cid not in PUB:
        continue
    text = "\n".join(PUB_TEXT.get(cid, {}).values())
    in_text = bool(s) and s in text
    doi_l = (PUB[cid]["doi"] or "").lower()
    DL_AUDIT[s]["doc_doi_in_header"] = ("True" if doi_l and doi_l in text.lower()
                                        else "False" if doi_l else "")
    DL_AUDIT[s]["publication_identity"] = ("verified: document's own DOI" if doi_l and doi_l in text.lower()
                                           else "verified: title and first author" if DL_AUDIT[s]["doc_title_matched"] == "True"
                                           else "not verified")
    group = sorted(x for x in STUDY if CANONICAL[x] == canon)
    LINKS.append(dict(study_id=s, corpus_id=cid,
                      role="DATA_DESCRIPTOR" if PUB[cid]["publication_type"] == "DATA_DESCRIPTOR" else "PRIMARY",
                      link_source="ACCESSION_IN_TEXT" if in_text else "PROJECTS_CSV", accession_in_text=in_text,
                      evidence_value_id="", link_status="CONFIRMED" if in_text else "UNCONFIRMED",
                      shared_with=json.dumps([x for x in group if x != s])))
    if not in_text and PUB[cid]["full_text_status"] == "CONVERTED":
        raise_flag("study_publication", f"{s}|{cid}", "ACCESSION_NOT_IN_TEXT", "WARNING",
                   f"{s} does not appear verbatim in {cid}; link rests on the corpus's exact_bioproject_accession match")
print("C2 publications:", len(PUB), "| files:", len(FILES), "| study links:", len(LINKS),
      "| confirmed:", sum(l["link_status"] == "CONFIRMED" for l in LINKS))
_unlinked = [pid for pid in PUB if pid not in {l["corpus_id"] for l in LINKS}]
if _unlinked:
    print(f"  note: {len(_unlinked)} publication(s) with no projects.csv row (no study to link): {_unlinked}")

# ---------- C3: sections ----------
SEC_META = {(r["paper_id"], r["section_id_original"]): r for r in SEC_ROWS}
# SECTION_DECISIONS is keyed (paper_id, section_id_original); Cells 12b/13b look sections up with the
# catalogue's composite id, so register the composite as an alias of the same decision row.
for (_cid, _sid), _d in list(SECTION_DECISIONS.items()):
    SECTION_DECISIONS.setdefault((_cid, f"{_cid}:{_sid}"), _d)
XCLS = {"NOT_APPLICABLE_EXCLUDED": "BACKMATTER", "UNKNOWN_NO_S7_DECISION": "OTHER_BODY"}
# The S5 crosswalk (R01-R17) does not match the rule vocabulary the surviving decision table uses
# (R10_PRIMARY_SCIENTIFIC / R21_CONTAINER / R11_CONTEXT / R20_CONTAINER_WITH_CONTENT_CUE), so
# own_section_class() never fires on it. Where that happens the class falls back to the RECORDED
# section_type - a derived mapping, labelled PROPOSED, never an approved crosswalk.
S7_TYPE_CLASS = {"abstract": "TITLE_ABSTRACT", "introduction": "INTRODUCTION", "methods": "METHODS",
                 "results": "RESULTS", "discussion": "DISCUSSION", "conclusion": "CONCLUSION",
                 "references": "BACKMATTER", "acknowledgments": "BACKMATTER", "acknowledgements": "BACKMATTER",
                 "funding": "BACKMATTER", "author_contributions": "BACKMATTER", "conflict_of_interest": "BACKMATTER",
                 "supplementary": "BACKMATTER", "data_availability": "BACKMATTER", "ethics": "BACKMATTER",
                 "abbreviations": "BACKMATTER"}
SECTION = {}
_xwalk_miss = _no_dec = 0
for cid in sorted(PUB_TEXT):
    for sid, text in PUB_TEXT[cid].items():
        key = f"{cid}:{sid}"
        m = SEC_META.get((cid, sid), {})
        d = SECTION_DECISIONS.get((cid, sid))
        path = " > ".join(m.get("heading_path") or []) or (d or {}).get("heading_path", "") or sid
        if "(preamble)" in path:
            cls, rule = "PREAMBLE", f"PREAMBLE@{XWALK_VERSION}"
        else:
            cls, rule = own_section_class((d or {}).get("rule_ids", ""), (d or {}).get("heading", ""))
            if not cls:
                _t = ((d or {}).get("section_type") or m.get("section_type") or "").strip().lower()
                cls = S7_TYPE_CLASS.get(_t, "OTHER_BODY")
                rule = f"section_type:{_t}@xwalk-0.2-PROPOSED" if _t in S7_TYPE_CLASS else f"NO_MATCH@{XWALK_VERSION}"
                _xwalk_miss += 1
            if d is None:
                _no_dec += 1
        if BODY_ONLY and is_backmatter(path) and cls not in ("PREAMBLE",):
            cls = "BACKMATTER"
        _jev = (d or {}).get("jev_score", "")
        SECTION[key] = dict(section_id=key, section_id_original=sid, corpus_id=cid, section_path=path,
                            ordinal=int(m.get("section_order") or (d or {}).get("order") or 0),
                            section_class=XCLS.get(cls, cls),
                            template_fields=json.dumps([x for x in ((d or {}).get("rule_ids") or "").split(";") if x]),
                            selection_status=(d or {}).get("final_status") or "REVIEW_REQUIRED",
                            jev_relevance=(float(_jev) if str(_jev).strip() else ""),
                            n_tokens=int(m.get("token_count") or 0),
                            text_sha256=hashlib.sha256(text.encode("utf-8")).hexdigest())
for cid, p in PUB.items():
    ss = [x for x in SECTION.values() if x["corpus_id"] == cid]
    p["n_sections"] = len(ss)
    p["n_sections_selected"] = sum(x["selection_status"] in ("RELEVANT", "RELEVANT_CONTEXT") for x in ss)
print("C3 sections:", len(SECTION), "| by class:", dict(_C(x["section_class"] for x in SECTION.values())),
      f"| crosswalk fallbacks: {_xwalk_miss} | no S7 decision: {_no_dec}")


## 14b. LLM field extraction over the desired sections (Cells 12L1, 12L2)

Uses an OpenAI-compatible endpoint, by default FhGenie on-prem (`https://fhgenie.fraunhofer.de/v1`) with `deepseek-ai/DeepSeek-V4.1-Flash`. Set the token in `FCS_ONPREM_KEY` (or `FHGENIE_API_TOKEN`); without it these cells skip and the affected fields stay NOT_ATTEMPTED. Only S7-kept, non-backmatter sections whose class is routed to a field (`LLM_FIELD_SECTIONS`) are sent, and each window asks only for its routed fields. Every value needs a verbatim quote; kept values become PAPER_TEXT candidates in Cell 12b under the usual selection rules. FhGenie on-prem is a pilot without SLA: draft releases only, and tell the FhGenie team before large runs.

**Backends** (`NB_LLM_BACKEND`): `llama_cpp` (default) runs the small local Qwen3.5-4B GGUF on llama-server, as in Cell 7c, so text stays on this machine and no token is needed; `fhgenie` uses the FhGenie on-prem API. `NB_LLM_DISABLE=1` skips the step.

In [ ]:
# ===== Cell 12K: LLM backend and FhGenie API key (edit this cell) =====
# Replace <API_Key> with your FhGenie on-prem token (FhGenie UI -> User Info -> Create API-Token; valid 30 days).
# The key is only put into this process's environment (FCS_ONPREM_KEY); redact() removes it from every log and
# output. It IS saved in the .ipynb file while it is typed here: put <API_Key> back before sharing or copying
# the notebook. Choose "llama_cpp" to use the local Qwen3.5-4B model instead (no key needed).
FHGENIE_API_KEY = (os.environ.get("FCS_ONPREM_KEY")
                    or os.environ.get("FHGENIE_API_TOKEN")
                    or "<API_Key>")
LLM_BACKEND_CHOICE = "fhgenie"            # "fhgenie" or "llama_cpp"

os.environ["NB_LLM_BACKEND"] = LLM_BACKEND_CHOICE
if FHGENIE_API_KEY.strip() and FHGENIE_API_KEY.strip() != "<API_Key>":
    os.environ["FCS_ONPREM_KEY"] = FHGENIE_API_KEY.strip()
    print("FhGenie key set from this cell (value not shown)")
elif LLM_BACKEND_CHOICE == "fhgenie":
    if os.environ.get("FCS_ONPREM_KEY") or os.environ.get("FHGENIE_API_TOKEN"):
        print("Using the FhGenie key already in the environment (value not shown)")
    else:
        print("No FhGenie key: replace <API_Key> above, or the LLM cells skip and fields stay NOT_ATTEMPTED")
del FHGENIE_API_KEY
print("LLM backend:", LLM_BACKEND_CHOICE)

In [ ]:
# ===== Cell 12L1: OpenAI-compatible LLM client (FhGenie on-prem) - model choice, cache, tests =====
# FhGenie exposes an OpenAI-compatible API (LiteLLM) on the Fraunhofer Private Cloud. Pilot service, no SLA:
# use it for draft catalogue releases, not production. Tokens are created in the FhGenie UI (User Info ->
# Create API-Token) and expire after 30 days. Tell the FhGenie team (ServiceDesk) before long or large runs.
#   FCS_ONPREM_KEY or FHGENIE_API_TOKEN   the token, environment only; never printed, logged or written.
#   FHGENIE_BASE_URL   default https://fhgenie.fraunhofer.de/v1
#   FHGENIE_MODEL      default deepseek-ai/DeepSeek-V4.1-Flash (ids as listed by /v1/models; the older
#                      deepseek-ai/DeepSeek-V4-Flash-0731 also exists). If the id is not offered, the newest
#                      'deepseek...flash' model is used; otherwise the cell stops and lists what is available.
#   FHGENIE_TEMPERATURE default 0 (extraction; the opencode configs use 1.0 for chat/coding).
# DeepSeek V4 Flash is a reasoning model: its thinking can use up the output budget, so max_tokens is large
# and an empty answer with JSON in the reasoning text is accepted; an empty reply is retried.
# Responses are cached only on success (key: model, prompt version, schema, window text); failures are
# logged to _work/llm_catalogue_failures.json and retried on the next run. 401/403 stops the run.
import urllib.request, urllib.error
LLM_BASE_URL = os.environ.get("FHGENIE_BASE_URL", "https://fhgenie.fraunhofer.de/v1").rstrip("/")
LLM_WANTED_MODEL = os.environ.get("FHGENIE_MODEL", "deepseek-ai/DeepSeek-V4.1-Flash")
LLM_TEMPERATURE = float(os.environ.get("FHGENIE_TEMPERATURE", "0"))
LLM_TOKEN_VARS = ("FCS_ONPREM_KEY", "FHGENIE_API_TOKEN")
LLM_TIMEOUT_S, LLM_RETRIES, LLM_WORKERS = 180, 3, 4
if os.environ.get("NB_LLM_BACKEND", "").strip().lower() == "fhgenie":
    LLM_TIMEOUT_S = 300                                  # DeepSeek V4.1 Flash reasons first; 180 s timed out on 2/29 windows
# Backend: llama_cpp (default) = the small local Qwen3.5-4B GGUF on llama-server (same files as Cell 7c);
# text never leaves the machine and no token is needed. fhgenie = the FhGenie on-prem API above.
LLM_BACKEND = os.environ.get("NB_LLM_BACKEND", "llama_cpp").strip().lower()   # llama_cpp | fhgenie
LLM_BACKEND = {"llama_cpp": "llama_cpp"}.get(LLM_BACKEND, LLM_BACKEND)        # old name still accepted
LOCAL_GGUF = Path(os.environ.get("NB_LLM_GGUF") or str(ROOT / "models" / "Qwen3.5-4B-Q4_K_M.gguf"))
LOCAL_SERVER = Path(os.environ.get("NB_LLM_SERVER") or str(ROOT / "models" / "llama-bin" / "llama-server"))
LOCAL_SLOTS, LOCAL_SLOT_CTX = 2, 16384
LLM_MAX_TOKENS = 16000 if LLM_BACKEND == "fhgenie" else 3072
LLM_LOCAL_PROC = None
if LLM_BACKEND == "llama_cpp":
    LLM_WORKERS = LOCAL_SLOTS
LLM_CAT_CACHE = WORK / "llm_catalogue_cache.jsonl"
LLM_CAT_FAILURES = WORK / "llm_catalogue_failures.json"
SECRET_ENV_VARS = tuple(dict.fromkeys(tuple(SECRET_ENV_VARS) + LLM_TOKEN_VARS))   # redact() scrubs them

def llm_token():
    if LLM_BACKEND == "llama_cpp":
        return "local"                                # llama-server needs no key
    for n in LLM_TOKEN_VARS:
        v = os.environ.get(n, "").strip()
        if v:
            return v
    raise RuntimeError("set FCS_ONPREM_KEY (or FHGENIE_API_TOKEN) in the environment; the value is never shown")

class LLMAuthError(RuntimeError):
    pass

def _llm_http(method, path, body=None, timeout=LLM_TIMEOUT_S):
    token = llm_token()
    req = urllib.request.Request(LLM_BASE_URL + path, method=method,
                                 data=json.dumps(body).encode("utf-8") if body is not None else None,
                                 headers={"Authorization": "Bearer " + token, "Content-Type": "application/json",
                                          "User-Agent": "cdm-catalogue/0.1"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, json.loads(r.read().decode("utf-8"))
    except urllib.error.HTTPError as ex:
        detail = redact(ex.read().decode("utf-8", errors="replace")[:400])
        if ex.code in (401, 403):
            raise LLMAuthError(f"HTTP {ex.code}: token rejected or expired (tokens last 30 days). {detail}")
        return ex.code, {"error": detail}

def pick_model(available, wanted):
    """Exact id (case-insensitive) > id containing `wanted` > newest 'deepseek...flash' id > None."""
    low = {m.lower(): m for m in available}
    if wanted.lower() in low:
        return low[wanted.lower()]
    part = [m for m in available if wanted.lower() in m.lower()]
    if part:
        return sorted(part, key=len)[0]
    ds = [m for m in available if "deepseek" in m.lower() and "flash" in m.lower()]
    return sorted(ds, reverse=True)[0] if ds else None

THINK_RX = re.compile(r"<think>.*?</think>", re.S | re.I)
def parse_json_reply(text):
    """Model text -> dict. Strips <think> blocks and ``` fences, then takes the outermost {...}."""
    t = THINK_RX.sub("", text or "").strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.I).strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        a, b = t.find("{"), t.rfind("}")
        if a >= 0 and b > a:
            return json.loads(t[a:b + 1])
        raise

def locate_quote(text, quote):
    """-> (start, end) of quote in text: exact first, then whitespace-tolerant; None if absent.
    The stored evidence is always text[start:end], so it equals the slice even when spacing differed."""
    q = (quote or "").strip()
    if not q:
        return None
    i = text.find(q)
    if i >= 0:
        return i, i + len(q)
    toks = q.split()
    if not toks:
        return None
    m = re.search(r"\s+".join(re.escape(t) for t in toks), text)
    return (m.start(), m.end()) if m else None

LLM_MODE = None            # json_schema | json_object | text - the first one the server accepts
def chat_json(messages, schema, max_tokens=16000):
    """One chat call -> (parsed dict, raw text, mode). Downgrades response_format if the server rejects it."""
    global LLM_MODE
    modes = [LLM_MODE] if LLM_MODE else ["json_schema", "json_object", "text"]
    last = None
    for mode in modes:
        body = dict(model=LLM_MODEL, messages=messages, temperature=LLM_TEMPERATURE, max_tokens=max_tokens)
        if mode == "json_schema":
            body["response_format"] = {"type": "json_schema", "json_schema": {"name": "extraction", "strict": True, "schema": schema}}
        elif mode == "json_object":
            body["response_format"] = {"type": "json_object"}
        for attempt in range(LLM_RETRIES):
            code, d = _llm_http("POST", "/chat/completions", body)
            if code == 200:
                msg = (d.get("choices") or [{}])[0].get("message") or {}
                text = msg.get("content") or ""
                if not text.strip():                       # reasoning used the budget, or answer only in reasoning
                    text = msg.get("reasoning_content") or msg.get("reasoning") or ""
                if not text.strip():
                    last = "empty reply"
                    continue
                try:
                    out = parse_json_reply(text)
                    LLM_MODE = mode
                    return out, text, mode
                except (json.JSONDecodeError, ValueError):
                    last = "unparseable JSON"
                    continue
            last = f"HTTP {code}: {str(d.get('error', ''))[:200]}"
            if code == 400 and mode != "text":
                break                                     # server rejects this response_format -> next mode
            if code in (429, 500, 502, 503, 504) or code is None:
                time.sleep(2 + 4 * attempt)
                continue
            break
    raise LookupFailure(last or "no response")

# ---------- offline tests (no network) ----------
class TestLLMHelpers(unittest.TestCase):
    def test_pick_model(self):
        av = ["nvidia/GLM-5.3-NVFP4", "deepseek-ai/DeepSeek-V4-Flash-0731", "deepseek-ai/DeepSeek-V4.1-Flash"]
        self.assertEqual(pick_model(av, "deepseek-ai/DeepSeek-V4.1-Flash"), "deepseek-ai/DeepSeek-V4.1-Flash")
        self.assertEqual(pick_model(av, "deepseek-ai/deepseek-v4.1-flash"), "deepseek-ai/DeepSeek-V4.1-Flash")
        self.assertEqual(pick_model(av[:2], "deepseek-ai/DeepSeek-V4.1-Flash"), "deepseek-ai/DeepSeek-V4-Flash-0731")
        self.assertIsNone(pick_model(["nvidia/GLM-5.3-NVFP4"], "deepseek-ai/DeepSeek-V4.1-Flash"))
    def test_parse_json_reply(self):
        self.assertEqual(parse_json_reply('<think>x</think>```json\n{"items": []}\n```'), {"items": []})
        self.assertEqual(parse_json_reply('Here: {"items": [1]} done'), {"items": [1]})
    def test_locate_quote_exact_and_spacing(self):
        t = "Stool samples were collected from 120\nhealthy adults."
        self.assertEqual(locate_quote(t, "collected from 120 healthy"), (19, 45))
        s, e = locate_quote(t, "collected from 120 healthy")
        self.assertEqual(t[s:e], "collected from 120\nhealthy")
        self.assertIsNone(locate_quote(t, "collected from 130"))
    def test_token_is_redacted(self):
        had = "FCS_ONPREM_KEY" in os.environ
        os.environ.setdefault("FCS_ONPREM_KEY", "tok-test-123")
        tok = os.environ["FCS_ONPREM_KEY"]
        self.assertNotIn(tok, redact("Authorization: Bearer " + tok))
        if not had:
            del os.environ["FCS_ONPREM_KEY"]
_resL = unittest.TextTestRunner(verbosity=1, stream=sys.stdout).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestLLMHelpers))
TEST_RESULTS["llm_client"] = dict(ran=_resL.testsRun, ok=_resL.wasSuccessful())
assert _resL.wasSuccessful(), "LLM client tests failed"

# ---------- connect + choose the model ----------
def _free_port_local():
    import socket
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def start_local_qwen():
    """Start llama-server with the local Qwen GGUF (thinking off) on a free port; wait until /health is OK."""
    global LLM_BASE_URL, LLM_LOCAL_PROC
    if not (LOCAL_GGUF.exists() and LOCAL_SERVER.exists()):
        raise RuntimeError(f"local backend needs {LOCAL_GGUF} and {LOCAL_SERVER}")
    port = (globals().get("_free_port") or _free_port_local)()   # never a fixed port: :8080/:8089 may be busy
    log = open(WORK / "llama_server_catalogue.log", "w", encoding="utf-8")
    LLM_LOCAL_PROC = subprocess.Popen(
        [str(LOCAL_SERVER), "-m", str(LOCAL_GGUF), "-ngl", "99", "-c", str(LOCAL_SLOTS * LOCAL_SLOT_CTX),
         "--parallel", str(LOCAL_SLOTS), "--jinja", "--reasoning-budget", "0",
         "--host", "127.0.0.1", "--port", str(port)], stdout=log, stderr=subprocess.STDOUT)
    LLM_BASE_URL = f"http://127.0.0.1:{port}/v1"
    for _ in range(240):
        if LLM_LOCAL_PROC.poll() is not None:
            raise RuntimeError("llama-server exited - see _work/llama_server_catalogue.log")
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=5) as r:
                if r.status == 200:
                    return
        except Exception:
            pass
        time.sleep(1)
    stop_local_qwen()
    raise RuntimeError("llama-server did not become healthy in 240 s - see _work/llama_server_catalogue.log")

def stop_local_qwen():
    global LLM_LOCAL_PROC
    if LLM_LOCAL_PROC is not None and LLM_LOCAL_PROC.poll() is None:
        LLM_LOCAL_PROC.terminate()
        try:
            LLM_LOCAL_PROC.wait(timeout=30)
        except Exception:
            LLM_LOCAL_PROC.kill()
    LLM_LOCAL_PROC = None
import atexit
atexit.register(stop_local_qwen)

LLM_MODEL, LLM_MODELS_AVAILABLE = None, []
if LLM_BACKEND == "llama_cpp":
    LLM_ENABLED = not os.environ.get("NB_LLM_DISABLE")
    if not LLM_ENABLED:
        print("NB_LLM_DISABLE set -> LLM extraction skipped; affected fields stay NOT_ATTEMPTED")
    elif not (LOCAL_GGUF.exists() and LOCAL_SERVER.exists()):
        # Host without the local model files: degrade exactly like Cell 7c does instead of
        # aborting the notebook. Set NB_LLM_GGUF / NB_LLM_SERVER (or install the files) to enable.
        LLM_ENABLED = False
        print(f"local LLM files missing ({LOCAL_GGUF}, {LOCAL_SERVER}) -> LLM extraction skipped; "
              "affected fields stay NOT_ATTEMPTED")
    else:
        start_local_qwen()
        LLM_MODEL = f"local:{LOCAL_GGUF.name}"          # llama-server serves one model; the id is for cache keys/provenance
        print("LLM backend: local llama-server", LLM_BASE_URL, "| model:", LOCAL_GGUF.name,
              f"| slots {LOCAL_SLOTS} x {LOCAL_SLOT_CTX} tokens | thinking off")
elif LLM_BACKEND == "fhgenie":
    LLM_ENABLED = any(os.environ.get(n, "").strip() for n in LLM_TOKEN_VARS)
    if not LLM_ENABLED:
        print("FCS_ONPREM_KEY / FHGENIE_API_TOKEN not set -> LLM extraction skipped; affected fields stay NOT_ATTEMPTED")
    else:
        code, d = _llm_http("GET", "/models", timeout=30)
        if code != 200:
            raise RuntimeError(f"/models returned HTTP {code}: {d}")
        LLM_MODELS_AVAILABLE = sorted(m.get("id", "") for m in d.get("data", []))
        LLM_MODEL = pick_model(LLM_MODELS_AVAILABLE, LLM_WANTED_MODEL)
        if not LLM_MODEL:
            raise RuntimeError(f"no model like {LLM_WANTED_MODEL!r}. Available: {LLM_MODELS_AVAILABLE}. "
                               "Set FHGENIE_MODEL to one of them.")
        print("LLM backend: FhGenie", LLM_BASE_URL, "| model:", LLM_MODEL,
              "(requested " + LLM_WANTED_MODEL + ")" if LLM_MODEL != LLM_WANTED_MODEL else "")
else:
    raise ValueError(f"NB_LLM_BACKEND must be llama_cpp or fhgenie, not {LLM_BACKEND!r}")

In [ ]:
# ===== Cell 12L2: LLM field extraction - only the desired sections, quote-verified candidates =====
# Which text is sent:
#   1) the study's publication (text processed once per publication, also for DUPLICATE_RECORD groups),
#   2) sections whose S7 final_status is in SEL_KEEP and that are not backmatter (same rule as the NER tools),
#   3) and whose section class (PROPOSED crosswalk) is routed to at least one field in LLM_FIELD_SECTIONS.
#      Each window asks ONLY for the fields routed to its section's class, e.g. kit and primers only from
#      METHODS. Papers without headings have one PREAMBLE section, which is routed to every field.
# Nothing else leaves the machine. Every section sent is listed in catalogue/llm_sections_sent.csv.
# The model proposes; it never decides. An item is kept only if its quote is found in the window text
# (stored evidence = the exact slice, at most 300 chars) and, for counts, the number appears in the quote.
# Kept items become PAPER_TEXT candidates in Cell 12b, where the normal selection rules apply
# (agreement -> SELECTED, disagreement -> CONFLICT + flag). Everything stays PROVISIONAL_UNVALIDATED.
LLM_SCOPE = "run"            # "run" = publications of PAPERS | "included" = every INCLUDED study | [study ids]
LLM_PROMPT_VERSION = "catalogue-extract-v1"
# local 4B model: 16k-token slots -> smaller windows, more of them
LLM_WINDOW_CHARS, LLM_WINDOW_OVERLAP, LLM_MAX_WINDOWS_PER_PAPER = (24000, 600, 16) if LLM_BACKEND == "fhgenie" \
    else (9000, 400, 40)
LLM_FIELD_SECTIONS = {       # field -> section classes it may be read from (PREAMBLE always allowed)
    "amplicon_region": {"METHODS"}, "primers": {"METHODS"}, "sequencing_platform": {"METHODS"},
    "read_layout": {"METHODS"}, "extraction_kit": {"METHODS"}, "bead_beating": {"METHODS"},
    "target_gene": {"METHODS", "TITLE_ABSTRACT"}, "sample_type": {"METHODS", "TITLE_ABSTRACT"},
    "body_site": {"METHODS", "TITLE_ABSTRACT"}, "host_species": {"METHODS", "TITLE_ABSTRACT"},
    "conditions": {"TITLE_ABSTRACT", "METHODS"}, "has_healthy_controls": {"TITLE_ABSTRACT", "METHODS"},
    "subjects_group": {"TITLE_ABSTRACT", "METHODS"}, "age_range_text": {"METHODS", "TITLE_ABSTRACT"},
    "n_subjects": {"METHODS", "TITLE_ABSTRACT"}, "n_samples": {"METHODS", "TITLE_ABSTRACT"},
    "country": {"METHODS", "TITLE_ABSTRACT"}, "study_design": {"TITLE_ABSTRACT", "METHODS"},
    "intervention": {"TITLE_ABSTRACT", "METHODS"},
}
LLM_OTHER_BODY_IF_NO_METHODS = True   # papers whose methods subsections did not match a class
SF = CAT_SCHEMA["tables"]["study"]["fields"]
INT_FIELDS = {"n_subjects", "n_samples"}
BOOL_FIELDS = {"bead_beating", "has_healthy_controls"}
FIELD_HINT = {f: SF[f]["description"] + (f" Allowed values: {', '.join(SF[f]['values'])}." if SF[f].get("values") else "")
              for f in LLM_FIELD_SECTIONS}
FIELD_HINT["read_layout"] += " Allowed values: PAIRED, SINGLE."
for f in BOOL_FIELDS:
    FIELD_HINT[f] += " Answer true or false."
for f in INT_FIELDS:
    FIELD_HINT[f] += " Answer a whole number."

SYSTEM_PROMPT = (
    "You extract study metadata from one excerpt of a microbiome research paper. Report only what the excerpt "
    "states about THIS study's own participants, samples and methods - never about cited or previous studies. "
    "For every value give `quote`: a short passage (at most 300 characters) copied character for character from "
    "the excerpt that states it. Leave a field out if the excerpt does not state it; do not infer, guess or use "
    "outside knowledge. List several conditions or countries as separate items. Answer with JSON only: "
    '{"items": [{"field": ..., "value": ..., "quote": ...}]}.')

def response_schema(fields):
    return {"type": "object", "additionalProperties": False, "required": ["items"],
            "properties": {"items": {"type": "array", "items": {
                "type": "object", "additionalProperties": False, "required": ["field", "value", "quote"],
                "properties": {"field": {"type": "string", "enum": sorted(fields)},
                               "value": {"type": "string"}, "quote": {"type": "string"}}}}}}

def windows(text, size=LLM_WINDOW_CHARS, overlap=LLM_WINDOW_OVERLAP):
    """[(offset, chunk)] covering text; cuts at a paragraph break when one is near the end of the window."""
    out, i = [], 0
    while i < len(text):
        j = min(len(text), i + size)
        if j < len(text):
            k = text.rfind("\n\n", i + size // 2, j)
            j = k if k > 0 else j
        out.append((i, text[i:j]))
        if j >= len(text):
            break
        i = max(j - overlap, i + 1)
    return out

# ---------- choose publications and sections ----------
if LLM_SCOPE == "run":
    llm_pubs = sorted({c for c in PAPERS if c in PUB})
elif LLM_SCOPE == "included":
    llm_pubs = sorted({l["corpus_id"] for l in LINKS if STUDY[l["study_id"]]["catalogue_status"] == "INCLUDED"})
else:
    llm_pubs = sorted({l["corpus_id"] for l in LINKS if l["study_id"] in set(LLM_SCOPE)})
LLM_JOBS, LLM_SENT = [], []
for cid in llm_pubs:
    secs_c = sorted((x for x in SECTION.values() if x["corpus_id"] == cid), key=lambda x: x["ordinal"])
    has_methods = any(x["section_class"] == "METHODS" for x in secs_c)
    jobs_c = []
    for x in secs_c:
        cls, sid = x["section_class"], x["section_id"]
        keep, reason = select_section(cid, sid, x["section_path"], SECTION_DECISIONS)
        if cls == "PREAMBLE":
            fields = set(LLM_FIELD_SECTIONS)
        elif cls == "OTHER_BODY" and LLM_OTHER_BODY_IF_NO_METHODS and not has_methods:
            fields = {f for f, c in LLM_FIELD_SECTIONS.items() if "METHODS" in c}
        else:
            fields = {f for f, c in LLM_FIELD_SECTIONS.items() if cls in c}
        sent = keep and bool(fields)
        LLM_SENT.append(dict(corpus_id=cid, section_id=sid, section_path=x["section_path"][-150:], section_class=cls,
                             s7_status=x["selection_status"], sent=int(sent),
                             reason=reason if not keep else ("no field routed to " + cls if not fields else "sent"),
                             fields="|".join(sorted(fields)) if sent else "", chars=len(PUB_TEXT[cid][sid])))
        if sent:
            for off, chunk in windows(PUB_TEXT[cid][sid]):
                jobs_c.append(dict(corpus_id=cid, section_id=sid, offset=off, text=chunk, fields=sorted(fields)))
    if len(jobs_c) > LLM_MAX_WINDOWS_PER_PAPER:
        print(f"{cid}: {len(jobs_c)} windows, capped at {LLM_MAX_WINDOWS_PER_PAPER} (rest recorded as not sent)")
        for j in jobs_c[LLM_MAX_WINDOWS_PER_PAPER:]:
            LLM_SENT.append(dict(corpus_id=cid, section_id=j["section_id"], section_path="", section_class="",
                                 s7_status="", sent=0, reason=f"window cap (offset {j['offset']})", fields="", chars=len(j["text"])))
        jobs_c = jobs_c[:LLM_MAX_WINDOWS_PER_PAPER]
    LLM_JOBS += jobs_c
dump_csv(CAT_DIR / "llm_sections_sent.csv", LLM_SENT, list(LLM_SENT[0]) if LLM_SENT else ["corpus_id"])
print(f"LLM scope: {len(llm_pubs)} publications | sections sent: {sum(r['sent'] for r in LLM_SENT)} of {len(LLM_SENT)} "
      f"| windows: {len(LLM_JOBS)} | chars: {sum(len(j['text']) for j in LLM_JOBS)}")

# ---------- run (cached, failures retried next run) ----------
llm_cache = {}
if LLM_CAT_CACHE.exists():
    for line in open(LLM_CAT_CACHE, encoding="utf-8"):
        if line.strip():
            try:
                e = json.loads(line); llm_cache[e["key"]] = e
            except json.JSONDecodeError:
                pass
llm_fail = json.loads(LLM_CAT_FAILURES.read_text(encoding="utf-8")) if LLM_CAT_FAILURES.exists() else {}

def job_key(j):
    return hashlib.sha256(json.dumps([LLM_MODEL, LLM_PROMPT_VERSION, j["fields"], j["text"]]).encode("utf-8")).hexdigest()

def run_job(j):
    k = job_key(j)
    if k in llm_cache:
        return j, llm_cache[k], "cached"
    hints = "\n".join(f"- {f}: {FIELD_HINT[f]}" for f in j["fields"])
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Fields to extract (only these):\n{hints}\n\nExcerpt:\n<<<\n{j['text']}\n>>>"}]
    try:
        out, raw, mode = chat_json(msgs, response_schema(j["fields"]), max_tokens=LLM_MAX_TOKENS)
    except LLMAuthError:
        raise
    except Exception as ex:
        return j, dict(key=k, error=redact(f"{type(ex).__name__}: {ex}")[:300]), "failed"
    return j, dict(key=k, model=LLM_MODEL, prompt_version=LLM_PROMPT_VERSION, mode=mode, at=_now(),
                   items=out.get("items", []) if isinstance(out, dict) else []), "new"

LLM_ITEMS, LLM_REJECTED, LLM_STATUS = [], [], _C()
LLM_OK_JOBS = []
# Progress + checkpoints. Every finished window is already cached (llm_catalogue_cache.jsonl), so a rerun
# resumes from where it stopped; the checkpoint file records how far the run got and what it kept so far.
try:
    from tqdm.auto import tqdm                         # widget bar in Jupyter, text bar elsewhere
except ImportError:
    tqdm = None
LLM_CHECKPOINT = CAT_DIR / "llm_checkpoint.json"
LLM_CHECKPOINT_EVERY = 5                               # windows between checkpoints

def llm_checkpoint(n_done, state, last=None):
    LLM_CHECKPOINT.write_text(redact(json.dumps(dict(
        state=state, at=_now(), backend=LLM_BACKEND, model=LLM_MODEL, prompt_version=LLM_PROMPT_VERSION,
        windows_total=len(LLM_JOBS), windows_done=n_done, windows_by_status=dict(LLM_STATUS),
        items_kept=len(LLM_ITEMS), items_rejected=len(LLM_REJECTED),
        last_window=(f"{last['corpus_id']}/{last['section_id']}@{last['offset']}" if last else None),
        resume="rerun this cell: cached windows are skipped, failed ones are retried"), indent=1)), encoding="utf-8")
    dump_csv(CAT_DIR / "llm_items_partial.csv", LLM_ITEMS,
             ["corpus_id", "section_id", "field", "value", "char_start", "char_end", "quote", "model"])
    LLM_CAT_FAILURES.write_text(json.dumps(llm_fail, indent=1), encoding="utf-8")

if LLM_ENABLED and LLM_JOBS:
    from concurrent.futures import ThreadPoolExecutor
    n_done, last_job = 0, None
    pbar = tqdm(total=len(LLM_JOBS), desc=f"LLM {LLM_BACKEND}", unit="window") if tqdm else None
    llm_checkpoint(0, "started")
    try:
        with ThreadPoolExecutor(max_workers=LLM_WORKERS) as ex, open(LLM_CAT_CACHE, "a", encoding="utf-8") as cache_fh:
            for n_done, (j, res, how) in enumerate(ex.map(run_job, LLM_JOBS), 1):
                last_job = j
                LLM_STATUS[how] += 1
                if how == "failed":
                    llm_fail[res["key"]] = dict(corpus_id=j["corpus_id"], section_id=j["section_id"], offset=j["offset"],
                                                error=res["error"], at=_now())
                else:
                    if how == "new":
                        cache_fh.write(json.dumps(res, ensure_ascii=False) + "\n"); cache_fh.flush()
                        llm_fail.pop(res["key"], None)
                    LLM_OK_JOBS.append(j)
                    for it in res.get("items", []):
                        f, val, q = (it.get("field") or "").strip(), str(it.get("value") or "").strip(), it.get("quote") or ""
                        base = dict(corpus_id=j["corpus_id"], section_id=j["section_id"], field=f, value=val, quote_model=q[:300])
                        if f not in j["fields"]:
                            LLM_REJECTED.append(dict(base, reason="field not asked for this section")); continue
                        if not val or val.lower() in ("not stated", "unknown", "n/a", "none stated", "null"):
                            LLM_REJECTED.append(dict(base, reason="empty value")); continue
                        if f in BOOL_FIELDS and val.lower() not in ("true", "false"):
                            LLM_REJECTED.append(dict(base, reason="value not true/false")); continue
                        loc = locate_quote(j["text"], q)
                        if not loc:
                            LLM_REJECTED.append(dict(base, reason="quote not found in the excerpt")); continue
                        s, e = j["offset"] + loc[0], j["offset"] + min(loc[1], loc[0] + 300)
                        slice_ = PUB_TEXT[j["corpus_id"]][j["section_id"]][s:e]
                        if f in INT_FIELDS and not re.sub(r'[^0-9]', '', val):
                            LLM_REJECTED.append(dict(base, reason="number not in quote")); continue   # e.g. value '.'
                        if f in INT_FIELDS and not re.search(rf"(?<!\d){re.escape(re.sub(r'[^0-9]', '', val))}(?!\d)",
                                                             slice_.replace(",", "")):
                            LLM_REJECTED.append(dict(base, reason="number not in quote")); continue
                        LLM_ITEMS.append(dict(base, char_start=s, char_end=e, quote=slice_, model=LLM_MODEL))
                if pbar is not None:
                    pbar.update(1)
                    pbar.set_postfix(kept=len(LLM_ITEMS), rejected=len(LLM_REJECTED), failed=LLM_STATUS["failed"])
                elif n_done % 10 == 0:
                    print(f"  LLM windows {n_done}/{len(LLM_JOBS)}", flush=True)
                if n_done % LLM_CHECKPOINT_EVERY == 0:
                    llm_checkpoint(n_done, "running", j)
    except BaseException as ex_:                       # Ctrl-C, token expiry (401), anything else
        llm_checkpoint(n_done, f"interrupted: {type(ex_).__name__}", last_job)
        print(f"stopped after {n_done}/{len(LLM_JOBS)} windows - checkpoint written to {LLM_CHECKPOINT.name}; "
              "rerun this cell to resume")
        raise
    finally:
        if pbar is not None:
            pbar.close()
    llm_checkpoint(n_done, "complete", last_job)

if LLM_BACKEND == "llama_cpp":
    stop_local_qwen()                                  # free the GPU for later cells

# fields counted as attempted only where a window that asked for them succeeded
LLM_ATTEMPTED = set()
for j in LLM_OK_JOBS:
    for l in LINKS:
        if l["corpus_id"] == j["corpus_id"]:
            LLM_ATTEMPTED.update((l["study_id"], f) for f in j["fields"])
LLM_RUN_ID = "LLM-" + _now().replace(":", "").replace("-", "")[:15] + "Z"
LLM_RUN_ROWS = [dict(run_id=LLM_RUN_ID, stage="MA-LLM", run_tag=RUN_TAG, started_at=CAT_CREATED_AT, ended_at=_now(),
                     run_status="DONE" if not LLM_STATUS["failed"] else ("PARTIAL" if LLM_OK_JOBS else "FAILED"),
                     tool_versions=json.dumps(dict(model=LLM_MODEL, backend=LLM_BACKEND, base_url=LLM_BASE_URL, response_mode=LLM_MODE,
                                                   prompt=LLM_PROMPT_VERSION, temperature=LLM_TEMPERATURE)),
                     input_manifest_sha256=hashlib.sha256(json.dumps(LLM_SENT).encode("utf-8")).hexdigest(),
                     output_manifest_sha256="", n_items=len(LLM_JOBS), n_failed=LLM_STATUS["failed"])] \
    if LLM_ENABLED and LLM_JOBS else []
dump_csv(CAT_DIR / "llm_items.csv", LLM_ITEMS,
         ["corpus_id", "section_id", "field", "value", "char_start", "char_end", "quote", "model"])
dump_csv(CAT_DIR / "llm_rejected.csv", LLM_REJECTED,
         ["corpus_id", "section_id", "field", "value", "quote_model", "reason"])
(CAT_DIR / "llm_report.json").write_text(redact(json.dumps(dict(
    enabled=LLM_ENABLED, backend=LLM_BACKEND, model=LLM_MODEL, requested=LLM_WANTED_MODEL, base_url=LLM_BASE_URL, response_mode=LLM_MODE,
    scope=LLM_SCOPE if isinstance(LLM_SCOPE, str) else "list", publications=len(llm_pubs), windows=len(LLM_JOBS),
    windows_by_status=dict(LLM_STATUS), items_kept=len(LLM_ITEMS), items_rejected=len(LLM_REJECTED),
    rejected_by_reason=dict(_C(r["reason"] for r in LLM_REJECTED)), kept_by_field=dict(_C(i["field"] for i in LLM_ITEMS)),
    status="PROVISIONAL_UNVALIDATED - review a sample before scaling"), indent=1)), encoding="utf-8")
print("LLM windows:", dict(LLM_STATUS), "| items kept:", len(LLM_ITEMS), "| rejected:", len(LLM_REJECTED),
      dict(_C(r["reason"] for r in LLM_REJECTED)))
# keep the top-level run manifest honest about the LLM stage (Cell 12C runs before this cell)
_man = OUT_DIR / "run_manifest.json"
if _man.exists():
    try:
        _m = json.loads(_man.read_text(encoding="utf-8"))
        _m["llm"] = dict(backend=LLM_BACKEND, model=LLM_MODEL, requested=LLM_WANTED_MODEL,
                         base_url=LLM_BASE_URL, response_mode=LLM_MODE, prompt=LLM_PROMPT_VERSION,
                         windows=len(LLM_JOBS), items_kept=len(LLM_ITEMS), items_rejected=len(LLM_REJECTED),
                         status="PROVISIONAL_UNVALIDATED")
        _man.write_text(redact(json.dumps(_m, indent=1, ensure_ascii=False)), encoding="utf-8")
        print("run manifest llm block refreshed:", _man)
    except Exception as _e:
        print("run manifest refresh skipped:", _e)


In [ ]:
# ===== Cell 12b: catalogue C4 field values + controlled terms, C5 value selection + flags =====
# Candidates come from two sources today:
#   PROJECTS_CSV : amplicon, avg_length, bead_beating, kit, sample_type, condition, subjects
#   PAPER_TEXT   : Methods-Dict spans (Cell 7d) for amplicon_region, primers, sequencing_platform,
#                  extraction_kit, target_gene, plus a bead-beating phrase match - papers in this run only.
# Every PAPER_TEXT value keeps corpus_id, section_id, offsets and a quote equal to the slice.
# Selection (schema rules.value_selection): curator decision > all sources agree -> SELECTED >
# disagreement -> CONFLICT + METADATA_CONFLICT flag (never pick a side) > nothing -> NOT_FOUND.
# Fields with no extractor yet are NOT_ATTEMPTED (body_site, host_species, n_subjects, country, ...).
EXTRACTOR_V = "0.1"
REVIEW_LOG_CSV = CAT_DIR / "review_log.csv"            # curators append here; the build only reads it
REVIEW_FIELDS_LOG = list(CAT_SCHEMA["tables"]["review_log"]["fields"])
if not REVIEW_LOG_CSV.exists():
    dump_csv(REVIEW_LOG_CSV, [], REVIEW_FIELDS_LOG)
REVIEW_LOG = load_csv(REVIEW_LOG_CSV)

# ---------- normalizers ----------
def norm_region(v):
    s = re.sub(r"\s+", "", (v or "").upper()).replace("–", "-").replace("—", "-").replace("/", "-")
    s = re.sub(r"^(V\d)(V\d)$", r"\1-\2", s)
    return s if re.fullmatch(r"V\d(-V\d)?", s) else ""

KIT_FAMILIES = [  # (all keywords, LOCAL id, label) - first match wins; order matters (fast before plain)
    (("qiaamp", "fast", "stool"), "KIT_QIAAMP_FAST_STOOL", "QIAamp Fast DNA Stool Mini Kit"),
    (("qiaamp", "stool"), "KIT_QIAAMP_STOOL", "QIAamp DNA Stool Mini Kit"),
    (("qiaamp", "powerfecal"), "KIT_QIAAMP_POWERFECAL", "QIAamp PowerFecal DNA Kit"),
    (("powerfecal",), "KIT_POWERFECAL", "PowerFecal DNA Kit"),
    (("powermicrobiome",), "KIT_POWERMICROBIOME", "MagAttract PowerMicrobiome DNA/RNA Kit"),
    (("powermag",), "KIT_POWERMAG", "PowerMag Microbiome RNA/DNA Isolation Kit"),
    (("powersoil",), "KIT_POWERSOIL", "PowerSoil DNA Isolation Kit"),
    (("fastdna",), "KIT_FASTDNA_SPIN", "FastDNA SPIN Kit"),
    (("e.z.n.a",), "KIT_EZNA_STOOL", "E.Z.N.A. Stool DNA Kit"),
    (("dneasy", "tissue"), "KIT_DNEASY_BLOOD_TISSUE", "DNeasy Blood & Tissue Kit"),
    (("magna pure",), "KIT_MAGNA_PURE", "MagNA Pure DNA Isolation Kit"),
    (("zymobiomics",), "KIT_ZYMOBIOMICS", "ZymoBIOMICS DNA Miniprep Kit"),
    (("nucleospin",), "KIT_NUCLEOSPIN_SOIL", "NucleoSpin Soil"),
    (("allprep",), "KIT_ALLPREP", "AllPrep DNA/RNA Mini Kit"),
    (("pi-480",), "KIT_GENEPREP_PI480", "Gene Prep Star PI-480"),
    (("magdea",), "KIT_MAGDEA", "MagDEA DNA 200 kit"),
    (("custom",), "KIT_IN_HOUSE", "in-house / custom protocol"),
    (("in-house",), "KIT_IN_HOUSE", "in-house / custom protocol"),
]
def norm_kit(v):
    low = (v or "").lower().replace("ezna", "e.z.n.a").replace("e.z.n.a.", "e.z.n.a")
    low = re.sub(r"[^\x00-\x7f]", "", low)
    for kws, tid, label in KIT_FAMILIES:
        if all(k in low for k in kws):
            return label, "LOCAL:" + tid
    return re.sub(r"\s*\(.*?\)\s*", " ", low).strip(" ;,."), ""

PLATFORMS = [(r"novaseq", "Illumina NovaSeq"), (r"nextseq", "Illumina NextSeq"), (r"hiseq", "Illumina HiSeq"),
             (r"miseq", "Illumina MiSeq"), (r"454|gs flx|pyrosequenc", "Roche 454"),
             (r"ion (torrent|pgm|s5)", "Ion Torrent"), (r"pacbio", "PacBio"), (r"nanopore|minion", "Oxford Nanopore"),
             (r"bgiseq|dnbseq", "BGISEQ/DNBSEQ"), (r"^illumina$", "Illumina")]
def norm_platform(v):
    low = (v or "").lower()
    for pat, lab in PLATFORMS:
        if re.search(pat, low):
            return lab
    return ""

SUBJ = [(r"infant|newborn|neonat|preterm|younger than 2", "INFANTS"), (r"child", "CHILDREN"),
        (r"adolescen", "ADOLESCENTS"), (r"elderly|older|centenarian|aged above", "OLDER_ADULTS"), (r"adult", "ADULTS")]
def norm_subjects(v):
    low = (v or "").lower()
    if re.search(r"newborn to centenarian", low):
        return "MIXED"
    hits = {lab for pat, lab in SUBJ if re.search(pat, low)}
    return "MIXED" if len(hits) > 1 else (hits.pop() if hits else "")

def split_list(v):
    return [x.strip() for x in re.split(r"\s*(?:\+|,|;|\band\b)\s*", v or "") if x.strip()]

HEALTHY_RX = re.compile(r"^(healthy( controls?| subjects)?|controls?)$", re.I)
BEAD_RX = re.compile(r"bead[- ]?beat(?:ing|en)?|beat[- ]?beating", re.I)

# ---------- controlled terms (ontology via OLS4 exact match, else LOCAL PROPOSED) ----------
FACET_ONTS = {"condition": "mondo,doid", "sample_type": "uberon", "platform": "efo,obi",
              "body_site": "uberon", "host_species": "ncbitaxon"}
facet_c = LookupCache("OLS4_FACET", WORK / "ols4_facet_cache_v4.json", LINK_FAILURES,
                      OLS4 + "/search?q={term}&ontology={onts}&exact=true&queryFields=label,synonym&rows=1")
def facet_fetch(key):
    onts, term = key.split("|", 1)
    code, d = http_json(f"{OLS4}/search?q={urllib.parse.quote(term)}&ontology={onts}&exact=true"
                        "&queryFields=label,synonym&rows=1")
    resp = (d or {}).get("response") if isinstance(d, dict) else None
    if code == 404 or not isinstance(resp, dict) or "docs" not in resp:
        raise LookupFailure(f"UNEXPECTED_PAYLOAD http={code}")
    if not resp["docs"]:
        return "NOT_FOUND", None
    d0 = resp["docs"][0]
    desc = [x for x in (d0.get("description") or []) if isinstance(x, str) and x.strip()]
    return "FOUND", dict(term_id=d0.get("obo_id", ""), label=d0.get("label", ""), vocab=d0.get("ontology_prefix", ""),
                         definition=desc[0] if desc else "",
                         synonyms=sorted(set((d0.get("exact_synonyms") or []) + (d0.get("related_synonyms") or [])))[:20])
TERMS = {}
VOCABS = set(CAT_SCHEMA["tables"]["controlled_term"]["fields"]["vocabulary"]["values"])
def term_for(facet, label):
    """-> term_id or ''. Ontology facets: exact OLS4 match; LOCAL facets: PROPOSED LOCAL term."""
    if not label:
        return ""
    if facet in FACET_ONTS:
        e = facet_c.get(f"{FACET_ONTS[facet]}|{label}", facet_fetch)
        if e and e["status"] == "FOUND" and e["value"]["term_id"]:
            v = e["value"]
            vocab = {"MONDO": "MONDO", "DOID": "DOID", "UBERON": "UBERON", "EFO": "EFO", "OBI": "OBI",
                     "NCBITAXON": "NCBITaxon", "CHEBI": "CHEBI"}.get(v["vocab"].upper(), "")
            if vocab in VOCABS:
                TERMS.setdefault(v["term_id"], dict(term_id=v["term_id"], vocabulary=vocab, facet=facet, label=v["label"],
                                                    definition=v["definition"],
                                                    definition_source=("OLS4:" + v["vocab"]) if v["definition"] else "",
                                                    synonyms=json.dumps(v["synonyms"]), parent_ids="[]", term_status="ACTIVE"))
                return v["term_id"]
        if facet != "platform":
            return ""                                   # no ontology hit -> value kept, no term (never a guess)
    tid = "LOCAL:" + re.sub(r"[^A-Z0-9]+", "_", f"{facet}_{label}".upper()).strip("_")
    TERMS.setdefault(tid, dict(term_id=tid, vocabulary="LOCAL", facet=facet, label=label, definition="",
                               definition_source="LOCAL", synonyms="[]", parent_ids="[]", term_status="PROPOSED"))
    return tid

def local_term(facet, tid, label):
    TERMS.setdefault(tid, dict(term_id=tid, vocabulary="LOCAL", facet=facet, label=label, definition="",
                               definition_source="LOCAL", synonyms="[]", parent_ids="[]", term_status="PROPOSED"))
    return tid

# ---------- C4: candidates ----------
VALUES = []
def add_value(study, field, source, raw, normalized, term_id, extractor, ev=None):
    ev = ev or {}
    VALUES.append(dict(value_id=stable_id("VAL_", study, field, source, raw, ev.get("char_start", "")),
                       study_id=study, field=field, value_raw=raw, value_normalized=normalized, term_id=term_id,
                       source=source, corpus_id=ev.get("corpus_id", ""), section_id=ev.get("section_id", ""),
                       char_start=ev.get("char_start", ""), char_end=ev.get("char_end", ""),
                       quote=ev.get("quote", ""), extractor=extractor, audit_result="NOT_CHECKED",
                       selection_status="CANDIDATE", review_status="PROVISIONAL_UNVALIDATED", reject_reason=""))

for p in PROJECTS:
    s = p["project"]
    X = f"projects_csv@{EXTRACTOR_V}"
    if p["amplicon"].strip():
        r_ = norm_region(p["amplicon"])
        add_value(s, "amplicon_region", "PROJECTS_CSV", p["amplicon"], r_,
                  local_term("amplicon_region", "LOCAL:REGION_" + r_.replace("-", "_"), r_) if r_ else "", X)
    if p["avg_length"].strip():
        add_value(s, "avg_read_length", "PROJECTS_CSV", p["avg_length"], str(round(float(p["avg_length"]), 2)), "", X)
    if p["bead_beating"].strip():
        add_value(s, "bead_beating", "PROJECTS_CSV", p["bead_beating"], p["bead_beating"].strip().lower(), "", X)
    if p["kit"].strip():
        lab, tid = norm_kit(p["kit"])
        add_value(s, "extraction_kit", "PROJECTS_CSV", p["kit"], lab,
                  local_term("extraction_kit", tid, lab) if tid else "", X)
    parts = sorted({x.lower() for x in split_list(p["sample_type"])} - {"mice", "na", "mostly stool", "possibly swab"})
    if parts:                       # one candidate per CSV cell; several specimens -> 'saliva|stool', no single term
        add_value(s, "sample_type", "PROJECTS_CSV", p["sample_type"], "|".join(parts),
                  term_for("sample_type", parts[0]) if len(parts) == 1 else "", X)
    conds = [c for c in split_list(p["condition"]) if c.upper() != "NA"]
    if conds and all(HEALTHY_RX.match(c) for c in conds):
        add_value(s, "conditions", "PROJECTS_CSV", p["condition"], "NOT_APPLICABLE_HEALTHY_ONLY", "", X)
    for c in conds:
        if HEALTHY_RX.match(c):
            if len(conds) > 1:
                add_value(s, "has_healthy_controls", "PROJECTS_CSV", p["condition"], "true", "", X)
            continue
        c_ = re.sub(r"[^\x00-\x7f]+", "'", c).strip()
        add_value(s, "conditions", "PROJECTS_CSV", c, c_.lower(), term_for("condition", c_), X)
    if p["subjects"].strip() and p["subjects"].strip().upper() != "NA":
        g_ = norm_subjects(p["subjects"])
        add_value(s, "subjects_group", "PROJECTS_CSV", p["subjects"], g_,
                  local_term("subjects_group", "LOCAL:SUBJECTS_" + g_, g_) if g_ else "", X)

# paper text (papers in this run); a paper's values attach to every study linked to that publication
STUDIES_OF = {}
for l in LINKS:
    STUDIES_OF.setdefault(l["corpus_id"], []).append(l["study_id"])
TEXT_FIELDS = ("amplicon_region", "primers", "sequencing_platform", "extraction_kit", "target_gene", "bead_beating")
PAPER_ATTEMPTED = {c for c in PAPERS if c in PUB}
for cid in PAPERS:
    if cid not in PUB:
        raise_flag("pipeline_run", NER_RUN_ID, "DUPLICATE_RECORD", "INFO",
                   f"{cid} is in the NER run but is not a catalogue publication (duplicate copy); its text is not used",
                   field=cid)
        continue
    dict_rows = [r for r in all_rows if r["cid"] == cid and r["model"] == "Methods-Dict"]
    primers = {}
    for r in dict_rows:
        ev = dict(corpus_id=cid, section_id=sec_key(cid, r["section_id"]), char_start=int(r["char_start"]),
                  char_end=int(r["char_end"]), quote=r["term"][:300])
        lab, term = r["label_raw"], r["term"]
        for s in STUDIES_OF.get(cid, []):
            X = f"methods_dict@{EXTRACTOR_V}"
            if lab == "amplicon_region" and norm_region(term):
                r_ = norm_region(term)
                add_value(s, "amplicon_region", "PAPER_TEXT", term, r_,
                          local_term("amplicon_region", "LOCAL:REGION_" + r_.replace("-", "_"), r_), X, ev)
            elif lab == "platform" and norm_platform(term):
                pl = norm_platform(term)
                add_value(s, "sequencing_platform", "PAPER_TEXT", term, pl, term_for("platform", pl), X, ev)
            elif lab == "kit":
                kl, tid = norm_kit(term)
                add_value(s, "extraction_kit", "PAPER_TEXT", term, kl, local_term("extraction_kit", tid, kl) if tid else "", X, ev)
            elif lab == "primer":
                primers.setdefault(s, []).append((term, ev))
            elif lab == "assay" and "16s" in term.lower():
                add_value(s, "target_gene", "PAPER_TEXT", term, "16S rRNA", local_term("target_gene", "LOCAL:GENE_16S", "16S rRNA"), X, ev)
    for s, ps in primers.items():
        names = sorted({t for t, _ in ps}, key=lambda t: (t[-1] != "F", t))
        add_value(s, "primers", "PAPER_TEXT", "/".join(names), "/".join(names), "", f"methods_dict@{EXTRACTOR_V}", ps[0][1])
        if any(t.upper().startswith("ITS") for t in names):
            add_value(s, "target_gene", "PAPER_TEXT", ps[0][0], "ITS", local_term("target_gene", "LOCAL:GENE_ITS", "ITS"),
                      f"methods_dict@{EXTRACTOR_V}", ps[0][1])
    for sid, text in body_secs[cid].items():
        m = BEAD_RX.search(text)
        if m:
            for s in STUDIES_OF.get(cid, []):
                add_value(s, "bead_beating", "PAPER_TEXT", m.group(0), "true", "", f"bead_phrase@{EXTRACTOR_V}",
                          dict(corpus_id=cid, section_id=sec_key(cid, sid), char_start=m.start(), char_end=m.end(), quote=m.group(0)))
            break

# ---------- LLM candidates (Cells 12L1/12L2): same normalizers, then the same selection rules ----------
LLM_ATTEMPTED = globals().get("LLM_ATTEMPTED", set())
_SF = CAT_SCHEMA["tables"]["study"]["fields"]
_ENUMS = {"subjects_group": set(_SF["subjects_group"]["values"]), "study_design": set(_SF["study_design"]["values"]),
          "read_layout": {"PAIRED", "SINGLE"}}
HOST_SYNONYMS = {"human": "homo sapiens", "humans": "homo sapiens", "mouse": "mus musculus", "mice": "mus musculus"}
def llm_normalize(f, val):
    """-> (normalized, term_id, '') or (None, None, reason)."""
    v = val.strip()
    if f == "amplicon_region":
        r_ = norm_region(v)
        return (r_, local_term("amplicon_region", "LOCAL:REGION_" + r_.replace("-", "_"), r_), "") if r_ else (None, None, "not a V-region")
    if f == "extraction_kit":
        lab, tid = norm_kit(v)
        return lab, (local_term("extraction_kit", tid, lab) if tid else ""), ""
    if f == "sequencing_platform":
        pl = norm_platform(v)
        return (pl, term_for("platform", pl), "") if pl else (None, None, "unknown platform")
    if f == "target_gene":
        low = v.lower()
        if "16s" in low:
            return "16S rRNA", local_term("target_gene", "LOCAL:GENE_16S", "16S rRNA"), ""
        if "its" in low.split() or low.startswith("its"):
            return "ITS", local_term("target_gene", "LOCAL:GENE_ITS", "ITS"), ""
        return v, "", ""
    if f in ("bead_beating", "has_healthy_controls"):
        lv = v.lower()
        return (lv, "", "") if lv in ("true", "false") else (None, None, "not true/false")
    if f in _ENUMS:
        u = re.sub(r"[\s-]+", "_", v.upper())
        if u not in _ENUMS[f]:
            return None, None, "not an allowed value"
        return u, (local_term(f, f"LOCAL:{f.upper()}_{u}", u) if f != "read_layout" else ""), ""
    if f in ("n_subjects", "n_samples"):
        d = re.sub(r"[^0-9]", "", v)
        return (str(int(d)), "", "") if d else (None, None, "not a number")
    if f == "sample_type":
        return v.lower(), term_for("sample_type", v.lower()), ""
    if f == "body_site":
        return v.lower(), term_for("body_site", v.lower()), ""
    if f == "host_species":
        lv = HOST_SYNONYMS.get(v.lower(), v.lower())
        return lv, term_for("host_species", lv), ""
    if f == "conditions":
        return v.lower(), term_for("condition", v), ""
    if f == "country":
        c_ = v.strip().title()
        return c_, local_term("country", "LOCAL:COUNTRY_" + re.sub(r"[^A-Z0-9]+", "_", c_.upper()).strip("_"), c_), ""
    return v, "", ""                                              # primers, age_range_text, intervention

LLM_NORM_REJECTED = []
# v4: overlapping windows can return the same item twice; keep the first
_llm_items, _llm_dups = dedupe_by(globals().get("LLM_ITEMS", []),
                                  lambda it: (it["corpus_id"], it["section_id"], it["field"], it["value"].lower(), it["char_start"]))
print("LLM items: duplicates from overlapping windows dropped:", _llm_dups)
for it in _llm_items:
    n_, tid, why = llm_normalize(it["field"], it["value"])
    if n_ is None:
        LLM_NORM_REJECTED.append(dict(it, reason=why)); continue
    ev = {k: it[k] for k in ("corpus_id", "section_id", "char_start", "char_end", "quote")}
    for s in STUDIES_OF.get(it["corpus_id"], []):
        add_value(s, it["field"], "PAPER_TEXT", it["value"], n_, tid, f"llm:{it['model']}|{LLM_PROMPT_VERSION}", ev)
        VALUES[-1]["run_id"] = LLM_RUN_ID
if globals().get("LLM_ITEMS"):
    print("LLM candidates added:", sum(1 for v in VALUES if v["extractor"].startswith("llm:")),
          "| rejected at normalization:", len(LLM_NORM_REJECTED), dict(_C(r["reason"] for r in LLM_NORM_REJECTED)))

# v4: one candidate per value_id (same source, field, raw value and offset)
_vals, _nv = dedupe_by(VALUES, lambda v: v["value_id"])
VALUES[:] = _vals
print("candidate values: duplicates merged:", _nv)

# paper-internal subsumption: 'V3' and 'V4' alongside 'V3-V4', or 'Illumina' alongside 'Illumina MiSeq'
_by_sf = {}
for v in VALUES:
    _by_sf.setdefault((v["study_id"], v["field"]), []).append(v)
for (s, f), vs in _by_sf.items():
    paper = [v for v in vs if v["source"] == "PAPER_TEXT"]
    if f == "amplicon_region":
        singles = sorted({int(v["value_normalized"][1]) for v in paper if "-" not in v["value_normalized"]})
        if paper and len(singles) > 1 and not any("-" in v["value_normalized"] for v in paper) and                 singles == list(range(singles[0], singles[-1] + 1)):
            first = next(v for v in paper if "-" not in v["value_normalized"])
            r_ = f"V{singles[0]}-V{singles[-1]}"
            add_value(s, f, "PAPER_TEXT", " + ".join(f"V{x}" for x in singles), r_,
                      local_term("amplicon_region", "LOCAL:REGION_" + r_.replace("-", "_"), r_),
                      f"methods_dict+adjacent_regions@{EXTRACTOR_V}",
                      {k: first[k] for k in ("corpus_id", "section_id", "char_start", "char_end", "quote")})
            vs.append(VALUES[-1]); paper.append(VALUES[-1])
        ranges = [v["value_normalized"] for v in paper if "-" in v["value_normalized"]]
        for v in paper:
            n = v["value_normalized"]
            if "-" not in n and any(int(r[1]) <= int(n[1]) <= int(r[-1]) for r in ranges):
                v.update(selection_status="REJECTED", reject_reason="single region inside a range stated in the same paper")
    if f == "sequencing_platform" and any(v["value_normalized"].startswith("Illumina ") for v in paper):
        for v in paper:
            if v["value_normalized"] == "Illumina":
                v.update(selection_status="REJECTED", reject_reason="vendor only; a specific Illumina instrument is stated")

# ---------- C5: selection ----------
CURATED_FIELDS = [k for k, v in CAT_SCHEMA["tables"]["study"]["fields"].items() if v.get("curated")]
LIST_FIELDS = {k for k in CURATED_FIELDS if CAT_SCHEMA["tables"]["study"]["fields"][k]["type"].startswith("list")}
ATTEMPTED_FIELDS = {"amplicon_region", "avg_read_length", "bead_beating", "extraction_kit", "sample_type", "conditions",
                    "subjects_group", "has_healthy_controls", "primers", "sequencing_platform", "target_gene"}
def _key(v):
    return v["term_id"] or v["value_normalized"].lower()

def _curated(v, status):
    return dict(value=v["value_normalized"] if v else None, term_id=(v or {}).get("term_id") or None,
                status=status, source=v["source"] if v else None, value_id=v["value_id"] if v else None)

reviews = {}
for rv in REVIEW_LOG:
    if rv.get("target_table") == "study" and rv.get("decision") in ("CORRECT", "ACCEPT"):
        reviews[(rv["target_id"], rv.get("field", ""))] = rv

for s, st in STUDY.items():
    canon_pub = ELIG[CANONICAL[s]]["corpus_id"]
    paper_in_run = canon_pub in PAPER_ATTEMPTED
    summary = _C()
    for f in CURATED_FIELDS:
        vs = [v for v in _by_sf.get((s, f), []) if v["selection_status"] != "REJECTED"]
        rv = reviews.get((s, f))
        if rv:                                                     # 1) curator decision wins
            if rv["decision"] == "CORRECT":
                add_value(s, f, "CURATOR", rv["new_value"], rv["new_value"], "", f"review_log:{rv['review_id']}")
                cv = VALUES[-1]; cv["selection_status"] = "SELECTED"; cv["review_status"] = "CORRECTED"
            else:
                cv = next((v for v in VALUES if v["value_id"] == rv["new_value"] or v["value_id"] == rv.get("old_value")), None)
                if cv: cv.update(selection_status="SELECTED", review_status="ACCEPTED")
            obj = _curated(cv, "SELECTED")
            st[f] = json.dumps([obj] if f in LIST_FIELDS else obj)
            summary["SELECTED"] += 1
            continue
        csv_vs = [v for v in vs if v["source"] == "PROJECTS_CSV"]
        txt_vs = [v for v in vs if v["source"] == "PAPER_TEXT"]
        for v in csv_vs:                                           # metadata-audit result for CSV values
            if (f in TEXT_FIELDS and paper_in_run) or (s, f) in LLM_ATTEMPTED:
                v["audit_result"] = ("MATCH" if any(_key(t) == _key(v) for t in txt_vs) else "MISMATCH") if txt_vs else "NOT_FOUND"
        for v in txt_vs:
            v["audit_result"] = "NOT_APPLICABLE"
        if f in LIST_FIELDS:
            groups = {}
            for v in vs:
                groups.setdefault(_key(v), []).append(v)
            healthy = [v for v in vs if v["value_normalized"] == "NOT_APPLICABLE_HEALTHY_ONLY"]
            if not vs:
                status = "NOT_FOUND" if (f in ATTEMPTED_FIELDS or (s, f) in LLM_ATTEMPTED) else "NOT_ATTEMPTED"
                st[f] = json.dumps([_curated(None, status)]); summary[status] += 1
            elif healthy and len(healthy) == len(vs):
                for v in vs: v["selection_status"] = "SELECTED"
                st[f] = json.dumps([_curated(None, "NOT_APPLICABLE")]); summary["NOT_APPLICABLE"] += 1
            elif healthy:                                          # CSV: healthy only; paper names a condition
                for v in vs: v["selection_status"] = "CONFLICT"
                for v in healthy: v["audit_result"] = "MISMATCH"
                st[f] = json.dumps([_curated(None, "CONFLICT")]); summary["CONFLICT"] += 1
                raise_flag("study", s, "METADATA_CONFLICT", "WARNING", "projects.csv says healthy only; paper text names: " +
                           ", ".join(v["value_normalized"] for v in vs if v not in healthy)[:400], field=f, raised_by="MA")
            else:
                objs = []
                for k, g in groups.items():
                    rep = next((v for v in g if v["source"] == "PAPER_TEXT"), g[0])
                    rep["selection_status"] = "SELECTED"
                    objs.append(_curated(rep, "SELECTED"))
                st[f] = json.dumps(objs); summary["SELECTED"] += 1
            continue
        groups = {}
        for v in vs:
            groups.setdefault(_key(v), []).append(v)
        if len(groups) == 1:                                       # 2) every source agrees
            g = next(iter(groups.values()))
            rep = next((v for v in g if v["source"] == "PAPER_TEXT"), g[0])
            rep["selection_status"] = "SELECTED"
            st[f] = json.dumps(_curated(rep, "SELECTED")); summary["SELECTED"] += 1
        elif len(groups) > 1:                                      # 3) disagreement -> CONFLICT, never pick
            for v in vs: v["selection_status"] = "CONFLICT"
            st[f] = json.dumps(_curated(None, "CONFLICT")); summary["CONFLICT"] += 1
            detail = "; ".join(f"{v['source']}: {v['value_raw'][:60]!r} -> {v['value_normalized'][:40]!r}" for v in vs[:6])
            raise_flag("study", s, "METADATA_CONFLICT", "WARNING", detail[:500], field=f, raised_by="MA")
            if csv_vs and txt_vs:
                raise_flag("study", s, "METADATA_MISMATCH", "WARNING",
                           f"projects.csv {csv_vs[0]['value_raw'][:80]!r} not supported by the paper text", field=f,
                           raised_by="MA")
        else:                                                      # 4) no source
            attempted = (s, f) in LLM_ATTEMPTED or (f in ATTEMPTED_FIELDS and (f not in TEXT_FIELDS or paper_in_run or
                                                   f in ("amplicon_region", "extraction_kit", "bead_beating")))
            status = "NOT_FOUND" if attempted else "NOT_ATTEMPTED"
            st[f] = json.dumps(_curated(None, status)); summary[status] += 1
            if status == "NOT_FOUND" and st["catalogue_status"] == "INCLUDED":
                raise_flag("study", s, "FIELD_MISSING", "INFO", f"no source gives {f}", field=f)
    st["audit_summary"] = json.dumps(dict(summary))
    if paper_in_run and any(v["field"] == "extraction_kit" and v["source"] == "PROJECTS_CSV"
                            for v in _by_sf.get((s, "extraction_kit"), [])) and \
            not any(v["source"] == "PAPER_TEXT" for v in _by_sf.get((s, "extraction_kit"), [])):
        raise_flag("study", s, "VALUE_IN_SUPPLEMENT_ONLY", "INFO",
                   "kit in projects.csv not found in the paper body (may be in a supplement)", field="extraction_kit")

print("C4 candidate values:", len(VALUES), "| by source:", dict(_C(v["source"] for v in VALUES)),
      "| controlled terms:", len(TERMS), dict(_C(t["vocabulary"] for t in TERMS.values())))
print("C5 selection:", dict(_C(v["selection_status"] for v in VALUES)),
      "| CSV audit:", dict(_C(v["audit_result"] for v in VALUES if v["source"] == "PROJECTS_CSV")))
print("facet lookups:", facet_c.counts, "| failures (retried next run):", len(facet_c.failures))
facet_c.save()

In [ ]:
# ===== Cell 12c: catalogue C6 findings, C7 validation, C8 views + draft release + exports =====
# C6: findings come from template_mapped_corpus/findings/candidates.jsonl (S9). Their offsets index the
#     harmonized Markdown, so each quote is re-located in the section text; PASSED only when the slice
#     equals the quote. Duplicate re-run rows collapse on finding_id. Every finding stays CANDIDATE.
# C7: the schema's 12 validation rules; any violation writes the report and stops the build.
# C8: catalogue_entry for INCLUDED studies without an open BLOCKER, facet_index, then a DRAFT release
#     (approved_by = NOT_APPROVED) cut into OUT_DIR/catalogue/releases/<release_id>/ - never overwritten.
FINDINGS_JSONL = ROOT / "template_mapped_corpus" / "findings" / "candidates.jsonl"
DIRECTION = {"increased": "INCREASED", "decreased": "DECREASED", "no_change": "NO_CHANGE",
             "associated": "ASSOCIATED", "mixed": "MIXED"}
def finding_type(f):
    d = (f.get("direction") or "").lower()
    txt = " ".join(str(f.get(k) or "") for k in ("statistic_text", "outcome")).lower()
    if d == "no_change":
        return "NO_DIFFERENCE"
    if "correlat" in txt:
        return "CORRELATION"
    if "predict" in txt or "auc" in txt:
        return "PREDICTION"
    if d in ("increased", "decreased"):
        return "DIFFERENCE"
    if d == "associated":
        return "ASSOCIATION"
    return "DESCRIPTIVE"
P_EXACT = re.compile(r"\bp\s*=\s*([0-9]*\.?[0-9]+(?:e-?\d+)?)", re.I)
def misattribution(q):
    if re.match(r"\s*(fig\.?|figure)\s*\d", q, re.I):
        return "FIGURE_LEGEND"
    if re.match(r"\s*table\s*\d", q, re.I):
        return "TABLE_LABEL"
    if re.search(r"\bet al\.?|\[\d+(?:[,–-]\d+)*\]|previous(ly)? (stud|report)", q, re.I):
        return "CITES_OTHER_WORK"
    return ""

ent_by_term = {}
for r in ner_corpus:
    for sf in r["surface_forms"].split("|"):
        ent_by_term.setdefault(sf.lower(), r["entity_id"])
FINDING = {}
if FINDINGS_JSONL.exists():
    for line in open(FINDINGS_JSONL, encoding="utf-8"):
        if not line.strip():
            continue
        try:
            c = json.loads(line)
        except json.JSONDecodeError:
            continue                                         # partial line from an interrupted run
        cid, sid, f = c["corpus_id"], c["section_id"], c["finding"]
        if cid not in PUB or sid not in SECTION:
            continue
        q = f.get("evidence_quote") or ""
        text = PUB_TEXT[cid][sid]
        i = text.find(q) if q else -1
        cs, ce = (i, i + len(q)) if i >= 0 else (-1, -1)
        ft = finding_type(f)
        fid = stable_id("FND_", cid, sid, cs, ce, ft) if i >= 0 else stable_id("FND_", cid, sid, q, ft)
        pv = P_EXACT.search(f.get("statistic_text") or "")
        FINDING.setdefault(fid, dict(
            finding_id=fid, corpus_id=cid, section_id=sid, finding_type=ft,
            subject_text=f.get("taxon") or "(not stated)",
            subject_entity_id=ent_by_term.get((f.get("taxon") or "").lower(), ""),
            object_text=f.get("exposure_or_group") or "",
            object_entity_id=ent_by_term.get((f.get("exposure_or_group") or "").lower(), ""),
            direction=DIRECTION.get((f.get("direction") or "").lower(), "NOT_STATED"),
            comparison=f.get("comparison") or "", statistic_text=f.get("statistic_text") or "",
            p_value=float(pv.group(1)) if pv else "", quote=q, char_start=cs, char_end=ce,
            extractor=f"{c.get('model', '')}|prompt:{c.get('prompt_version', '')}",
            evidence_check="PASSED" if i >= 0 else "FAILED", misattribution_flag=misattribution(q),
            status="CANDIDATE"))
for cid, p in PUB.items():
    fs = [x for x in FINDING.values() if x["corpus_id"] == cid]
    p["n_findings"] = json.dumps(dict(_C(x["status"] for x in fs)))
    p["n_entities"] = json.dumps(dict(_C(r["cdm_type"] for r in ner_paper if r["cid"] == cid)))
print("C6 findings:", len(FINDING), "| evidence:", dict(_C(x["evidence_check"] for x in FINDING.values())),
      "| misattribution:", dict(_C(x["misattribution_flag"] for x in FINDING.values() if x["misattribution_flag"])))

# ---------- summaries on study ----------
for s, st in STUDY.items():
    st["n_publications"] = sum(1 for l in LINKS if l["study_id"] == s and l["link_status"] == "CONFIRMED")
    st["open_flags"] = json.dumps(dict(_C(f["severity"] for f in CAT_FLAGS
                                          if f["target_table"] == "study" and f["target_id"] == s and f["flag_status"] == "OPEN")))

# ---------- tables ----------
CAT_FLAGS = list({f["flag_id"]: f for f in CAT_FLAGS}.values())
pipeline_rows = [
    dict(run_id=NER_RUN_ID, stage="N1-N7", run_tag=RUN_TAG, started_at=RUN_STARTED_AT, ended_at=built_at,
         run_status="DONE" if all(t.get("status") == "OK" for t in TOOL_STATUS.values()) else "PARTIAL",
         tool_versions=json.dumps({k: v.get("status", "") for k, v in TOOL_STATUS.items()}),
         input_manifest_sha256=file_sha256(OUT_DIR / "run_manifest.json") if (OUT_DIR / "run_manifest.json").exists() else "",
         output_manifest_sha256="", n_items=len(PAPERS), n_failed=sum(1 for c in PAPERS if not TT[c])),
    dict(run_id=CAT_RUN_ID, stage="CAT", run_tag=RUN_TAG, started_at=CAT_CREATED_AT, ended_at="", run_status="DONE",
         tool_versions=json.dumps(dict(catalogue_schema=CAT_VERSION, extractors=EXTRACTOR_V, crosswalk=XWALK_VERSION,
                                       ner_schema=NER_SCHEMA_VERSION)),
         input_manifest_sha256=hashlib.sha256(CAT_SCHEMA_PATH.read_bytes()).hexdigest(), output_manifest_sha256="",
         n_items=len(STUDY), n_failed=0)]
pipeline_rows += globals().get("LLM_RUN_ROWS", [])           # MA-LLM run (Cell 12L2), when it ran
CAT.update(
    study=list(STUDY.values()), study_field_value=VALUES, archive_run=[],
    study_publication=LINKS, publication=list(PUB.values()), source_file=list(FILES.values()),
    section=list(SECTION.values()), finding=list(FINDING.values()), controlled_term=list(TERMS.values()),
    quality_flag=CAT_FLAGS, review_log=REVIEW_LOG, pipeline_run=pipeline_rows)
for t, rows in CAT.items():
    cat_stamp(rows)

# ---------- C8a: views ----------
def _obj(st, f):
    return json.loads(st[f])
blockers = {f["target_id"] for f in CAT_FLAGS if f["severity"] == "BLOCKER" and f["flag_status"] == "OPEN"
            and f["target_table"] == "study"}
ENTRIES = []
for s, st in STUDY.items():
    if st["catalogue_status"] != "INCLUDED" or s in blockers:
        continue
    pubs = [l for l in LINKS if l["study_id"] == s]
    top = {}
    for l in pubs:
        for r in sorted((r for r in ner_paper if r["cid"] == l["corpus_id"]), key=lambda r: -r["frequency"]):
            lst = top.setdefault(r["cdm_type"], [])
            if len(lst) < 10:
                lst.append(dict(entity_id=r["entity_id"], normalized_term=r["normalized_term"], frequency=r["frequency"]))
    fs = [x for l in pubs for x in FINDING.values() if x["corpus_id"] == l["corpus_id"]]
    facets = {}
    for f, facet in (("amplicon_region", "amplicon_region"), ("extraction_kit", "extraction_kit"),
                     ("sequencing_platform", "platform"), ("sample_type", "sample_type"), ("conditions", "condition"),
                     ("subjects_group", "subjects_group"), ("target_gene", "target_gene")):
        o = _obj(st, f)
        for x in (o if isinstance(o, list) else [o]):
            if x.get("status") == "SELECTED" and x.get("term_id"):
                facets.setdefault(facet, []).append(x["term_id"])
    p0 = PUB[pubs[0]["corpus_id"]] if pubs else {}
    labels = [TERMS[t]["label"] for v in facets.values() for t in v if t in TERMS]
    ENTRIES.append(dict(
        study_id=s, title=st["title"] or p0.get("title", ""),
        links=dict(archive_url=st["archive_url"], doi_url=("https://doi.org/" + p0["doi"]) if p0.get("doi") else None,
                   pmc_url=None),
        publications=[dict(corpus_id=l["corpus_id"], doi=PUB[l["corpus_id"]]["doi"], title=PUB[l["corpus_id"]]["title"],
                           journal=PUB[l["corpus_id"]]["journal"] or None, year=PUB[l["corpus_id"]]["year"] or None,
                           role=l["role"], identity_status=PUB[l["corpus_id"]]["identity_status"]) for l in pubs],
        methods={f: _obj(st, f) for f in ("target_gene", "amplicon_region", "primers", "sequencing_platform",
                                         "read_layout", "avg_read_length", "extraction_kit", "bead_beating")},
        population={f: _obj(st, f) for f in ("sample_type", "body_site", "host_species", "conditions",
                                             "has_healthy_controls", "subjects_group", "age_range_text",
                                             "n_subjects", "country")},
        design=dict(study_design=_obj(st, "study_design"), intervention=_obj(st, "intervention"),
                    n_samples=_obj(st, "n_samples"), n_runs=st["n_runs"] or None),
        top_entities=top,
        findings=dict(n_accepted=sum(x["status"] == "ACCEPTED" for x in fs),
                      n_candidate=sum(x["status"] == "CANDIDATE" for x in fs),
                      examples=[dict(quote=x["quote"][:300], section_id=x["section_id"]) for x in fs
                                if x["status"] == "ACCEPTED"][:3]),
        quality=dict(audit_summary=json.loads(st["audit_summary"]), open_flags=json.loads(st["open_flags"]),
                     flag_codes=sorted({f["code"] for f in CAT_FLAGS if f["target_id"] == s})),
        facets=facets,
        search_text=" ".join([st["title"] or p0.get("title", "")] + labels),
        provenance=dict(release_id=RELEASE_ID, schema_version=CAT_VERSION, run_ids=[CAT_RUN_ID, NER_RUN_ID])))
FACET_INDEX = {}
for e in ENTRIES:
    for facet, tids in e["facets"].items():
        for tid in set(tids):
            k = (facet, tid)
            fi = FACET_INDEX.setdefault(k, dict(facet=facet, term_id=tid, label=TERMS.get(tid, {}).get("label", ""),
                                               n_studies=0, pubs=set(), parent_term_id=""))
            fi["n_studies"] += 1
            fi["pubs"].update(p["corpus_id"] for p in e["publications"])
facet_rows = [dict({k: v for k, v in fi.items() if k != "pubs"}, n_publications=len(fi["pubs"]))
              for fi in FACET_INDEX.values()]
print("C8 entries:", len(ENTRIES), "of", sum(s["catalogue_status"] == "INCLUDED" for s in STUDY.values()),
      "INCLUDED studies | facet values:", len(facet_rows))

# ---------- C7: validation (schema rules, in order) ----------
def _pk(t):
    pk = CAT_SCHEMA["tables"][t]["primary_key"]
    return pk if isinstance(pk, list) else [pk]
def _is_blank(v):
    return v is None or v == ""
V = {}
bad = []
for t, rows in CAT.items():                                     # 1 PK unique/non-null; FK resolve
    pk = _pk(t)
    keys = [tuple(r.get(k) for k in pk) for r in rows]
    bad += [f"{t}: null PK {k}" for k in keys if any(_is_blank(x) for x in k)]
    bad += [f"{t}: duplicate PK {k}" for k, n in _C(keys).items() if n > 1]
    for fn, spec in CAT_SCHEMA["tables"][t]["fields"].items():
        key = spec.get("key", "")
        if key.startswith("FK"):
            tt, tf = key.split("->")[1].strip().split(".")
            if tt in ("ner_entities_corpus",):
                targets = {r["entity_id"] for r in ner_corpus}
            elif tt in CAT:
                targets = {r.get(tf) for r in CAT[tt]}
            else:
                continue
            bad += [f"{t}.{fn}={r[fn]!r} -> {tt}.{tf} missing" for r in rows if not _is_blank(r.get(fn)) and r[fn] not in targets]
V["01_keys_unique_and_fks_resolve"] = bad
bad = []
for t, rows in CAT.items():                                     # 2 required + enums
    fields = dict(CAT_SCHEMA["common_fields"], **CAT_SCHEMA["tables"][t]["fields"])
    for fn, spec in fields.items():
        if fn == "...":
            continue
        for r in rows:
            v = r.get(fn)
            if spec.get("required") and _is_blank(v) and not (t == "study" and fn in ("n_runs",)):
                bad.append(f"{t}.{fn} required, blank ({r.get(_pk(t)[0])})")
            if spec.get("type") == "enum" and not _is_blank(v) and v not in spec.get("values", []):
                bad.append(f"{t}.{fn}={v!r} not in enum")
V["02_required_present_enums_allowed"] = bad[:200]
V["03_one_study_per_projects_row"] = [] if len(STUDY) == len(PROJECTS) == len({p["project"] for p in PROJECTS}) else \
    [f"{len(STUDY)} studies vs {len(PROJECTS)} rows"]
V["04_duplicates_share_canonical_publication"] = [
    s for s in STUDY if CANONICAL[s] != s and
    {l["corpus_id"] for l in LINKS if l["study_id"] == s} != {l["corpus_id"] for l in LINKS if l["study_id"] == CANONICAL[s]}
] + [ELIG[s]["corpus_id"] for s in STUDY if CANONICAL[s] != s and ELIG[s]["corpus_id"] in PUB]
vids = {v["value_id"] for v in VALUES}
bad = []
for s, st in STUDY.items():                                     # 5 curated objects
    for f in CURATED_FIELDS:
        o = json.loads(st[f])
        for x in (o if isinstance(o, list) else [o]):
            if "status" not in x:
                bad.append(f"{s}.{f} has no status")
            elif x["status"] == "SELECTED" and x.get("value_id") and x["value_id"] not in vids:
                bad.append(f"{s}.{f} SELECTED value_id missing")
            elif x["status"] == "SELECTED" and f not in LIST_FIELDS and not x.get("value_id"):
                bad.append(f"{s}.{f} SELECTED without value_id")
V["05_curated_objects_have_status"] = bad
bad = []
for (s, f), vs in _by_sf.items():                               # 6 no SELECTED while sources disagree
    live = [v for v in vs if v["selection_status"] != "REJECTED"]
    if f not in LIST_FIELDS and len({_key(v) for v in live}) > 1 and any(v["selection_status"] == "SELECTED" for v in live) \
            and (s, f) not in reviews:
        bad.append(f"{s}.{f}")
V["06_no_selected_while_sources_disagree"] = bad
bad = [v["value_id"] for v in VALUES if v["source"] in ("PAPER_TEXT", "SUPPLEMENT") and
       PUB_TEXT.get(v["corpus_id"], {}).get(v["section_id"], "")[int(v["char_start"]):int(v["char_end"])] != v["quote"]]
bad += [x["finding_id"] for x in FINDING.values() if x["evidence_check"] == "PASSED" and
        PUB_TEXT[x["corpus_id"]][x["section_id"]][x["char_start"]:x["char_end"]] != x["quote"]]
bad += [v["value_id"] for v in VALUES if len(v["quote"]) > 300]
V["07_evidence_spans_reproduce_quotes"] = bad
V["08_facets_point_to_terms_local_terms_defined"] = (
    [f"{e['study_id']}:{t}" for e in ENTRIES for ts in e["facets"].values() for t in ts if t not in TERMS] +
    [t["term_id"] for t in TERMS.values() if t["vocabulary"] == "LOCAL" and t["term_status"] == "ACTIVE"
     and not (t["label"] and t["definition"])])
V["09_entries_only_included_without_blocker"] = [e["study_id"] for e in ENTRIES
                                                 if STUDY[e["study_id"]]["catalogue_status"] != "INCLUDED" or e["study_id"] in blockers]
V["10_summary_counts_match"] = (
    [s for s, st in STUDY.items() if st["n_publications"] != sum(1 for l in LINKS if l["study_id"] == s and l["link_status"] == "CONFIRMED")] +
    [c for c, p in PUB.items() if p["n_sections"] != sum(1 for x in SECTION.values() if x["corpus_id"] == c)])
run_ids = {r["run_id"] for r in pipeline_rows}
V["11_run_and_release_ids_exist"] = [f"{t}" for t, rows in CAT.items() for r in rows
                                     if r["run_id"] not in run_ids or r["release_id"] != RELEASE_ID][:50]
# 12 is checked on the written export below
for k, v in V.items():
    print(f"  {k:48s} {'ok' if not v else f'FAIL ({len(v)})'}")

# ---------- C8b: draft release + exports ----------
REL_DIR = CAT_DIR / "releases" / RELEASE_ID
assert not REL_DIR.exists(), f"release {RELEASE_ID} already exists - releases are immutable"
(REL_DIR / "tables").mkdir(parents=True)
EXPORT_DROP = {"source_file": {"local_path"}}
checksums, counts = {}, {}
all_tables = dict(CAT, catalogue_entry=[dict(e, created_at=CAT_CREATED_AT) for e in ENTRIES],
                  facet_index=facet_rows)
for t, rows in all_tables.items():
    spec = CAT_SCHEMA["tables"].get(t) or CAT_SCHEMA["views"][t]
    cols = [c for c in list(spec["fields"]) + (list(CAT_SCHEMA["common_fields"]) if t in CAT_SCHEMA["tables"] else [])
            if c != "..." and c not in EXPORT_DROP.get(t, set())]
    out = []
    for r in rows:
        out.append({c: (json.dumps(r.get(c), ensure_ascii=False) if isinstance(r.get(c), (dict, list)) else r.get(c, ""))
                    for c in cols})
    p = REL_DIR / "tables" / f"{t}.csv"
    dump_csv(p, out, cols)
    checksums[t], counts[t] = file_sha256(p), len(rows)
    try:
        import pyarrow as _pa, pyarrow.parquet as _pq
        _pq.write_table(_pa.Table.from_pylist([{k: (None if v == "" else str(v)) for k, v in r.items()} for r in out]
                                              or [{c: None for c in cols}]), REL_DIR / "tables" / f"{t}.parquet")
    except ImportError:
        pass
for t in ("ner_entities_paper", "ner_entities_corpus"):          # referenced from the NER build, not copied
    counts[t] = len(ner_paper if t.endswith("paper") else ner_corpus)
    checksums[t] = file_sha256(OUT_DIR / f"{t}.csv")
dd = [dict(table=t, field=f, type=s.get("type", ""), required=s.get("required", False), key=s.get("key", ""),
           curated=s.get("curated", False), values="|".join(map(str, s.get("values", []))) if isinstance(s.get("values"), list) else "",
           source=s.get("source", ""), description=s.get("description", ""))
      for t, spec in list(CAT_SCHEMA["tables"].items()) + list(CAT_SCHEMA["views"].items())
      for f, s in spec["fields"].items()]
dump_csv(REL_DIR / "data_dictionary.csv", dd, list(dd[0]))
(REL_DIR / "entries").mkdir()
(REL_DIR / "jsonld").mkdir()
with open(REL_DIR / "entries.jsonl", "w", encoding="utf-8") as fh:
    for e in ENTRIES:
        fh.write(json.dumps(e, ensure_ascii=False) + "\n")
        (REL_DIR / "entries" / f"{e['study_id']}.json").write_text(json.dumps(e, indent=1, ensure_ascii=False), encoding="utf-8")
        m = e["methods"]
        (REL_DIR / "jsonld" / f"{e['study_id']}.jsonld").write_text(json.dumps({
            "@context": "https://schema.org/", "@type": "Dataset", "name": e["title"], "identifier": e["study_id"],
            "url": e["links"]["archive_url"],
            "citation": [e["links"]["doi_url"]] if e["links"]["doi_url"] else [],
            "measurementTechnique": [x["value"] for x in (m["target_gene"], m["sequencing_platform"]) if x.get("value")],
            "keywords": sorted({TERMS[t]["label"] for ts in e["facets"].values() for t in ts if t in TERMS})},
            indent=1, ensure_ascii=False), encoding="utf-8")
dump_csv(REL_DIR / "mixs_crosswalk_PROPOSED.csv",
         [dict(study_field=a, mixs_term=b, status="PROPOSED - verify against the current MIxS release")
          for a, b in (("target_gene", "target_gene"), ("amplicon_region", "target_subfragment"),
                       ("primers", "pcr_primers"), ("sequencing_platform", "seq_meth"),
                       ("extraction_kit", "nucl_acid_ext"))], ["study_field", "mixs_term", "status"])
export_text = "".join(p.read_text(encoding="utf-8", errors="ignore") for p in REL_DIR.rglob("*") if p.suffix in (".csv", ".json", ".jsonl", ".jsonld"))
V["12_exports_no_local_path_or_full_text"] = (["local_path column exported"] if "local_path" in
                                               (REL_DIR / "tables" / "source_file.csv").read_text(encoding="utf-8").splitlines()[0] else []) + \
    [x["finding_id"] for x in FINDING.values() if len(x["quote"]) > 1000]
print(f"  {'12_exports_no_local_path_or_full_text':48s} {'ok' if not V['12_exports_no_local_path_or_full_text'] else 'FAIL'}")
release = dict(release_id=RELEASE_ID, created_at=CAT_CREATED_AT, schema_version=CAT_VERSION,
               ner_template_version=NER_SCHEMA_VERSION + " (ner_entity_template.yaml not on disk)",
               counts=counts, checksums=checksums, approved_by="NOT_APPROVED",
               notes="Draft cut by the notebook. Values are PROVISIONAL_UNVALIDATED; LOCAL terms are PROPOSED and "
                     "have no team definitions yet; archive titles, runs and licences are not fetched.")
(REL_DIR / "release.json").write_text(json.dumps(release, indent=1), encoding="utf-8")
CAT_VALIDATION = dict(status="PASS" if not any(V.values()) else "FAIL", release_id=RELEASE_ID,
                      checks={k: dict(violations=len(v), examples=[str(x)[:200] for x in v[:5]]) for k, v in V.items()},
                      open_items=dict(local_terms_without_definition=sum(1 for t in TERMS.values() if t["vocabulary"] == "LOCAL" and not t["definition"]),
                                      not_attempted_fields=sorted({f for s in STUDY.values() for f in CURATED_FIELDS
                                                                   if '"NOT_ATTEMPTED"' in s[f]})))
(REL_DIR / "validation_report.json").write_text(json.dumps(CAT_VALIDATION, indent=1), encoding="utf-8")
(CAT_DIR / "LATEST_RELEASE.txt").write_text(RELEASE_ID + "\n", encoding="utf-8")
print("release:", REL_DIR, "| tables:", counts)
print("flags:", dict(_C((f["code"], f["severity"]) for f in CAT_FLAGS)))
if CAT_VALIDATION["status"] != "PASS":
    REL_DIR.rename(REL_DIR.with_name(RELEASE_ID + "_FAILED"))
assert CAT_VALIDATION["status"] == "PASS", "catalogue validation failed - see validation_report.json in the release folder"

## 15. Corpus building template (Cells 13a-13c)

Fills `human_microbiome_corpus_building_template.md` 1.0.0 for the papers in this run: the §2 tracking table (all rows), one §3 Markdown record per paper, §3G study blocks and §4 findings extracted by the LLM chosen in Cell 12K (quote-verified, every number must appear in its quote), the §5 curation log and the §7 corpus report. Outputs: `OUT_DIR/corpus_template/`. Everything is `extracted_unverified`.

In [ ]:
# ===== Cell 13a: corpus template (human_microbiome_corpus_building_template.md 1.0.0) - §2 tracking table, §3 metadata =====
# Builds the template's outputs from what the pipeline already holds. Nothing is guessed: a field with no source
# stays blank (null), "not_reported" is used only where the relevant text was checked by an extractor, and
# "not_applicable" only where the field cannot apply. Scope: the tracking table covers every corpus row
# (482); paper records, study blocks and findings cover the papers in this run (PAPERS).
# Outputs: OUT_DIR/corpus_template/ (tracking_table.csv, papers/*.md, studies.csv, study_fields.csv,
#          findings.csv, curation_log.csv, corpus_report.json, corpus_validation.json).
TPL_VERSION = "1.0.0"
TPL_DIR = OUT_DIR / "corpus_template"
(TPL_DIR / "papers").mkdir(parents=True, exist_ok=True)
CURATION_LOG = []

def log_event(cid, stage, tool, version, action, result, problem="", resolution=""):
    CURATION_LOG.append(dict(corpus_id=cid, date=_now()[:10], stage=stage, tool_person=tool, tool_version=version,
                             action_check=action, result=result, problem_found=problem, resolution=resolution))

# template status vocabularies (§2 "Standard status values")
def tpl_processing(e, pv):
    if e["final_category"] == "excluded" or STUDY.get(e["expected_study_id"], {}).get("catalogue_status") == "EXCLUDED_WRONG_PAPER":
        return "excluded"
    if e["final_category"] == "placeholder_no_full_text":
        return "not_started"
    st = pv.get("status", "")
    if st in ("failed",):
        return "needs_repair"
    if st in ("validated", "validated_with_warnings") and e["final_category"] == "eligible":
        return "automated_checks_passed"            # never manual_review_passed: no human review recorded
    if st:
        return "text_extracted"
    return "source_acquired" if DL_AUDIT.get(e["expected_study_id"], {}).get("matched_local_file") else "not_started"

def tpl_fulltext(pv):
    return {"validated": "complete", "validated_with_warnings": "complete", "needs_review": "partial",
            "failed": "failed"}.get(pv.get("status", ""), "not_checked")

def tpl_inclusion(e):
    return {"eligible": "include", "excluded": "exclude"}.get(e["final_category"], "pending")

TRACKING = []
for sid_, e in sorted(ELIG.items(), key=lambda kv: kv[1]["corpus_id"]):
    cid, a, pv = e["corpus_id"], DL_AUDIT[sid_], _prov(sid_)
    canon = ELIG[CANONICAL[sid_]]["corpus_id"]
    notes = [e["exclusion_or_review_reason"]] if e["exclusion_or_review_reason"] else []
    if canon != cid:
        notes.append(f"duplicate record of {canon} ({CANONICAL[sid_]})")
    TRACKING.append({"Corpus ID": cid, "Title": a["expected_title"], "DOI / PMID / PMCID": a["expected_doi"],
                     "Year": a["expected_year"], "Source / file path": pv.get("chosen_source", a["matched_local_file"]),
                     "Inclusion decision": tpl_inclusion(e), "Processing status": tpl_processing(e, pv),
                     "Full-text status": tpl_fulltext(pv), "Annotation status": "not_started",
                     "Reviewer / notes": "; ".join(notes)[:500], "_study_id": sid_, "_canonical": canon})
TRACK_BY_CID = {r["Corpus ID"]: r for r in TRACKING}
print("§2 tracking rows:", len(TRACKING), "| inclusion:", dict(_C(r["Inclusion decision"] for r in TRACKING)),
      "| processing:", dict(_C(r["Processing status"] for r in TRACKING)))
print("note: the template expects 453 records (HMPAPER_000001-000453); this corpus has", len(TRACKING), "rows")

# §3 B/C/D/E per paper in this run (from the audit, provenance, segmentation and catalogue tables)
# Scope (NB_TPL_SCOPE): "run" = the papers in this run (PAPERS); "included" = every eligible publication with text
# (no NER rerun needed: study blocks and findings read the section text directly). A whole-corpus run is long:
# about 10 LLM windows per paper - tell the FhGenie team first, and watch the 30-day token expiry.
TPL_SCOPE = os.environ.get("NB_TPL_SCOPE", "run").strip().lower()
if TPL_SCOPE == "included":
    TPL_PAPERS = sorted(c for c, t in TRACK_BY_CID.items() if t["Inclusion decision"] == "include" and c in PUB and c in PUB_TEXT)
else:
    TPL_PAPERS = [c for c in PAPERS if c in TRACK_BY_CID]
PAPER_META = {}
for cid in TPL_PAPERS:
    t = TRACK_BY_CID[cid]
    sid_ = t["_study_id"]
    a, e, pv = DL_AUDIT[sid_], ELIG[sid_], _prov(sid_)
    srcs = pv.get("all_sources", [])
    by_fmt = {s["format"]: s for s in srcs}
    p = PUB.get(cid, {})
    PAPER_META[cid] = dict(
        title=a["expected_title"], doi=a["expected_doi"], pmid=None, pmcid=None, authors=None, journal=None,
        year=a["expected_year"], first_author=a["expected_first_author"],
        publication_type="primary_research" if p.get("publication_type") == "RESEARCH_ARTICLE" else (
            "other" if p else None),
        language=None, abstract=None, keywords=None,
        source_repository=None, source_url=a["link"], retrieval_date=None, licence=None, licence_url=None,
        reuse_checked="no",
        pdf_path=by_fmt.get("pdf", {}).get("path"), xml_path=by_fmt.get("xml", {}).get("path"),
        text_path=f"segmented_sections_markdown/{cid}.md", supplementary=None,
        source_sha256=pv.get("source_sha256"),
        extraction_tool=f"markdown_harmonization {pv.get('pipeline_version', '')} ({pv.get('chosen_format', '')})",
        text_checksum=p.get("text_sha256"),
        relevance="yes" if e["final_category"] == "eligible" else "unclear",
        inclusion=t["Inclusion decision"], inclusion_reason="eligibility audit: " + e["current_outcome"],
        exclusion_reason=e["exclusion_or_review_reason"] or None, protocol="segmentation eligibility gate (not frozen)",
        duplicate_of=t["_canonical"] if t["_canonical"] != cid else None,
        title_match="pass" if a["doc_title_matched"] == "True" else ("fail" if a["doc_title_matched"] == "False" else "not_checked"),
        abstract_match="not_checked", body="complete" if pv.get("status", "").startswith("validated") else "not_checked",
        hierarchy="yes" if p.get("segmentation_status") == "OK" else "not_checked",
        tables="no", captions="no", supplements="no", references="not_checked",
        problems="; ".join(pv.get("warnings", []) or []) or None)
    log_event(cid, "metadata", "catalogue Cell 12a", CAT_VERSION, "copy bibliographic and provenance fields", "done")
    log_event(cid, "section_validation", "segmentation", "seg_core", "exact-slice validation",
              PUB.get(cid, {}).get("segmentation_status", "NOT_RUN"))
print("§3 paper metadata for:", TPL_PAPERS)

In [ ]:
# ===== Cell 13b: corpus template §3G study blocks and §4 findings - LLM over the desired sections =====
# Uses the backend chosen in Cell 12K (FhGenie or local llama.cpp) through chat_json() from Cell 12L1.
# Sections sent: S7-kept, non-backmatter, routed by class (STUDY_ROUTES / FINDING_CLASSES). The one exception:
# a Data availability section is opened ONLY for data_accessions and code_data_availability.
# Every item needs a verbatim quote found in the excerpt, and every number in a value must appear in its quote.
# A study block can hold several studies ("study_label"); findings name the study they belong to.
# Everything stays extracted_unverified. Results are cached per window (successes only).
TPL_PROMPT_VERSION = "corpus-template-v1"
STUDY_FIELDS = {   # field -> (section classes, hint)
    "study_design": ({"TITLE_ABSTRACT", "METHODS"}, "design as stated, e.g. case-control, cross-sectional, cohort, RCT"),
    "research_question": ({"TITLE_ABSTRACT", "INTRODUCTION"}, "the study's stated objective"),
    "country_region": ({"METHODS", "TITLE_ABSTRACT"}, "where participants were recruited/sampled"),
    "recruitment_setting": ({"METHODS"}, "hospital, clinic, community, ..."),
    "population_description": ({"METHODS", "TITLE_ABSTRACT"}, "who was studied"),
    "n_participants": ({"METHODS", "TITLE_ABSTRACT", "RESULTS"}, "number of participants, as a number"),
    "n_samples": ({"METHODS", "RESULTS"}, "number of biological samples, as a number (not participants)"),
    "n_technical_replicates": ({"METHODS"}, "number of technical replicates, as a number"),
    "n_time_points": ({"METHODS"}, "number of sampling time points, as a number"),
    "age_information": ({"METHODS", "RESULTS"}, "age range/mean as written"),
    "sex_gender_information": ({"METHODS", "RESULTS"}, "sex/gender as reported"),
    "health_conditions": ({"TITLE_ABSTRACT", "METHODS"}, "conditions studied"),
    "exposures_interventions": ({"METHODS", "TITLE_ABSTRACT"}, "exposures or interventions"),
    "comparator_control_group": ({"METHODS", "TITLE_ABSTRACT"}, "control/comparator group"),
    "eligibility_criteria": ({"METHODS"}, "inclusion/exclusion criteria"),
    "confounders_covariates": ({"METHODS"}, "covariates adjusted for"),
    "limitations_reported": ({"DISCUSSION", "CONCLUSION"}, "limitations the authors state"),
    "body_sites": ({"METHODS", "TITLE_ABSTRACT"}, "body site(s) sampled"),
    "specimen_types": ({"METHODS", "TITLE_ABSTRACT"}, "specimen types, e.g. stool, saliva, biopsy"),
    "collection_method": ({"METHODS"}, "how samples were collected"),
    "collection_time_points": ({"METHODS"}, "when samples were collected"),
    "storage_transport": ({"METHODS"}, "storage/transport conditions"),
    "extraction_method": ({"METHODS"}, "DNA/RNA extraction method or kit"),
    "assay_type": ({"METHODS", "TITLE_ABSTRACT"}, "one of: 16S, shotgun_metagenomics, metatranscriptomics, metabolomics, culture, other"),
    "sequencing_platform": ({"METHODS"}, "sequencing instrument"),
    "target_gene_region": ({"METHODS"}, "target gene and region, e.g. 16S V4"),
    "negative_positive_controls": ({"METHODS"}, "negative/positive controls used"),
    "contamination_handling": ({"METHODS"}, "how contamination was handled"),
    "taxonomic_profiling_method": ({"METHODS"}, "e.g. DADA2, QIIME 2, MetaPhlAn"),
    "taxonomic_database_version": ({"METHODS"}, "reference database and version, e.g. SILVA 138"),
    "functional_profiling_method": ({"METHODS"}, "e.g. PICRUSt2, HUMAnN"),
    "quality_filtering_normalization": ({"METHODS"}, "quality filtering, rarefaction, normalization"),
    "statistical_methods": ({"METHODS"}, "statistical tests/models"),
    "multiple_testing_correction": ({"METHODS"}, "e.g. Benjamini-Hochberg FDR"),
    "software_pipeline_version": ({"METHODS"}, "software and versions"),
    "data_accessions": ({"DATA_AVAILABILITY", "METHODS"}, "BioProject/SRA/ENA/other accessions"),
    "code_data_availability": ({"DATA_AVAILABILITY"}, "code/data availability statement"),
}
NUMERIC_STUDY = {"n_participants", "n_samples", "n_technical_replicates", "n_time_points"}
ASSAY_TYPES = {"16S", "shotgun_metagenomics", "metatranscriptomics", "metabolomics", "culture", "other"}
FINDING_CLASSES = {"RESULTS", "TITLE_ABSTRACT"}
CLAIM_TYPES = {"result", "null_result", "author_interpretation", "other"}
DIRECTIONS = {"increased", "decreased", "no_association_reported", "mixed", "unclear"}
FINDING_FIELDS = ["study_label", "claim_type", "body_site", "population_subgroup", "exposure_comparison", "outcome",
                  "taxon", "taxonomic_rank", "function_pathway", "direction", "direction_refers_to", "effect_measure",
                  "effect_estimate", "p_value", "q_value", "confidence_interval", "adjusted_covariates", "evidence_quote"]
NUMERIC_FINDING = {"effect_estimate", "p_value", "q_value", "confidence_interval"}

STUDY_SYSTEM = (
    "You extract the study metadata of ONE microbiome paper from one excerpt. Report only what the excerpt states "
    "about this paper's own participants, samples and methods - never about cited studies. If the paper has several "
    "distinct cohorts or experiments, give each a short study_label (e.g. 'main', 'validation cohort'); otherwise use "
    "'main'. For every value give `quote`: a passage of at most 300 characters copied character for character from "
    "the excerpt. Leave out any field the excerpt does not state; never infer or guess. JSON only: "
    '{"studies": [{"study_label": ..., "fields": [{"field": ..., "value": ..., "quote": ...}]}]}')
FINDING_SYSTEM = (
    "You extract the reported scientific findings of ONE microbiome paper from one excerpt of its results. One item "
    "per distinct result; include null and contradictory results; never include background claims about other "
    "studies. Copy statistics exactly as written; leave a field empty if not reported - never compute or infer a "
    "statistic. evidence_quote: at most 300 characters copied character for character from the excerpt. claim_type: "
    "result | null_result | author_interpretation | other. direction: increased | decreased | "
    "no_association_reported | mixed | unclear. JSON only: {\"findings\": [{...}]}")

def study_schema(fields):
    item = {"type": "object", "additionalProperties": False, "required": ["field", "value", "quote"],
            "properties": {"field": {"type": "string", "enum": sorted(fields)}, "value": {"type": "string"},
                           "quote": {"type": "string"}}}
    return {"type": "object", "additionalProperties": False, "required": ["studies"], "properties": {"studies": {
        "type": "array", "items": {"type": "object", "additionalProperties": False, "required": ["study_label", "fields"],
                                   "properties": {"study_label": {"type": "string"},
                                                  "fields": {"type": "array", "items": item}}}}}}

def finding_schema():
    props = {f: {"type": "string"} for f in FINDING_FIELDS}
    props["claim_type"] = {"type": "string", "enum": sorted(CLAIM_TYPES)}
    props["direction"] = {"type": "string", "enum": sorted(DIRECTIONS)}
    return {"type": "object", "additionalProperties": False, "required": ["findings"], "properties": {"findings": {
        "type": "array", "items": {"type": "object", "additionalProperties": False,
                                   "required": ["claim_type", "direction", "evidence_quote"], "properties": props}}}}

def numbers_in(value):
    return re.findall(r"\d+(?:\.\d+)?(?:[eE][-−]?\d+)?", (value or "").replace("−", "-"))

def numbers_supported(value, quote):
    q = quote.replace(",", "").replace("−", "-")
    return all(re.search(rf"(?<![\d.]){re.escape(n)}(?![\d])", q) for n in numbers_in(value.replace(",", "")))

def tpl_section_class(x, whole_paper=False):
    """Section class for routing. A '(preamble)' section, or an unclassified section in a paper without any
    Methods/Results heading, holds the whole paper text, so it is routed like PREAMBLE (all fields + findings)."""
    if re.search(r"data availability|availability of data|accession", x["section_path"], re.I):
        return "DATA_AVAILABILITY"
    if x["section_path"].strip() == "(preamble)" or (whole_paper and x["section_class"] == "OTHER_BODY"):
        return "PREAMBLE"
    return x["section_class"]

# ---------- jobs ----------
TPL_JOBS, TPL_SENT = [], []
for cid in TPL_PAPERS:
    if cid not in PUB:
        continue
    secs_c = sorted((x for x in SECTION.values() if x["corpus_id"] == cid), key=lambda x: x["ordinal"])
    whole = not any(x["section_class"] in ("METHODS", "RESULTS") for x in secs_c)
    for x in secs_c:
        sid, cls = x["section_id"], tpl_section_class(x, whole)
        keep, reason = select_section(cid, sid, x["section_path"], SECTION_DECISIONS)
        if cls == "DATA_AVAILABILITY" and x["selection_status"] not in ("IRRELEVANT", "CONTAINER_ONLY"):
            keep, reason = True, "data availability (accession fields only)"
        sfields = sorted(set(STUDY_FIELDS) if cls == "PREAMBLE" else {f for f, (c, _) in STUDY_FIELDS.items() if cls in c})
        do_find = cls in FINDING_CLASSES or cls == "PREAMBLE"
        TPL_SENT.append(dict(corpus_id=cid, section_id=sid, section_path=x["section_path"][-150:], section_class=cls,
                             sent=int(keep and bool(sfields or do_find)), reason=reason,
                             study_fields="|".join(sfields) if keep else "", findings=int(keep and do_find)))
        if not keep:
            continue
        for off, chunk in windows(PUB_TEXT[cid][sid]):
            if sfields:
                TPL_JOBS.append(dict(kind="study", corpus_id=cid, section_id=sid, offset=off, text=chunk, fields=sfields))
            if do_find:
                TPL_JOBS.append(dict(kind="finding", corpus_id=cid, section_id=sid, offset=off, text=chunk, fields=FINDING_FIELDS))
dump_csv(TPL_DIR / "llm_sections_sent.csv", TPL_SENT, list(TPL_SENT[0]) if TPL_SENT else ["corpus_id"])
print(f"§3G/§4 jobs: {len(TPL_JOBS)} windows ({dict(_C(j['kind'] for j in TPL_JOBS))}) from "
      f"{sum(r['sent'] for r in TPL_SENT)} of {len(TPL_SENT)} sections")

def tpl_key(j):
    return hashlib.sha256(json.dumps([LLM_MODEL, TPL_PROMPT_VERSION, j["kind"], j["fields"], j["text"]]).encode()).hexdigest()

def tpl_run(j):
    k = tpl_key(j)
    if k in llm_cache:
        return j, llm_cache[k], "cached"
    if j["kind"] == "study":
        hints = "\n".join(f"- {f}: {STUDY_FIELDS[f][1]}" for f in j["fields"])
        msgs = [{"role": "system", "content": STUDY_SYSTEM},
                {"role": "user", "content": f"STUDY FIELDS (only these):\n{hints}\n\nExcerpt:\n<<<\n{j['text']}\n>>>"}]
        schema = study_schema(j["fields"])
    else:
        msgs = [{"role": "system", "content": FINDING_SYSTEM},
                {"role": "user", "content": f"FINDINGS fields: {', '.join(FINDING_FIELDS)}\n\nExcerpt:\n<<<\n{j['text']}\n>>>"}]
        schema = finding_schema()
    try:
        out, raw, mode = chat_json(msgs, schema, max_tokens=LLM_MAX_TOKENS)
    except LLMAuthError:
        raise
    except Exception as ex:
        return j, dict(key=k, error=redact(f"{type(ex).__name__}: {ex}")[:300]), "failed"
    return j, dict(key=k, model=LLM_MODEL, prompt_version=TPL_PROMPT_VERSION, mode=mode, at=_now(),
                   items=out if isinstance(out, dict) else {}), "new"

# ---------- run ----------
TPL_STUDY_ITEMS, TPL_FINDINGS, TPL_REJECTED, TPL_STATUS, TPL_OK = [], [], [], _C(), []
_restarted = False
if LLM_ENABLED and TPL_JOBS:
    if LLM_BACKEND == "llama_cpp" and LLM_LOCAL_PROC is None:
        start_local_qwen(); _restarted = True                 # Cell 12L2 stopped it
    from concurrent.futures import ThreadPoolExecutor
    try:
        with ThreadPoolExecutor(max_workers=LLM_WORKERS) as ex, open(LLM_CAT_CACHE, "a", encoding="utf-8") as fh:
            for n, (j, res, how) in enumerate(ex.map(tpl_run, TPL_JOBS), 1):
                TPL_STATUS[how] += 1
                if how == "failed":
                    log_event(j["corpus_id"], "study_annotation" if j["kind"] == "study" else "finding_extraction",
                              LLM_MODEL, TPL_PROMPT_VERSION, f"{j['kind']} window {j['section_id']}@{j['offset']}",
                              "failed", res["error"], "retried next run")
                    continue
                if how == "new":
                    fh.write(json.dumps(res, ensure_ascii=False) + "\n"); fh.flush()
                    llm_cache[res["key"]] = res
                TPL_OK.append(j)
                full = PUB_TEXT[j["corpus_id"]][j["section_id"]]
                def _locate(q):
                    loc = locate_quote(j["text"], q)
                    if not loc:
                        return None
                    s, e = j["offset"] + loc[0], j["offset"] + min(loc[1], loc[0] + 300)
                    return s, e, full[s:e]
                if j["kind"] == "study":
                    for stu in (res["items"].get("studies") or []):
                        label = (stu.get("study_label") or "main").strip()[:60] or "main"
                        for it in (stu.get("fields") or []):
                            f, v, q = (it.get("field") or "").strip(), str(it.get("value") or "").strip(), it.get("quote") or ""
                            base = dict(kind="study", corpus_id=j["corpus_id"], section_id=j["section_id"], study_label=label,
                                        field=f, value=v, quote_model=q[:300])
                            if f not in j["fields"]:
                                TPL_REJECTED.append(dict(base, reason="field not routed to this section")); continue
                            if not v or v.lower() in ("not stated", "unknown", "n/a", "none", "null", "not reported"):
                                TPL_REJECTED.append(dict(base, reason="empty value")); continue
                            if f == "assay_type" and v not in ASSAY_TYPES:
                                TPL_REJECTED.append(dict(base, reason="assay_type not in template list")); continue
                            loc = _locate(q)
                            if not loc:
                                TPL_REJECTED.append(dict(base, reason="quote not found in the excerpt")); continue
                            if (f in NUMERIC_STUDY and not numbers_in(v)) or not numbers_supported(v, loc[2]):
                                TPL_REJECTED.append(dict(base, reason="number not in quote")); continue
                            TPL_STUDY_ITEMS.append(dict(base, char_start=loc[0], char_end=loc[1], quote=loc[2]))
                else:
                    for it in (res["items"].get("findings") or []):
                        it = {k: str(v).strip() if v is not None else "" for k, v in it.items() if k in FINDING_FIELDS}
                        base = dict(kind="finding", corpus_id=j["corpus_id"], section_id=j["section_id"],
                                    value=it.get("taxon", ""), quote_model=it.get("evidence_quote", "")[:300])
                        if it.get("claim_type") not in CLAIM_TYPES or it.get("direction") not in DIRECTIONS:
                            TPL_REJECTED.append(dict(base, field="finding", reason="claim_type/direction not allowed")); continue
                        loc = _locate(it.get("evidence_quote", ""))
                        if not loc:
                            TPL_REJECTED.append(dict(base, field="finding", reason="quote not found in the excerpt")); continue
                        if not (it.get("taxon") or it.get("function_pathway") or it.get("outcome")):
                            TPL_REJECTED.append(dict(base, field="finding", reason="no taxon, function or outcome")); continue
                        bad = [f for f in NUMERIC_FINDING if it.get(f) and not numbers_supported(it[f], loc[2])]
                        if bad:
                            TPL_REJECTED.append(dict(base, field="finding", reason="statistic not in quote: " + ",".join(bad))); continue
                        TPL_FINDINGS.append(dict(it, corpus_id=j["corpus_id"], section_id=j["section_id"],
                                                 char_start=loc[0], char_end=loc[1], evidence_text=loc[2]))
                if n % 10 == 0:
                    print(f"  template windows {n}/{len(TPL_JOBS)}", flush=True)
    finally:
        if LLM_BACKEND == "llama_cpp" and _restarted:
            stop_local_qwen()
print("template LLM windows:", dict(TPL_STATUS), "| study items:", len(TPL_STUDY_ITEMS), "| findings:", len(TPL_FINDINGS),
      "| rejected:", len(TPL_REJECTED), dict(_C(r["reason"].split(":")[0] for r in TPL_REJECTED)))
TPL_ATTEMPTED = {(j["corpus_id"], f) for j in TPL_OK if j["kind"] == "study" for f in j["fields"]}
TPL_FIND_ATTEMPTED = {j["corpus_id"] for j in TPL_OK if j["kind"] == "finding"}

In [ ]:
# ===== Cell 13c: corpus template §3 paper records, §4 finding records, §5 curation log, §7 report + checks =====
# Study IDs: <corpus_id>_STUDY_NNN, numbered in order of first appearance of each study_label ('main' first).
# Finding IDs: <corpus_id>_FIND_NNN, ordered by section then offset (stable for the same text and extraction).
# Several values for one study field are all kept (with their evidence); none is chosen automatically.
# Missing: left blank with a note - "no value found by the extractor; unconfirmed" where the routed sections were
# sent, "not checked" where they were not. `not_reported` is never set by the machine (template §2 rule).
def _label_key(s):
    return re.sub(r"[^a-z0-9]+", " ", (s or "main").lower()).strip() or "main"

STUDIES, STUDY_FIELD_ROWS = {}, []
for cid in TPL_PAPERS:
    labels = []
    for it in TPL_STUDY_ITEMS:
        if it["corpus_id"] == cid and _label_key(it["study_label"]) not in labels:
            labels.append(_label_key(it["study_label"]))
    labels = sorted(labels, key=lambda l: (l != "main", labels.index(l))) or ["main"]
    for n, lab in enumerate(labels, 1):
        STUDIES[(cid, lab)] = dict(study_id=f"{cid}_STUDY_{n:03d}", corpus_id=cid, study_label=lab)
seen = set()
for it in TPL_STUDY_ITEMS:
    st = STUDIES[(it["corpus_id"], _label_key(it["study_label"]))]
    k = (st["study_id"], it["field"], it["value"].lower(), it["char_start"])
    if k in seen:
        continue                                            # same value from overlapping windows
    seen.add(k)
    STUDY_FIELD_ROWS.append(dict(study_id=st["study_id"], corpus_id=it["corpus_id"], field=it["field"], value=it["value"],
                                 section_id=it["section_id"], char_start=it["char_start"], char_end=it["char_end"],
                                 evidence_text=it["quote"], offset_scope="section-local text (segmented_sections_markdown, markers removed)",
                                 extraction_method="automated", extractor=f"{LLM_MODEL}|{TPL_PROMPT_VERSION}",
                                 verification_status="pending"))

def _norm_taxon(t):
    if not t or "TAXON_MATCHER" not in globals():
        return "", ""
    hits = TAXON_MATCHER.find(t)
    if hits and hits[0][1] - hits[0][0] == len(t.strip()):
        p = hits[0][3][0]
        return p["name"], p["id"]
    return "", ""

FIND_ROWS = []
_fs, _ = dedupe_by(sorted(TPL_FINDINGS, key=lambda f: (f["corpus_id"], f["section_id"], f["char_start"])),
                   lambda f: (f["corpus_id"], f["section_id"], f["char_start"], f["char_end"], f.get("taxon", "").lower(),
                              f.get("direction")))
_counter = _C()
for f in _fs:
    cid = f["corpus_id"]
    _counter[cid] += 1
    st = STUDIES.get((cid, _label_key(f.get("study_label"))), STUDIES.get((cid, "main")))
    tn, tid = _norm_taxon(f.get("taxon", ""))
    bs = f.get("body_site", "")
    FIND_ROWS.append({
        "finding_id": f"{cid}_FIND_{_counter[cid]:03d}", "corpus_id": cid, "study_id": st["study_id"] if st else "",
        "study_link": "matched" if (cid, _label_key(f.get("study_label"))) in STUDIES else "defaulted_to_first_study",
        "claim_type": f.get("claim_type"), "body_site_original": bs,
        "body_site_normalized": TERMS.get(term_for("body_site", bs.lower()), {}).get("label", "") if bs else "",
        "population_subgroup": f.get("population_subgroup", ""), "exposure_comparison": f.get("exposure_comparison", ""),
        "outcome": f.get("outcome", ""), "taxon_original": f.get("taxon", ""), "taxon_normalized": tn, "taxon_id": tid,
        "taxonomic_rank": f.get("taxonomic_rank", ""), "function_pathway": f.get("function_pathway", ""),
        "association_direction": f.get("direction"), "direction_refers_to": f.get("direction_refers_to", ""),
        "effect_measure": f.get("effect_measure", ""), "effect_estimate": f.get("effect_estimate", ""),
        "p_value": f.get("p_value", ""), "q_value": f.get("q_value", ""), "confidence_interval": f.get("confidence_interval", ""),
        "adjusted_covariates": f.get("adjusted_covariates", ""), "evidence_text": f["evidence_text"],
        "evidence_section_id": f["section_id"], "evidence_locator_type": "section",
        "evidence_locator": f"{f['section_id']}[{f['char_start']}:{f['char_end']}]",
        "text_version_id": PUB.get(cid, {}).get("text_sha256", ""), "extraction_method": "automated",
        "extractor_model_version": f"{LLM_MODEL}|{TPL_PROMPT_VERSION}", "confidence": "",
        "verification_status": "pending", "reviewer": "", "review_date": "", "notes": ""})

for cid in TPL_PAPERS:
    n_items = sum(1 for r in STUDY_FIELD_ROWS if r["corpus_id"] == cid) + sum(1 for r in FIND_ROWS if r["corpus_id"] == cid)
    if n_items:
        TRACK_BY_CID[cid]["Annotation status"] = "extracted_unverified"
    elif any(j["corpus_id"] == cid for j in TPL_OK):
        TRACK_BY_CID[cid]["Annotation status"] = "in_progress"
    log_event(cid, "study_annotation", LLM_MODEL or "-", TPL_PROMPT_VERSION, "study fields extracted",
              f"{sum(1 for r in STUDY_FIELD_ROWS if r['corpus_id'] == cid)} values" if LLM_ENABLED else "skipped (no LLM)")
    log_event(cid, "finding_extraction", LLM_MODEL or "-", TPL_PROMPT_VERSION, "findings extracted",
              f"{sum(1 for r in FIND_ROWS if r['corpus_id'] == cid)} findings" if LLM_ENABLED else "skipped (no LLM)")
    log_event(cid, "evidence_validation", "Cell 13c", TPL_VERSION, "every quote re-sliced from the section text",
              "pass" if all(PUB_TEXT[cid][r["evidence_section_id"]][r["char_start"]:r["char_end"]] == r["evidence_text"]
                            for r in [dict(x, char_start=int(x["evidence_locator"].split("[")[1].split(":")[0]),
                                           char_end=int(x["evidence_locator"].split(":")[1].rstrip("]")))
                                      for x in FIND_ROWS if x["corpus_id"] == cid]) else "fail")

# ---------- write tables ----------
dump_csv(TPL_DIR / "tracking_table.csv", TRACKING, [k for k in TRACKING[0] if not k.startswith("_")])
dump_csv(TPL_DIR / "studies.csv", list(STUDIES.values()), ["study_id", "corpus_id", "study_label"])
dump_csv(TPL_DIR / "study_fields.csv", STUDY_FIELD_ROWS, list(STUDY_FIELD_ROWS[0]) if STUDY_FIELD_ROWS else ["study_id"])
dump_csv(TPL_DIR / "findings.csv", FIND_ROWS, list(FIND_ROWS[0]) if FIND_ROWS else ["finding_id"])
dump_csv(TPL_DIR / "rejected_items.csv", TPL_REJECTED,
         ["kind", "corpus_id", "section_id", "study_label", "field", "value", "quote_model", "reason"])

# ---------- §3 paper records (one .md each) ----------
def _v(x):
    return "" if x in (None, "") else str(x)

def _study_value(st_id, cid, f):
    vals = [r for r in STUDY_FIELD_ROWS if r["study_id"] == st_id and r["field"] == f]
    if vals:
        return "; ".join(f"{r['value']} [{r['section_id']}]" for r in vals[:4]) + (" ..." if len(vals) > 4 else "")
    # template rule: `not_reported` only after a person confirms the paper does not report it
    return "_(no value found by the extractor; unconfirmed)_" if (cid, f) in TPL_ATTEMPTED else "_(not checked)_"

for cid in TPL_PAPERS:
    m, t = PAPER_META[cid], TRACK_BY_CID[cid]
    secs_c = sorted((x for x in SECTION.values() if x["corpus_id"] == cid), key=lambda x: x["ordinal"])
    sent = {r["section_id"] for r in TPL_SENT if r["corpus_id"] == cid and r["sent"]}
    L = [f"# Paper Record: `{cid}`", "", "### A. Record control", "",
         f"- **Corpus ID:** `{cid}`", f"- **Schema version:** `{TPL_VERSION}`",
         f"- **Processing status:** `{t['Processing status']}`", f"- **Full-text completeness:** `{t['Full-text status']}`",
         f"- **Annotation status:** `{t['Annotation status']}`", "- **Reviewer:** ", "- **Review date:** ",
         f"- **Curation notes:** generated by pipeline_standalone.ipynb ({RUN_TAG}); machine-filled, not human-reviewed. {_v(t['Reviewer / notes'])}",
         "", "### B. Publication metadata", "",
         f"- **Title:** {_v(m['title'])}", f"- **DOI:** {_v(m['doi'])}", f"- **PMID:** {_v(m['pmid'])}",
         f"- **PMCID:** {_v(m['pmcid'])}", f"- **Authors:** {_v(m['authors'])} (first author: {_v(m['first_author'])})",
         f"- **Journal:** {_v(m['journal'])}", f"- **Publication year:** {_v(m['year'])}",
         f"- **Publication type:** `{_v(m['publication_type'])}`", f"- **Language:** {_v(m['language'])}",
         f"- **Abstract:** {'see section ' + next((x['section_id'] for x in secs_c if x['section_class'] == 'TITLE_ABSTRACT' and 'abstract' in x['section_path'].lower()), '') if any(x['section_class'] == 'TITLE_ABSTRACT' for x in secs_c) else ''}",
         f"- **Keywords:** {_v(m['keywords'])}", "", "### C. Source and file provenance", "",
         f"- **Source database/repository:** {_v(m['source_repository'])}", f"- **Source URL:** {_v(m['source_url'])}",
         f"- **Retrieval date:** {_v(m['retrieval_date'])}", f"- **License name:** {_v(m['licence'])}",
         f"- **License URL:** {_v(m['licence_url'])}", f"- **Reuse permission checked:** `{m['reuse_checked']}`",
         f"- **Original PDF path:** {_v(m['pdf_path'])}", f"- **Original XML path:** {_v(m['xml_path'])}",
         f"- **Extracted text path:** {_v(m['text_path'])}", f"- **Supplementary file paths:** {_v(m['supplementary'])}",
         f"- **Original file SHA-256:** {_v(m['source_sha256'])}", f"- **Extraction tool and version:** {_v(m['extraction_tool'])}",
         "- **Extraction date:** ", f"- **Text version ID/checksum:** {_v(m['text_checksum'])}",
         "", "### D. Eligibility and relevance", "",
         f"- **Human microbiome relevance:** `{m['relevance']}`", f"- **Inclusion decision:** `{m['inclusion']}`",
         f"- **Inclusion reason(s):** {_v(m['inclusion_reason'])}", f"- **Exclusion reason(s):** {_v(m['exclusion_reason'])}",
         f"- **Protocol version applied:** {_v(m['protocol'])}",
         f"- **Duplicate/canonical record ID, if applicable:** {_v(m['duplicate_of'])}",
         "", "### E. Full-text integrity checks", "",
         f"- **Title matches source:** `{m['title_match']}`", f"- **Abstract matches source:** `{m['abstract_match']}`",
         f"- **Body text completeness:** `{m['body']}`", f"- **Section hierarchy preserved:** `{m['hierarchy']}`",
         f"- **Tables checked:** `{m['tables']}`", f"- **Figure captions checked:** `{m['captions']}`",
         f"- **Supplementary materials checked:** `{m['supplements']}`", f"- **References retained:** `{m['references']}`",
         f"- **Known extraction problems:** {_v(m['problems'])}",
         "", "### F. Full-text section inventory", "",
         "| Section ID | Heading | Heading path | Order | Text path/reference | Checked |", "|---|---|---|---:|---|---|"]
    for x in secs_c:
        head = x["section_path"].split(" > ")[-1]
        L.append(f"| `{x['section_id']}` | {head[:80]} | {x['section_path'][-120:]} | {x['ordinal']} | "
                 f"{m['text_path']}#{x['section_id']} | `{'yes' if x['section_id'] in sent else 'no'}` |")
    L += ["", "### G. Study metadata", ""]
    for (c, lab), st in STUDIES.items():
        if c != cid:
            continue
        L += [f"#### Study ID: `{st['study_id']}` ({lab})", ""]
        for f in STUDY_FIELDS:
            L.append(f"- **{f.replace('_', ' ').capitalize()}:** {_study_value(st['study_id'], cid, f)}")
        L.append("")
    L += ["### H. Findings index", "", "| Finding ID | Study ID | Short description | Evidence locator | Extraction status | Verification status |",
          "|---|---|---|---|---|---|"]
    for r in FIND_ROWS:
        if r["corpus_id"] == cid:
            desc = f"{r['taxon_original'] or r['outcome']} {r['association_direction']} ({r['exposure_comparison']})"[:90]
            L.append(f"| `{r['finding_id']}` | `{r['study_id']}` | {desc} | {r['evidence_locator']} | `extracted_unverified` | `pending` |")
    if not any(r["corpus_id"] == cid for r in FIND_ROWS):
        L.append("| | | " + ("no findings extracted" if cid in TPL_FIND_ATTEMPTED else "findings not extracted (no LLM run)") + " | | | |")
    L += ["", "### I. Paper-level final review checklist", ""] + [f"- [ ] {x}" for x in (
        "Bibliographic identifiers checked", "Duplicate status resolved or documented", "Source and license information recorded",
        "Full text checked against source", "Sections, tables, figure captions and supplements checked or explicitly marked",
        "Participant counts distinguished from sample counts", "Reported facts distinguished from missing information",
        "Every finding has a stable ID and study link", "Evidence text and locator verified against the correct text version",
        "Reviewer and review date recorded", "Remaining limitations documented")] + ["", "**Final paper status:** `pending`", ""]
    (TPL_DIR / "papers" / f"{cid}.md").write_text("\n".join(L), encoding="utf-8")
log_event("ALL", "final_review", "-", "-", "human review", "pending", "no reviewer has signed off", "")
dump_csv(TPL_DIR / "curation_log.csv", CURATION_LOG,
         ["corpus_id", "date", "stage", "tool_person", "tool_version", "action_check", "result", "problem_found", "resolution"])

# ---------- §7 corpus-level report + checks ----------
def _rate(n, d):
    return dict(n=n, of=d, rate=round(n / d, 4) if d else None)
N = len(TRACKING)
report = dict(
    template_version=TPL_VERSION, run_tag=RUN_TAG, written_at=_now(), llm_backend=LLM_BACKEND, llm_model=LLM_MODEL,
    status="PROVISIONAL_UNVALIDATED - schema-valid is not scientifically correct; no annotation accuracy evaluated",
    corpus_rows=N, template_expected=453,
    processing=dict(_C(r["Processing status"] for r in TRACKING)), inclusion=dict(_C(r["Inclusion decision"] for r in TRACKING)),
    full_text=dict(_C(r["Full-text status"] for r in TRACKING)),
    missing_title=_rate(sum(1 for r in TRACKING if not r["Title"]), N),
    missing_doi=_rate(sum(1 for r in TRACKING if not r["DOI / PMID / PMCID"]), N),
    missing_pmid_pmcid=_rate(N, N), missing_abstract_field="not extracted (section reference only)",
    duplicate_identifier_groups=sum(1 for v in _C(r["_canonical"] for r in TRACKING).values() if v > 1),
    full_text_completeness=_rate(sum(1 for r in TRACKING if r["Full-text status"] == "complete"), N),
    tables_checked=_rate(0, len(TPL_PAPERS)), supplements_checked=_rate(0, len(TPL_PAPERS)),
    papers_in_run=len(TPL_PAPERS), study_blocks=len(STUDIES), study_field_values=len(STUDY_FIELD_ROWS),
    findings_extracted=len(FIND_ROWS), findings_human_verified=_rate(0, len(FIND_ROWS)),
    llm_items_rejected=dict(_C(r["reason"].split(":")[0] for r in TPL_REJECTED)),
    accuracy_vs_gold="UNKNOWN - no reviewed gold subset")
checks = {
    "unique_finding_ids": [k for k, v in _C(r["finding_id"] for r in FIND_ROWS).items() if v > 1],
    "findings_link_to_existing_study": [r["finding_id"] for r in FIND_ROWS if r["study_id"] not in {s["study_id"] for s in STUDIES.values()}],
    "evidence_recoverable_from_text_version": [r["finding_id"] for r in FIND_ROWS
        if PUB_TEXT[r["corpus_id"]][r["evidence_section_id"]][int(r["evidence_locator"].split("[")[1].split(":")[0]):
                                                               int(r["evidence_locator"].split(":")[1].rstrip("]"))] != r["evidence_text"]],
    "study_values_recoverable": [r["study_id"] + ":" + r["field"] for r in STUDY_FIELD_ROWS
                                 if PUB_TEXT[r["corpus_id"]][r["section_id"]][r["char_start"]:r["char_end"]] != r["evidence_text"]],
    "statuses_in_template_vocab": [r["Corpus ID"] for r in TRACKING if r["Processing status"] not in
        {"not_started", "source_acquired", "text_extracted", "automated_checks_passed", "manual_review_passed", "needs_repair", "excluded"}
        or r["Full-text status"] not in {"not_checked", "complete", "partial", "failed", "not_applicable"}
        or r["Annotation status"] not in {"not_started", "in_progress", "extracted_unverified", "human_verified", "needs_revision"}
        or r["Inclusion decision"] not in {"pending", "include", "exclude"}],
    "finding_enums": [r["finding_id"] for r in FIND_ROWS if r["claim_type"] not in CLAIM_TYPES or r["association_direction"] not in DIRECTIONS],
    "no_manual_review_claimed": [r["Corpus ID"] for r in TRACKING if r["Processing status"] == "manual_review_passed"
                                 or r["Annotation status"] == "human_verified"],
}
validation_t = dict(status="PASS" if not any(checks.values()) else "FAIL",
                    checks={k: dict(violations=len(v), examples=v[:5]) for k, v in checks.items()})
(TPL_DIR / "corpus_report.json").write_text(redact(json.dumps(report, indent=1, ensure_ascii=False)), encoding="utf-8")
(TPL_DIR / "corpus_validation.json").write_text(json.dumps(validation_t, indent=1), encoding="utf-8")
for k, v in checks.items():
    print(f"  {k:42s} {'ok' if not v else f'FAIL ({len(v)})'}")
print("template outputs:", TPL_DIR, "| studies:", len(STUDIES), "| study values:", len(STUDY_FIELD_ROWS),
      "| findings:", len(FIND_ROWS), "| paper records:", len(TPL_PAPERS))
assert validation_t["status"] == "PASS", "corpus template checks failed - see corpus_validation.json"

## 16. Audit of the Oct 20-28 work: registry and helpers (Cell 14)


In [ ]:
# ===== Cell 14: audit helpers + artifact registry (Oct 20-28 work) =====
# Read-only audit of the work done 2026-10-20..28: NER expansion, LLM runs, judge,
# terminology freeze, corpus 2.2.0, linking, calibration sheets, frozen bundle.
# Nothing here writes outside OUT_DIR/audit/ and nothing here modifies the artifacts.
import csv, io, json, os, re, sys
from collections import Counter, defaultdict
from pathlib import Path

AUDIT_DIR = OUT_DIR / "audit"
AUDIT_DIR.mkdir(exist_ok=True)
AUDIT_RECOMPUTE = os.environ.get("NB_AUDIT_RECOMPUTE", "1").strip().lower() in ("1", "true", "yes")
AUDIT_MAX_ROWS = int(os.environ.get("NB_AUDIT_MAX_ROWS", "0") or 0)   # 0 = no cap
csv.field_size_limit(10 ** 9)                                          # long evidence/quote fields

# --- repo import path (Cell 1 added it too; re-assert because a fresh kernel may skip checks) ---
for _p in (ROOT / "src", ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# section loader shared with the NER runners (harness-free: the audit cells import their own)
from cdm_pipeline.ner.runner_sections import load_sections


def load_ner_csv(path, limit=0):
    """NUL-safe CSV reader. Several NER CSVs and 4 source .md files contain NUL bytes,
    which make csv.DictReader raise `_csv.Error: line contains NUL`; strip them first.
    Same contract as cdm_pipeline.utils.load_csv_safe, but honours `limit` for big files."""
    txt = Path(path).read_text(encoding="utf-8", errors="replace").replace("\x00", "")
    rows = list(csv.DictReader(io.StringIO(txt)))
    return rows[:limit] if limit else rows


def _safe_norm(t):
    return re.sub(r"\s+", " ", (t or "").lower().strip())


# The fifteen section-level NER variants added in this window (jobs 28643785, 28643869,
# 28647499, 28649224, 28649661 - the last three COMPLETED 2026-10-28, after the first audit
# snapshot; they get the same span-fidelity / dedup / novelty treatment as the first six).
# `key` is the short name used in this audit; `model` is the value in the CSV `model` column.
NER_NEW = [
    ("bert_disease",     ROOT / "results/28643785_ner_bert_variants/bert_disease_ner.csv",
     "biobert_diseases_wl", "bert_disease"),
    ("bert_chemical",    ROOT / "results/28643785_ner_bert_variants/bert_chemical_ner.csv",
     "biobert_chemical", "bert_chemical"),
    ("bert_genetic",     ROOT / "results/28643785_ner_bert_variants/bert_genetic_ner.csv",
     "biobert_genetic", "bert_genetic"),
    ("scibert_jnlpba",   ROOT / "results/28643785_ner_bert_variants/scibert_jnlpba_ner.csv",
     "scibert_jnlpba", "scibert_jnlpba"),
    ("bionlp13cg",       ROOT / "results/28643869_ner_extra_variants/scispacy_ner_bionlp13cg.csv",
     "en_ner_bionlp13cg_md", "scispacy_bionlp13cg"),
    # HunFlair2 replaced by the span-trim re-run (job 28649224).  The 28643869 CSV is
    # superseded and must not be read: it carried 34,378 junk rows and silently dropped
    # 785 good spans whose raw token window failed the old byte-exact slice check.
    ("hunflair2",        ROOT / "results/28649224_ner_hunflair2_trim/hunflair2_ner.csv",
     "hunflair2", "hunflair2"),
    ("gliner2_base",     ROOT / "results/28647499_ner_gliner_new/gliner2_base_ner.csv",
     "gliner2_base", "gliner2_base"),
    ("gliner2_large",    ROOT / "results/28647499_ner_gliner_new/gliner2_large_ner.csv",
     "gliner2_large", "gliner2_large"),
    ("gliner_v25_biomed", ROOT / "results/28647499_ner_gliner_new/gliner_v25_biomed_ner.csv",
     "gliner_v25_biomed", "gliner_v25_biomed"),
    ("gliner_v25_medium", ROOT / "results/28647499_ner_gliner_new/gliner_v25_medium_ner.csv",
     "gliner_v25_medium", "gliner_v25_medium"),
    ("dnorm",            ROOT / "results/28649661_ner_java_tools/dnorm_ner.csv",
     "DNorm", "DNorm"),
    ("taggerone_disease", ROOT / "results/28649661_ner_java_tools/taggerone_disease_ner.csv",
     "TaggerOne-Disease", "TaggerOne-Disease"),
    ("taggerone_chemical", ROOT / "results/28649661_ner_java_tools/taggerone_chemical_ner.csv",
     "TaggerOne-Chemical", "TaggerOne-Chemical"),
    ("tmchem",           ROOT / "results/28649661_ner_java_tools/tmchem_ner.csv",
     "tmChem", "tmChem"),
    ("gnormplus",        ROOT / "results/28649661_ner_java_tools/gnormplus_ner.csv",
     "GNormPlus", "GNormPlus"),
]
# Jobs that were still running at the first audit snapshot; both have since COMPLETED.
# Cell 18 keeps a per-file inventory (with the runner's own summary JSON for reconciliation);
# Cell 15 deep-audits their CSVs exactly like the six above.  The Java entry is job 28649661,
# which ships all five tools (tmChem added, Ab3P restored); 28647522 shipped only four and
# its CSVs are superseded - never read them here.
NER_LIVE = [
    ("gliner_new", ROOT / "results/28647499_ner_gliner_new", "28647499"),
    ("java_tools", ROOT / "results/28649661_ner_java_tools", "28649661"),
]

# Artifacts produced in this window, with the file that is authoritative for each count.
ARTIFACTS = {
    "cpu_ensemble":      ROOT / "zenodo_v1/ner_453_sections/all_ner_sections_combined.csv",
    "llm_vllm_body":     ROOT / "results/llm_ner_body/vllm_sections_ner.csv",
    "mesh_ner":          ROOT / "results/mesh_ner/mesh_ner_terms.csv",
    "llm_vs_ner":        ROOT / "results/llm_vs_ner_comparison/full_body/comparison.json",
    "judge_full":        ROOT / "logs/llm_judge/_judge_summary_full.json",
    "termpipe_v1":       ROOT / "results/terminology_v1/terminology_extraction_manifest.json",
    "release_1_0_0":     ROOT / "releases/terminology/1.0.0/terminology_extraction_manifest.json",
    "release_gate":      ROOT / "logs/release_gate_report.json",
    "corpus_v2_2_2_0":   ROOT / "corpus_v2/releases/2.2.0/corpus_manifest.json",
    "frozen_bundle":     ROOT / "frozen_output/BUNDLE_INFO.json",
    "cv_linking":        ROOT / "results/cv_linking/cv_summary.json",
    "omop_linking":      ROOT / "results/omop_linking/omop_summary.json",
    "taxonomy_pilot":    ROOT / "results/taxonomy_pilot/term_categories_pilot_summary.json",
    "validation_sample": ROOT / "results/validation/sample_v1/_manifest.json",
    "error_triage":      ROOT / "results/ner_errors_triage_735/enrichment_report.json",
    "adj_pilot":         ROOT / "results/adjudication_pilot_triples_500/triage_summary.json",
    "gold_standard":     ROOT / "logs/gold_standard/sheet_C1_records.csv",
}
ARTIFACT_STATUS = {k: ("PRESENT" if Path(v).exists() else "MISSING") for k, v in ARTIFACTS.items()}
print("audit artifacts:", dict(Counter(ARTIFACT_STATUS.values())))
for k, v in ARTIFACT_STATUS.items():
    if v == "MISSING":
        print("   MISSING", k, "->", ARTIFACTS[k])


def read_json(path, default=None):
    """Read a JSON artifact; a missing/blank file returns `default` (audit never hard-fails on it)."""
    p = Path(path)
    if not p.exists():
        return default
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, UnicodeDecodeError):
        return default


def write_audit_csv(name, rows, fields):
    """Write an audit table under OUT_DIR/audit/ (inside OUT_DIR, so the Path guard holds)."""
    dump_csv(AUDIT_DIR / name, rows, fields)
    print("wrote", AUDIT_DIR / name, f"({len(rows)} rows)")


print("RECOMPUTE:", AUDIT_RECOMPUTE, "| max_rows:", AUDIT_MAX_ROWS or "no cap")
print("new NER artifacts:", {k: p.exists() for k, p, _, _ in NER_NEW})
print("live  NER dirs   :", {k: p.exists() for k, p, _ in NER_LIVE})


## 17. NER expansion audit: span fidelity, dedup survival, novelty (Cell 15)


In [ ]:
# ===== Cell 15: NER expansion audit - span fidelity, dedup survival, novelty vs CPU baseline =====
# Scope: the fifteen section-level NER artifacts added 2026-10-27/28 (jobs 28643785, 28643869,
# 28647499 gliner-new, 28649661 java-tools). The java-tools CSVs carry extra norm/cdm_type columns;
# DictReader ignores them here and the same eight columns drive every check.
# Three independent questions, three tables:
#   (1) SPAN FIDELITY  - is text[char_start:char_end] really the term? (slice rule, per model)
#   (2) DEDUP SURVIVAL - how many rows survive cdm_pipeline.dedup's real filters? Where do the rest go?
#   (3) NOVELTY        - which terms are new versus the frozen CPU ensemble baseline (zenodo_v1)?
# All three are read-only. (2) is the important one: it measures an actual defect (see below).
from cdm_pipeline import dedup as _dd
from cdm_pipeline.utils import is_junk, norm as _norm

# ---- (1) span fidelity -------------------------------------------------------
# The pipeline contract is text[char_start:char_end] == term against the SECTION body.
# We re-derive the body text from the same loader the NER runners use, so this checks the
# artifacts against the exact geometry they claim, not against the whole .md file.
_units, _ustats = load_sections(SEG_DIR, body_only=BODY_ONLY)
UNIT_TEXT = {(u["cid"], u["section_id"]): u["text"] for u in _units}
print("section universe for validation:", len(UNIT_TEXT), "units /",
      len({k[0] for k in UNIT_TEXT}), "papers /", sum(len(t) for t in UNIT_TEXT.values()), "chars")

SPAN_ROWS, NER_ROWS = [], {}
_HYG_PUNCT = ",.;:)]}\"'-"
for key, path, model_col, reg in NER_NEW:
    if not path.exists():
        print("   MISSING artifact", key, path)
        continue
    rows = load_ner_csv(path, AUDIT_MAX_ROWS)
    NER_ROWS[key] = rows
    bad = nul = nosect = 0
    dup = 0
    hyg_punct = hyg_nl = 0
    seen = set()
    for r in rows:
        t, s, e = r["term"], int(r["char_start"]), int(r["char_end"])
        body = UNIT_TEXT.get((r["paper"], r["section_id"]))
        if body is None:
            nosect += 1
            continue
        got = body[s:e]
        if got != t:
            if "\x00" in t or "\x00" in got:
                nul += 1          # source .md NUL bytes shift the slice; an artifact, not a mis-span
            else:
                bad += 1
        # span hygiene (same measures as the hunflair2 defect, Cell 19, for every model)
        tt = t.strip()
        if tt and tt[-1] in _HYG_PUNCT:
            hyg_punct += 1
        if "\n" in t or "\r" in t:
            hyg_nl += 1
        k = (r["paper"], r["section_id"], s, e, t, r["labels"])
        if k in seen:
            dup += 1
        else:
            seen.add(k)
    SPAN_ROWS.append(dict(model=key, registry_key=reg, csv_model=model_col, rows=len(rows),
                          exact=int(len(rows) - bad - nul - nosect), mismatched=bad,
                          nul_artifact=nul, section_missing=nosect, duplicate_spans=dup,
                          exact_pct=round(100.0 * (len(rows) - bad - nul - nosect) / max(len(rows), 1), 3),
                          trailing_punct=hyg_punct, trailing_punct_pct=round(100.0 * hyg_punct / max(len(rows), 1), 1),
                          contains_newline=hyg_nl,
                          contains_newline_pct=round(100.0 * hyg_nl / max(len(rows), 1), 1)))
    print(f"  {key:19s} rows={len(rows):7d} exact={SPAN_ROWS[-1]['exact_pct']:6.2f}% "
          f"mismatch={bad} nul={nul} nosect={nosect} dup={dup} "
          f"trail_punct={hyg_punct} newline={hyg_nl}")
write_audit_csv("ner_span_fidelity.csv", SPAN_ROWS,
                ["model", "registry_key", "csv_model", "rows", "exact", "exact_pct",
                 "mismatched", "nul_artifact", "section_missing", "duplicate_spans",
                 "trailing_punct", "trailing_punct_pct", "contains_newline", "contains_newline_pct"])

# ---- (2) dedup survival through the REAL filter path -------------------------
# cdm_pipeline.dedup drops a row when (a) the term is junk/too short/too long,
# (b) the label is not in VALID_TYPES, or (c) norm() is empty. We call the module's own
# fix_label/VALID_TYPES/MAX_TERM_LENGTH so this is the production decision, not a copy.
# STATUS 2026-10-28: the approved mappings (java-tools case-folds + taxon->ORGANISM,
# GENETIC->GENOMICS) are IN dedup.fix_label, so `kept` includes them (15-tool total (see TOTAL line)).
# The remaining counterfactual is the still-unapproved wiring of common.LABEL_TO_CDM
# (kept_map_only), which would additionally recover DNA/RNA/CELL_LINE/GENE/TAXON labels.
DEDUP_ROWS, DEDUP_TOT = [], dict(rows=0, kept=0, junk=0, label=0)
_UNWIRED = sorted({r["labels"] for rows in NER_ROWS.values() for r in rows[:5000]
                   if _dd.fix_label(r["labels"]) not in _dd.VALID_TYPES})
# Counterfactual tier, NOT applied to production code (no sign-off yet):
#   map_only  = additionally wire the translations common.LABEL_TO_CDM already defines
try:
    from common import LABEL_TO_CDM as _L2C
except ImportError:                       # notebook must still run without the repo package
    _L2C = {}
TYPE_DECISION = {"GENETIC": "GENOMICS"}   # applied 2026-10-28; kept here for provenance
FIX4 = {k: _L2C[k] for k in ("DNA", "RNA", "CELL_LINE") if k in _L2C}
FIX4.update(TYPE_DECISION)
KEPT_TERMS = set()
for key, rows in NER_ROWS.items():
    junk = label = kept = kept_map = kept_fix4 = 0
    for r in rows:
        t = r["term"].strip()
        if len(t) < 2 or is_junk(t) or len(t) > _dd.MAX_TERM_LENGTH or not _norm(t):
            junk += 1
            continue
        lab = _dd.fix_label(r["labels"])
        if lab in _dd.VALID_TYPES:                   # ships today
            kept += 1
            kept_map += 1
            kept_fix4 += 1
            KEPT_TERMS.add(_norm(t))
            continue
        if _L2C.get(lab) in _dd.VALID_TYPES:         # tier 1: the map already covers this label
            kept_map += 1
        if FIX4.get(lab) in _dd.VALID_TYPES:         # tier 2: map + the GENETIC decision
            kept_fix4 += 1
            continue
        label += 1                                   # no CDM home; correctly dropped
    DEDUP_ROWS.append(dict(model=key, rows=len(rows), kept=kept, dropped_junk=junk,
                           dropped_label=label, kept_pct=round(100.0 * kept / max(len(rows), 1), 1),
                           kept_map_only=kept_map,
                           kept_pct_map_only=round(100.0 * kept_map / max(len(rows), 1), 1),
                           kept_with_fix4=kept_fix4,
                           kept_pct_with_fix4=round(100.0 * kept_fix4 / max(len(rows), 1), 1)))
    DEDUP_TOT["rows"] += len(rows); DEDUP_TOT["kept"] += kept
    DEDUP_TOT["junk"] += junk; DEDUP_TOT["label"] += label
    print(f"  {key:15s} rows={len(rows):7d} kept={kept:7d} ({DEDUP_ROWS[-1]['kept_pct']:5.1f}%) "
          f"junk={junk:6d} label_drop={label:6d} | map_only={kept_map:7d} "
          f"| +decision={kept_fix4:7d} ({DEDUP_ROWS[-1]['kept_pct_with_fix4']:5.1f}%)")
KEPT_ALL = sum(r["kept_with_fix4"] for r in DEDUP_ROWS)
KEPT_MAP = sum(r["kept_map_only"] for r in DEDUP_ROWS)
DEDUP_TOT.update(kept_pct=round(100.0 * DEDUP_TOT["kept"] / max(DEDUP_TOT["rows"], 1), 1),
                 kept_map_only=KEPT_MAP,
                 kept_pct_map_only=round(100.0 * KEPT_MAP / max(DEDUP_TOT["rows"], 1), 1),
                 kept_with_fix4=KEPT_ALL,
                 kept_pct_with_fix4=round(100.0 * KEPT_ALL / max(DEDUP_TOT["rows"], 1), 1))
print("TOTAL:", DEDUP_TOT, "| label values dropped by VALID_TYPES:", _UNWIRED)
write_audit_csv("ner_dedup_survival.csv", DEDUP_ROWS,
                ["model", "rows", "kept", "kept_pct", "dropped_junk", "dropped_label",
                 "kept_map_only", "kept_pct_map_only", "kept_with_fix4", "kept_pct_with_fix4"])

# ---- (3) novelty vs the frozen CPU baseline ---------------------------------
# zenodo_v1/ is a frozen export: read it, never write to it. 1.7M rows; ~4 s.
BASE_CSV = ROOT / "zenodo_v1/ner_453_sections/all_ner_sections_combined.csv"
NOVELTY_ROWS = []
if BASE_CSV.exists():
    _base = load_ner_csv(BASE_CSV)
    BASE_RAW = {r["term"].strip() for r in _base}
    BASE_NORM = {_safe_norm(r["term"]) for r in _base}
    print(f"baseline: {len(_base)} rows / {len(BASE_RAW)} raw distinct / {len(BASE_NORM)} norm distinct")
    uni_raw, uni_norm = set(), set()
    for key, rows in NER_ROWS.items():
        raw = {r["term"].strip() for r in rows}
        nrm = {_safe_norm(r["term"]) for r in rows}
        uni_raw |= raw; uni_norm |= nrm
        new_raw, new_norm = raw - BASE_RAW, nrm - BASE_NORM
        NOVELTY_ROWS.append(dict(model=key, rows=len(rows), distinct_raw=len(raw),
                                 new_raw=len(new_raw),
                                 new_raw_pct=round(100.0 * len(new_raw) / max(len(raw), 1), 1),
                                 distinct_norm=len(nrm), new_norm=len(new_norm),
                                 new_norm_pct=round(100.0 * len(new_norm) / max(len(nrm), 1), 1)))
        print(f"  {key:15s} distinct={len(raw):6d} new={len(new_raw):6d} "
              f"({NOVELTY_ROWS[-1]['new_raw_pct']:5.1f}%)")
    UNION = dict(models=len(NER_ROWS), distinct_raw=len(uni_raw), new_raw=len(uni_raw - BASE_RAW),
                 new_raw_pct=round(100.0 * len(uni_raw - BASE_RAW) / max(len(uni_raw), 1), 1),
                 distinct_norm=len(uni_norm), new_norm=len(uni_norm - BASE_NORM),
                 new_norm_pct=round(100.0 * len(uni_norm - BASE_NORM) / max(len(uni_norm), 1), 1),
                 baseline_rows=len(_base), baseline_distinct_raw=len(BASE_RAW),
                 baseline_distinct_norm=len(BASE_NORM))
    print("UNION:", UNION)
    write_audit_csv("ner_novelty_vs_baseline.csv", NOVELTY_ROWS,
                    ["model", "rows", "distinct_raw", "new_raw", "new_raw_pct",
                     "distinct_norm", "new_norm", "new_norm_pct"])
    (AUDIT_DIR / "ner_novelty_union.json").write_text(json.dumps(UNION, indent=1), encoding="utf-8")
else:
    UNION = {}
    print("baseline CSV missing:", BASE_CSV)

# ---- (4) bert_disease vs the existing pubmedbert/biobert_diseases run --------
# Same weights (alvaroalon2/biobert_diseases_ner), different wrapper. If the two agree poorly,
# the "new" model is really a different extraction, not a rerun of the old one.
BERT_COMPARE = {}
_old = [r for r in (_base if BASE_CSV.exists() else []) if r["model"] == "biobert_diseases"]
if _old and "bert_disease" in NER_ROWS:
    old_t = {_safe_norm(r["term"]) for r in _old}
    new_t = {_safe_norm(r["term"]) for r in NER_ROWS["bert_disease"]}
    inter = old_t & new_t
    BERT_COMPARE = dict(old_model="biobert_diseases", new_model="biobert_disease",
                        old_rows=len(_old), old_distinct=len(old_t),
                        new_rows=len(NER_ROWS["bert_disease"]), new_distinct=len(new_t),
                        shared=len(inter), new_only=len(new_t - old_t), old_only=len(old_t - new_t),
                        jaccard=round(len(inter) / max(len(old_t | new_t), 1), 4))
    print("bert_disease vs pubmedbert:", BERT_COMPARE)
    (AUDIT_DIR / "bert_disease_vs_pubmedbert.json").write_text(json.dumps(BERT_COMPARE, indent=1),
                                                              encoding="utf-8")

# ---- summary + the defect statement -----------------------------------------
AUDIT_NER = dict(span_fidelity=SPAN_ROWS, dedup=DEDUP_ROWS, dedup_total=DEDUP_TOT,
                 labels_dropped_by_valid_types=_UNWIRED, map_only_translations=FIX4 and
                 {k: v for k, v in _L2C.items() if k in ("DNA", "RNA", "CELL_LINE")},
                 type_decision=TYPE_DECISION, fix4=FIX4,
                 novelty=NOVELTY_ROWS, novelty_union=UNION, bert_compare=BERT_COMPARE)
(AUDIT_DIR / "ner_expansion_summary.json").write_text(json.dumps(AUDIT_NER, indent=1), encoding="utf-8")

print("\n--- interpretation (read before quoting any number) ---")
print("1. Span fidelity is a HARD check: exact means text[start:end] == term in the source section.")
print("2. Dedup survival is a DEFECT measurement, not a quality score. `kept` is what ships")
print("   today, INCLUDING the 2026-10-28 approved mappings (java case-folds, taxon->ORGANISM,")
print("   GENETIC->GENOMICS). kept_map_only shows what the still-unapproved LABEL_TO_CDM wiring")
print("   would ADD on top (DNA/RNA/CELL_LINE/GENE/TAXON). Cell 19 records the decisions.")
print("3. Novelty is coverage, not accuracy: no gold standard covers these fourteen models.")


## 18. LLM NER, LLM-vs-CPU agreement and LLM judges (Cell 16)


In [ ]:
# ===== Cell 16: LLM NER (vLLM full-body) + LLM-vs-CPU agreement + LLM judges =====
# Three artifacts, three very different kinds of claim. Read the caveat line printed at the end.
#   (1) vLLM Qwen body extraction  - 366k located spans over the full section body (not a sample)
#   (2) LLM vs CPU-ensemble        - INTER-MODEL AGREEMENT. Not accuracy. No gold standard exists.
#   (3) LLM judges C1-C4           - MODEL-judged agreement with the released artifacts.
#                                    Also not accuracy, and not human gold (all sheets unlabelled).
VLLM_SUM = ROOT / "results/llm_ner_body/_vllm_sections_summary.json"
VLLM_CSV = ROOT / "results/llm_ner_body/vllm_sections_ner.csv"
AGREE_DIR = ROOT / "results/llm_vs_ner_comparison/full_body"
JUDGE_JSON = ROOT / "logs/llm_judge/_judge_summary_full.json"

# ---- (1) vLLM full-body extraction ------------------------------------------
VLLM = read_json(VLLM_SUM, {}) or {}
VLLM_ROWS = []
if VLLM:
    _st = VLLM.get("stats", {})
    _cov = VLLM.get("coverage", {})
    VLLM_ROWS.append(dict(
        model=VLLM.get("model", ""), prompt_version=VLLM.get("prompt_version", ""),
        sections=VLLM.get("sections", 0), segments=VLLM.get("segments", 0),
        cached_segments=VLLM.get("cached", 0), full_section_max_chars=VLLM.get("full_section_max_chars", 0),
        returned=_st.get("returned", 0), not_in_text=_st.get("not_in_text", 0),
        bad_type=_st.get("bad_type", 0), located_spans=_st.get("located_spans", 0),
        first_match=_st.get("first_match_rows", 0), expanded=_st.get("expanded_rows", 0),
        segments_failed=_st.get("segments_failed", 0),
        coverage_status=_cov.get("coverage_status", ""),
        char_covered=_cov.get("processed_chars", ""), char_total=_cov.get("expected_chars", ""),
        char_coverage_pct=_cov.get("char_coverage_pct", ""),
        sections_complete=_cov.get("sections_complete", 0),
        sections_empty=_cov.get("sections_empty", 0),
        sections_incomplete=_cov.get("sections_incomplete", 0),
        truncation_warnings=len(_cov.get("truncation_warnings") or [])))
    print("vLLM:", VLLM_ROWS[0]["model"], "| sections", VLLM_ROWS[0]["sections"],
          "| returned", VLLM_ROWS[0]["returned"], "-> located", VLLM_ROWS[0]["located_spans"],
          f"(not_in_text {VLLM_ROWS[0]['not_in_text']}, bad_type {VLLM_ROWS[0]['bad_type']})",
          "| coverage", VLLM_ROWS[0]["coverage_status"], "| failed", VLLM_ROWS[0]["segments_failed"])
else:
    print("vLLM summary missing:", VLLM_SUM)
write_audit_csv("llm_vllm_body.csv", VLLM_ROWS,
                ["model", "prompt_version", "sections", "segments", "cached_segments",
                 "full_section_max_chars", "returned", "not_in_text", "bad_type", "located_spans",
                 "first_match", "expanded", "segments_failed", "coverage_status",
                 "char_covered", "char_total", "char_coverage_pct", "sections_complete",
                 "sections_empty", "sections_incomplete", "truncation_warnings"])

# Per-CDM-type composition of the vLLM output (which types carry the expansion)
VLLM_TYPES = []
if VLLM_CSV.exists():
    _vr = load_ner_csv(VLLM_CSV, AUDIT_MAX_ROWS)
    _c = Counter(r["labels"] for r in _vr)
    _exp = Counter(r["labels"] for r in _vr if r.get("llm_occurrence_expanded") in ("1", "True", "true"))
    for lab, n in _c.most_common():
        VLLM_TYPES.append(dict(cdm_type=lab, rows=n,
                               pct=round(100.0 * n / max(len(_vr), 1), 2),
                               occurrence_expanded=_exp.get(lab, 0)))
    print("vLLM rows:", len(_vr), "| distinct terms:", len({_safe_norm(r['term']) for r in _vr}),
          "| types:", len(VLLM_TYPES))
    write_audit_csv("llm_vllm_body_by_type.csv", VLLM_TYPES,
                    ["cdm_type", "rows", "pct", "occurrence_expanded"])
else:
    print("vLLM CSV missing:", VLLM_CSV)

# ---- (2) LLM vs CPU-ensemble agreement --------------------------------------
AGREE = read_json(AGREE_DIR / "comparison.json", {}) or {}
AGREE_ROWS, TYPE_ROWS = [], []
if AGREE:
    _sc = AGREE.get("scope", {})
    for _variant, _label in (("agreement_llm_vs_cpu_ensemble", "all_terms"),
                             ("agreement_llm_vs_cpu_ensemble_excl_generic", "excl_generic")):
        _a = AGREE.get(_variant) or {}
        if not _a:
            continue
        AGREE_ROWS.append(dict(
            variant=_label,
            jaccard_global=_a.get("jaccard_global_terms"),
            jaccard_pooled=_a.get("jaccard_pooled_within_unit"),
            jaccard_mean_per_unit=_a.get("jaccard_mean_per_unit"),
            jaccard_typed_pooled=_a.get("jaccard_typed_pooled"),
            shared_terms=_a.get("shared_terms"), llm_only=_a.get("llm_only_terms"),
            cpu_only=_a.get("cpu_only_terms"), llm_distinct=_a.get("llm_distinct_terms"),
            cpu_distinct=_a.get("cpu_distinct_terms"),
            span_exact_rate=_a.get("llm_span_exact_match_rate"),
            units_llm_unique=_a.get("units_with_llm_unique_terms"),
            units_cpu_unique=_a.get("units_with_cpu_unique_terms")))
        print(f"agreement [{_label}]: J_global={AGREE_ROWS[-1]['jaccard_global']} "
              f"J_pooled={AGREE_ROWS[-1]['jaccard_pooled']} shared={AGREE_ROWS[-1]['shared_terms']} "
              f"llm_only={AGREE_ROWS[-1]['llm_only']} cpu_only={AGREE_ROWS[-1]['cpu_only']}")
    print("agreement scope:", _sc)
    write_audit_csv("llm_vs_cpu_agreement.csv", AGREE_ROWS,
                    ["variant", "jaccard_global", "jaccard_pooled", "jaccard_mean_per_unit",
                     "jaccard_typed_pooled", "shared_terms", "llm_only", "cpu_only",
                     "llm_distinct", "cpu_distinct", "span_exact_rate",
                     "units_llm_unique", "units_cpu_unique"])
    _ta = AGREE_DIR / "type_agreement.csv"
    if _ta.exists():
        TYPE_ROWS = load_ner_csv(_ta)
        print("type-agreement rows:", len(TYPE_ROWS), "->", _ta.name)
else:
    _sc = {}
    print("agreement comparison missing:", AGREE_DIR / "comparison.json")

# ---- (3) LLM judges C1-C4 ---------------------------------------------------
JUDGE, JUDGE_ROWS = read_json(JUDGE_JSON, {}) or {}, []
if JUDGE:
    for _art, _d in (JUDGE.get("artifacts") or {}).items():
        for _metric, _m in (_d.get("metrics") or {}).items():
            JUDGE_ROWS.append(dict(
                artifact=_art, metric=_metric, rate_pct=_m.get("rate"),
                ci_low=(_m.get("ci") or [None, None])[0], ci_high=(_m.get("ci") or [None, None])[1],
                k=_m.get("k"), n=_m.get("n"), uncertain=_m.get("uncertain"),
                judged=_d.get("judged"), expected=_d.get("expected"),
                complete=_d.get("complete"), confidence_mean=round(_d.get("confidence_mean") or 0, 4)))
        print(f"judge {_art}: judged {_d.get('judged')}/{_d.get('expected')} complete={_d.get('complete')}")
    _tot = JUDGE.get("totals", {})
    print("judge totals:", _tot, "| incomplete:", JUDGE.get("incomplete"))
    write_audit_csv("llm_judge_metrics.csv", JUDGE_ROWS,
                    ["artifact", "metric", "rate_pct", "ci_low", "ci_high", "k", "n", "uncertain",
                     "judged", "expected", "complete", "confidence_mean"])
else:
    _tot = {}
    print("judge summary missing:", JUDGE_JSON)

AUDIT_LLM = dict(vllm=VLLM_ROWS, vllm_by_type=VLLM_TYPES, agreement=AGREE_ROWS,
                 agreement_scope=_sc, judge=JUDGE_ROWS, judge_totals=_tot,
                 judge_caveat="model-judged agreement, all human calibration sheets are EMPTY -> not calibrated")
(AUDIT_DIR / "llm_runs_summary.json").write_text(json.dumps(AUDIT_LLM, indent=1), encoding="utf-8")

print("\n--- caveat ---")
print("Nothing in this cell is an accuracy measurement. (2) compares two systems to each other;")
print("(3) asks an LLM to judge artifacts and every human calibration sheet is still empty.")
print("C4 was 23.0% in an earlier pass and 17.5% in this one -> treat C4 as unstable, not as a score.")


## 19. Terminology v1, frozen release, corpus 2.2.0, frozen bundle (Cell 17)


In [ ]:
# ===== Cell 17: terminology v1, frozen release 1.0.0, release gate, corpus 2.2.0, frozen bundle =====
# The delivery chain end to end, with the one known defect surfaced rather than hidden:
#   terminology_v1 (job 28619916) -> releases/terminology/1.0.0 (frozen) -> release gates
#   corpus_v2 2.2.0 -> frozen_output/ bundle (copied + derived, 0 missing)
# Counts come from the manifests, not from re-counting 2.5M JSONL rows (that is release-gate work).
TERM_DIR = ROOT / "results/terminology_v1"
REL_DIR = ROOT / "releases/terminology/1.0.0"
GATE_JSON = ROOT / "logs/release_gate_report.json"
CORPUS_MANIFEST = ROOT / "corpus_v2/releases/2.2.0/corpus_manifest.json"
BUNDLE_JSON = ROOT / "frozen_output/BUNDLE_INFO.json"

# ---- (1) terminology v1 build ----------------------------------------------
TERM_MAN = read_json(TERM_DIR / "terminology_extraction_manifest.json", {}) or {}
TERM_ROWS = []
if TERM_MAN:
    _cov = TERM_MAN.get("coverage") or {}
    _cmp = TERM_MAN.get("completeness") or {}
    _art = TERM_MAN.get("artifact_status") or {}
    for _name, _st in _art.items():
        TERM_ROWS.append(dict(artifact=_name, status=_st))
    print("terminology manifest version:", TERM_MAN.get("manifest_version"),
          "| pipeline:", TERM_MAN.get("pipeline_version"),
          "| generated:", TERM_MAN.get("generated_at"))
    print("coverage:", {k: _cov.get(k) for k in _cov if isinstance(_cov.get(k), (int, float))})
    print("artifact_status: present", _art.get("n_present"), "/ expected", _art.get("n_expected"),
          "| missing", _art.get("n_missing"), _art.get("missing"))
    print("completeness:", _cmp.get("overall_status"),
          "| retrieval evaluation:", _cmp.get("retrieval_evaluation"))
    print("stages not run:", (_cmp.get("corpus_stages_not_run") or {}))
    # These two are the honesty checks: all artifacts present, and nothing silently assumed.
    assert _art.get("n_missing") == 0, "terminology manifest reports missing artifacts"
    assert _cmp.get("overall_status") == "COMPLETE", "terminology build is not COMPLETE"
else:
    print("terminology manifest missing:", TERM_DIR / "terminology_extraction_manifest.json")

# Per-artifact line counts (physical lines; cheap even for 2.5M rows)
TERM_FILES = []
for _f in sorted(TERM_DIR.glob("*.jsonl")):
    _n = sum(1 for _ in open(_f, "rb"))
    TERM_FILES.append(dict(file=_f.name, physical_lines=_n,
                           mb=round(_f.stat().st_size / 1024 / 1024, 1)))
    print(f"  {_f.name:32s} {_n:9d} lines  {TERM_FILES[-1]['mb']:7.1f} MB")
write_audit_csv("terminology_v1_files.csv", TERM_FILES, ["file", "physical_lines", "mb"])

# ---- (2) frozen release -----------------------------------------------------
FROZEN = {}
if REL_DIR.exists():
    _prov = read_json(REL_DIR / "PROVENANCE.json", {}) or {}
    _arti = _prov.get("artifacts") or {}
    _art_sha = (_arti.get("terminology_extraction_manifest.json")
                or _arti.get("terminology_extraction_manifest")
                or file_sha256(REL_DIR / "terminology_extraction_manifest.json"))
    # Calibration state: 1.0.0 froze with zero auto-accepted mappings and an empty human sheet.
    _auto_acc = (read_json(REL_DIR / "terminology_extraction_manifest.json", {}) or {}).get(
        "coverage", {}).get("auto_accepted_mappings")
    FROZEN = dict(
        dir=str(REL_DIR.relative_to(ROOT)),
        files=sum(1 for _ in REL_DIR.rglob("*") if _.is_file()),
        frozen_at=_prov.get("frozen_at"),
        job=_prov.get("job_id", _prov.get("slurm_job_id")),
        corpus_release=_prov.get("corpus_release", _prov.get("corpus_version")),
        manifest_sha256=_art_sha[:16],
        auto_accepted_mappings=_auto_acc,
        calibration_status="NOT_CALIBRATED" if not _auto_acc else "CALIBRATED",
        has_calibration_dir=(REL_DIR / "calibration_v0_pre_fix").exists())
    print("frozen release:", FROZEN)
    # The release ledger row is the audit trail; show it if the ledger parses.
    _led = ROOT / "releases/RELEASES.md"
    if _led.exists():
        _rows = [ln for ln in _led.read_text(encoding="utf-8").splitlines()
                 if ln.startswith("|") and "terminology" in ln and "1.0.0" in ln]
        for ln in _rows:
            print("  ledger:", ln.strip()[:200])
else:
    print("frozen release missing:", REL_DIR)

# ---- (3) release gates -----------------------------------------------------
GATE, GATE_ROWS = read_json(GATE_JSON, {}) or {}, []
if GATE:
    for _g, _d in (GATE.get("gates") or {}).items():
        GATE_ROWS.append(dict(gate=_g, status=_d.get("status"),
                              targets=len(_d.get("targets") or []),
                              note=(((_d.get("details") or {}).get("note")) or "")[:180]))
        print(f"  gate {_g}: {_d.get('status')}")
    _st = Counter(r["status"] for r in GATE_ROWS)
    print("gate totals:", dict(_st), "| generated", GATE.get("generated_at"))
    write_audit_csv("release_gates.csv", GATE_ROWS, ["gate", "status", "targets", "note"])
else:
    print("release gate report missing:", GATE_JSON)

# ---- (4) corpus v2 2.2.0 ----------------------------------------------------
CORPUS = read_json(CORPUS_MANIFEST, {}) or {}
if CORPUS:
    _c = CORPUS.get("counts") or CORPUS.get("artifacts") or {}
    CORPUS_ROWS = [dict(metric=k, value=(v if isinstance(v, (int, float, str)) else json.dumps(v)[:120]))
                   for k, v in _c.items()]
    print("corpus manifest:", CORPUS.get("corpus_version", CORPUS.get("corpus_release")),
          "| generated:", CORPUS.get("generated_at"), "| keys:", sorted(CORPUS)[:8])
    for r in CORPUS_ROWS[:14]:
        print(f"  {r['metric']:28s} {r['value']}")
    write_audit_csv("corpus_v2_counts.csv", CORPUS_ROWS, ["metric", "value"])
else:
    CORPUS_ROWS = []
    print("corpus manifest missing:", CORPUS_MANIFEST)

# ---- (5) frozen bundle ------------------------------------------------------
BUNDLE, BUNDLE_ROWS = read_json(BUNDLE_JSON, {}) or {}, []
if BUNDLE:
    _cnt = BUNDLE.get("counts") or {}
    BUNDLE_ROWS = [dict(item=k, value=v) for k, v in _cnt.items()]
    _man = ROOT / "frozen_output/MANIFEST.sha256"
    print("bundle:", BUNDLE.get("generated_at"),
          "| copied", BUNDLE.get("files_copied"), "derived", BUNDLE.get("files_derived"),
          "missing", BUNDLE.get("files_missing"),
          "| manifest lines", sum(1 for _ in open(_man)) if _man.exists() else "n/a",
          "| calibration", BUNDLE.get("calibration_status"),
          "| secrets_excluded", BUNDLE.get("secrets_excluded"))
    print("bundle counts:", _cnt)
    print("bundle size:", round((BUNDLE.get("total_bytes") or 0) / 1e6, 0), "MB")
    write_audit_csv("frozen_bundle_counts.csv", BUNDLE_ROWS, ["item", "value"])
    # files_missing is a LIST of paths in BUNDLE_INFO.json (empty == nothing missing).
    _miss = BUNDLE.get("files_missing")
    assert not _miss, f"frozen bundle reports missing files: {_miss}"
    assert BUNDLE.get("secrets_excluded") is True, "frozen bundle does not assert secrets_excluded"
else:
    print("bundle info missing:", BUNDLE_JSON)

AUDIT_RELEASE = dict(terminology_manifest_version=TERM_MAN.get("manifest_version"),
                     terminology_files=TERM_FILES, frozen=FROZEN, gates=GATE_ROWS,
                     corpus=CORPUS_ROWS, bundle=BUNDLE_ROWS)
(AUDIT_DIR / "release_chain_summary.json").write_text(json.dumps(AUDIT_RELEASE, indent=1),
                                                      encoding="utf-8")

print("\n--- what the gate failure means ---")
print("G2 compares manifest coverage to physical line counts. term_mentions.jsonl is off by one:")
print("coverage says 2,505,736, the artifact holds 2,505,735. The three other artifacts match exactly.")
print("This is a KNOWN defect in the frozen build. Do not repair releases/terminology/1.0.0;")
print("fix it in the next build. The frozen bundle ships with calibration_status='pending'.")


## 20. Human-review state, error triage, linking pilots, late NER jobs (Cell 18)


In [ ]:
# ===== Cell 18: human-review state, error triage, linking pilots, and live NER jobs =====
# This cell is mostly about what is NOT done. Every human-review artifact in the repo is
# still empty, so nothing downstream of them can be called calibrated or gold.
# It also picks up the two SLURM jobs that were still running when this notebook was written;
# those sections are env-guarded and print whatever exists at run time.

# ---- (1) human calibration / gold: are the verdicts filled in? --------------
HUMAN_SHEETS = [
    ("judge_sheet_C1", ROOT / "logs/gold_standard/sheet_C1_records.csv"),
    ("judge_sheet_C2", ROOT / "logs/gold_standard/sheet_C2_types.csv"),
    ("judge_sheet_C3", ROOT / "logs/gold_standard/sheet_C3_mappings.csv"),
    ("judge_sheet_C4", ROOT / "logs/gold_standard/sheet_C4_triples.csv"),
    ("calibration_sheet", ROOT / "logs/llm_judge/HUMAN_CALIBRATION_SHEET.csv"),
    ("validation_mentions", ROOT / "results/validation/sample_v1/mentions.csv"),
    ("validation_definitions", ROOT / "results/validation/sample_v1/definitions.csv"),
    ("validation_mappings", ROOT / "results/validation/sample_v1/mappings.csv"),
    ("validation_findings", ROOT / "results/validation/sample_v1/findings.csv"),
    ("frozen_calibration_v0", ROOT / "releases/terminology/1.0.0/calibration_v0_pre_fix/calibration_sample.csv"),
]
# A review column is "filled" only if some row holds a non-blank value. Match the HUMAN
# namespaces only: a bare "confidence"/"notes" column is model output, not a reviewer verdict.
REVIEW_RE = re.compile(r"^(human_|review_|adjudicat)|^(verdict|notes)$")

HUMAN_ROWS = []
for _name, _p in HUMAN_SHEETS:
    if not _p.exists():
        HUMAN_ROWS.append(dict(sheet=_name, path=str(_p.relative_to(ROOT)), rows="", filled_cells="",
                               review_columns="", status="MISSING"))
        continue
    _rows = load_ner_csv(_p, AUDIT_MAX_ROWS)
    _cols = [c for c in (_rows[0].keys() if _rows else []) if REVIEW_RE.search(c.lower())]
    _filled = sum(1 for r in _rows for c in _cols if str(r.get(c, "")).strip())
    _status = "FILLED" if _filled else ("EMPTY" if _cols else "NO_REVIEW_COLUMNS")
    HUMAN_ROWS.append(dict(sheet=_name, path=str(_p.relative_to(ROOT)), rows=len(_rows),
                           filled_cells=_filled, review_columns=len(_cols), status=_status))
    print(f"  {_name:22s} rows={len(_rows):6d} review_cols={len(_cols):2d} filled={_filled:6d} -> {_status}")
write_audit_csv("human_review_state.csv", HUMAN_ROWS,
                ["sheet", "path", "rows", "review_columns", "filled_cells", "status"])
_all_empty = all(r["status"] in ("EMPTY", "NO_REVIEW_COLUMNS", "MISSING") for r in HUMAN_ROWS)
print("all human review sheets empty:", _all_empty)

# ---- (2) validation sample v1 (the sheet that supersedes calibration_v0) ---
VAL_MAN = read_json(ROOT / "results/validation/sample_v1/_manifest.json", {}) or {}
VAL_ROWS = []
if VAL_MAN:
    print("validation sample:", VAL_MAN.get("sample_version"), "| seed", VAL_MAN.get("seed"),
          "| replaces", str(VAL_MAN.get("replaces"))[:90])
    for _task, _d in (VAL_MAN.get("tasks") or {}).items():
        for _k in ("status", "population", "strata", "target", "realised", "path"):
            VAL_ROWS.append(dict(metric=f"{_task}.{_k}", value=_d.get(_k, "")))
    for r in VAL_ROWS:
        print(f"  {r['metric']:40s} {r['value']}")
    write_audit_csv("validation_sample_counts.csv", VAL_ROWS, ["metric", "value"])

# ---- (3) NER error triage ---------------------------------------------------
TRIAGE = read_json(ROOT / "results/ner_errors_triage_735/enrichment_report.json", {}) or {}
TRIAGE_ROWS = []
if TRIAGE:
    print("triage:", TRIAGE.get("n_records"), "records /", TRIAGE.get("n_batches"), "batches")
    for _k, _v in (TRIAGE.get("evidence_kind_counts") or {}).items():
        TRIAGE_ROWS.append(dict(kind="evidence", name=_k, n=_v))
    for _k, _v in (TRIAGE.get("model_counts") or {}).items():
        TRIAGE_ROWS.append(dict(kind="model", name=_k, n=_v))
    for _k, _v in (TRIAGE.get("grounding_stats") or {}).items():
        if _k != "n":
            TRIAGE_ROWS.append(dict(kind="grounding", name=_k, n=_v))
    for r in TRIAGE_ROWS:
        if r["kind"] == "grounding":
            print(f"  grounding {r['name']:32s} {r['n']}")
    write_audit_csv("ner_error_triage.csv", TRIAGE_ROWS, ["kind", "name", "n"])

# ---- (4) mapping / relation pilots -----------------------------------------
PILOT_ROWS = []
CV = read_json(ROOT / "results/cv_linking/cv_summary.json", {}) or {}
OMOP = read_json(ROOT / "results/omop_linking/omop_summary.json", {}) or {}
TAX = read_json(ROOT / "results/taxonomy_pilot/term_categories_pilot_summary.json", {}) or {}
ADJ = read_json(ROOT / "results/adjudication_pilot_triples_500/triage_summary.json", {}) or {}
if CV:
    _w, _c = CV.get("work_set") or {}, CV.get("coverage") or {}
    PILOT_ROWS.append(dict(pilot="cv_linking", metric="typed_pairs", value=_w.get("typed_pairs_total")))
    PILOT_ROWS.append(dict(pilot="cv_linking", metric="reused_from_cpu", value=_w.get("reused_from_cpu_ensemble")))
    PILOT_ROWS.append(dict(pilot="cv_linking", metric="newly_mapped", value=_w.get("newly_mapped")))
    PILOT_ROWS.append(dict(pilot="cv_linking", metric="pct_mapped", value=_c.get("pct_mapped")))
    PILOT_ROWS.append(dict(pilot="cv_linking", metric="generic_skipped", value=_w.get("generic_entity_pairs_skipped")))
if OMOP:
    _c = OMOP.get("coverage") or {}
    PILOT_ROWS.append(dict(pilot="omop_linking", metric="pairs_with_hit", value=_c.get("pairs_with_omop_hit")))
    PILOT_ROWS.append(dict(pilot="omop_linking", metric="pct_mapped", value=_c.get("pct_mapped")))
    PILOT_ROWS.append(dict(pilot="omop_linking", metric="pct_standard_concept", value=_c.get("pct_with_standard_concept")))
if TAX:
    for k in ("terms", "assigned", "uncertain", "invented", "needs_review", "batch_errors"):
        PILOT_ROWS.append(dict(pilot="taxonomy_pilot", metric=k, value=TAX.get(k)))
if ADJ:
    _d = ADJ.get("triage_label_distribution") or {}
    for k, v in _d.items():
        PILOT_ROWS.append(dict(pilot="adjudication_pilot", metric=k, value=v))
    PILOT_ROWS.append(dict(pilot="adjudication_pilot", metric="n",
                           value=ADJ.get("n")))
    PILOT_ROWS.append(dict(pilot="adjudication_pilot", metric="human_review_status",
                           value=ADJ.get("human_review_status_all")))
for r in PILOT_ROWS:
    print(f"  {r['pilot']:18s} {r['metric']:32s} {r['value']}")
if PILOT_ROWS:
    write_audit_csv("pilots_and_linking.csv", PILOT_ROWS, ["pilot", "metric", "value"])

# ---- (5) completed late NER jobs (28647499, 28649661) ------------------------
# Both jobs were still running at the first audit snapshot and COMPLETED on 2026-10-28
# (gliner-new 09:39:54 CET, java-tools 14:12:50 CET; sacct State=COMPLETED, queue empty).
# Their per-tool CSVs are deep-audited in Cell 15 via NER_ROWS; this block reconciles the
# three independent row counts that exist for each tool:
#   csv_rows    = actual data rows in the artifact (authoritative; counted in Cell 15)
#   summary     = the runner's own _summary.json (written after a reanchor/dedup pass)
#   checkpoint  = the "Per-checkpoint row counts" section of logs/<name>_<job>.out
# Deltas between the three are audit findings, not rounding: for gliner-new the .out
# checkpoint counts exceed the summary by 67,558 rows in total.
# This audit covers job 28649661, which ships all five tools (tmChem added, Ab3P restored).
# Job 28647522 shipped only four and is superseded: mixing it in would attribute DNorm's
# pre-Ab3P output (34,703 rows) to the current tool set (48,203 rows).
_SHORT2JSON = {"gliner2_base": "gliner2_base", "gliner2_large": "gliner2_large",
               "gliner_v25_biomed": "gliner_v25_biomed", "gliner_v25_medium": "gliner_v25_medium",
               "dnorm": "DNorm", "taggerone_disease": "TaggerOne-Disease",
               "taggerone_chemical": "TaggerOne-Chemical", "tmchem": "tmChem",
               "gnormplus": "GNormPlus"}
_JOBSPEC = {"gliner_new": ("28647499", "ner_gliner_new_28647499.out",
                           "_gliner_new_summary.json"),
            "java_tools": ("28649661", "ner_java_tools_28649661.out",
                           "_java_tools_summary_j28649661.json")}
LIVE_ROWS = []
for _key, _dir, _job in NER_LIVE:
    if not _dir.exists():
        print("live NER missing:", _key, _dir)
        continue
    _jobid, _logname, _sumname = _JOBSPEC[_key]
    _sum = read_json(_dir / _sumname) or {}
    _ckpt = {}
    _logf = ROOT / "outputs/logs" / _logname
    if _logf.exists():
        _in_ck = False
        for _ln in _logf.read_text(encoding="utf-8", errors="replace").splitlines():
            if "Per-checkpoint row counts" in _ln:
                _in_ck = True; continue
            if _in_ck:
                _m = re.match(r"(\S+)\s+(\d+)(?:\s+rows)?\s*$", _ln.strip())
                if _m:
                    _ckpt[_m.group(1)] = int(_m.group(2))
                elif _ln.strip().startswith("---"):
                    break
    for _short, _jname in _SHORT2JSON.items():
        if _key == "gliner_new" and not _short.startswith("gliner"):
            continue
        if _key == "java_tools" and _short.startswith("gliner"):
            continue
        _csvf = _dir / f"{_short}_ner.csv"
        _csv_n = len(NER_ROWS.get(_short, [])) if _csvf.exists() else ""
        _sum_n = (_sum.get("models", {}).get(_jname, {}) or {}).get("rows", "")
        _ck_n = _ckpt.get(f"{_short}_ner.csv", _ckpt.get(_short, ""))
        LIVE_ROWS.append(dict(job=_jobid, kind="reconcile", file=_csvf.name,
                              rows=_csv_n, count="csv_rows", summary_rows=_sum_n,
                              checkpoint_rows=_ck_n))
        print(f"  [{_key}] {_short:18s} csv={_csv_n} summary={_sum_n} checkpoint={_ck_n}")
    if "combined_rows" in _sum:
        _comb = sum(len(NER_ROWS.get(s, [])) for s in _SHORT2JSON if s.startswith("gliner"))
        LIVE_ROWS.append(dict(job=_jobid, kind="reconcile", file="all_ner_gliner_new_combined.csv",
                              rows=_comb, count="csv_rows(sum of 4 tools)",
                              summary_rows=_sum["combined_rows"], checkpoint_rows=""))
        print(f"  [{_key}] combined            csv={_comb} summary={_sum['combined_rows']}")
    for _k2, _v2 in sorted(_sum.items()):
        if _k2 != "models" and not isinstance(_v2, dict):
            LIVE_ROWS.append(dict(job=_jobid, kind="summary_field", file=_k2,
                                  rows="", count="", summary_rows=_v2, checkpoint_rows=""))
    _tm = (_sum.get("tmChem") or {})
    if _tm:
        print(f"  [{_key}] tmChem: {_tm.get('status')}: {_tm.get('reason', '')[:100]}")
if LIVE_ROWS:
    write_audit_csv("live_ner_jobs.csv", LIVE_ROWS,
                    ["job", "kind", "file", "rows", "count", "summary_rows", "checkpoint_rows"])

AUDIT_REVIEW = dict(human_sheets=HUMAN_ROWS, all_empty=_all_empty,
                    validation_manifest_counts=VAL_ROWS, triage=TRIAGE_ROWS,
                    pilots=PILOT_ROWS, live_jobs=LIVE_ROWS)
(AUDIT_DIR / "review_state_summary.json").write_text(json.dumps(AUDIT_REVIEW, indent=1),
                                                     encoding="utf-8")

print("\n--- the honest summary ---")
print("Every review sheet above is EMPTY. That means: the LLM judge metrics are uncalibrated,")
print("T5c is NOT_CALIBRATED, and no precision/recall/F1 claim can be made for any NER model here.")
print("Linking coverage (CV 83.95%, OMOP 13.44%) measures how many pairs got a candidate mapping,")
print("not how many were correct: auto_accepted_mappings is 0 in the 1.0.0 release.")


## 21. Known defects and fix proposals (Cell 19)


In [ ]:
# ===== Cell 19: known defects, re-derived from source (not asserted from memory) =====
# Three defect groups were found in this work window. This cell re-checks them against the
# live code and the live artifacts, so the notebook cannot silently drift out of date.
# STATUS 2026-10-28: four fixes were APPROVED (user sign-off recorded in this cell's output)
# and applied to production before this cell runs:
#   - java-tools case-folds (Disease/Chemical/CellLine) + Species/Genus/Strain -> ORGANISM
#   - GENETIC -> GENOMICS
#   - hunflair2 span-trim (ner/hunflair2.py _trim_span) + full re-run (job 28649224)
# NOT applied (still unapproved): wiring common.LABEL_TO_CDM into dedup (DNA/RNA/CELL_LINE/
# GENE/TAXON), so scibert_jnlpba stays at 48.2% and bionlp13cg keeps its 10,563 drops.
# Scope is the fifteen variant CSVs of job set through 28649661 (tmChem included); the
# superseded 28647522 Java run and the pre-trim 28643869 hunflair2 CSV are not read here.
# The checks below therefore PASS for the fixed labels and still flag the unwired map.
# It CHANGES NO PRODUCTION CODE ITSELF. Each finding is emitted as a flag row for review.
import ast
from cdm_pipeline import dedup as _dd
from cdm_pipeline.ner import hunflair2 as _hf2
import common as _common

DEFECT_ROWS = []

# ---- Defect 1: hunflair2 emits spans with trailing punctuation / newlines ----
# Cause: the wrapper asks flair for tokenized output (use_tokenizer=False path), so the
# returned span is the raw token window, not the trimmed entity. The span still satisfies
# text[start:end] == term, so span-fidelity checks pass; the text is just dirty.
HF_CSV = ROOT / "results/28643869_ner_extra_variants/hunflair2_ner.csv"
HF_STATS = {}
if HF_CSV.exists():
    _hf = load_ner_csv(HF_CSV, AUDIT_MAX_ROWS)
    _PUNCT = ",.;:)]}\"'-"
    _trail_punct = sum(1 for r in _hf if r["term"].strip() and r["term"].rstrip()[-1] in _PUNCT)
    _newline = sum(1 for r in _hf if "\n" in r["term"] or "\r" in r["term"])
    _lead_ws = sum(1 for r in _hf if r["term"] != r["term"].lstrip())
    _trail_ws = sum(1 for r in _hf if r["term"] != r["term"].rstrip())
    _short = sum(1 for r in _hf if len(r["term"].strip()) < 3)
    HF_STATS = dict(rows=len(_hf), trailing_punct=_trail_punct,
                    trailing_punct_pct=round(100.0 * _trail_punct / max(len(_hf), 1), 1),
                    contains_newline=_newline,
                    contains_newline_pct=round(100.0 * _newline / max(len(_hf), 1), 1),
                    leading_ws=_lead_ws, trailing_ws=_trail_ws, under_3_chars=_short)
    print("hunflair2 dirty-span check:", HF_STATS)
    print("  source:", str(HF_CSV.relative_to(ROOT)), "(PRE-FIX artifact, job 28643869;")
    print("  the span-trim fix shipped 2026-10-28 and its re-run is job 28649224 -")
    print("  once that artifact lands, add it to Cell 14's NER_NEW and compare rates here)")
    # Show the wrapper setting that causes it, straight from the module.
    _src = Path(_hf2.__file__).read_text(encoding="utf-8").splitlines()
    for _i, _ln in enumerate(_src, 1):
        if "use_tokenizer" in _ln:
            print(f"  {Path(_hf2.__file__).name}:{_i}: {_ln.strip()}")
    DEFECT_ROWS.append(dict(defect="hunflair2_span_hygiene", severity="MEDIUM",
                            metric="rows_with_trailing_punctuation",
                            value=_trail_punct, denominator=len(_hf),
                            pct=HF_STATS["trailing_punct_pct"],
                            location=f"{Path(_hf2.__file__).name}: use_tokenizer=False",
                            impact="raw labels reach dedup/terminology with punctuation and "
                                   "embedded newlines; same term fragments into many norm() keys"))
    DEFECT_ROWS.append(dict(defect="hunflair2_span_hygiene", severity="MEDIUM",
                            metric="rows_containing_newline", value=_newline,
                            denominator=len(_hf), pct=HF_STATS["contains_newline_pct"],
                            location=f"{Path(_hf2.__file__).name}: use_tokenizer=False",
                            impact="multi-line spans break CSV/JSONL readability and term identity"))
else:
    print("hunflair2 CSV missing:", HF_CSV)

# ---- Defect 2: the label map exists but dedup never imports it --------------
# common.LABEL_TO_CDM maps sciSpaCy/ERNIE-style fine labels (DNA, RNA, CELL_LINE, GENE, TAXON)
# onto the 14 CDM types. dedup.py defines its own two-entry LABEL_FIX and validates against
# VALID_TYPES, so every label the map would have fixed is dropped as if it were invalid.
# IMPORTANT: the map is necessary but NOT sufficient. Three label groups have no home even
# after wiring it, and they need three different kinds of decision:
#   GENETIC (all of bert_genetic)          -> semantic choice: GENOMICS | DNA_RNA | PROTEIN
#   java tools' title-case labels          -> mostly pure case-folds (Disease->DISEASE,
#   (all five tools, job 28649661:           Chemical->CHEMICAL, CellLine->CELL_LINE) plus the
#   180,480 rows, 0 of 170,580 kept unfixed) conventional Species/Genus/Strain -> ORGANISM
#   sciSpaCy fine anatomy labels           -> correctly droppable (no CDM home)
# Measured three tiers, recomputed below from the shipped CSVs, not asserted:
#   six transformer models  85.5% current / 89.9% map-wired / 89.9% map+decision
#   all fifteen variants    92.5% current / 93.4% map-wired / 93.4% map+decision
# The map and map+decision tiers coincide because GENETIC->GENOMICS is already applied, so
# every remaining dropped label is one that LABEL_TO_CDM cannot fold either.
_LABEL_TABLE = dict(_common.LABEL_TO_CDM)
print(f"common.LABEL_TO_CDM entries: {len(_LABEL_TABLE)}")

# Explicit decision table for labels that either are not in the map, or whose map target is
# itself questionable. Nothing here is applied; it documents what a human must decide.
_TYPE_DECISIONS = {
    "GENETIC": dict(options=["GENOMICS", "DNA_RNA", "PROTEIN"], audit_choice="GENOMICS",
                    rationale="biobert_genetic_ner tags genes/gene products. common maps GENE->PROTEIN "
                              "and GENE_OR_GENE_PRODUCT->DNA_RNA, so GENOMICS is the least-consistent "
                              "of the three; the label space is too coarse for one CDM type."),
    "Disease": dict(options=["DISEASE"], audit_choice="DISEASE",
                    rationale="pure case-fold; DNorm and TaggerOne-Disease emit title-case CDM names."),
    "Chemical": dict(options=["CHEMICAL"], audit_choice="CHEMICAL",
                    rationale="pure case-fold; TaggerOne-Chemical emits title-case CDM names."),
    "CellLine": dict(options=["CELL_LINE"], audit_choice="CELL_LINE",
                    rationale="pure case-fold; GNormPlus emits title-case CDM names."),
    "Species": dict(options=["ORGANISM"], audit_choice="ORGANISM",
                    rationale="conventional: NCBI Taxonomy entries are organisms; GNormPlus 'Species' "
                              "is its taxon category, along with Genus and Strain."),
    "Genus": dict(options=["ORGANISM"], audit_choice="ORGANISM",
                  rationale="conventional: same NCBI Taxonomy provenance as Species."),
    "Strain": dict(options=["ORGANISM"], audit_choice="ORGANISM",
                   rationale="conventional: same NCBI Taxonomy provenance as Species."),
}
lbl_rows = []
for _raw in ("GENETIC", "DNA", "RNA", "CELL_LINE", "GENE", "TAXON", "PROTEIN",
             "CELLULAR_COMPONENT", "MULTI_TISSUE_STRUCTURE", "ANATOMICAL_SYSTEM", "DISEASE",
             "Disease", "Chemical", "CellLine", "Species", "Genus", "Strain"):
    _mapped = _LABEL_TABLE.get(_raw)
    _dedup_says = _dd.fix_label(_raw)
    _survives = _dedup_says in _dd.VALID_TYPES
    if _survives:
        _v = "OK"
    elif _raw in _TYPE_DECISIONS:
        _v = "NEEDS_TYPE_DECISION"
    elif _mapped in _dd.VALID_TYPES:
        _v = "RECOVERABLE_BY_MAP"
    else:
        _v = "DROPPED_NO_HOME"
    lbl_rows.append(dict(raw_label=_raw, label_to_cdm=_mapped or "",
                         dedup_fix_label=_dedup_says, survives_valid_types=_survives,
                         cdm_home=_mapped if _mapped else "NONE", verdict=_v))
    print(f"  {_raw:28s} LABEL_TO_CDM->{(_mapped or '-'):14s} dedup.fix_label->{_dedup_says:26s} "
          f"kept={str(_survives):5s} {_v}")
_BY_MAP = [r["raw_label"] for r in lbl_rows if r["verdict"] == "RECOVERABLE_BY_MAP"]
_NEEDS_DEC = [r["raw_label"] for r in lbl_rows if r["verdict"] == "NEEDS_TYPE_DECISION"]
_NO_HOME = [r["raw_label"] for r in lbl_rows if r["verdict"] == "DROPPED_NO_HOME"]
print("recoverable by importing the existing map:", _BY_MAP)
print("needs an explicit type decision:", _NEEDS_DEC)
print("no CDM home (correctly droppable):", _NO_HOME)

# Confirm the map is genuinely unused by dedup CODE (not just mentioned in a comment).
# AST-based: a Name load of LABEL_TO_CDM anywhere in dedup.py means the map is wired in;
# comments and docstrings never produce Name nodes, so they cannot false-positive here.
_dd_src = Path(_dd.__file__).read_text(encoding="utf-8")
_uses_map = any(isinstance(n, ast.Name) and n.id == "LABEL_TO_CDM"
                for n in ast.walk(ast.parse(_dd_src)))
print("dedup.py uses common.LABEL_TO_CDM in code:", _uses_map)
assert _uses_map is False, "dedup.py now uses LABEL_TO_CDM - defect 2 may be fixed; re-audit"

for r in lbl_rows:
    DEFECT_ROWS.append(dict(defect="dedup_label_map_unwired", severity="HIGH" if r["survives_valid_types"] is False else "NONE",
                            metric=f"label:{r['raw_label']}", value=1, denominator="",
                            pct="", location="src/cdm_pipeline/dedup.py: fix_label/VALID_TYPES",
                            impact=(f"kept: {r['dedup_fix_label']}" if r["verdict"] == "OK" else
                                    f"LABEL_TO_CDM maps {r['raw_label']}->{r['label_to_cdm']} but dedup "
                                    f"keeps {r['dedup_fix_label']}; row is dropped" if r["verdict"] == "RECOVERABLE_BY_MAP"
                                    else f"{r['raw_label']} is not in LABEL_TO_CDM; needs a type decision"
                                    if r["verdict"] == "NEEDS_TYPE_DECISION"
                                    else f"{r['raw_label']} has no CDM home")))
write_audit_csv("known_defects.csv", DEFECT_ROWS,
                ["defect", "severity", "metric", "value", "denominator", "pct", "location", "impact"])
write_audit_csv("label_map_check.csv", lbl_rows,
                ["raw_label", "label_to_cdm", "dedup_fix_label", "survives_valid_types",
                 "cdm_home", "verdict"])

# ---- Defect 1b: dirty spans are NOT hunflair2-specific (Cell 15 measured every model) ----
# Embedded newlines inside term strings hit 13k-19k rows per GLiNER model - more rows in absolute
# terms than hunflair2, at a lower rate. Same consequence: one surface term fragments into many
# norm() keys. Reported once, cross-model, from Cell 15's SPAN_ROWS (guard: Cell 15 may not have
# run in a shortened session).
if "SPAN_ROWS" in globals() and SPAN_ROWS:
    _nl_rows = sorted((r for r in SPAN_ROWS if r["contains_newline"] > 0),
                      key=lambda r: -r["contains_newline"])
    _tot_nl = sum(r["contains_newline"] for r in _nl_rows)
    _top = ", ".join(f"{r['model']}={r['contains_newline']} ({r['contains_newline_pct']}%)"
                     for r in _nl_rows[:4])
    print(f"newline-in-term across models: {_tot_nl} rows total | {_top}")
    DEFECT_ROWS.append(dict(defect="span_newline_pollution_multi_model", severity="MEDIUM",
                            metric="rows_containing_newline", value=_tot_nl,
                            denominator=sum(r["rows"] for r in SPAN_ROWS), pct="",
                            location="all NER wrappers (see Cell 15 ner_span_fidelity.csv)",
                            impact=f"dirty term strings fragment norm() keys; worst: {_top}. "
                                   "A shared span-trim/whitespace-normalise step at CSV write time "
                                   "would fix every model at once (needs sign-off; changes spans)."))

# ---- what was decided and applied (2026-10-28, user sign-off) ----------------
# Each row records the decision this audit proposed; `verified` is re-derived live
# from the production code above, so a later revert would flip it back to False.
_wired = _dd.fix_label("Disease") == "DISEASE" and _dd.fix_label("GENETIC") == "GENOMICS" \
    and _dd.fix_label("Species") == "ORGANISM"
FIX_PROPOSALS = [
    dict(defect="java_tools_labels_have_no_cdm_home",
         change="extended dedup.fix_label: case-folds (Disease->DISEASE, Chemical->CHEMICAL, "
                "CellLine->CELL_TYPE) + conventional taxon mapping (Species/Genus/Strain->ORGANISM)",
         compute_cost="none (label mapping only; no re-run needed)",
         rows_recovered="170,580 java-tools rows now kept (0 before the fix); 15-variant "
                        "survival rises accordingly - see Cell 15's TOTAL line",
         risk="LOW for the case-folds; MEDIUM accepted for Species/Genus/Strain->ORGANISM",
         verified=_wired),
    dict(defect="genetic_label_has_no_cdm_home",
         change="decided GENETIC -> GENOMICS and mapped it in dedup.fix_label",
         compute_cost="none (decision + one-line mapping)",
         rows_recovered="bert_genetic 0.0% -> 92.0% (18,720 rows)",
         risk="MEDIUM accepted - GENOMICS conflicts with common's own GENE->PROTEIN / "
              "GENE_OR_GENE_PRODUCT->DNA_RNA; recorded as the team's choice",
         verified=_dd.fix_label("GENETIC") == "GENOMICS"),
    dict(defect="hunflair2_span_hygiene",
         change="ner/hunflair2.py: _trim_span() trims every whitespace-token window to its "
                "non-whitespace, non-punctuation core before emitting",
         compute_cost="one full re-run (job 28649224, ~25 min CPU); results dir "
                      "results/28649224_ner_hunflair2_trim",
         rows_recovered="~24k of 106,011 pre-fix rows become clean term strings",
         risk="MEDIUM accepted - changes spans, so the re-run artifact is re-validated in Cell 15",
         verified="_trim_span" in Path(_hf2.__file__).read_text(encoding="utf-8")),
    dict(defect="dedup_label_map_unwired",
         change="in dedup.fix_label, fall back to common.LABEL_TO_CDM before returning the raw label: "
                "return LABEL_FIX.get(label) or LABEL_TO_CDM.get(label) or label",
         compute_cost="none (pure label mapping)",
         rows_recovered="would lift scibert_jnlpba 48.2% -> 90.2% and the 15-variant total "
                        "92.5% -> 93.4% (measured, Cell 15 kept_map_only). bionlp13cg's "
                        "10,563 drops are fine anatomy labels (CELLULAR_COMPONENT, "
                        "MULTI_TISSUE_STRUCTURE, ...) with no map entry and no CDM home - "
                        "correctly droppable, NOT recoverable by this change",
         risk="LOW - target types already in CDM_TYPES; NOT YET APPROVED, still unwired",
         verified=False),
]
write_audit_csv("defect_fix_proposals.csv", FIX_PROPOSALS,
                ["defect", "change", "compute_cost", "rows_recovered", "risk", "verified"])
for f in FIX_PROPOSALS:
    print(f"\n{'APPLIED + VERIFIED' if f['verified'] else 'PROPOSED (not applied)'}: {f['defect']}"
          f"\n  {f['change']}\n  cost: {f['compute_cost']}\n  risk: {f['risk']}")

print("\nApproved and applied 2026-10-28 (user sign-off): java-tools case-folds + taxon->ORGANISM,")
print("GENETIC->GENOMICS, hunflair2 span-trim + re-run (job 28649224). Still open, needs sign-off:")
print("wiring common.LABEL_TO_CDM into dedup.py: measured upside 92.5% -> 93.4% (Cell 15).")
